# Load in and clean data

## Loading in Packages and Data

In [0]:
%pip install "matplotlib==3.8.4" matplotlib-venn
%pip install upsetplot
%pip install statsmodels patsy
dbutils.library.restartPython()

In [0]:
import pandas as pd

results_GPT_full_df = pd.read_json(("/Workspace/Users/user/user/UMD_HPO_GPT_Output_Results_full_final.json"), orient="records")
display(results_GPT_full_df)
print(len(results_GPT_full_df))

results_mini_full_df = pd.read_json("/Workspace/Users/user/user/UMD_HPO_GPT_Mini_Output_Results_full_final.json", orient="records")
display(results_mini_full_df)
print(len(results_mini_full_df))

results_claude_full_df = pd.read_json("/Workspace/Users/user/user/UMD_HPO_Sonnet_Output_Results_full_final.json", orient="records")
display(results_claude_full_df)
print(len(results_claude_full_df))

manual_score = pd.read_csv("/Workspace/Users/user/user/manual_score.csv")
display(manual_score)

notes_enrolled4 = pd.read_csv("/Workspace/Users/user/user/notes_enrolled4.csv")
display(notes_enrolled4)
print(len(notes_enrolled4))

checklist_filtered = pd.read_csv("/Workspace/Users/user/user/checklist_filtered.csv")
display(checklist_filtered)

## Fixing manual_score inner_join Error

In [0]:
checklist_from_disk = pd.read_csv("/Workspace/Users/user/user/hposetta_stone2.csv")
disk_terms = set(checklist_from_disk['Term from checklist'].dropna())

pairs_to_check = [
    ('Autism Spectrum Disorder (ASD)', 'Autism Spectrum Disorder (ASD'),
    ('Diabetes Mellitus Type I (insulin dependent)', 'Diabetes Mellitus Type I (insulin dependent'),
    ('Diabetes Mellitus Type II (non-insulin dependent)', 'Diabetes Mellitus Type II (non-insulin dependent'),
    ('Double Outlet Right Ventricle (DORV)', 'Double Outlet Right Ventricle (DORV'),
    ('Hypertrophic cardiomyopathy (HCM)', 'Hypertrophic cardiomyopathy (HCM'),
    ('Tetralogy of Fallot (TOF)', 'Tetralogy of Fallot (TOF'),
    ('CPEO (opthalmoplegia)', 'CPEO (opthalmoplegia'),
]

print(f"Rows in hposetta_stone2.csv as read fresh from disk: {len(checklist_from_disk)}")
print()
for closed, unclosed in pairs_to_check:
    print(f"closed form   {'PRESENT' if closed in disk_terms else 'absent  '}  {closed!r}")
    print(f"unclosed form {'PRESENT' if unclosed in disk_terms else 'absent  '}  {unclosed!r}")
    print()

In [0]:
manual_score.columns

In [0]:
remap = {
    'Double Outlet Right Ventricle (DORV': 'Double Outlet Right Ventricle (DORV)',
    'Hypertrophic cardiomyopathy (HCM': 'Hypertrophic cardiomyopathy (HCM)',
    'Tetralogy of Fallot (TOF': 'Tetralogy of Fallot (TOF)',
    'CPEO (opthalmoplegia': 'CPEO (opthalmoplegia)',
    'Autism Spectrum Disorder (ASD': 'Autism Spectrum Disorder (ASD)',
    'Diabetes Mellitus Type I (insulin dependent': 'Diabetes Mellitus Type I (insulin dependent)',
    'Diabetes Mellitus Type II (non-insulin dependent': 'Diabetes Mellitus Type II (non-insulin dependent)',
    'Chiari I malformation': 'Chiari malformation',
}

checklist_fresh = pd.read_csv("/Workspace/Users/user/user/hposetta_stone2.csv")
checklist_fresh = checklist_fresh[checklist_fresh['Ignore or not'] != 'ignore'].drop(columns=['Ignore or not'])
checklist_fresh['Term from checklist'] = checklist_fresh['Term from checklist'].str.strip().map(lambda x: remap.get(x, x))

dup_terms = checklist_fresh[checklist_fresh.duplicated(subset=['Term from checklist'], keep=False) & checklist_fresh['Term from checklist'].notna()]
if len(dup_terms) > 0:
    print(f"{dup_terms['Term from checklist'].nunique()} term(s) appear more than once; keeping first occurrence of each.")
    checklist_fresh = checklist_fresh.drop_duplicates(subset=['Term from checklist'], keep='first')

print(f"checklist_fresh: {checklist_fresh.shape}")
print(f"Unique non-null 'Term from checklist' values: {checklist_fresh['Term from checklist'].nunique()}")

In [0]:
n_null_terms = checklist_fresh['Term from checklist'].isna().sum()
n_unique_terms = checklist_fresh['Term from checklist'].nunique()

print(f"Rows in checklist_fresh: {len(checklist_fresh)}")
print(f"Rows where 'Term from checklist' is null: {n_null_terms}")
print(f"Unique non-null terms: {n_unique_terms}")
print()
if n_null_terms > 0:
    display(checklist_fresh[checklist_fresh['Term from checklist'].isna()])

In [0]:
names_to_check = [
    'checklist_full', 'checklist_filtered', 'data_labels',
    'data_labels_filtered', 'data_labels_filtered4',
    'hposetta_stone', 'hposetta_stone2',
    'REDCap_mapping', 'records_formatted', 'manual_list', 'manual_df', 'manual_score',
    'results_GPT_full_df', 'results_mini_full_df', 'results_claude_full_df',
    'combined_df', 'notes_enrolled4', 'notes_enrolled', 'adjudication_df', 'group_assignments',
    'group0_df', 'group1_df', 'group2_df', 'group3_df',
    'checklist_fresh', 'manual_score_before_fix',
    'pipeline_join', 'pipeline_missing', 'pipeline_missing_cohort',
    'adjudicated_pairs', 'all_adjudications', 'to_append'
]

for name in names_to_check:
    if name in globals():
        obj = globals()[name]
        shape = obj.shape if hasattr(obj, 'shape') else type(obj).__name__
        print(f"DEFINED   {name}: {shape}")
    else:
        print(f"MISSING   {name}")

In [0]:
print(list(checklist_filtered.columns))
print()
dup_terms = checklist_filtered.duplicated(subset=['Term from checklist'], keep=False).sum()
print(f"Duplicate 'Term from checklist' rows in checklist_filtered: {dup_terms}")
print()
dup_pairs = manual_score.duplicated(subset=['Subject ID', 'Phenotype']).sum()
print(f"Duplicate Subject ID x Phenotype pairs in manual_score: {dup_pairs}")

In [0]:
import pandas as pd

# Step 1: reload REDCap_mapping fresh — identical to your cell
REDCap_mapping_full_path = "/Workspace/Users/user/user/data_labels_filtered4.csv"
REDCap_mapping = pd.read_csv(REDCap_mapping_full_path)

# Step 2: reload checklist_full fresh — identical to your cell
checklist_full_path = "/Workspace/Users/user/user/hposetta_stone2.csv"
checklist_full = pd.read_csv(checklist_full_path)

# Step 3: build checklist_filtered — identical to your cell
checklist_filtered = checklist_full[checklist_full['Ignore or not'] != 'ignore'].copy()
checklist_filtered.rename(columns={'HPO version: hpo-web@1.7.0 - hpo-obo@2020-08-11': 'Text from Checklist'}, inplace=True)
checklist_filtered.drop(columns=['Ignore or not'], inplace=True)

# THE FIX (1 of 2): normalize the checklist's join key unconditionally, before it's used —
# instead of leaving it dependent on whether a separate cell happened to run first
remap = {
    'Double Outlet Right Ventricle (DORV': 'Double Outlet Right Ventricle (DORV)',
    'Hypertrophic cardiomyopathy (HCM': 'Hypertrophic cardiomyopathy (HCM)',
    'Tetralogy of Fallot (TOF': 'Tetralogy of Fallot (TOF)',
    'CPEO (opthalmoplegia': 'CPEO (opthalmoplegia)',
    'Autism Spectrum Disorder (ASD': 'Autism Spectrum Disorder (ASD)',
    'Diabetes Mellitus Type I (insulin dependent': 'Diabetes Mellitus Type I (insulin dependent)',
    'Diabetes Mellitus Type II (non-insulin dependent': 'Diabetes Mellitus Type II (non-insulin dependent)',
    'Chiari I malformation': 'Chiari malformation',
}
checklist_filtered['Term from checklist'] = checklist_filtered['Term from checklist'].str.strip().map(lambda x: remap.get(x, x))

# Step 4: records_formatted — identical to your cell
records = []
for i in range(0, len(REDCap_mapping)):
    phen = []
    for j in range(0, len(REDCap_mapping.columns)):
        if pd.isnull(REDCap_mapping.iloc[i][j]) == False:
            phen.append(REDCap_mapping.iloc[i][j])
    records.append(phen)
records_formatted = pd.DataFrame(records)

# Step 5: manual_list — identical to your cell
manual_list = []
for j in range(0, len(records_formatted)):
    for i in range(0, len(records_formatted.columns)):
        manual_list.append(records_formatted[i][j])
manual_list = [x for x in manual_list if x is not None]

# Step 6: manual_df — identical to your cell
rows = []
current_id = None
for item in manual_list:
    if item.startswith('WES'):
        current_id = item
    else:
        rows.append((current_id, item))
manual_df = pd.DataFrame(rows, columns=['Subject ID', 'Phenotype'])

# THE FIX (2 of 2): same normalization applied to the other side of the join
manual_df['Phenotype'] = manual_df['Phenotype'].str.strip().map(lambda x: remap.get(x, x))

# Step 7: the join — identical structure to your cell
manual_score_fixed = manual_df.merge(
    checklist_filtered[['Term from checklist', 'HPO term (ID)']],
    left_on='Phenotype',
    right_on='Term from checklist',
    how='inner'
).drop(columns=['Term from checklist'])

# Step 8: your existing post-join strip, kept as-is (now a no-op, left in for fidelity)
manual_score_fixed['Phenotype'] = manual_score_fixed['Phenotype'].str.strip()

print(f"REDCap_mapping: {REDCap_mapping.shape}")
print(f"checklist_filtered: {checklist_filtered.shape}")
dup = checklist_filtered['Term from checklist'].value_counts()
dup = dup[dup > 1]
if len(dup) > 0:
    print(f"  {len(dup)} term(s) still repeat in checklist_filtered (untouched, separate issue):")
    print(dup)
print(f"manual_df: {manual_df.shape}")
print()
print(f"manual_score_fixed rows: {len(manual_score_fixed)}")
print(f"manual_score_fixed unique Subject ID x Phenotype pairs: {manual_score_fixed.drop_duplicates(subset=['Subject ID','Phenotype']).shape[0]}")

In [0]:
old_pairs = set(zip(manual_score['Subject ID'], manual_score['Phenotype']))
new_pairs = set(zip(manual_score_fixed['Subject ID'], manual_score_fixed['Phenotype']))

recovered = new_pairs - old_pairs
recovered_df = pd.DataFrame(sorted(recovered), columns=['Subject ID', 'Phenotype'])

print(f"Pairs present in manual_score_fixed but scored 0 in manual_score: {len(recovered_df)}")
print()
print(recovered_df.groupby('Phenotype').size().sort_values(ascending=False))
print()
display(recovered_df)

## Data Cleaning

In [0]:
# list of notes that are in notes_enrolled4 but were not in the manual score. These notes are kept in the analysis. 
gpt_not_manual_ids = set(results_GPT_full_df['Subject ID']) - set(manual_score_fixed['Subject ID'])
display(notes_enrolled4[notes_enrolled4['Subject ID'].isin(gpt_not_manual_ids)])
len(gpt_not_manual_ids)

In [0]:
# Printing details about those in notes_enrolled4 but not in manual_score_fixed as well as in GPT results but not in manual_score_fixed

in_notes_not_manual = set(notes_enrolled4['Subject ID']) - set(manual_score_fixed['Subject ID'])
in_gpt_not_manual = set(results_GPT_full_df['Subject ID']) - set(manual_score_fixed['Subject ID'])

print(f"In notes_enrolled4 but not manual_score_fixed: {len(in_notes_not_manual)}")
print(f"In GPT results but not manual_score_fixed: {len(in_gpt_not_manual)}")
print(f"GPT-not-manual is subset of notes-not-manual: {in_gpt_not_manual.issubset(in_notes_not_manual)}")
print(in_notes_not_manual)

In [0]:
#Loading in redcap mapping
REDCap_mapping_full_path = "/Workspace/Users/user/user/data_labels_filtered4.csv"

REDCap_mapping = pd.read_csv(REDCap_mapping_full_path)
print('Number of rows:', len(REDCap_mapping))
print('Number of columns:', len(REDCap_mapping.columns))
display(REDCap_mapping)

In [0]:
# Check if the notes missing in the manual review appear in REDCap_mapping or in checklist_filtered
missing_in_redcap = set(in_notes_not_manual) - set(REDCap_mapping.values.flatten())
missing_in_checklist = set(in_notes_not_manual) - missing_in_redcap

print(f"Not in REDCap_mapping at all: {len(missing_in_redcap)}")
print(missing_in_redcap)
print(f"\nIn REDCap_mapping but phenotypes didn't match checklist_filtered: {len(missing_in_checklist)}")
print(missing_in_checklist)

In [0]:
# For those in REDCap_mapping, what phenotypes did they have?
# Recreate records from REDCap_mapping
records = []
for i in range(0, len(REDCap_mapping)):
    phen = []
    for j in range(0, len(REDCap_mapping.columns)):
        if pd.isnull(REDCap_mapping.iloc[i][j]) == False:
            phen.append(REDCap_mapping.iloc[i][j])
    records.append(phen)

# Now check phenotypes for the 11 missing Subject IDs
for sid in missing_in_checklist:
    sid_row = [i for i, record in enumerate(records) if record and record[0] == sid]
    if sid_row:
        phenotypes = [x for x in records[sid_row[0]] if not str(x).startswith('WES')]
        print(f"\n{sid}: {phenotypes}")
    else:
        print(f"\n{sid}: not found in records")

In [0]:
print(f"notes_enrolled4 subjects: {notes_enrolled4['Subject ID'].nunique()}")
print(f"results_GPT_full_df subjects: {results_GPT_full_df['Subject ID'].nunique()}")
print(f"results_mini_full_df subjects: {results_mini_full_df['Subject ID'].nunique()}")
print(f"results_claude_full_df subjects: {results_claude_full_df['Subject ID'].nunique()}")
print(f"manual_score_fixed subjects: {manual_score_fixed['Subject ID'].nunique()}")

In [0]:
# Finding phenotypes that are not represented in gpt, claude, mini, and manual

phenotypes_results_gpt = set(results_GPT_full_df['Phenotype'].dropna().unique())
phenotypes_results_mini = set(results_mini_full_df['Phenotype'].dropna().unique())
phenotypes_results_claude = set(results_claude_full_df['Phenotype'].dropna().unique())

phenotypes_manual = set(manual_score_fixed['Phenotype'].dropna().unique())

missing_from_manual = sorted(phenotypes_results_gpt - phenotypes_manual)
missing_from_gpt = sorted(phenotypes_manual - phenotypes_results_gpt)
missing_from_mini = sorted(phenotypes_manual - phenotypes_results_mini)
missing_from_claude = sorted(phenotypes_manual - phenotypes_results_claude)

print(len(missing_from_manual), "values in GPT but not in manual_score_fixed")
print(missing_from_manual)

print()

print(len(missing_from_gpt), "values in manual_score_fixed but not in GPT")
print(missing_from_gpt)

print()

print(len(missing_from_mini), "values in manual_score_fixed but not in mini")
print(missing_from_mini)

print()

print(len(missing_from_claude), "values in manual_score_fixed but not in claude")
print(missing_from_claude)

In [0]:
# Remapping phenotypes such that any parenthesis/spacing issues are normalized and Chairi I malformation (which claude hallucinated and is a subset of the original phenotype that was input into the model) is renamed to chiari malformation, the original name

remap = {
    # Parenthesis fixes — all should have closing )
    'Double Outlet Right Ventricle (DORV': 'Double Outlet Right Ventricle (DORV)',
    'Hypertrophic cardiomyopathy (HCM': 'Hypertrophic cardiomyopathy (HCM)',
    'Tetralogy of Fallot (TOF': 'Tetralogy of Fallot (TOF)',
    'CPEO (opthalmoplegia': 'CPEO (opthalmoplegia)',
    'Autism Spectrum Disorder (ASD': 'Autism Spectrum Disorder (ASD)',
    'Diabetes Mellitus Type I (insulin dependent': 'Diabetes Mellitus Type I (insulin dependent)',
    'Diabetes Mellitus Type II (non-insulin dependent': 'Diabetes Mellitus Type II (non-insulin dependent)',
    # Hallucination fix
    'Chiari I malformation': 'Chiari malformation',
}

manual_score_fixed['Phenotype'] = manual_score_fixed['Phenotype'].str.strip().map(lambda x: remap.get(x, x))
results_GPT_full_df['Phenotype'] = results_GPT_full_df['Phenotype'].str.strip().map(lambda x: remap.get(x, x))
results_mini_full_df['Phenotype'] = results_mini_full_df['Phenotype'].str.strip().map(lambda x: remap.get(x, x))
results_claude_full_df['Phenotype'] = results_claude_full_df['Phenotype'].str.strip().map(lambda x: remap.get(x, x))
checklist_filtered['Term from checklist'] = checklist_filtered['Term from checklist'].str.strip().map(lambda x: remap.get(x, x))

In [0]:
# Showing how many duplicated phenotypes remain
results_GPT_full_df.duplicated(subset=['Subject ID', 'Phenotype']).sum()

In [0]:
# Finding phenotypes that are not represented in gpt, claude, mini, and manual after deduplication

phenotypes_results_gpt = set(results_GPT_full_df['Phenotype'].dropna().unique())
phenotypes_results_mini = set(results_mini_full_df['Phenotype'].dropna().unique())
phenotypes_results_claude = set(results_claude_full_df['Phenotype'].dropna().unique())

phenotypes_manual = set(manual_score_fixed['Phenotype'].dropna().unique())

missing_from_manual = sorted(phenotypes_results_gpt - phenotypes_manual)
missing_from_gpt = sorted(phenotypes_manual - phenotypes_results_gpt)
missing_from_mini = sorted(phenotypes_manual - phenotypes_results_mini)
missing_from_claude = sorted(phenotypes_manual - phenotypes_results_claude)

print(len(missing_from_manual), "values in results_GPT_full_df but not in manual_score_fixed")
print(missing_from_manual)

print()

print(len(missing_from_gpt), "values in manual_score_fixed but not in GPT")
print(missing_from_gpt)

print()

print(len(missing_from_mini), "values in manual_score_fixed but not in mini")
print(missing_from_mini)

print()

print(len(missing_from_claude), "values in manual_score_fixed but not in claude")
print(missing_from_claude)

In [0]:
#Renaming columns
results_GPT_full_df = results_GPT_full_df.rename(columns={'Score': 'GPT_Score'})
results_mini_full_df = results_mini_full_df.rename(columns={'Score': 'GPT_Mini_Score'})
results_claude_full_df = results_claude_full_df.rename(columns={'Score': 'Claude_Sonnet_Score'})

In [0]:
# Viewing score totals across entire dataframes (no subsetting)

score_totals = {}

for name, df, score_col, agg in [
    ('Manual', manual_score_fixed, 'Phenotype', 'count'),
    ('GPT', results_GPT_full_df, 'GPT_Score', 'sum'),
    ('GPT Mini', results_mini_full_df, 'GPT_Mini_Score', 'sum'),
    ('Claude Sonnet', results_claude_full_df, 'Claude_Sonnet_Score', 'sum'),
]:
    if agg == 'count':
        score_totals[name] = df[score_col].count()
    else:
        score_totals[name] = df[score_col].sum()

print("Total phenotypes scored positive, per model, across all subjects:")
for model, total in score_totals.items():
    print(f"  {model}: {total}")

In [0]:
# Displaying notes under 150 characters long 
short_notes = notes_enrolled4[notes_enrolled4['NOTE_TEXT'].str.len() < 150]
display(short_notes)

In [0]:
manual_score_fixed.columns

In [0]:
# Viewing score totals for rows to be dropped in ids_to_drop

score_totals = {}

for name, df, score_col, agg in [
    ('Manual', manual_score_fixed, 'Phenotype', 'count'),
    ('GPT', results_GPT_full_df, 'GPT_Score', 'sum'),
    ('GPT Mini', results_mini_full_df, 'GPT_Mini_Score', 'sum'),
    ('Claude Sonnet', results_claude_full_df, 'Claude_Sonnet_Score', 'sum'),
]:
    subset = df[df['Subject ID'].isin(short_notes['Subject ID'])]
    if agg == 'count':
        score_totals[name] = subset[score_col].count()
    else:
        score_totals[name] = subset[score_col].sum()

print(f"Subjects with notes under 150 characters: {short_notes['Subject ID'].nunique()}")
print("\nTotal phenotypes scored positive, per model, for these subjects:")
for model, total in score_totals.items():
    print(f"  {model}: {total}")

In [0]:
# Displayign the phenotypes that the manual had scored to these short notes
display(manual_score_fixed[manual_score_fixed['Subject ID'].isin(short_notes['Subject ID'])])

In [0]:
# Dropping notes under 150 characters long
ids_to_drop = short_notes['Subject ID']

manual_score_fixed = manual_score_fixed[~manual_score_fixed['Subject ID'].isin(ids_to_drop)]
results_GPT_full_df = results_GPT_full_df[~results_GPT_full_df['Subject ID'].isin(ids_to_drop)]
results_mini_full_df = results_mini_full_df[~results_mini_full_df['Subject ID'].isin(ids_to_drop)]
results_claude_full_df = results_claude_full_df[~results_claude_full_df['Subject ID'].isin(ids_to_drop)]

In [0]:
# Subjects where GPT scored 0 phenotypes total
gpt_totals = results_GPT_full_df.groupby('Subject ID')['GPT_Score'].sum()
zero_score_subjects = gpt_totals[gpt_totals == 0].index.tolist()

print(f"Subjects with 0 phenotypes scored by GPT: {len(zero_score_subjects)}")
print(zero_score_subjects)

# Pull their notes from notes_enrolled4
zero_score_notes = notes_enrolled4[notes_enrolled4['Subject ID'].isin(zero_score_subjects)]

print(f"\nMatching note rows found: {len(zero_score_notes)}")
display(zero_score_notes[['Subject ID', 'NOTE_TEXT', 'NOTE_TYPE']])

In [0]:
# Viewing score totals for rows dropped in next cell
subject_ids = [8 WES IDs]

score_totals = {}

for name, df, score_col, agg in [
    ('Manual', manual_score_fixed, 'Phenotype', 'count'),
    ('GPT', results_GPT_full_df, 'GPT_Score', 'sum'),
    ('GPT Mini', results_mini_full_df, 'GPT_Mini_Score', 'sum'),
    ('Claude Sonnet', results_claude_full_df, 'Claude_Sonnet_Score', 'sum'),
]:
    subset = df[df['Subject ID'].isin(subject_ids)]
    if agg == 'count':
        score_totals[name] = subset[score_col].count()
    else:
        score_totals[name] = subset[score_col].sum()

print(f"Subjects to be dropped: {len(subject_ids)}")
print("\nTotal phenotypes scored positive, per model, for these subjects:")
for model, total in score_totals.items():
    print(f"  {model}: {total}")

In [0]:
manual_score_fixed = manual_score_fixed[~manual_score_fixed['Subject ID'].isin(['9 WES IDs])]
results_GPT_full_df = results_GPT_full_df[~results_GPT_full_df['Subject ID'].isin(['9 WES IDs])]
results_mini_full_df = results_mini_full_df[~results_mini_full_df['Subject ID'].isin(['9 WES IDs])]
results_claude_full_df = results_claude_full_df[~results_claude_full_df['Subject ID'].isin(['9 WES IDs])]

In [0]:
# Subjects where mini scored 0 phenotypes total
mini_totals = results_mini_full_df.groupby('Subject ID')['GPT_Mini_Score'].sum()
zero_score_subjects = mini_totals[mini_totals == 0].index.tolist()

print(f"Subjects with 0 phenotypes scored by mini: {len(zero_score_subjects)}")
print(zero_score_subjects)

# Pull their notes from notes_enrolled4
zero_score_notes = notes_enrolled4[notes_enrolled4['Subject ID'].isin(zero_score_subjects)]

print(f"\nMatching note rows found: {len(zero_score_notes)}")
display(zero_score_notes[['Subject ID', 'NOTE_TEXT']])

In [0]:
# Subjects where claude scored 0 phenotypes total
claude_totals = results_claude_full_df.groupby('Subject ID')['Claude_Sonnet_Score'].sum()
zero_score_subjects = claude_totals[claude_totals == 0].index.tolist()

print(f"Subjects with 0 phenotypes scored by claude: {len(zero_score_subjects)}")
print(zero_score_subjects)

# Pull their notes from notes_enrolled4
zero_score_notes = notes_enrolled4[notes_enrolled4['Subject ID'].isin(zero_score_subjects)]

print(f"\nMatching note rows found: {len(zero_score_notes)}")
display(zero_score_notes[['Subject ID', 'NOTE_TEXT']])

In [0]:
display(notes_enrolled4[notes_enrolled4['Subject ID'] == 1 WES ID])
display(manual_score_fixed[manual_score_fixed['Subject ID'] == 1 WES ID])
display(results_GPT_full_df[results_GPT_full_df['Subject ID'] == 1 WES ID])
display(results_mini_full_df[results_mini_full_df['Subject ID'] == 1 WES ID])
display(results_claude_full_df[results_claude_full_df['Subject ID'] == 1 WES ID])

In [0]:
matches = notes_enrolled4[notes_enrolled4['NOTE_TEXT'].str.contains('consent forms', case=True, na=False)]

print(f"Rows matching 'consent forms' (case-sensitive): {len(matches)}")
display(matches)

In [0]:
# Check that results_GPT_full_df, results_mini_full_df, and results_claude_full_df
# all have the exact same set of Subject IDs (no extras, no missing), and that
# all of those Subject IDs are also present in manual_score_fixed.

gpt_ids = set(results_GPT_full_df['Subject ID'].unique())
mini_ids = set(results_mini_full_df['Subject ID'].unique())
claude_ids = set(results_claude_full_df['Subject ID'].unique())
manual_ids = set(manual_score_fixed['Subject ID'].unique())

print(f"GPT subjects: {len(gpt_ids)}")
print(f"Mini subjects: {len(mini_ids)}")
print(f"Claude subjects: {len(claude_ids)}")
print(f"Manual subjects: {len(manual_ids)}")
print()

# --- Pairwise comparison across the three result sets ---
all_three_equal = (gpt_ids == mini_ids == claude_ids)
print(f"GPT, Mini, and Claude all have identical Subject ID sets: {all_three_equal}")
print()

if not all_three_equal:
    print("GPT vs Mini:")
    print(f"  In GPT but not Mini: {sorted(gpt_ids - mini_ids)}")
    print(f"  In Mini but not GPT: {sorted(mini_ids - gpt_ids)}")
    print()
    print("GPT vs Claude:")
    print(f"  In GPT but not Claude: {sorted(gpt_ids - claude_ids)}")
    print(f"  In Claude but not GPT: {sorted(claude_ids - gpt_ids)}")
    print()
    print("Mini vs Claude:")
    print(f"  In Mini but not Claude: {sorted(mini_ids - claude_ids)}")
    print(f"  In Claude but not Mini: {sorted(claude_ids - mini_ids)}")
    print()

# --- Check all three are fully contained in manual_score_fixed ---
combined_result_ids = gpt_ids | mini_ids | claude_ids
missing_from_manual = combined_result_ids - manual_ids

print(f"All GPT/Mini/Claude subjects present in manual_score_fixed: {len(missing_from_manual) == 0}")
if missing_from_manual:
    print(f"  Subjects missing from manual_score_fixed: {sorted(missing_from_manual)}")

# --- Optional: also flag if manual_score_fixed has extra subjects not in any result set ---
extra_in_manual = manual_ids - combined_result_ids
if extra_in_manual:
    print(f"\nSubjects in manual_score_fixed but not in any of GPT/Mini/Claude: {len(extra_in_manual)}")
    print(f"  {sorted(extra_in_manual)}")

In [0]:
notes_enrolled4.columns

In [0]:
# View the notes for GPT/Mini/Claude subjects not present in manual_score_fixed

subject_ids_to_check = combined_result_ids - manual_ids

notes_subset = notes_enrolled4[
    notes_enrolled4['Subject ID'].isin(subject_ids_to_check)
][['Subject ID', 'NOTE_FILE_TIME_DTTM', 'NOTE_TEXT', 'Enrollment Status', 'Date of last chart review:']].dropna(subset=['NOTE_TEXT']).reset_index(drop=True)

display(notes_subset)

In [0]:
import pandas as pd

output_folder = "/Workspace/Users/user/user/"

data_labels_filtered4 = pd.read_csv(output_folder + "data_labels_filtered4.csv")

matching_ids = set(data_labels_filtered4['Record ID #:']).intersection(set(subject_ids_to_check))

print(f"Total rows in data_labels_filtered4: {len(data_labels_filtered4)}")
print(f"Subject IDs in subject_ids_to_check: {len(subject_ids_to_check)}")
print(f"Matching subject IDs found: {len(matching_ids)}")

if matching_ids:
    print(matching_ids)
    display(data_labels_filtered4[data_labels_filtered4['Record ID #:'].isin(matching_ids)])
else:
    print("No matching subject IDs found.")

In [0]:
# It looks like these were evaluated, just not scored a phenotype by the manual scorer

# manual_score_fixed = manual_score_fixed[~manual_score_fixed['Subject ID'].isin(subject_ids_to_check)]
#results_GPT_full_df = results_GPT_full_df[~results_GPT_full_df['Subject ID'].isin(subject_ids_to_check)]
#results_mini_full_df = results_mini_full_df[~results_mini_full_df['Subject ID'].isin(subject_ids_to_check)]
#results_claude_full_df = results_claude_full_df[~results_claude_full_df['Subject ID'].isin(subject_ids_to_check)]

In [0]:
# Pull the 10 shortest unique notes from notes_enrolled4, restricted to subjects that also appear in results_GPT_full_df

valid_subjects = results_GPT_full_df['Subject ID'].unique()

filtered_notes = notes_enrolled4[
    notes_enrolled4['Subject ID'].isin(valid_subjects)
].dropna(subset=['NOTE_TEXT']).drop_duplicates(subset=['NOTE_TEXT']).copy()

filtered_notes['char_count'] = filtered_notes['NOTE_TEXT'].str.len()

smallest_10 = filtered_notes.nsmallest(10, 'char_count')[['Subject ID', 'NOTE_TEXT', 'char_count']]
display(smallest_10)

In [0]:
manual_score_fixed = manual_score_fixed[~manual_score_fixed['Subject ID'].isin([5 WES IDs])]
results_GPT_full_df = results_GPT_full_df[~results_GPT_full_df['Subject ID'].isin([5 WES IDs])]
results_mini_full_df = results_mini_full_df[~results_mini_full_df['Subject ID'].isin([5 WES IDs])]
results_claude_full_df = results_claude_full_df[~results_claude_full_df['Subject ID'].isin([5 WES IDs])]

In [0]:
# Drop rows from manual_score_fixed that don't have a matching Subject ID in results_GPT_full_df

gpt_ids = set(results_GPT_full_df['Subject ID'].unique())

before_count = len(manual_score_fixed)
manual_score_fixed = manual_score_fixed[manual_score_fixed['Subject ID'].isin(gpt_ids)].reset_index(drop=True)
after_count = len(manual_score_fixed)

print(f"Rows before: {before_count}")
print(f"Rows after: {after_count}")
print(f"Rows dropped: {before_count - after_count}")

In [0]:
# Find all notes (in notes_enrolled4) belonging to subjects present in any of the
# three model result sets, where the note text contains three consecutive asterisks ("***")

all_model_subject_ids = (
    set(results_GPT_full_df['Subject ID'])
    | set(results_mini_full_df['Subject ID'])
    | set(results_claude_full_df['Subject ID'])
)

notes_subset = notes_enrolled4[
    notes_enrolled4['Subject ID'].isin(all_model_subject_ids)
].dropna(subset=['NOTE_TEXT']).copy()

triple_asterisk_notes = notes_subset[
    notes_subset['NOTE_TEXT'].str.contains(r'\*\*\*', regex=True, na=False)
][['Subject ID', 'NOTE_TEXT']].reset_index(drop=True)

print(f"Notes containing '***': {len(triple_asterisk_notes)}")

if triple_asterisk_notes.empty:
    print("No matching notes found.")
else:
    display(triple_asterisk_notes)

In [0]:
# Viewing score totals across entire dataframes (no subsetting)

score_totals = {}

for name, df, score_col, agg in [
    ('Manual', manual_score_fixed, 'Phenotype', 'count'),
    ('GPT', results_GPT_full_df, 'GPT_Score', 'sum'),
    ('GPT Mini', results_mini_full_df, 'GPT_Mini_Score', 'sum'),
    ('Claude Sonnet', results_claude_full_df, 'Claude_Sonnet_Score', 'sum'),
]:
    if agg == 'count':
        score_totals[name] = df[score_col].count()
    else:
        score_totals[name] = df[score_col].sum()

print("Total phenotypes scored positive, per model, across all subjects:")
for model, total in score_totals.items():
    print(f"  {model}: {total}")

In [0]:
print(f"notes_enrolled4 subjects: {notes_enrolled4['Subject ID'].nunique()}")
print(f"results_GPT_full_df subjects: {results_GPT_full_df['Subject ID'].nunique()}")
print(f"results_mini_full_df subjects: {results_mini_full_df['Subject ID'].nunique()}")
print(f"results_claude_full_df subjects: {results_claude_full_df['Subject ID'].nunique()}")
print(f"manual_score_fixed subjects: {manual_score_fixed['Subject ID'].nunique()}")

In [0]:
# Create sets of (Subject ID, Phenotype) tuples for each model
gpt_pairs = set(zip(results_GPT_full_df['Subject ID'], results_GPT_full_df['Phenotype']))
mini_pairs = set(zip(results_mini_full_df['Subject ID'], results_mini_full_df['Phenotype']))
claude_pairs = set(zip(results_claude_full_df['Subject ID'], results_claude_full_df['Phenotype']))

# Check if all three are identical
print(f"All three models have identical Subject ID/Phenotype pairs: {gpt_pairs == mini_pairs == claude_pairs}")

# If not, see what's different
print("In GPT but not Mini:", gpt_pairs - mini_pairs)
print("In Mini but not GPT:", mini_pairs - gpt_pairs)
print("In GPT but not Claude:", gpt_pairs - claude_pairs)
print("In Claude but not GPT:", claude_pairs - gpt_pairs)

In [0]:
# Finding parse error details
parse_error_ids = results_claude_full_df[results_claude_full_df['Phenotype'] == 'PARSE ERROR']['Subject ID']
print(len(parse_error_ids))
results_GPT_full_df = results_GPT_full_df[~results_GPT_full_df['Subject ID'].isin(parse_error_ids)]
results_mini_full_df = results_mini_full_df[~results_mini_full_df['Subject ID'].isin(parse_error_ids)]
results_claude_full_df = results_claude_full_df[~results_claude_full_df['Subject ID'].isin(parse_error_ids)]
manual_score_fixed = manual_score_fixed[~manual_score_fixed['Subject ID'].isin(parse_error_ids)]

In [0]:
manual_score_fixed = manual_score_fixed[~manual_score_fixed['Subject ID'].isin([6 WES IDs])]
results_GPT_full_df = results_GPT_full_df[~results_GPT_full_df['Subject ID'].isin([6 WES IDs])]
results_mini_full_df = results_mini_full_df[~results_mini_full_df['Subject ID'].isin([6 WES IDs])]
results_claude_full_df = results_claude_full_df[~results_claude_full_df['Subject ID'].isin([6 WES IDs])]

In [0]:
print(f"results_GPT_full_df subjects: {results_GPT_full_df['Subject ID'].nunique()}")
print(f"results_mini_full_df subjects: {results_mini_full_df['Subject ID'].nunique()}")
print(f"results_claude_full_df subjects: {results_claude_full_df['Subject ID'].nunique()}")
print(f"manual_score_fixed subjects: {manual_score_fixed['Subject ID'].nunique()}")

In [0]:
# Phenotypes in manual_score_fixed but not GPT results
manual_score_not_in_GPT = manual_score_fixed[~manual_score_fixed['Phenotype'].isin(results_GPT_full_df['Phenotype'])]
print(manual_score_not_in_GPT)

In [0]:
# Removing phenotypes in manual_score_fixed that aren't in the GPT results. 
manual_score_fixed = manual_score_fixed[manual_score_fixed['Phenotype'].isin(results_GPT_full_df['Phenotype'])]

In [0]:
# Seeing which Subject IDs are in each AI but not the manual score
print("In GPT but not manual_score_fixed:", set(results_GPT_full_df['Subject ID']) - set(manual_score_fixed['Subject ID']))
print("In Mini but not manual_score_fixed:", set(results_mini_full_df['Subject ID']) - set(manual_score_fixed['Subject ID']))
print("In Claude but not manual_score_fixed:", set(results_claude_full_df['Subject ID']) - set(manual_score_fixed['Subject ID']))

In [0]:
# Subject IDs in notes_enrolled4 but not in manual_score_fixed
in_notes_not_manual = set(notes_enrolled4['Subject ID']) - set(manual_score_fixed['Subject ID'])
print(f"Subject IDs in notes_enrolled4 but not manual_score_fixed: {len(in_notes_not_manual)}")
print(in_notes_not_manual)

In [0]:
# Subject IDs in results_GPT_full_df but not in manual_score_fixed
gpt_not_manual_ids = set(results_GPT_full_df['Subject ID']) - set(manual_score_fixed['Subject ID'])

subset = notes_enrolled4[notes_enrolled4['Subject ID'].isin(gpt_not_manual_ids)]

if subset.empty:
    print("No subjects found in results_GPT_full_df that are missing from manual_score_fixed.")
else:
    display(subset)

In [0]:
#Combining all AI scoring to one dataframe

combined_df = results_GPT_full_df.merge(
    results_mini_full_df[['Subject ID', 'Phenotype', 'GPT_Mini_Score']],
    on=['Subject ID', 'Phenotype']
).merge(
    results_claude_full_df[['Subject ID', 'Phenotype', 'Claude_Sonnet_Score']],
    on=['Subject ID', 'Phenotype']
)

display(combined_df)

In [0]:
# matches = notes_enrolled4[notes_enrolled4['Subject ID'].isin([3 WES IDs])]
# display(matches[['Subject ID', 'NOTE_TEXT']])

In [0]:
#combined_df = combined_df[~combined_df['Subject ID'].isin([3 WES IDs])]

In [0]:
# Normalizing manual_score_fixed to match format of combined_df/AI scoring

# Get all unique subject IDs and phenotypes
all_subject_ids = combined_df['Subject ID'].unique()
all_phenotypes = checklist_filtered['Term from checklist'].dropna().unique()

# Create a complete grid of every (Subject ID, Phenotype) combination
full_grid = pd.MultiIndex.from_product(
    [all_subject_ids, all_phenotypes], 
    names=['Subject ID', 'Phenotype']
).to_frame(index=False)

# Mark which combinations were manually scored as present
manual_score_fixed['Manual_Score'] = 1
full_grid = full_grid.merge(
    manual_score_fixed[['Subject ID', 'Phenotype', 'Manual_Score']], 
    on=['Subject ID', 'Phenotype'], 
    how='left'
)
full_grid['Manual_Score'] = full_grid['Manual_Score'].fillna(0).astype(int)

display(full_grid)
print(len(full_grid))

In [0]:
# Quality auditing the normalized manual score output

print("Unique Subject IDs in all_subject_ids:", len(all_subject_ids))
print("Unique Phenotypes in all_phenotypes:", len(all_phenotypes))
print(f"Expected grid size: {len(all_subject_ids)} x {len(all_phenotypes)} = {len(all_subject_ids) * len(all_phenotypes)}")

# Check for duplicates in manual_score_fixed after the merge
print("\nDuplicate (Subject ID, Phenotype) pairs in full_grid:")
print(full_grid.duplicated(subset=['Subject ID', 'Phenotype']).sum())

# Check if any phenotypes in manual_score_fixed aren't in checklist_filtered
manual_phens = set(manual_score_fixed['Phenotype'])
checklist_phens = set(checklist_filtered['Term from checklist'])
print("\nPhenotypes in manual_score_fixed but not in checklist_filtered:")
print(manual_phens - checklist_phens)

In [0]:
# Troubleshooting 126 phenotypes. Possibly null values?
print("Null phenotypes in manual_score_fixed:", manual_score_fixed['Phenotype'].isna().sum())
print("Null phenotypes in checklist_filtered:", checklist_filtered['Term from checklist'].isna().sum())

In [0]:
# Dropping null phenotype values from checklist_filtered
checklist_filtered = checklist_filtered.dropna(subset=['Term from checklist'])
print(f"Checklist phenotypes after dropping nulls: {len(checklist_filtered)}")

In [0]:
# Showing and dropping duplicate phenotypes
# Show the duplicate terms
print("Duplicate terms:")
print(checklist_filtered[checklist_filtered['Term from checklist'].duplicated(keep=False)]['Term from checklist'].unique())

# Deduplicate
checklist_filtered = checklist_filtered.drop_duplicates(subset='Term from checklist')
print(f"\nChecklist phenotypes after deduplication: {len(checklist_filtered)}")

In [0]:
# Find the duplicate phenotype in checklist_filtered
print("Duplicate phenotypes in checklist_filtered:")
print(checklist_filtered[checklist_filtered['Term from checklist'].duplicated(keep=False)]['Term from checklist'].unique())

# Find duplicate (Subject ID, Phenotype) pairs in manual_score_fixed
print("\nDuplicate (Subject ID, Phenotype) pairs in manual_score_fixed:")
print(manual_score_fixed[manual_score_fixed.duplicated(subset=['Subject ID', 'Phenotype'], keep=False)])

## Building combined_df

In [0]:
#Rebuilding manual_score_fixed with clean checklist_filtered

# Recreate records from REDCap_mapping
records = []
for i in range(0, len(REDCap_mapping)):
    phen = []
    for j in range(0, len(REDCap_mapping.columns)):
        if pd.isnull(REDCap_mapping.iloc[i][j]) == False:
            phen.append(REDCap_mapping.iloc[i][j])
    records.append(phen)

records_formatted = pd.DataFrame(records)

# Recreate manual_list from records_formatted
manual_list = []
for j in range(0, len(records_formatted)):
    for i in range(0, len(records_formatted.columns)):
        manual_list.append(records_formatted[i][j])

manual_list = [x for x in manual_list if x is not None]

# Rebuild manual_score_fixed with clean checklist_filtered
rows = []
current_id = None
for item in manual_list:
    if item.startswith('WES'):
        current_id = item
    else:
        rows.append((current_id, item))

manual_df = pd.DataFrame(rows, columns=['Subject ID', 'Phenotype'])

# <<< RESTORED LINE — normalize BEFORE the join, not after >>>
manual_df['Phenotype'] = manual_df['Phenotype'].str.strip().map(lambda x: remap.get(x, x))

manual_score_fixed = manual_df.merge(
    checklist_filtered[['Term from checklist', 'HPO term (ID)']],
    left_on='Phenotype',
    right_on='Term from checklist',
    how='inner'
).drop(columns=['Term from checklist'])

manual_score_fixed['Phenotype'] = manual_score_fixed['Phenotype'].str.strip()
print(f"manual_score_fixed rows: {len(manual_score_fixed)}")
print(f"Duplicate (Subject ID, Phenotype) pairs: {manual_score_fixed.duplicated(subset=['Subject ID', 'Phenotype']).sum()}")

nine_terms = [
    'Autism Spectrum Disorder (ASD)', 'Chronic diarrhea', 'Undescended testis',
    'Diabetes Mellitus Type I (insulin dependent)', 'Diabetes Mellitus Type II (non-insulin dependent)',
    'Hypertrophic cardiomyopathy (HCM)', 'Double Outlet Right Ventricle (DORV)', 'Ketosis',
    'Tetralogy of Fallot (TOF)'
]
found = manual_score_fixed[manual_score_fixed['Phenotype'].isin(nine_terms)].groupby('Phenotype').size().sort_values(ascending=False)
print()
print("Rows for the 9 bug-affected terms (expect ~231/48/47/7/6/6/4/3/3):")
print(found)

In [0]:
# Normalizing the manual_score_fixed so it can be joined onto the combined dataframe 
# Get all unique subject IDs and phenotypes
all_subject_ids = combined_df['Subject ID'].unique()
all_phenotypes = checklist_filtered['Term from checklist'].unique()

# Create a complete grid of every (Subject ID, Phenotype) combination
full_grid = pd.MultiIndex.from_product(
    [all_subject_ids, all_phenotypes], 
    names=['Subject ID', 'Phenotype']
).to_frame(index=False)

# Mark which combinations were manually scored as present
manual_score_fixed['Manual_Score'] = 1
full_grid = full_grid.merge(
    manual_score_fixed[['Subject ID', 'Phenotype', 'Manual_Score']], 
    on=['Subject ID', 'Phenotype'], 
    how='left'
)
full_grid['Manual_Score'] = full_grid['Manual_Score'].fillna(0).astype(int)

print(len(full_grid))
print("Duplicate (Subject ID, Phenotype) pairs:", full_grid.duplicated(subset=['Subject ID', 'Phenotype']).sum())
display(full_grid)

In [0]:
# Adding the manual score values to the AI combined dataframe from earlier
combined_df = combined_df.merge(full_grid, on=['Subject ID', 'Phenotype'], how='left')
print(f"combined_df rows: {len(combined_df)}")
print(f"Null Manual_Score values: {combined_df['Manual_Score'].isna().sum()}")
display(combined_df)

In [0]:
print(f"combined_df shape:           {combined_df.shape}")
print()

nine_terms = [
    'Autism Spectrum Disorder (ASD)', 'Chronic diarrhea', 'Undescended testis',
    'Diabetes Mellitus Type I (insulin dependent)', 'Diabetes Mellitus Type II (non-insulin dependent)',
    'Hypertrophic cardiomyopathy (HCM)', 'Double Outlet Right Ventricle (DORV)', 'Ketosis',
    'Tetralogy of Fallot (TOF)'
]
found = manual_score_fixed[manual_score_fixed['Phenotype'].isin(nine_terms)].groupby('Phenotype').size().sort_values(ascending=False)
print("Rows for the 9 bug-affected terms in the CURRENT manual_score_fixed (expect ~231/48/47/7/6/6/4/3/3):")
print(found if len(found) else "NONE FOUND -- all 9 terms failed to match in this rebuild")

In [0]:
print(f"combined_df shape: {combined_df.shape}")
print()

all_checklist_terms = checklist_filtered['Term from checklist'].dropna().unique()
found_counts = manual_score_fixed[manual_score_fixed['Phenotype'].isin(all_checklist_terms)].groupby('Phenotype').size()

zero_terms = sorted(set(all_checklist_terms) - set(found_counts.index))

print(f"Total checklist terms checked: {len(all_checklist_terms)}")
print(f"Terms with at least 1 row in manual_score_fixed: {len(found_counts)}")
print(f"Terms with ZERO rows in manual_score_fixed: {len(zero_terms)}")
print()
if zero_terms:
    print("Terms with zero rows:")
    for t in zero_terms:
        print(f"  {t!r}")
print()

In [0]:
combined_df.columns

In [0]:
# Record counts before removing Stenosis
before_len = len(combined_df)
previous_unique_subject_ids = combined_df["Subject ID"].nunique()

expected_before_len = 125 * previous_unique_subject_ids
expected_after_len = 124 * previous_unique_subject_ids

# Remove all rows where Phenotype is Stenosis
combined_df = combined_df[
    ~combined_df["Phenotype"]
    .astype(str)
    .str.strip()
    .str.casefold()
    .eq("stenosis")
].copy()

after_len = len(combined_df)

print(f"Unique Subject IDs: {previous_unique_subject_ids:,}")
print()
print(f"Actual length before:   {before_len:,}")
print(f"Expected length before: {expected_before_len:,}")
print()
print(f"Actual length after:    {after_len:,}")
print(f"Expected length after:  {expected_after_len:,}")
print()
print(f"Rows removed:           {before_len - after_len:,}")

## Disagreements With Manual

In [0]:
# Flag disagreements for each model vs manual
combined_df['GPT_disagree'] = (combined_df['GPT_Score'] != combined_df['Manual_Score']).astype(int)
combined_df['GPT_Mini_disagree'] = (combined_df['GPT_Mini_Score'] != combined_df['Manual_Score']).astype(int)
combined_df['Claude_Sonnet_disagree'] = (combined_df['Claude_Sonnet_Score'] != combined_df['Manual_Score']).astype(int)

# Total disagreements — any model disagrees with manual
combined_df['any_disagree'] = (
    (combined_df['GPT_disagree'] == 1) | 
    (combined_df['GPT_Mini_disagree'] == 1) | 
    (combined_df['Claude_Sonnet_disagree'] == 1)
).astype(int)

print(f"Total rows where at least one model disagrees with manual: {combined_df['any_disagree'].sum()}")

# Unique disagreements — only this model disagrees with manual, other two agree
combined_df['GPT_unique_disagree'] = (
    (combined_df['GPT_disagree'] == 1) & 
    (combined_df['GPT_Mini_disagree'] == 0) & 
    (combined_df['Claude_Sonnet_disagree'] == 0)
).astype(int)

combined_df['GPT_Mini_unique_disagree'] = (
    (combined_df['GPT_Mini_disagree'] == 1) & 
    (combined_df['GPT_disagree'] == 0) & 
    (combined_df['Claude_Sonnet_disagree'] == 0)
).astype(int)

combined_df['Claude_Sonnet_unique_disagree'] = (
    (combined_df['Claude_Sonnet_disagree'] == 1) & 
    (combined_df['GPT_disagree'] == 0) & 
    (combined_df['GPT_Mini_disagree'] == 0)
).astype(int)

print("\nTotal disagreements vs manual per model:")
print(f"  GPT: {combined_df['GPT_disagree'].sum()}")
print(f"  GPT Mini: {combined_df['GPT_Mini_disagree'].sum()}")
print(f"  Claude Sonnet: {combined_df['Claude_Sonnet_disagree'].sum()}")

print("\nUnique disagreements vs manual (only this model disagrees):")
print(f"  GPT: {combined_df['GPT_unique_disagree'].sum()}")
print(f"  GPT Mini: {combined_df['GPT_Mini_unique_disagree'].sum()}")
print(f"  Claude Sonnet: {combined_df['Claude_Sonnet_unique_disagree'].sum()}")

In [0]:
print("Unique values in Claude_Sonnet_Score:", combined_df['Claude_Sonnet_Score'].unique())
print("Unique values in GPT_Score:", combined_df['GPT_Score'].unique())
print("Unique values in GPT_Mini_Score:", combined_df['GPT_Mini_Score'].unique())
print("Unique values in Manual_Score:", combined_df['Manual_Score'].unique())

In [0]:
# GPT agrees but Mini disagrees
gpt_agree_mini_disagree = (
    (combined_df['GPT_disagree'] == 0) & 
    (combined_df['GPT_Mini_disagree'] == 1)
).sum()

# GPT agrees but Claude disagrees
gpt_agree_claude_disagree = (
    (combined_df['GPT_disagree'] == 0) & 
    (combined_df['Claude_Sonnet_disagree'] == 1)
).sum()

print(f"GPT agrees, Mini disagrees: {gpt_agree_mini_disagree}")
print(f"GPT agrees, Claude disagrees: {gpt_agree_claude_disagree}")

In [0]:
import matplotlib
print(matplotlib.__version__)
print(matplotlib.__file__)

In [0]:
# Scoring Venn Diagram
import matplotlib
print(matplotlib.__version__)  # should show 3.8.4

import matplotlib.pyplot as plt
from matplotlib_venn import venn3

gpt = set(combined_df[combined_df['GPT_disagree'] == 1].index)
mini = set(combined_df[combined_df['GPT_Mini_disagree'] == 1].index)
claude = set(combined_df[combined_df['Claude_Sonnet_disagree'] == 1].index)

# Compute actual subset sizes but normalize to equal circle sizes
subsets = (
    len(gpt - mini - claude),
    len(mini - gpt - claude),
    len(gpt & mini - claude),
    len(claude - gpt - mini),
    len(gpt & claude - mini),
    len(mini & claude - gpt),
    len(gpt & mini & claude),
)

# Use equal values to force equal circle sizes, but label with real counts
equal_subsets = (1, 1, 1, 1, 1, 1, 1)

fig, ax = plt.subplots(figsize=(8, 8))

v = venn3(
    subsets=equal_subsets,
    set_labels=('GPT', 'GPT Mini', 'Claude Sonnet'),
    set_colors=('#4C9BE8', '#57C4A7', '#E87B4C'),
    alpha=0.5,
    ax=ax
)

# Overwrite labels with real counts
labels = list(subsets)
for i, label in enumerate(v.subset_labels):
    if label:
        label.set_text(str(labels[i]))
        label.set_fontsize(11)

for patch in v.patches:
    if patch:
        patch.set_edgecolor('white')
        patch.set_linewidth(2)

for label in v.set_labels:
    if label:
        label.set_fontsize(13)
        label.set_fontweight('bold')

ax.set_title('Disagreements with manual score', fontsize=15, pad=20)
plt.tight_layout()
plt.show()

In [0]:
# Precision and recall calculations
from sklearn.metrics import precision_score, recall_score

total_rows = len(combined_df)
total_disagree = (combined_df['GPT_disagree'] | combined_df['GPT_Mini_disagree'] | combined_df['Claude_Sonnet_disagree']).sum()
pct_disagree = total_disagree / total_rows * 100

print(f"Total rows: {total_rows}")
print(f"Total disagreements (any model): {total_disagree}")
print(f"Percent disagreements: {pct_disagree:.1f}%")

print("\n--- Using manual score as ground truth ---\n")

for model, col in [('GPT', 'GPT_Score'), ('GPT Mini', 'GPT_Mini_Score'), ('Claude Sonnet', 'Claude_Sonnet_Score')]:
    y_true = combined_df['Manual_Score']
    y_pred = combined_df[col].astype(int)

    TP = ((y_pred == 1) & (y_true == 1)).sum()
    TN = ((y_pred == 0) & (y_true == 0)).sum()
    FP = ((y_pred == 1) & (y_true == 0)).sum()
    FN = ((y_pred == 0) & (y_true == 1)).sum()

    precision = TP / (TP + FP) if (TP + FP) > 0 else 0
    recall = TP / (TP + FN) if (TP + FN) > 0 else 0

    print(f"=== {model} ===")
    print(f"TP = {TP} (both manual and {model} scored a 1)")
    print(f"TN = {TN} (both manual and {model} scored a 0)")
    print(f"FP = {FP} ({model} scored a 1 but manual scored a 0)")
    print(f"FN = {FN} ({model} scored a 0 but manual scored a 1)")
    print(f"Precision = TP / (TP + FP) = {TP} / ({TP} + {FP}) = {precision:.3f}")
    print(f"Recall    = TP / (TP + FN) = {TP} / ({TP} + {FN}) = {recall:.3f}")
    print()

In [0]:
# Sensitivity, Specificity, and F1 calculations

from sklearn.metrics import f1_score

print("--- Using manual score as ground truth ---\n")

for model, col in [('GPT', 'GPT_Score'), ('GPT Mini', 'GPT_Mini_Score'), ('Claude Sonnet', 'Claude_Sonnet_Score')]:
    y_true = combined_df['Manual_Score']
    y_pred = combined_df[col].astype(int)

    TP = ((y_pred == 1) & (y_true == 1)).sum()
    TN = ((y_pred == 0) & (y_true == 0)).sum()
    FP = ((y_pred == 1) & (y_true == 0)).sum()
    FN = ((y_pred == 0) & (y_true == 1)).sum()

    sensitivity = TP / (TP + FN) if (TP + FN) > 0 else 0
    specificity = TN / (TN + FP) if (TN + FP) > 0 else 0
    f1 = f1_score(y_true, y_pred)

    print(f"=== {model} ===")
    print(f"TP = {TP} (both manual and {model} scored a 1)")
    print(f"TN = {TN} (both manual and {model} scored a 0)")
    print(f"FP = {FP} ({model} scored a 1 but manual scored a 0)")
    print(f"FN = {FN} ({model} scored a 0 but manual scored a 1)")
    print(f"Sensitivity = TP / (TP + FN) = {TP} / ({TP} + {FN}) = {sensitivity:.3f}")
    print(f"Specificity = TN / (TN + FP) = {TN} / ({TN} + {FP}) = {specificity:.3f}")
    print(f"F1          = 2*TP / (2*TP + FP + FN) = 2*{TP} / (2*{TP} + {FP} + {FN}) = {f1:.3f}")
    print()

In [0]:
# Number of subjects with a disagreement
disagree_by_subject = combined_df.groupby('Subject ID').agg(
    any_disagree=('any_disagree', 'sum')
).reset_index()

subjects_with_disagree = (disagree_by_subject['any_disagree'] > 0).sum()
subjects_no_disagree = (disagree_by_subject['any_disagree'] == 0).sum()

print(f"Total subjects: {len(disagree_by_subject)}")
print(f"Subjects with at least one disagreement: {subjects_with_disagree}")
print(f"Subjects with no disagreements: {subjects_no_disagree}")
print(f"Percent of subjects with at least one disagreement: {subjects_with_disagree / len(disagree_by_subject) * 100:.1f}%")

# Figures

## Venn Diagrams, Upset Plot

In [0]:
# ============================================================
# EDIT THESE TWO LINES TO SWAP MODELS
model = "GPT 5.4 Mini"
score_col = "GPT_Mini_Score"      # options: 'GPT_Score', 'GPT_Mini_Score', 'Claude_Sonnet_Score'
# ============================================================

import matplotlib.pyplot as plt
from matplotlib_venn import venn2

manual_set = set(combined_df[combined_df['Manual_Score'] == 1].index)
model_set  = set(combined_df[combined_df[score_col] == 1].index)

manual_only = len(manual_set - model_set)
model_only  = len(model_set - manual_set)
both        = len(manual_set & model_set)

# Use equal values to force equal circle sizes, label with real counts
subsets = (manual_only, model_only, both)
equal_subsets = (1, 1, 1)

fig, ax = plt.subplots(figsize=(7, 7))

v = venn2(
    subsets=equal_subsets,
    set_labels=('Manual', model),
    set_colors=('#D4B896', '#57C4A7'),
    alpha=0.5,
    ax=ax
)

# Overwrite labels with real counts
labels = list(subsets)
for i, label in enumerate(v.subset_labels):
    if label:
        label.set_text(str(labels[i]))
        label.set_fontsize(12)

for patch in v.patches:
    if patch:
        patch.set_edgecolor('white')
        patch.set_linewidth(2)

for label in v.set_labels:
    if label:
        label.set_fontsize(14)
        label.set_fontweight('bold')

ax.set_title(f'Positive phenotype calls: Manual vs {model}', fontsize=15, pad=20)

# Precision / Recall text beneath the figure
precision = both / (both + model_only) if (both + model_only) > 0 else 0
recall    = both / (both + manual_only) if (both + manual_only) > 0 else 0

formula_text = (
    f"Precision = TP / (TP + FP) = {both} / ({both} + {model_only}) = {precision:.3f}\n"
    f"Recall    = TP / (TP + FN) = {both} / ({both} + {manual_only}) = {recall:.3f}"
)

fig.text(
    0.5, 0.15, formula_text,
    ha='center', va='bottom',
    fontsize=15, fontfamily='monospace',
    linespacing=1.8
)

plt.subplots_adjust(bottom=0.18)
plt.show()

print(f"Manual total positives: {len(manual_set)}")
print(f"{model} total positives: {len(model_set)}")
print(f"Manual only (FN): {manual_only}")
print(f"{model} only (FP): {model_only}")
print(f"Both (TP): {both}")

In [0]:
# ============================================================
# EDIT THESE TWO LINES TO SWAP MODELS
model = "GPT 5.4"
score_col = "GPT_Score"      # options: 'GPT_Score', 'GPT_Mini_Score', 'Claude_Sonnet_Score'
# ============================================================

import matplotlib.pyplot as plt
from matplotlib_venn import venn2

manual_set = set(combined_df[combined_df['Manual_Score'] == 1].index)
model_set  = set(combined_df[combined_df[score_col] == 1].index)

manual_only = len(manual_set - model_set)   # FN
model_only  = len(model_set - manual_set)   # FP
both        = len(manual_set & model_set)   # TP

precision = both / (both + model_only) if (both + model_only) > 0 else 0
recall    = both / (both + manual_only) if (both + manual_only) > 0 else 0

# Use equal values to force equal circle sizes, label with real counts
subsets = (manual_only, model_only, both)
equal_subsets = (1, 1, 1)

fig, ax = plt.subplots(figsize=(7, 8.5))

v = venn2(
    subsets=equal_subsets,
    set_labels=('Manual', model),
    set_colors=('#D4B896', '#4C9BE8'),
    alpha=0.5,
    ax=ax
)

# Overwrite labels with real counts
labels = list(subsets)
for i, label in enumerate(v.subset_labels):
    if label:
        label.set_text(str(labels[i]))
        label.set_fontsize(12)

for patch in v.patches:
    if patch:
        patch.set_edgecolor('white')
        patch.set_linewidth(2)

for label in v.set_labels:
    if label:
        label.set_fontsize(14)
        label.set_fontweight('bold')

ax.set_title(f'Positive phenotype calls: Manual vs {model}', fontsize=15, pad=20)

# Precision / Recall text beneath the figure
precision = both / (both + model_only) if (both + model_only) > 0 else 0
recall    = both / (both + manual_only) if (both + manual_only) > 0 else 0

formula_text = (
    f"Precision = TP / (TP + FP) = {both} / ({both} + {model_only}) = {precision:.3f}\n"
    f"Recall    = TP / (TP + FN) = {both} / ({both} + {manual_only}) = {recall:.3f}"
)

fig.text(
    0.5, 0.2, formula_text,
    ha='center', va='bottom',
    fontsize=15, fontfamily='monospace',
    linespacing=1.8
)

plt.subplots_adjust(bottom=0.18)
plt.show()

print(f"Manual total positives: {len(manual_set)}")
print(f"{model} total positives: {len(model_set)}")
print(f"Manual only (FN): {manual_only}")
print(f"{model} only (FP): {model_only}")
print(f"Both (TP): {both}")
print(f"score_col in use: {score_col}")
print(f"TP={both}, FP={model_only}, FN={manual_only}")

In [0]:
# ============================================================
# EDIT THESE TWO LINES TO SWAP MODELS
model = "Claude Sonnet 4.6"
score_col = "Claude_Sonnet_Score"      # options: 'GPT_Score', 'GPT_Mini_Score', 'Claude_Sonnet_Score'
# ============================================================

import matplotlib.pyplot as plt
from matplotlib_venn import venn2

manual_set = set(combined_df[combined_df['Manual_Score'] == 1].index)
model_set  = set(combined_df[combined_df[score_col] == 1].index)

manual_only = len(manual_set - model_set)
model_only  = len(model_set - manual_set)
both        = len(manual_set & model_set)

# Use equal values to force equal circle sizes, label with real counts
subsets = (manual_only, model_only, both)
equal_subsets = (1, 1, 1)

fig, ax = plt.subplots(figsize=(7, 7))

v = venn2(
    subsets=equal_subsets,
    set_labels=('Manual', model),
    set_colors=('#D4B896', '#E87B4C'),
    alpha=0.5,
    ax=ax
)


# Overwrite labels with real counts
labels = list(subsets)
for i, label in enumerate(v.subset_labels):
    if label:
        label.set_text(str(labels[i]))
        label.set_fontsize(12)

for patch in v.patches:
    if patch:
        patch.set_edgecolor('white')
        patch.set_linewidth(2)

for label in v.set_labels:
    if label:
        label.set_fontsize(14)
        label.set_fontweight('bold')

ax.set_title(f'Positive phenotype calls: Manual vs {model}', fontsize=15, pad=20)

# Precision / Recall text beneath the figure
precision = both / (both + model_only) if (both + model_only) > 0 else 0
recall    = both / (both + manual_only) if (both + manual_only) > 0 else 0
formula_text = (
    f"Precision = TP / (TP + FP) = {both} / ({both} + {model_only}) = {precision:.3f}\n"
    f"Recall    = TP / (TP + FN) = {both} / ({both} + {manual_only}) = {recall:.3f}"
)

fig.text(
    0.5, 0.15, formula_text,
    ha='center', va='bottom',
    fontsize=15, fontfamily='monospace',
    linespacing=1.8
)

plt.subplots_adjust(bottom=0.18)
plt.show()

print(f"Manual total positives: {len(manual_set)}")
print(f"{model} total positives: {len(model_set)}")
print(f"Manual only (FN): {manual_only}")
print(f"{model} only (FP): {model_only}")
print(f"Both (TP): {both}")
print(f"score_col in use: {score_col}")
print(f"TP={both}, FP={model_only}, FN={manual_only}")

In [0]:
# Scoring Venn Diagram

import matplotlib.pyplot as plt
from matplotlib_venn import venn3

gpt = set(combined_df[combined_df['GPT_disagree'] == 1].index)
mini = set(combined_df[combined_df['GPT_Mini_disagree'] == 1].index)
claude = set(combined_df[combined_df['Claude_Sonnet_disagree'] == 1].index)

# Real counts now drive circle size directly (no more equal_subsets)
subsets = (
    len(gpt - mini - claude),
    len(mini - gpt - claude),
    len(gpt & mini - claude),
    len(claude - gpt - mini),
    len(gpt & claude - mini),
    len(mini & claude - gpt),
    len(gpt & mini & claude),
)

fig, ax = plt.subplots(figsize=(8, 8))

v = venn3(
    subsets=subsets,
    set_labels=('GPT', 'GPT Mini', 'Claude Sonnet'),
    set_colors=('#4C9BE8', '#57C4A7', '#E87B4C'),
    alpha=0.5,
    ax=ax
)

for label in v.subset_labels:
    if label:
        label.set_fontsize(11)

for patch in v.patches:
    if patch:
        patch.set_edgecolor('white')
        patch.set_linewidth(2)

for label in v.set_labels:
    if label:
        label.set_fontsize(13)
        label.set_fontweight('bold')

ax.set_title('Disagreements with manual score', fontsize=15, pad=20)
plt.tight_layout()
plt.show()

In [0]:
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

gpt = set(combined_df[combined_df['GPT_disagree'] == 1].index)
mini = set(combined_df[combined_df['GPT_Mini_disagree'] == 1].index)
claude = set(combined_df[combined_df['Claude_Sonnet_disagree'] == 1].index)

# Define each intersection: (label tuple of which sets are TRUE, count)
intersections = [
    (('GPT',), len(gpt - mini - claude)),
    (('GPT Mini',), len(mini - gpt - claude)),
    (('Claude Sonnet',), len(claude - gpt - mini)),
    (('GPT', 'GPT Mini'), len(gpt & mini - claude)),
    (('GPT', 'Claude Sonnet'), len(gpt & claude - mini)),
    (('GPT Mini', 'Claude Sonnet'), len(mini & claude - gpt)),
    (('GPT', 'GPT Mini', 'Claude Sonnet'), len(gpt & mini & claude)),
]

# Sort largest to smallest (like sort_by='cardinality')
intersections.sort(key=lambda x: x[1], reverse=True)

sets = ['GPT', 'GPT Mini', 'Claude Sonnet']
counts = [c for _, c in intersections]
memberships = [members for members, _ in intersections]

fig = plt.figure(figsize=(10, 7))
gs = gridspec.GridSpec(2, 1, height_ratios=[3, 1.2], hspace=0.05)

# Top: bar chart of intersection sizes
ax_bar = fig.add_subplot(gs[0])
bar_positions = range(len(counts))
ax_bar.bar(bar_positions, counts, color='#4C9BE8', width=0.6)

for i, c in enumerate(counts):
    ax_bar.text(i, c + max(counts) * 0.01, str(c), ha='center', fontsize=10)

ax_bar.set_xticks([])
ax_bar.set_ylabel('Disagreement count', fontsize=12)
ax_bar.set_title('Disagreements with manual score', fontsize=15, pad=15)
ax_bar.spines['top'].set_visible(False)
ax_bar.spines['right'].set_visible(False)
ax_bar.spines['bottom'].set_visible(False)

# Bottom: dot matrix showing set membership per bar
ax_matrix = fig.add_subplot(gs[1], sharex=ax_bar)

for i, members in enumerate(memberships):
    active_idx = [sets.index(s) for s in members]
    # faint dots for inactive sets
    for j in range(len(sets)):
        color = '#333333' if j in active_idx else '#dddddd'
        ax_matrix.plot(i, j, 'o', color=color, markersize=10)
    # connecting line between active dots
    if len(active_idx) > 1:
        ax_matrix.plot([i, i], [min(active_idx), max(active_idx)], color='#333333', linewidth=1.5, zorder=0)

ax_matrix.set_yticks(range(len(sets)))
ax_matrix.set_yticklabels(sets, fontsize=12)
ax_matrix.set_xticks([])
ax_matrix.set_ylim(-0.5, len(sets) - 0.5)
ax_matrix.invert_yaxis()
for spine in ax_matrix.spines.values():
    spine.set_visible(False)

plt.tight_layout()
plt.show()

In [0]:
# ============================================================
# EDIT THESE TWO LINES TO SWAP MODELS
model = "GPT 5.4 Mini"
score_col = "GPT_Mini_Score"      # options: 'GPT_Score', 'GPT_Mini_Score', 'Claude_Sonnet_Score'
# ============================================================

import matplotlib.pyplot as plt
from matplotlib_venn import venn2

manual_set = set(combined_df[combined_df['Manual_Score'] == 1].index)
model_set  = set(combined_df[combined_df[score_col] == 1].index)

manual_only = len(manual_set - model_set)
model_only  = len(model_set - manual_set)
both        = len(manual_set & model_set)

precision = both / (both + model_only) if (both + model_only) > 0 else 0
recall    = both / (both + manual_only) if (both + manual_only) > 0 else 0

# Real counts now drive circle size directly (no more equal_subsets)
subsets = (manual_only, model_only, both)

this_total = manual_only + model_only + both

totals_by_model = {
    'GPT_Score': len(manual_set - set(combined_df[combined_df['GPT_Score'] == 1].index)) +
                 len(set(combined_df[combined_df['GPT_Score'] == 1].index) - manual_set) +
                 len(manual_set & set(combined_df[combined_df['GPT_Score'] == 1].index)),
    'GPT_Mini_Score': len(manual_set - set(combined_df[combined_df['GPT_Mini_Score'] == 1].index)) +
                       len(set(combined_df[combined_df['GPT_Mini_Score'] == 1].index) - manual_set) +
                       len(manual_set & set(combined_df[combined_df['GPT_Mini_Score'] == 1].index)),
    'Claude_Sonnet_Score': len(manual_set - set(combined_df[combined_df['Claude_Sonnet_Score'] == 1].index)) +
                             len(set(combined_df[combined_df['Claude_Sonnet_Score'] == 1].index) - manual_set) +
                             len(manual_set & set(combined_df[combined_df['Claude_Sonnet_Score'] == 1].index)),
}
max_total = max(totals_by_model.values())
normalize_to = this_total / max_total

fig, ax = plt.subplots(figsize=(7, 7))

v = venn2(
    subsets=subsets,
    set_labels=('Manual', model),
    set_colors=('#D4B896', '#57C4A7'),
    alpha=0.5,
    normalize_to=normalize_to,   # <-- add this line
    ax=ax
)

for label in v.subset_labels:
    if label:
        label.set_fontsize(12)

for patch in v.patches:
    if patch:
        patch.set_edgecolor('white')
        patch.set_linewidth(2)

for label in v.set_labels:
    if label:
        label.set_fontsize(14)
        label.set_fontweight('bold')

ax.set_title(f'Positive phenotype calls: Manual vs {model}', fontsize=15, pad=20)

formula_text = (
    f"Precision = TP / (TP + FP) = {both} / ({both} + {model_only}) = {precision:.3f}\n"
    f"Recall    = TP / (TP + FN) = {both} / ({both} + {manual_only}) = {recall:.3f}"
)

fig.text(
    0.5, 0.1, formula_text,
    ha='center', va='bottom',
    fontsize=15, fontfamily='monospace',
    linespacing=1.8
)

plt.subplots_adjust(bottom=0.18)
plt.show()

print(f"Manual total positives: {len(manual_set)}")
print(f"{model} total positives: {len(model_set)}")
print(f"Manual only (FN): {manual_only}")
print(f"{model} only (FP): {model_only}")
print(f"Both (TP): {both}")

In [0]:
print(f"{model}: this_total = {this_total}, normalize_to = {normalize_to:.4f}")

In [0]:
print(f"Manual constant check: {len(manual_set)} / {max_total} = {len(manual_set)/max_total:.5f}")

In [0]:
# ============================================================
# EDIT THESE TWO LINES TO SWAP MODELS
model = "GPT 5.4"
score_col = "GPT_Score"      # options: 'GPT_Score', 'GPT_Mini_Score', 'Claude_Sonnet_Score'
# ============================================================

import matplotlib.pyplot as plt
from matplotlib_venn import venn2

manual_set = set(combined_df[combined_df['Manual_Score'] == 1].index)
model_set  = set(combined_df[combined_df[score_col] == 1].index)

manual_only = len(manual_set - model_set)
model_only  = len(model_set - manual_set)
both        = len(manual_set & model_set)

precision = both / (both + model_only) if (both + model_only) > 0 else 0
recall    = both / (both + manual_only) if (both + manual_only) > 0 else 0

# Real counts now drive circle size directly (no more equal_subsets)
subsets = (manual_only, model_only, both)

this_total = manual_only + model_only + both

totals_by_model = {
    'GPT_Score': len(manual_set - set(combined_df[combined_df['GPT_Score'] == 1].index)) +
                 len(set(combined_df[combined_df['GPT_Score'] == 1].index) - manual_set) +
                 len(manual_set & set(combined_df[combined_df['GPT_Score'] == 1].index)),
    'GPT_Mini_Score': len(manual_set - set(combined_df[combined_df['GPT_Mini_Score'] == 1].index)) +
                       len(set(combined_df[combined_df['GPT_Mini_Score'] == 1].index) - manual_set) +
                       len(manual_set & set(combined_df[combined_df['GPT_Mini_Score'] == 1].index)),
    'Claude_Sonnet_Score': len(manual_set - set(combined_df[combined_df['Claude_Sonnet_Score'] == 1].index)) +
                             len(set(combined_df[combined_df['Claude_Sonnet_Score'] == 1].index) - manual_set) +
                             len(manual_set & set(combined_df[combined_df['Claude_Sonnet_Score'] == 1].index)),
}
max_total = max(totals_by_model.values())
normalize_to = this_total / max_total

fig, ax = plt.subplots(figsize=(7, 7))

v = venn2(
    subsets=subsets,
    set_labels=('Manual', model),
    set_colors=('#D4B896', '#4C9BE8'),
    alpha=0.5,
    normalize_to=normalize_to,   # <-- add this line
    ax=ax
)

for label in v.subset_labels:
    if label:
        label.set_fontsize(12)

for patch in v.patches:
    if patch:
        patch.set_edgecolor('white')
        patch.set_linewidth(2)

for label in v.set_labels:
    if label:
        label.set_fontsize(14)
        label.set_fontweight('bold')

ax.set_title(f'Positive phenotype calls: Manual vs {model}', fontsize=15, pad=20)

formula_text = (
    f"Precision = TP / (TP + FP) = {both} / ({both} + {model_only}) = {precision:.3f}\n"
    f"Recall    = TP / (TP + FN) = {both} / ({both} + {manual_only}) = {recall:.3f}"
)

fig.text(
    0.5, 0.1, formula_text,
    ha='center', va='bottom',
    fontsize=15, fontfamily='monospace',
    linespacing=1.8
)

plt.subplots_adjust(bottom=0.18)
plt.show()

print(f"Manual total positives: {len(manual_set)}")
print(f"{model} total positives: {len(model_set)}")
print(f"Manual only (FN): {manual_only}")
print(f"{model} only (FP): {model_only}")
print(f"Both (TP): {both}")

In [0]:
print(f"{model}: this_total = {this_total}, normalize_to = {normalize_to:.4f}")

In [0]:
print(f"Manual constant check: {len(manual_set)} / {max_total} = {len(manual_set)/max_total:.5f}")

In [0]:
# ============================================================
# EDIT THESE TWO LINES TO SWAP MODELS
model = "Claude Sonnet 4.6"
score_col = "Claude_Sonnet_Score"      # options: 'GPT_Score', 'GPT_Mini_Score', 'Claude_Sonnet_Score'
# ============================================================

import matplotlib.pyplot as plt
from matplotlib_venn import venn2

manual_set = set(combined_df[combined_df['Manual_Score'] == 1].index)
model_set  = set(combined_df[combined_df[score_col] == 1].index)

manual_only = len(manual_set - model_set)
model_only  = len(model_set - manual_set)
both        = len(manual_set & model_set)

precision = both / (both + model_only) if (both + model_only) > 0 else 0
recall    = both / (both + manual_only) if (both + manual_only) > 0 else 0

# Real counts now drive circle size directly (no more equal_subsets)
subsets = (manual_only, model_only, both)

this_total = manual_only + model_only + both

totals_by_model = {
    'GPT_Score': len(manual_set - set(combined_df[combined_df['GPT_Score'] == 1].index)) +
                 len(set(combined_df[combined_df['GPT_Score'] == 1].index) - manual_set) +
                 len(manual_set & set(combined_df[combined_df['GPT_Score'] == 1].index)),
    'GPT_Mini_Score': len(manual_set - set(combined_df[combined_df['GPT_Mini_Score'] == 1].index)) +
                       len(set(combined_df[combined_df['GPT_Mini_Score'] == 1].index) - manual_set) +
                       len(manual_set & set(combined_df[combined_df['GPT_Mini_Score'] == 1].index)),
    'Claude_Sonnet_Score': len(manual_set - set(combined_df[combined_df['Claude_Sonnet_Score'] == 1].index)) +
                             len(set(combined_df[combined_df['Claude_Sonnet_Score'] == 1].index) - manual_set) +
                             len(manual_set & set(combined_df[combined_df['Claude_Sonnet_Score'] == 1].index)),
}
max_total = max(totals_by_model.values())
normalize_to = this_total / max_total

fig, ax = plt.subplots(figsize=(7, 7))

v = venn2(
    subsets=subsets,
    set_labels=('Manual', model),
    set_colors=('#D4B896', '#E87B4C'),
    alpha=0.5,
    normalize_to=normalize_to,   # <-- add this line
    ax=ax
)

for label in v.subset_labels:
    if label:
        label.set_fontsize(12)

for patch in v.patches:
    if patch:
        patch.set_edgecolor('white')
        patch.set_linewidth(2)

for label in v.set_labels:
    if label:
        label.set_fontsize(14)
        label.set_fontweight('bold')

ax.set_title(f'Positive phenotype calls: Manual vs {model}', fontsize=15, pad=20)

formula_text = (
    f"Precision = TP / (TP + FP) = {both} / ({both} + {model_only}) = {precision:.3f}\n"
    f"Recall    = TP / (TP + FN) = {both} / ({both} + {manual_only}) = {recall:.3f}"
)

fig.text(
    0.5, 0.1, formula_text,
    ha='center', va='bottom',
    fontsize=15, fontfamily='monospace',
    linespacing=1.8
)

plt.subplots_adjust(bottom=0.18)
plt.show()

print(f"Manual total positives: {len(manual_set)}")
print(f"{model} total positives: {len(model_set)}")
print(f"Manual only (FN): {manual_only}")
print(f"{model} only (FP): {model_only}")
print(f"Both (TP): {both}")

In [0]:
print(f"{model}: this_total = {this_total}, normalize_to = {normalize_to:.4f}")

In [0]:
print(f"Manual constant check: {len(manual_set)} / {max_total} = {len(manual_set)/max_total:.5f}")

In [0]:
import matplotlib.pyplot as plt
from matplotlib_venn import venn2

score_cols_map = {
    'GPT 5.4': 'GPT_Score',
    'GPT 5.4 Mini': 'GPT_Mini_Score',
    'Claude Sonnet 4.6': 'Claude_Sonnet_Score',
}
color_map = {
    'GPT 5.4': '#4C9BE8',
    'GPT 5.4 Mini': '#57C4A7',
    'Claude Sonnet 4.6': '#E87B4C',
}

manual_set = set(combined_df[combined_df['Manual_Score'] == 1].index)

# --- Compute all three models' data first, so we know shared totals/limits up front ---
results = {}
for model, score_col in score_cols_map.items():
    model_set = set(combined_df[combined_df[score_col] == 1].index)
    manual_only = len(manual_set - model_set)
    model_only  = len(model_set - manual_set)
    both        = len(manual_set & model_set)
    this_total  = manual_only + model_only + both
    results[model] = {
        'score_col': score_col, 'manual_only': manual_only, 'model_only': model_only,
        'both': both, 'this_total': this_total,
        'precision': both / (both + model_only) if (both + model_only) > 0 else 0,
        'recall': both / (both + manual_only) if (both + manual_only) > 0 else 0,
    }

max_total = max(r['this_total'] for r in results.values())

# --- First pass: draw the largest-total model to capture reference axis limits ---
reference_model = max(results, key=lambda m: results[m]['this_total'])
r = results[reference_model]
normalize_to = r['this_total'] / max_total

fig_ref, ax_ref = plt.subplots(figsize=(7, 7))
venn2(
    subsets=(r['manual_only'], r['model_only'], r['both']),
    set_labels=('Manual', reference_model),
    set_colors=('#D4B896', color_map[reference_model]),
    alpha=0.5, normalize_to=normalize_to, ax=ax_ref
)
REFERENCE_XLIM = ax_ref.get_xlim()
REFERENCE_YLIM = ax_ref.get_ylim()
plt.close(fig_ref)  # discard this draft, we'll redraw all three properly below

# --- Second pass: draw all three with shared, fixed axis limits ---
for model, score_col in score_cols_map.items():
    r = results[model]
    normalize_to = r['this_total'] / max_total

    fig, ax = plt.subplots(figsize=(7, 7))

    v = venn2(
        subsets=(r['manual_only'], r['model_only'], r['both']),
        set_labels=('Manual', model),
        set_colors=('#D4B896', color_map[model]),
        alpha=0.5,
        normalize_to=normalize_to,
        ax=ax
    )

    ax.set_xlim(REFERENCE_XLIM)
    ax.set_ylim(REFERENCE_YLIM)
    ax.set_aspect('equal')

    for label in v.subset_labels:
        if label:
            label.set_fontsize(12)

    for patch in v.patches:
        if patch:
            patch.set_edgecolor('white')
            patch.set_linewidth(2)

    for label in v.set_labels:
        if label:
            label.set_fontsize(14)
            label.set_fontweight('bold')

    ax.set_title(f'Positive phenotype calls: Manual vs {model}', fontsize=15, pad=20)

    formula_text = (
        f"Precision = TP / (TP + FP) = {r['both']} / ({r['both']} + {r['model_only']}) = {r['precision']:.3f}\n"
        f"Recall    = TP / (TP + FN) = {r['both']} / ({r['both']} + {r['manual_only']}) = {r['recall']:.3f}"
    )
    fig.text(0.5, 0.1, formula_text, ha='center', va='bottom',
              fontsize=15, fontfamily='monospace', linespacing=1.8)

    plt.subplots_adjust(bottom=0.18)
    plt.show()

    print(f"Manual total positives: {len(manual_set)}")
    print(f"{model} total positives: {r['manual_only'] + r['both'] + (r['model_only'])}")  # note: model total = model_only + both
    print(f"Manual only (FN): {r['manual_only']}")
    print(f"{model} only (FP): {r['model_only']}")
    print(f"Both (TP): {r['both']}\n")

In [0]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib_venn import venn2

df = combined_df.copy()

score_cols_map = {
    "GPT 5.4": "GPT_Score",
    "GPT 5.4 Mini": "GPT_Mini_Score",
    "Claude Sonnet 4.6": "Claude_Sonnet_Score"
}

color_map = {
    "GPT 5.4": "#4C9BE8",
    "GPT 5.4 Mini": "#57C4A7",
    "Claude Sonnet 4.6": "#E87B4C"
}

manual_color = "#D4B896"

required_columns = [
    "Manual_Score"
] + list(score_cols_map.values())

missing_columns = [
    column
    for column in required_columns
    if column not in df.columns
]

if missing_columns:
    raise KeyError(
        f"Missing required columns: {missing_columns}"
    )

if not df.index.is_unique:
    raise ValueError(
        "combined_df must have a unique index because the "
        "Venn sets are constructed using dataframe indices."
    )

# ------------------------------------------------------------
# Convert scores to numeric and confirm that they are binary
# ------------------------------------------------------------

df[required_columns] = (
    df[required_columns]
    .apply(pd.to_numeric, errors="coerce")
)

for column in required_columns:

    unexpected_values = (
        df.loc[
            df[column].notna()
            & ~df[column].isin([0, 1]),
            column
        ]
        .unique()
        .tolist()
    )

    if unexpected_values:
        raise ValueError(
            f"{column} contains values other than 0 and 1: "
            f"{sorted(unexpected_values)}"
        )

# ------------------------------------------------------------
# Calculate Venn regions and performance metrics
# ------------------------------------------------------------

results = {}

for model, score_col in score_cols_map.items():

    # Retain only observations with both scores available
    evaluable = df[
        df["Manual_Score"].notna()
        & df[score_col].notna()
    ].copy()

    manual_set = set(
        evaluable[
            evaluable["Manual_Score"].eq(1)
        ].index
    )

    model_set = set(
        evaluable[
            evaluable[score_col].eq(1)
        ].index
    )

    manual_only = len(
        manual_set - model_set
    )

    model_only = len(
        model_set - manual_set
    )

    both = len(
        manual_set & model_set
    )

    positive_union = (
        manual_only
        + model_only
        + both
    )

    true_negative = (
        len(evaluable)
        - positive_union
    )

    precision = (
        both / (both + model_only)
        if both + model_only > 0
        else np.nan
    )

    recall = (
        both / (both + manual_only)
        if both + manual_only > 0
        else np.nan
    )

    results[model] = {
        "score_col": score_col,
        "n_evaluable": len(evaluable),
        "manual_positives": len(manual_set),
        "model_positives": len(model_set),
        "manual_only": manual_only,
        "model_only": model_only,
        "both": both,
        "true_negative": true_negative,
        "this_total": positive_union,
        "precision": precision,
        "recall": recall
    }

# ------------------------------------------------------------
# Establish shared Venn-diagram limits
# ------------------------------------------------------------

max_total = max(
    result["this_total"]
    for result in results.values()
)

if max_total == 0:
    raise ValueError(
        "No positive calls were found."
    )

draft_limits = []

for model, result in results.items():

    normalize_to = (
        result["this_total"] / max_total
    )

    draft_fig, draft_ax = plt.subplots(
        figsize=(7, 7)
    )

    venn2(
        subsets=(
            result["manual_only"],
            result["model_only"],
            result["both"]
        ),
        set_labels=(
            "Manual",
            model
        ),
        set_colors=(
            manual_color,
            color_map[model]
        ),
        alpha=0.5,
        normalize_to=normalize_to,
        ax=draft_ax
    )

    draft_limits.append({
        "xlim": draft_ax.get_xlim(),
        "ylim": draft_ax.get_ylim()
    })

    plt.close(draft_fig)

reference_xlim = (
    min(item["xlim"][0] for item in draft_limits),
    max(item["xlim"][1] for item in draft_limits)
)

reference_ylim = (
    min(item["ylim"][0] for item in draft_limits),
    max(item["ylim"][1] for item in draft_limits)
)

# ------------------------------------------------------------
# Draw the three Venn diagrams
# ------------------------------------------------------------

for model, result in results.items():

    normalize_to = (
        result["this_total"] / max_total
    )

    fig, ax = plt.subplots(
        figsize=(7, 7)
    )

    venn = venn2(
        subsets=(
            result["manual_only"],
            result["model_only"],
            result["both"]
        ),
        set_labels=(
            "Manual",
            model
        ),
        set_colors=(
            manual_color,
            color_map[model]
        ),
        alpha=0.5,
        normalize_to=normalize_to,
        ax=ax
    )

    ax.set_xlim(reference_xlim)
    ax.set_ylim(reference_ylim)
    ax.set_aspect("equal")

    # Label the Venn regions
    region_labels = {
        "10": (
            "FN\n"
            f"n = {result['manual_only']:,}"
        ),
        "01": (
            "FP\n"
            f"n = {result['model_only']:,}"
        ),
        "11": (
            "TP\n"
            f"n = {result['both']:,}"
        )
    }

    for region_id, label_text in region_labels.items():

        label = venn.get_label_by_id(region_id)

        if label is not None:
            label.set_text(label_text)
            label.set_fontsize(12)
            label.set_fontweight("bold")

    for patch in venn.patches:
        if patch is not None:
            patch.set_edgecolor("white")
            patch.set_linewidth(2)

    for label in venn.set_labels:
        if label is not None:
            label.set_fontsize(14)
            label.set_fontweight("bold")

    ax.set_title(
        f"Positive Phenotype Calls: Manual vs {model}",
        fontsize=15,
        fontweight="bold",
        pad=20
    )

    plt.tight_layout()
    plt.show()

# ------------------------------------------------------------
# Create numeric performance summary
# ------------------------------------------------------------

performance_summary = pd.DataFrame([
    {
        "Scoring method": model,
        "Evaluable, n": result["n_evaluable"],
        "TP": result["both"],
        "FP": result["model_only"],
        "FN": result["manual_only"],
        "Precision": result["precision"],
        "Recall": result["recall"]
    }
    for model, result in results.items()
])

# Create a formatted copy for the publication table
publication_table = performance_summary.copy()

count_columns = [
    "Evaluable, n",
    "TP",
    "FP",
    "FN"
]

for column in count_columns:
    publication_table[column] = (
        publication_table[column]
        .map(lambda value: f"{int(value):,}")
    )

for column in ["Precision", "Recall"]:
    publication_table[column] = (
        publication_table[column]
        .map(
            lambda value:
                "NA"
                if pd.isna(value)
                else f"{value:.3f}"
        )
    )

# ------------------------------------------------------------
# Render publication-ready table
# ------------------------------------------------------------

fig_table, ax_table = plt.subplots(
    figsize=(12.5, 4.4)
)

ax_table.axis("off")

fig_table.text(
    0.05,
    0.91,
    "Performance of LLM Phenotype Scoring Methods Against Manual Annotation",
    fontsize=17,
    fontweight="bold",
    color="#202833",
    ha="left"
)

table_bbox = [
    0.05,
    0.29,
    0.90,
    0.47
]

column_widths = [
    0.28,   # Scoring method
    0.16,   # Evaluable n
    0.10,   # TP
    0.10,   # FP
    0.10,   # FN
    0.13,   # Precision
    0.13    # Recall
]

table = ax_table.table(
    cellText=publication_table.values,
    colLabels=publication_table.columns,
    colLoc="center",
    cellLoc="center",
    colWidths=column_widths,
    bbox=table_bbox
)

table.auto_set_font_size(False)
table.set_fontsize(12)

header_color = "#E5EBF2"
alternate_row_color = "#F4F6F8"
text_color = "#29313A"
line_color = "#D2D8DE"
border_color = "#7F8A94"

n_table_rows = len(publication_table) + 1

for (row, column), cell in table.get_celld().items():

    cell.set_text_props(color=text_color)
    cell.set_edgecolor(line_color)
    cell.set_linewidth(0.7)

    if row == 0:
        cell.set_facecolor(header_color)
        cell.set_text_props(
            fontweight="bold",
            color="#202833"
        )
    elif row % 2 == 0:
        cell.set_facecolor(alternate_row_color)
    else:
        cell.set_facecolor("white")

    if column == 0:
        cell.get_text().set_ha("left")
        cell.PAD = 0.05

# ------------------------------------------------------------
# Add publication-style horizontal borders
# ------------------------------------------------------------

left, bottom, width, height = table_bbox
row_height = height / n_table_rows

ax_table.plot(
    [left, left + width],
    [bottom + height, bottom + height],
    color=border_color,
    linewidth=1.5,
    transform=ax_table.transAxes
)

ax_table.plot(
    [left, left + width],
    [
        bottom + height - row_height,
        bottom + height - row_height
    ],
    color=border_color,
    linewidth=1.2,
    transform=ax_table.transAxes
)

ax_table.plot(
    [left, left + width],
    [bottom, bottom],
    color=border_color,
    linewidth=1.5,
    transform=ax_table.transAxes
)

# ------------------------------------------------------------
# Table note
# ------------------------------------------------------------

fig_table.text(
    0.05,
    0.15,
    (
        "Note: Manual annotation is treated as the reference. "
        "Each observation represents one evaluable "
        "Subject ID–Phenotype pair. "
        "Precision = TP/(TP + FP); recall = TP/(TP + FN).\n"
        "TP, true positive; FP, false positive; "
        "FN, false negative."
    ),
    fontsize=10.5,
    color="#59636F",
    ha="left",
    va="top"
)

plt.tight_layout()
plt.show()

# Numeric version retained for subsequent analyses
display(performance_summary)

## Graphs

In [0]:
notes_enrolled4.columns

In [0]:
display(combined_df.head())

In [0]:
# Check for duplicate Subject IDs in notes_enrolled4
dup_check = notes_enrolled4['Subject ID'].duplicated().sum()
print(f"Total rows in notes_enrolled4: {len(notes_enrolled4)}")
print(f"Unique Subject IDs: {notes_enrolled4['Subject ID'].nunique()}")
print(f"Duplicate Subject ID rows: {dup_check}")

In [0]:
combined_df_full_info = combined_df.merge(
    notes_enrolled4,
    on='Subject ID',
    how='left'
)

print(f"combined_df rows: {len(combined_df)}")
print(f"combined_df_full_info rows: {len(combined_df_full_info)}")
print(f"Row count matches: {len(combined_df) == len(combined_df_full_info)}")
print(f"Null values introduced by merge (any column): {combined_df_full_info.isna().sum().sum() - combined_df.isna().sum().sum()}")

display(combined_df_full_info)

In [0]:
print(combined_df_full_info.shape)
print(combined_df_full_info['NOTE_TEXT'].duplicated().sum())
print(combined_df_full_info['NOTE_TEXT'].nunique())

In [0]:
import matplotlib.pyplot as plt
import numpy as np

# Deduplicate first, then compute character counts on unique notes only
unique_notes = combined_df_full_info['NOTE_TEXT'].dropna().drop_duplicates()
char_counts = unique_notes.str.len()

print(f"Total notes with text: {len(char_counts)}")
print(f"Min: {char_counts.min()}")
print(f"Max: {char_counts.max()}")
print(f"Mean: {char_counts.mean():.0f}")
print(f"Median: {char_counts.median():.0f}")
print(f"Std dev: {char_counts.std():.0f}")

# Fixed bin edges: equal-width bins from 0 to 25,000, plus one overflow bin
x_max = 25000
n_bins = 50
bin_edges = np.linspace(0, x_max, n_bins + 1)
bin_edges = np.append(bin_edges, char_counts.max() + 1)  # overflow bin

n_over = (char_counts > x_max).sum()
print(f"Notes above {x_max:,} chars (folded into overflow bin): {n_over}")

# Compute counts manually (not via ax.hist) so every bin -- including the
# wide overflow bin -- can be drawn at the same visual width
counts, edges = np.histogram(char_counts, bins=bin_edges)
bin_positions = np.arange(len(counts))

fig, ax = plt.subplots(figsize=(11, 5))
ax.bar(bin_positions, counts, width=1.0, color='#4C9BE8', edgecolor='white')

# Label only every 5th regular bin to avoid overcrowding, plus the overflow bin
tick_step = 5
tick_positions = list(range(0, n_bins, tick_step)) + [n_bins]  # last index = overflow bin
tick_labels = [f'{int(edges[i]):,}' if i < n_bins else f'>{x_max:,}' for i in tick_positions]

ax.set_xticks(tick_positions)
ax.set_xticklabels(tick_labels, rotation=45, ha='right')

ax.set_xlabel('Character count', fontsize=12)
ax.set_ylabel('Number of notes', fontsize=12)
ax.set_title('Distribution of NOTE_TEXT character counts', fontsize=14, pad=15)
ax.set_ylim(bottom=0)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

plt.tight_layout()
plt.show()

In [0]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

df = combined_df_full_info.dropna(subset=['NOTE_TEXT']).copy()
df['char_count'] = df['NOTE_TEXT'].str.len()

score_cols = ['Manual_Score', 'GPT_Score', 'GPT_Mini_Score', 'Claude_Sonnet_Score']
colors = {
    'Manual_Score': '#D4B896',
    'GPT_Score': '#4C9BE8',
    'GPT_Mini_Score': '#57C4A7',
    'Claude_Sonnet_Score': '#E87B4C',
}
labels = {
    'Manual_Score': 'Manual',
    'GPT_Score': 'GPT 5.4',
    'GPT_Mini_Score': 'GPT 5.4 Mini',
    'Claude_Sonnet_Score': 'Claude Sonnet 4.6',
}

subject_level = df.groupby('Subject ID').agg(
    char_count=('char_count', 'first'),
    Manual_Score=('Manual_Score', 'sum'),
    GPT_Score=('GPT_Score', 'sum'),
    GPT_Mini_Score=('GPT_Mini_Score', 'sum'),
    Claude_Sonnet_Score=('Claude_Sonnet_Score', 'sum'),
).reset_index()

# Fixed bin edges: equal-width bins from actual min to 25,000, plus one overflow bin
x_min = subject_level['char_count'].min()
x_max = 25000
n_bins = 10
bin_edges = np.linspace(x_min, x_max, n_bins + 1)
bin_edges = np.append(bin_edges, subject_level['char_count'].max() + 1)  # overflow bin

subject_level['char_bin'] = pd.cut(subject_level['char_count'], bins=bin_edges, include_lowest=True)

binned = subject_level.groupby('char_bin', observed=True)[score_cols].mean()
bin_counts = subject_level.groupby('char_bin', observed=True).size()

# Evenly-spaced categorical positions (0, 1, 2, ...) instead of numeric midpoints —
# this makes it visually clear the x-axis is binned/discrete, not continuous
bin_positions = np.arange(len(binned))

# Label each bin with its actual range, e.g. "2,443–4,700"
tick_labels = []
for interval in binned.index:
    left = int(round(interval.left))
    if interval.right > x_max:
        tick_labels.append(f'{left:,}+')
    else:
        right = int(round(interval.right))
        tick_labels.append(f'{left:,}\u2013{right:,}')

print("Subjects per bin (check for sparse bins):")
print(bin_counts)

fig, ax = plt.subplots(figsize=(11, 6.5))

for col in score_cols:
    ax.plot(bin_positions, binned[col].values, marker='o', label=labels[col], color=colors[col], linewidth=2)

ax.set_xticks(bin_positions)
ax.set_xticklabels(tick_labels, rotation=45, ha='right')

ax.set_xlabel('Note character count (binned)', fontsize=12)
ax.set_ylabel('Average phenotypes scored per subject', fontsize=12)
ax.set_title('Average phenotypes scored by note length', fontsize=15, pad=15)
ax.legend(loc='center left', bbox_to_anchor=(1.02, 0.5), frameon=False, fontsize=10)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

fig.text(0.5, -0.05, f"X-axis shows {n_bins} equal-width character-count ranges plus an overflow bin for notes above {x_max:,} characters. "
                       f"Bins are evenly spaced categorically, not scaled to character count.",
          ha='center', va='top', fontsize=10, style='italic', color='#555555')

plt.subplots_adjust(right=0.82)
plt.tight_layout()
plt.show()

display(binned)

In [0]:
# Pull the 10 unique notes with the smallest character counts, including Subject ID
unique_notes_df = combined_df_full_info.dropna(subset=['NOTE_TEXT']).drop_duplicates(subset=['NOTE_TEXT']).copy()
unique_notes_df['char_count'] = unique_notes_df['NOTE_TEXT'].str.len()

smallest_10 = unique_notes_df.nsmallest(10, 'char_count')[['Subject ID', 'NOTE_TEXT', 'char_count']]
display(smallest_10)

In [0]:
# How many rows per subject, and how many are exact duplicates?
rows_per_subject = combined_df_full_info.groupby('Subject ID').size()
print(rows_per_subject.describe())

# If you have a phenotype/category column, compare row count vs unique phenotype count per subject
if 'Phenotype' in combined_df_full_info.columns:
    check = combined_df_full_info.groupby('Subject ID').agg(
        total_rows=('Phenotype', 'size'),
        unique_phenotypes=('Phenotype', 'nunique')
    )
    print(check[check['total_rows'] != check['unique_phenotypes']])

In [0]:
# Confirm each subject has exactly one distinct NOTE_TEXT (and one char_count) across their 125 rows
notes_per_subject = combined_df_full_info.groupby('Subject ID')['NOTE_TEXT'].nunique()
print(notes_per_subject.value_counts())

In [0]:
import matplotlib.pyplot as plt
import pandas as pd

df = combined_df_full_info.copy()
df['review_date'] = pd.to_datetime(df['Date of last chart review:'], errors='coerce')

n_unparsed = df['Date of last chart review:'].notna().sum() - df['review_date'].notna().sum()
print(f"Rows with a date that failed to parse: {n_unparsed}")

# Choose granularity: 'Q' for quarter, 'M' for month
period_freq = 'Q'   # change to 'M' for monthly
df['review_period'] = df['review_date'].dt.to_period(period_freq)

# ------------------------------------------------------------
# Figure 1: Distribution of notes by period
# ------------------------------------------------------------
subject_level = df.dropna(subset=['review_period']).drop_duplicates(subset='Subject ID')
notes_by_period = subject_level['review_period'].value_counts().sort_index()

fig1, ax1 = plt.subplots(figsize=(11, 5))
ax1.bar(notes_by_period.index.astype(str), notes_by_period.values, color='#4C9BE8', edgecolor='white')
ax1.set_xlabel(f"Review date period ({'quarter' if period_freq=='Q' else 'month'})", fontsize=12)
ax1.set_ylabel('Number of notes', fontsize=12)
ax1.set_title('Distribution of notes by period of review date', fontsize=15, pad=15)
ax1.set_ylim(bottom=0)
ax1.spines['top'].set_visible(False)
ax1.spines['right'].set_visible(False)
for i, v in enumerate(notes_by_period.values):
    ax1.text(i, v + notes_by_period.max() * 0.01, str(v), ha='center', va='bottom', fontsize=11)
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

# ------------------------------------------------------------
# Figure 2: Average phenotypes scored per note, by period, per model
# ------------------------------------------------------------
score_cols = ['Manual_Score', 'GPT_Score', 'GPT_Mini_Score', 'Claude_Sonnet_Score']

subject_scores = df.dropna(subset=['review_period']).groupby('Subject ID').agg(
    review_period=('review_period', 'first'),
    Manual_Score=('Manual_Score', 'sum'),
    GPT_Score=('GPT_Score', 'sum'),
    GPT_Mini_Score=('GPT_Mini_Score', 'sum'),
    Claude_Sonnet_Score=('Claude_Sonnet_Score', 'sum'),
).reset_index()

avg_by_period = subject_scores.groupby('review_period')[score_cols].mean()

colors = {
    'Manual_Score': '#D4B896',
    'GPT_Score': '#4C9BE8',
    'GPT_Mini_Score': '#57C4A7',
    'Claude_Sonnet_Score': '#E87B4C',
}
labels = {
    'Manual_Score': 'Manual',
    'GPT_Score': 'GPT 5.4',
    'GPT_Mini_Score': 'GPT 5.4 Mini',
    'Claude_Sonnet_Score': 'Claude Sonnet 4.6',
}

fig2, ax2 = plt.subplots(figsize=(11, 6))
x_labels = avg_by_period.index.astype(str)

for col in score_cols:
    ax2.plot(x_labels, avg_by_period[col], marker='o', label=labels[col], color=colors[col], linewidth=2)

ax2.set_xlabel(f"Review period ({'quarter' if period_freq=='Q' else 'month'})", fontsize=12)
ax2.set_ylabel('Average phenotypes scored per note', fontsize=12)
ax2.set_title('Average phenotypes scored per note, by period of review date', fontsize=15, pad=15)
ax2.legend(frameon=False, fontsize=11)
ax2.set_ylim(bottom=0)
ax2.spines['top'].set_visible(False)
ax2.spines['right'].set_visible(False)
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

display(notes_by_period)
display(avg_by_period)

In [0]:
notes_enrolled4.columns

In [0]:
import matplotlib.pyplot as plt
import pandas as pd

df = combined_df_full_info.copy()
df['review_date'] = pd.to_datetime(df['NOTE_FILE_TIME_DTTM'], errors='coerce')

n_unparsed = df['NOTE_FILE_TIME_DTTM'].notna().sum() - df['review_date'].notna().sum()
print(f"Rows with a date that failed to parse: {n_unparsed}")

# Choose granularity: 'Q' for quarter, 'M' for month
period_freq = 'Q'   # change to 'M' for monthly
df['review_period'] = df['review_date'].dt.to_period(period_freq)

# ------------------------------------------------------------
# Figure 1: Distribution of notes by period
# ------------------------------------------------------------
subject_level = df.dropna(subset=['review_period']).drop_duplicates(subset='Subject ID')
notes_by_period = subject_level['review_period'].value_counts().sort_index()

fig1, ax1 = plt.subplots(figsize=(11, 5))
ax1.bar(notes_by_period.index.astype(str), notes_by_period.values, color='#4C9BE8', edgecolor='white')
ax1.set_xlabel(f"Note file time period ({'quarter' if period_freq=='Q' else 'month'})", fontsize=12)
ax1.set_ylabel('Number of notes', fontsize=12)
ax1.set_title('Distribution of notes by period of note file time', fontsize=15, pad=15)
ax1.set_ylim(bottom=0)
ax1.spines['top'].set_visible(False)
ax1.spines['right'].set_visible(False)
for i, v in enumerate(notes_by_period.values):
    ax1.text(i, v + notes_by_period.max() * 0.01, str(v), ha='center', va='bottom', fontsize=11)
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

# ------------------------------------------------------------
# Figure 2: Average phenotypes scored per note, by period, per model
# ------------------------------------------------------------
score_cols = ['Manual_Score', 'GPT_Score', 'GPT_Mini_Score', 'Claude_Sonnet_Score']

subject_scores = df.dropna(subset=['review_period']).groupby('Subject ID').agg(
    review_period=('review_period', 'first'),
    Manual_Score=('Manual_Score', 'sum'),
    GPT_Score=('GPT_Score', 'sum'),
    GPT_Mini_Score=('GPT_Mini_Score', 'sum'),
    Claude_Sonnet_Score=('Claude_Sonnet_Score', 'sum'),
).reset_index()

avg_by_period = subject_scores.groupby('review_period')[score_cols].mean()

colors = {
    'Manual_Score': '#D4B896',
    'GPT_Score': '#4C9BE8',
    'GPT_Mini_Score': '#57C4A7',
    'Claude_Sonnet_Score': '#E87B4C',
}
labels = {
    'Manual_Score': 'Manual',
    'GPT_Score': 'GPT 5.4',
    'GPT_Mini_Score': 'GPT 5.4 Mini',
    'Claude_Sonnet_Score': 'Claude Sonnet 4.6',
}

fig2, ax2 = plt.subplots(figsize=(11, 6))
x_labels = avg_by_period.index.astype(str)

for col in score_cols:
    ax2.plot(x_labels, avg_by_period[col], marker='o', label=labels[col], color=colors[col], linewidth=2)

ax2.set_xlabel(f"Note file time period ({'quarter' if period_freq=='Q' else 'month'})", fontsize=12)
ax2.set_ylabel('Average phenotypes scored per note', fontsize=12)
ax2.set_title('Average phenotypes scored per note, by period of note file time', fontsize=15, pad=15)
ax2.legend(frameon=False, fontsize=11)
ax2.set_ylim(bottom=0)
ax2.spines['top'].set_visible(False)
ax2.spines['right'].set_visible(False)
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

display(notes_by_period)
display(avg_by_period)

In [0]:
categorical_cols = ['CURRENT_AUTHOR', 'AUTHOR_ROLE', 'UPDATER', 'COSIGNER']

for col in categorical_cols:
    n_unique = combined_df_full_info[col].nunique(dropna=True)
    n_missing = combined_df_full_info[col].isna().sum()
    print(f"{col}: {n_unique} unique values, {n_missing} missing rows")
    print(combined_df_full_info[col].value_counts(dropna=True).head(15))
    print()

In [0]:
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

df = combined_df_full_info.copy()
score_cols = ['Manual_Score', 'GPT_Score', 'GPT_Mini_Score', 'Claude_Sonnet_Score']
colors = {'Manual_Score': '#D4B896', 'GPT_Score': '#4C9BE8', 'GPT_Mini_Score': '#57C4A7', 'Claude_Sonnet_Score': '#E87B4C'}
labels = {'Manual_Score': 'Manual', 'GPT_Score': 'GPT 5.4', 'GPT_Mini_Score': 'GPT 5.4 Mini', 'Claude_Sonnet_Score': 'Claude Sonnet 4.6'}

subject_level = df.dropna(subset=['CURRENT_AUTHOR']).drop_duplicates(subset='Subject ID').copy()
top_authors = subject_level['CURRENT_AUTHOR'].value_counts().head(14).index.tolist()
subject_level['CURRENT_AUTHOR_grouped'] = subject_level['CURRENT_AUTHOR'].apply(lambda x: x if x in top_authors else 'Other')

order = top_authors + ['Other']
notes_by_cat = subject_level['CURRENT_AUTHOR_grouped'].value_counts().reindex(order).dropna()

fig1, ax1 = plt.subplots(figsize=(12, 5))
ax1.bar(notes_by_cat.index.astype(str), notes_by_cat.values, color='#4C9BE8', edgecolor='white')
ax1.set_xlabel('Current author (top 14 + other)', fontsize=12)
ax1.set_ylabel('Number of notes', fontsize=12)
ax1.set_title('Distribution of notes by current author', fontsize=15, pad=15)
ax1.spines['top'].set_visible(False)
ax1.spines['right'].set_visible(False)
for i, v in enumerate(notes_by_cat.values):
    ax1.text(i, v + notes_by_cat.max() * 0.01, str(v), ha='center', va='bottom', fontsize=11)
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

subject_scores = df.dropna(subset=['CURRENT_AUTHOR']).groupby('Subject ID').agg(
    CURRENT_AUTHOR=('CURRENT_AUTHOR', 'first'),
    Manual_Score=('Manual_Score', 'sum'), GPT_Score=('GPT_Score', 'sum'),
    GPT_Mini_Score=('GPT_Mini_Score', 'sum'), Claude_Sonnet_Score=('Claude_Sonnet_Score', 'sum'),
).reset_index()
subject_scores['CURRENT_AUTHOR_grouped'] = subject_scores['CURRENT_AUTHOR'].apply(lambda x: x if x in top_authors else 'Other')
avg_by_cat = subject_scores.groupby('CURRENT_AUTHOR_grouped')[score_cols].mean().reindex(order).dropna()

x = np.arange(len(avg_by_cat))
width = 0.2
fig2, ax2 = plt.subplots(figsize=(13, 6))
for i, col in enumerate(score_cols):
    ax2.bar(x + (i - 1.5) * width, avg_by_cat[col], width, label=labels[col], color=colors[col])

ax2.set_xlabel('Current author (top 14 + other)', fontsize=12)
ax2.set_ylabel('Average phenotypes scored per note', fontsize=12)
ax2.set_title('Average phenotypes scored per note, by current author', fontsize=15, pad=15)
ax2.set_xticks(x)
ax2.set_xticklabels(avg_by_cat.index.astype(str), rotation=45, ha='right')
ax2.legend(loc='center left', bbox_to_anchor=(1.02, 0.5), frameon=False, fontsize=10)
ax2.spines['top'].set_visible(False)
ax2.spines['right'].set_visible(False)
plt.subplots_adjust(right=0.82)
plt.tight_layout()
plt.show()

display(notes_by_cat)
display(avg_by_cat)

In [0]:
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

df = combined_df_full_info.copy()
score_cols = ['Manual_Score', 'GPT_Score', 'GPT_Mini_Score', 'Claude_Sonnet_Score']
colors = {'Manual_Score': '#D4B896', 'GPT_Score': '#4C9BE8', 'GPT_Mini_Score': '#57C4A7', 'Claude_Sonnet_Score': '#E87B4C'}
labels = {'Manual_Score': 'Manual', 'GPT_Score': 'GPT 5.4', 'GPT_Mini_Score': 'GPT 5.4 Mini', 'Claude_Sonnet_Score': 'Claude Sonnet 4.6'}

subject_level = df.dropna(subset=['AUTHOR_ROLE']).drop_duplicates(subset='Subject ID').copy()
notes_by_cat = subject_level['AUTHOR_ROLE'].value_counts()

fig1, ax1 = plt.subplots(figsize=(10, 5))
ax1.bar(notes_by_cat.index.astype(str), notes_by_cat.values, color='#4C9BE8', edgecolor='white')
ax1.set_xlabel('Author role', fontsize=12)
ax1.set_ylabel('Number of notes', fontsize=12)
ax1.set_title('Distribution of notes by author role', fontsize=15, pad=15)
ax1.spines['top'].set_visible(False)
ax1.spines['right'].set_visible(False)
for i, v in enumerate(notes_by_cat.values):
    ax1.text(i, v + notes_by_cat.max() * 0.01, str(v), ha='center', va='bottom', fontsize=11)
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()


subject_scores = df.dropna(subset=['AUTHOR_ROLE']).groupby('Subject ID').agg(
    AUTHOR_ROLE=('AUTHOR_ROLE', 'first'),
    Manual_Score=('Manual_Score', 'sum'), GPT_Score=('GPT_Score', 'sum'),
    GPT_Mini_Score=('GPT_Mini_Score', 'sum'), Claude_Sonnet_Score=('Claude_Sonnet_Score', 'sum'),
).reset_index()
avg_by_cat = subject_scores.groupby('AUTHOR_ROLE')[score_cols].mean().reindex(notes_by_cat.index)

x = np.arange(len(avg_by_cat))
width = 0.2
fig2, ax2 = plt.subplots(figsize=(11, 6))
for i, col in enumerate(score_cols):
    ax2.bar(x + (i - 1.5) * width, avg_by_cat[col], width, label=labels[col], color=colors[col])

ax2.set_xlabel('Author role', fontsize=12)
ax2.set_ylabel('Average phenotypes scored per note', fontsize=12)
ax2.set_title('Average phenotypes scored per note, by author role', fontsize=15, pad=15)
ax2.set_xticks(x)
ax2.set_xticklabels(avg_by_cat.index.astype(str), rotation=45, ha='right')
ax2.legend(loc='center left', bbox_to_anchor=(1.02, 0.5), frameon=False, fontsize=10)
ax2.spines['top'].set_visible(False)
ax2.spines['right'].set_visible(False)
plt.subplots_adjust(right=0.82)
plt.tight_layout()
plt.show()

display(notes_by_cat)
display(avg_by_cat)

In [0]:
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

df = combined_df_full_info.copy()
score_cols = ['Manual_Score', 'GPT_Score', 'GPT_Mini_Score', 'Claude_Sonnet_Score']
colors = {'Manual_Score': '#D4B896', 'GPT_Score': '#4C9BE8', 'GPT_Mini_Score': '#57C4A7', 'Claude_Sonnet_Score': '#E87B4C'}
labels = {'Manual_Score': 'Manual', 'GPT_Score': 'GPT 5.4', 'GPT_Mini_Score': 'GPT 5.4 Mini', 'Claude_Sonnet_Score': 'Claude Sonnet 4.6'}

subject_level = df.dropna(subset=['UPDATER']).drop_duplicates(subset='Subject ID').copy()
notes_by_cat = subject_level['UPDATER'].value_counts()

fig1, ax1 = plt.subplots(figsize=(10, 5))
ax1.bar(notes_by_cat.index.astype(str), notes_by_cat.values, color='#4C9BE8', edgecolor='white')
ax1.set_xlabel('Updater', fontsize=12)
ax1.set_ylabel('Number of notes', fontsize=12)
ax1.set_title('Distribution of notes by updater', fontsize=15, pad=15)
ax1.spines['top'].set_visible(False)
ax1.spines['right'].set_visible(False)
for i, v in enumerate(notes_by_cat.values):
    ax1.text(i, v + notes_by_cat.max() * 0.01, str(v), ha='center', va='bottom', fontsize=11)
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

subject_scores = df.dropna(subset=['UPDATER']).groupby('Subject ID').agg(
    UPDATER=('UPDATER', 'first'),
    Manual_Score=('Manual_Score', 'sum'), GPT_Score=('GPT_Score', 'sum'),
    GPT_Mini_Score=('GPT_Mini_Score', 'sum'), Claude_Sonnet_Score=('Claude_Sonnet_Score', 'sum'),
).reset_index()
avg_by_cat = subject_scores.groupby('UPDATER')[score_cols].mean().reindex(notes_by_cat.index)

x = np.arange(len(avg_by_cat))
width = 0.2
fig2, ax2 = plt.subplots(figsize=(11, 6))
for i, col in enumerate(score_cols):
    ax2.bar(x + (i - 1.5) * width, avg_by_cat[col], width, label=labels[col], color=colors[col])

ax2.set_xlabel('Updater', fontsize=12)
ax2.set_ylabel('Average phenotypes scored per note', fontsize=12)
ax2.set_title('Average phenotypes scored per note, by updater', fontsize=15, pad=15)
ax2.set_xticks(x)
ax2.set_xticklabels(avg_by_cat.index.astype(str), rotation=45, ha='right')
ax2.legend(loc='center left', bbox_to_anchor=(1.02, 0.5), frameon=False, fontsize=10)
ax2.spines['top'].set_visible(False)
ax2.spines['right'].set_visible(False)
plt.subplots_adjust(right=0.82)
plt.tight_layout()
plt.show()

display(notes_by_cat)
display(avg_by_cat)

In [0]:
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

df = combined_df_full_info.copy()
score_cols = ['Manual_Score', 'GPT_Score', 'GPT_Mini_Score', 'Claude_Sonnet_Score']
colors = {'Manual_Score': '#D4B896', 'GPT_Score': '#4C9BE8', 'GPT_Mini_Score': '#57C4A7', 'Claude_Sonnet_Score': '#E87B4C'}
labels = {'Manual_Score': 'Manual', 'GPT_Score': 'GPT 5.4', 'GPT_Mini_Score': 'GPT 5.4 Mini', 'Claude_Sonnet_Score': 'Claude Sonnet 4.6'}

subject_level = df.dropna(subset=['COSIGNER']).drop_duplicates(subset='Subject ID').copy()
notes_by_cat = subject_level['COSIGNER'].value_counts()

fig1, ax1 = plt.subplots(figsize=(10, 5))
ax1.bar(notes_by_cat.index.astype(str), notes_by_cat.values, color='#4C9BE8', edgecolor='white')
ax1.set_xlabel('Cosigner', fontsize=12)
ax1.set_ylabel('Number of notes', fontsize=12)
ax1.set_title('Distribution of notes by cosigner', fontsize=15, pad=15)
ax1.spines['top'].set_visible(False)
ax1.spines['right'].set_visible(False)
for i, v in enumerate(notes_by_cat.values):
    ax1.text(i, v + notes_by_cat.max() * 0.01, str(v), ha='center', va='bottom', fontsize=11)
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

subject_scores = df.dropna(subset=['COSIGNER']).groupby('Subject ID').agg(
    COSIGNER=('COSIGNER', 'first'),
    Manual_Score=('Manual_Score', 'sum'), GPT_Score=('GPT_Score', 'sum'),
    GPT_Mini_Score=('GPT_Mini_Score', 'sum'), Claude_Sonnet_Score=('Claude_Sonnet_Score', 'sum'),
).reset_index()
avg_by_cat = subject_scores.groupby('COSIGNER')[score_cols].mean().reindex(notes_by_cat.index)

x = np.arange(len(avg_by_cat))
width = 0.2
fig2, ax2 = plt.subplots(figsize=(11, 6))
for i, col in enumerate(score_cols):
    ax2.bar(x + (i - 1.5) * width, avg_by_cat[col], width, label=labels[col], color=colors[col])

ax2.set_xlabel('Cosigner', fontsize=12)
ax2.set_ylabel('Average phenotypes scored per note', fontsize=12)
ax2.set_title('Average phenotypes scored per note, by cosigner', fontsize=15, pad=15)
ax2.set_xticks(x)
ax2.set_xticklabels(avg_by_cat.index.astype(str), rotation=45, ha='right')
ax2.legend(loc='center left', bbox_to_anchor=(1.02, 0.5), frameon=False, fontsize=10)
ax2.spines['top'].set_visible(False)
ax2.spines['right'].set_visible(False)
plt.subplots_adjust(right=0.82)
plt.tight_layout()
plt.show()

display(notes_by_cat)
display(avg_by_cat)

In [0]:
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

df = combined_df_full_info.copy()
score_cols = ['Manual_Score', 'GPT_Score', 'GPT_Mini_Score', 'Claude_Sonnet_Score']
colors = {'Manual_Score': '#D4B896', 'GPT_Score': '#4C9BE8', 'GPT_Mini_Score': '#57C4A7', 'Claude_Sonnet_Score': '#E87B4C'}
labels = {'Manual_Score': 'Manual', 'GPT_Score': 'GPT 5.4', 'GPT_Mini_Score': 'GPT 5.4 Mini', 'Claude_Sonnet_Score': 'Claude Sonnet 4.6'}

# Collapse COSIGNER into a binary category: has a cosigner (non-null) vs. no cosigner (null)
df['COSIGNER_STATUS'] = np.where(df['COSIGNER'].notna(), 'Has Cosigner', 'No Cosigner')

subject_level = df.drop_duplicates(subset='Subject ID').copy()
notes_by_cat = subject_level['COSIGNER_STATUS'].value_counts().reindex(['Has Cosigner', 'No Cosigner'])

fig1, ax1 = plt.subplots(figsize=(7, 5))
ax1.bar(notes_by_cat.index.astype(str), notes_by_cat.values, color='#4C9BE8', edgecolor='white')
ax1.set_xlabel('Cosigner status', fontsize=12)
ax1.set_ylabel('Number of notes', fontsize=12)
ax1.set_title('Distribution of notes by cosigner status', fontsize=15, pad=15)
ax1.set_ylim(bottom=0)
ax1.spines['top'].set_visible(False)
ax1.spines['right'].set_visible(False)
for i, v in enumerate(notes_by_cat.values):
    ax1.text(i, v + notes_by_cat.max() * 0.01, str(v), ha='center', va='bottom', fontsize=11)
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

subject_scores = df.groupby('Subject ID').agg(
    COSIGNER_STATUS=('COSIGNER_STATUS', 'first'),
    Manual_Score=('Manual_Score', 'sum'), GPT_Score=('GPT_Score', 'sum'),
    GPT_Mini_Score=('GPT_Mini_Score', 'sum'), Claude_Sonnet_Score=('Claude_Sonnet_Score', 'sum'),
).reset_index()
avg_by_cat = subject_scores.groupby('COSIGNER_STATUS')[score_cols].mean().reindex(notes_by_cat.index)

x = np.arange(len(avg_by_cat))
width = 0.2
fig2, ax2 = plt.subplots(figsize=(9, 6))
for i, col in enumerate(score_cols):
    ax2.bar(x + (i - 1.5) * width, avg_by_cat[col], width, label=labels[col], color=colors[col])

ax2.set_xlabel('Cosigner status', fontsize=12)
ax2.set_ylabel('Average phenotypes scored per note', fontsize=12)
ax2.set_title('Average phenotypes scored per note, by cosigner status', fontsize=15, pad=15)
ax2.set_xticks(x)
ax2.set_xticklabels(avg_by_cat.index.astype(str), rotation=0)
ax2.set_ylim(bottom=0)
ax2.legend(loc='center left', bbox_to_anchor=(1.02, 0.5), frameon=False, fontsize=10)
ax2.spines['top'].set_visible(False)
ax2.spines['right'].set_visible(False)
plt.subplots_adjust(right=0.82)
plt.tight_layout()
plt.show()

display(notes_by_cat)
display(avg_by_cat)

In [0]:
n_unique = notes_enrolled4['COSIGNER'].nunique(dropna=True)
n_missing = notes_enrolled4['COSIGNER'].isna().sum()

print(f"COSIGNER: {n_unique} unique values, {n_missing} missing rows")
print(notes_enrolled4['COSIGNER'].value_counts(dropna=True))

In [0]:
# ============================================================
# Variable-width age bins: 1-year bins up to 14, 2-year bins up to 20,
# 5-year bins up to 30, then 10-year bins beyond that
# ============================================================

import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

df = combined_df_full_info.copy()
score_cols = ['Manual_Score', 'GPT_Score', 'GPT_Mini_Score', 'Claude_Sonnet_Score']
colors = {'Manual_Score': '#D4B896', 'GPT_Score': '#4C9BE8', 'GPT_Mini_Score': '#57C4A7', 'Claude_Sonnet_Score': '#E87B4C'}
labels = {'Manual_Score': 'Manual', 'GPT_Score': 'GPT 5.4', 'GPT_Mini_Score': 'GPT 5.4 Mini', 'Claude_Sonnet_Score': 'Claude Sonnet 4.6'}

df['review_date'] = pd.to_datetime(df['Date of last chart review:'], errors='coerce')
df['dob'] = pd.to_datetime(df['D.O.B.'], errors='coerce')
df['age_at_note_years'] = (df['review_date'] - df['dob']).dt.days / 365.25

n_parsed = df[['review_date', 'dob']].notna().all(axis=1).sum()
print(f"Rows with both dates successfully parsed: {n_parsed} / {len(df)}")

# --- Build variable-width bin edges: 1yr up to 14, 2yr up to 20, 5yr up to 30, 10yr beyond ---
max_age = df['age_at_note_years'].max()

edges = list(range(0, 15, 1))                     # 0,1,2,...,14  -> 1-year bins
edges += list(range(16, 21, 2))                    # 16,18,20      -> 2-year bins (14-16,16-18,18-20)
edges += [25, 30]                                  # 25,30         -> 5-year bins (20-25,25-30)
next_edge = 40
while next_edge - 10 < max_age:
    edges.append(next_edge)                        # 40,50,60,...  -> 10-year bins
    next_edge += 10

edges = sorted(set(edges))
print(f"Bin edges: {edges}")

df['age_bin'] = pd.cut(df['age_at_note_years'], bins=edges, right=False, include_lowest=True)

subject_level = df.dropna(subset=['age_bin']).drop_duplicates(subset='Subject ID').copy()

# Figure 1: distribution of notes by age bin
notes_by_bin = subject_level['age_bin'].value_counts().sort_index()
notes_by_bin = notes_by_bin[notes_by_bin.index.isin(subject_level['age_bin'].cat.categories)]

# Evenly-spaced categorical positions; 1-year bins get a single-number label,
# wider bins keep a range label since two edges are needed to disambiguate them
bin_positions = np.arange(len(notes_by_bin))
tick_labels = []
for iv in notes_by_bin.index:
    width = iv.right - iv.left
    if width == 1:
        tick_labels.append(f'{int(iv.right)}')
    else:
        tick_labels.append(f'{int(iv.left)}\u2013{int(iv.right)}')

fig1, ax1 = plt.subplots(figsize=(12, 5))
ax1.plot(bin_positions, notes_by_bin.values, marker='o', color='#4C9BE8', linewidth=2)
ax1.set_xticks(bin_positions)
ax1.set_xticklabels(tick_labels, rotation=45, ha='right')
ax1.set_xlabel('Age at note (years, binned)', fontsize=12)
ax1.set_ylabel('Number of notes', fontsize=12)
ax1.set_title('Distribution of notes by age at note', fontsize=15, pad=15)
ax1.set_ylim(bottom=0)
ax1.spines['top'].set_visible(False)
ax1.spines['right'].set_visible(False)
fig1.text(0.5, -0.05, "X-axis bins: 1-year intervals up to 14 (labeled by single age), 2-year intervals up to 20, "
                       "5-year intervals up to 30, then 10-year intervals (labeled by range). "
                       "Bins are evenly spaced categorically, not scaled to age.",
          ha='center', va='top', fontsize=10, style='italic', color='#555555')
plt.tight_layout()
plt.show()

# Figure 2: average phenotypes scored per note, by age bin, per model
subject_scores = df.dropna(subset=['age_bin']).groupby('Subject ID').agg(
    age_bin=('age_bin', 'first'),
    Manual_Score=('Manual_Score', 'sum'), GPT_Score=('GPT_Score', 'sum'),
    GPT_Mini_Score=('GPT_Mini_Score', 'sum'), Claude_Sonnet_Score=('Claude_Sonnet_Score', 'sum'),
).reset_index()
avg_by_bin = subject_scores.groupby('age_bin', observed=True)[score_cols].mean()
avg_by_bin = avg_by_bin.reindex(notes_by_bin.index)

x_vals = bin_positions

fig2, ax2 = plt.subplots(figsize=(13, 6))

# Extend the right margin so end-of-line labels have room to sit outside the plotted data
x_range = x_vals.max() - x_vals.min()
ax2.set_xlim(x_vals.min(), x_vals.max() + x_range * 0.18)

for col in score_cols:
    y_vals = avg_by_bin[col].values
    ax2.plot(x_vals, y_vals, marker='o', color=colors[col], linewidth=2)

    # Label directly at the last valid point of each line
    last_valid_idx = np.where(~np.isnan(y_vals))[0]
    if len(last_valid_idx) > 0:
        last_idx = last_valid_idx[-1]
        ax2.annotate(
            labels[col],
            xy=(x_vals[last_idx], y_vals[last_idx]),
            xytext=(8, 0), textcoords='offset points',
            va='center', fontsize=11, color=colors[col], fontweight='bold'
        )

ax2.set_xticks(bin_positions)
ax2.set_xticklabels(tick_labels, rotation=45, ha='right')
ax2.set_xlabel('Age at note (years, binned)', fontsize=12)
ax2.set_ylabel('Average phenotypes scored per note', fontsize=12)
ax2.set_title('Average phenotypes scored per note, by age at note', fontsize=15, pad=15)
ax2.set_ylim(bottom=0)
ax2.spines['top'].set_visible(False)
ax2.spines['right'].set_visible(False)

fig2.text(0.5, -0.05, "X-axis bins: 1-year intervals up to 14 (labeled by single age), 2-year intervals up to 20, "
                       "5-year intervals up to 30, then 10-year intervals (labeled by range). "
                       "Bins are evenly spaced categorically, not scaled to age.",
          ha='center', va='top', fontsize=10, style='italic', color='#555555')

plt.tight_layout()
plt.show()

display(notes_by_bin)
display(avg_by_bin)

In [0]:
# ============================================================
# Variable-width age bins: 1-year bins up to 14, 2-year bins up to 20,
# 5-year bins up to 30, 10-year bins up to 70, then a single 70+ overflow bin
# ============================================================

import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

df = combined_df_full_info.copy()
score_cols = ['Manual_Score', 'GPT_Score', 'GPT_Mini_Score', 'Claude_Sonnet_Score']
colors = {'Manual_Score': '#D4B896', 'GPT_Score': '#4C9BE8', 'GPT_Mini_Score': '#57C4A7', 'Claude_Sonnet_Score': '#E87B4C'}
labels = {'Manual_Score': 'Manual', 'GPT_Score': 'GPT 5.4', 'GPT_Mini_Score': 'GPT 5.4 Mini', 'Claude_Sonnet_Score': 'Claude Sonnet 4.6'}

df['note_date'] = pd.to_datetime(
    df['NOTE_FILE_TIME_DTTM'],
    errors='coerce'
)

df['dob'] = pd.to_datetime(
    df['D.O.B.'],
    errors='coerce'
)

df['age_at_note_years'] = (
    (df['note_date'] - df['dob']).dt.days / 365.25
)
df['dob'] = pd.to_datetime(df['D.O.B.'], errors='coerce')
df['age_at_note_years'] = (df['note_date'] - df['dob']).dt.days / 365.25

n_parsed = df[['note_date', 'dob']].notna().all(axis=1).sum()
print(f"Rows with both dates successfully parsed: {n_parsed} / {len(df)}")

# --- Build variable-width bin edges: 1yr up to 14, 2yr up to 20, 5yr up to 30,
#     10yr up to 70, then a single overflow bin for everything 70+ ---
max_age = df['age_at_note_years'].max()

edges = list(range(0, 15, 1))                     # 0,1,2,...,14  -> 1-year bins
edges += list(range(16, 21, 2))                    # 16,18,20      -> 2-year bins (14-16,16-18,18-20)
edges += [25, 30]                                  # 25,30         -> 5-year bins (20-25,25-30)
edges += list(range(40, 71, 10))                   # 40,50,60,70   -> 10-year bins (30-40,...,60-70)
edges.append(max(max_age + 1, 71))                 # final edge -> overflow bin covering 70+

edges = sorted(set(edges))
print(f"Bin edges: {edges}")

df['age_bin'] = pd.cut(df['age_at_note_years'], bins=edges, right=False, include_lowest=True)

subject_level = df.dropna(subset=['age_bin']).drop_duplicates(subset='Subject ID').copy()

# Figure 1: distribution of notes by age bin
notes_by_bin = subject_level['age_bin'].value_counts().sort_index()
notes_by_bin = notes_by_bin[notes_by_bin.index.isin(subject_level['age_bin'].cat.categories)]

# Evenly-spaced categorical positions; 1-year bins get a single-number label,
# wider bins get a range label, and the final bin is labeled "70+"
bin_positions = np.arange(len(notes_by_bin))
tick_labels = []
for iv in notes_by_bin.index:
    width = iv.right - iv.left
    if iv.left >= 70:
        tick_labels.append('70+')
    elif width == 1:
        tick_labels.append(f'{int(iv.right)}')
    else:
        tick_labels.append(f'{int(iv.left)}\u2013{int(iv.right)}')

fig1, ax1 = plt.subplots(figsize=(12, 5))
ax1.plot(bin_positions, notes_by_bin.values, marker='o', color='#4C9BE8', linewidth=2)
ax1.set_xticks(bin_positions)
ax1.set_xticklabels(tick_labels, rotation=45, ha='right')
ax1.set_xlabel('Age at note (years, binned)', fontsize=12)
ax1.set_ylabel('Number of notes', fontsize=12)
ax1.set_title('Distribution of notes by age at note', fontsize=15, pad=15)
ax1.set_ylim(bottom=0)
ax1.spines['top'].set_visible(False)
ax1.spines['right'].set_visible(False)
fig1.text(0.5, -0.05, "X-axis bins: 1-year intervals up to 14, 2-year intervals up to 20, 5-year intervals up to 30, "
                       "10-year intervals up to 70, then a single 70+ bin. "
                       "Bins are evenly spaced categorically, not scaled to age.",
          ha='center', va='top', fontsize=10, style='italic', color='#555555')
plt.tight_layout()
plt.show()

# Figure 2: average phenotypes scored per note, by age bin, per model
subject_scores = df.dropna(subset=['age_bin']).groupby('Subject ID').agg(
    age_bin=('age_bin', 'first'),
    Manual_Score=('Manual_Score', 'sum'), GPT_Score=('GPT_Score', 'sum'),
    GPT_Mini_Score=('GPT_Mini_Score', 'sum'), Claude_Sonnet_Score=('Claude_Sonnet_Score', 'sum'),
).reset_index()
avg_by_bin = subject_scores.groupby('age_bin', observed=True)[score_cols].mean()
avg_by_bin = avg_by_bin.reindex(notes_by_bin.index)

x_vals = bin_positions

fig2, ax2 = plt.subplots(figsize=(12, 6))

for col in score_cols:
    y_vals = avg_by_bin[col].values
    ax2.plot(x_vals, y_vals, marker='o', label=labels[col], color=colors[col], linewidth=2)

ax2.set_xticks(bin_positions)
ax2.set_xticklabels(tick_labels, rotation=45, ha='right')
ax2.set_xlabel('Age at note (years, binned)', fontsize=12)
ax2.set_ylabel('Average phenotypes scored per note', fontsize=12)
ax2.set_title('Average phenotypes scored per note, by age at note', fontsize=15, pad=15)
ax2.set_ylim(bottom=0)
ax2.legend(loc='center left', bbox_to_anchor=(1.02, 0.5), frameon=False, fontsize=10)
ax2.spines['top'].set_visible(False)
ax2.spines['right'].set_visible(False)

fig2.text(0.5, -0.05, "X-axis bins: 1-year intervals up to 14, 2-year intervals up to 20, 5-year intervals up to 30, "
                       "10-year intervals up to 70, then a single 70+ bin. "
                       "Bins are evenly spaced categorically, not scaled to age.",
          ha='center', va='top', fontsize=10, style='italic', color='#555555')

plt.subplots_adjust(right=0.82)
plt.tight_layout()
plt.show()

display(notes_by_bin)
display(avg_by_bin)

In [0]:
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

df = combined_df_full_info.copy()
score_cols = ['Manual_Score', 'GPT_Score', 'GPT_Mini_Score', 'Claude_Sonnet_Score']
colors = {'Manual_Score': '#D4B896', 'GPT_Score': '#4C9BE8', 'GPT_Mini_Score': '#57C4A7', 'Claude_Sonnet_Score': '#E87B4C'}
labels = {'Manual_Score': 'Manual', 'GPT_Score': 'GPT 5.4', 'GPT_Mini_Score': 'GPT 5.4 Mini', 'Claude_Sonnet_Score': 'Claude Sonnet 4.6'}

subject_level = df.dropna(subset=['Enrollment Status']).drop_duplicates(subset='Subject ID').copy()
notes_by_cat = subject_level['Enrollment Status'].value_counts()

fig1, ax1 = plt.subplots(figsize=(10, 5))
ax1.bar(notes_by_cat.index.astype(str), notes_by_cat.values, color='#4C9BE8', edgecolor='white')
ax1.set_xlabel('Enrollment status', fontsize=12)
ax1.set_ylabel('Number of notes', fontsize=12)
ax1.set_title('Distribution of notes by enrollment status', fontsize=15, pad=15)
ax1.set_ylim(bottom=0)
ax1.spines['top'].set_visible(False)
ax1.spines['right'].set_visible(False)
for i, v in enumerate(notes_by_cat.values):
    ax1.text(i, v + notes_by_cat.max() * 0.01, str(v), ha='center', va='bottom', fontsize=11)
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

subject_scores = df.dropna(subset=['Enrollment Status']).groupby('Subject ID').agg(
    **{'Enrollment Status': ('Enrollment Status', 'first')},
    Manual_Score=('Manual_Score', 'sum'), GPT_Score=('GPT_Score', 'sum'),
    GPT_Mini_Score=('GPT_Mini_Score', 'sum'), Claude_Sonnet_Score=('Claude_Sonnet_Score', 'sum'),
).reset_index()
avg_by_cat = subject_scores.groupby('Enrollment Status')[score_cols].mean().reindex(notes_by_cat.index)

x = np.arange(len(avg_by_cat))
width = 0.2
fig2, ax2 = plt.subplots(figsize=(11, 6))
for i, col in enumerate(score_cols):
    ax2.bar(x + (i - 1.5) * width, avg_by_cat[col], width, label=labels[col], color=colors[col])

ax2.set_xlabel('Enrollment status', fontsize=12)
ax2.set_ylabel('Average phenotypes scored per note', fontsize=12)
ax2.set_title('Average phenotypes scored per note, by enrollment status', fontsize=15, pad=15)
ax2.set_xticks(x)
ax2.set_xticklabels(avg_by_cat.index.astype(str), rotation=45, ha='right')
ax2.set_ylim(bottom=0)
ax2.legend(loc='center left', bbox_to_anchor=(1.02, 0.5), frameon=False, fontsize=10)
ax2.spines['top'].set_visible(False)
ax2.spines['right'].set_visible(False)
plt.subplots_adjust(right=0.82)
plt.tight_layout()
plt.show()

display(notes_by_cat)
display(avg_by_cat)

In [0]:
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

df = combined_df_full_info.copy()
score_cols = ['Manual_Score', 'GPT_Score', 'GPT_Mini_Score', 'Claude_Sonnet_Score']
colors = {'Manual_Score': '#D4B896', 'GPT_Score': '#4C9BE8', 'GPT_Mini_Score': '#57C4A7', 'Claude_Sonnet_Score': '#E87B4C'}
labels = {'Manual_Score': 'Manual', 'GPT_Score': 'GPT 5.4', 'GPT_Mini_Score': 'GPT 5.4 Mini', 'Claude_Sonnet_Score': 'Claude Sonnet 4.6'}

subject_level = df.dropna(subset=['NOTE_TYPE']).drop_duplicates(subset='Subject ID').copy()
notes_by_cat = subject_level['NOTE_TYPE'].value_counts()

fig1, ax1 = plt.subplots(figsize=(10, 5))
ax1.bar(notes_by_cat.index.astype(str), notes_by_cat.values, color='#4C9BE8', edgecolor='white')
ax1.set_xlabel('Note type', fontsize=12)
ax1.set_ylabel('Number of notes', fontsize=12)
ax1.set_title('Distribution of notes by note type', fontsize=15, pad=15)
ax1.set_ylim(bottom=0)
ax1.spines['top'].set_visible(False)
ax1.spines['right'].set_visible(False)
for i, v in enumerate(notes_by_cat.values):
    ax1.text(i, v + notes_by_cat.max() * 0.01, str(v), ha='center', va='bottom', fontsize=11)
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

subject_scores = df.dropna(subset=['NOTE_TYPE']).groupby('Subject ID').agg(
    **{'NOTE_TYPE': ('NOTE_TYPE', 'first')},
    Manual_Score=('Manual_Score', 'sum'), GPT_Score=('GPT_Score', 'sum'),
    GPT_Mini_Score=('GPT_Mini_Score', 'sum'), Claude_Sonnet_Score=('Claude_Sonnet_Score', 'sum'),
).reset_index()
avg_by_cat = subject_scores.groupby('NOTE_TYPE')[score_cols].mean().reindex(notes_by_cat.index)

x = np.arange(len(avg_by_cat))
width = 0.2
fig2, ax2 = plt.subplots(figsize=(11, 6))
for i, col in enumerate(score_cols):
    ax2.bar(x + (i - 1.5) * width, avg_by_cat[col], width, label=labels[col], color=colors[col])

ax2.set_xlabel('Note type', fontsize=12)
ax2.set_ylabel('Average phenotypes scored per note', fontsize=12)
ax2.set_title('Average phenotypes scored per note, by note type', fontsize=15, pad=15)
ax2.set_xticks(x)
ax2.set_xticklabels(avg_by_cat.index.astype(str), rotation=45, ha='right')
ax2.set_ylim(bottom=0)
ax2.legend(loc='center left', bbox_to_anchor=(1.02, 0.5), frameon=False, fontsize=10)
ax2.spines['top'].set_visible(False)
ax2.spines['right'].set_visible(False)
plt.subplots_adjust(right=0.82)
plt.tight_layout()
plt.show()

display(notes_by_cat)
display(avg_by_cat)

In [0]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

score_cols = [
    "Manual_Score",
    "GPT_Score",
    "GPT_Mini_Score",
    "Claude_Sonnet_Score"
]

colors = {
    "Manual_Score": "#D4B896",
    "GPT_Score": "#4C9BE8",
    "GPT_Mini_Score": "#57C4A7",
    "Claude_Sonnet_Score": "#E87B4C"
}

labels = {
    "Manual_Score": "Manual",
    "GPT_Score": "GPT 5.4",
    "GPT_Mini_Score": "GPT 5.4 Mini",
    "Claude_Sonnet_Score": "Claude Sonnet 4.6"
}

# Use a distinctive name so another cell does not overwrite it
positive_calls_by_subject = (
    combined_df_full_info
    .groupby(
        "Subject ID",
        as_index=False,
        observed=True
    )[score_cols]
    .sum()
)

expected_subjects = combined_df_full_info["Subject ID"].nunique()
actual_subjects = positive_calls_by_subject["Subject ID"].nunique()

print(f"Subjects expected: {expected_subjects:,}")
print(f"Subjects summarized: {actual_subjects:,}")

assert actual_subjects == expected_subjects, (
    f"Expected {expected_subjects} subjects but obtained "
    f"{actual_subjects}."
)

display(positive_calls_by_subject[score_cols].describe())

# One bin per integer value
all_vals = pd.concat(
    [
        positive_calls_by_subject[column].dropna()
        for column in score_cols
    ]
)

bin_edges = (
    np.arange(all_vals.min(), all_vals.max() + 2) - 0.5
)

fig, axes = plt.subplots(
    2,
    2,
    figsize=(12, 8),
    sharex=True,
    sharey=True
)

axes = axes.flatten()

for ax, column in zip(axes, score_cols):

    ax.hist(
        positive_calls_by_subject[column].dropna(),
        bins=bin_edges,
        color=colors[column],
        edgecolor="white"
    )

    ax.set_title(labels[column], fontsize=13)
    ax.set_ylim(bottom=0)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.tick_params(axis="x", labelbottom=True)

fig.supxlabel(
    "Phenotypes scored per subject",
    fontsize=12
)

fig.supylabel(
    "Number of subjects",
    fontsize=12
)

fig.suptitle(
    "Distribution of phenotypes scored, by model",
    fontsize=15
)

plt.tight_layout()
plt.show()

In [0]:
import math

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib_venn import venn2
from matplotlib.colors import to_rgb

# ============================================================
# Configuration
# ============================================================

# Reference scoring that every model is compared against
reference_name = "Manual"
reference_column = "Manual_Score"
reference_color = "#D4B896"

# The order here sets the Venn panel order (A-C) and the ordering of
# rows and bars in the UpSet plot (panel D)
score_columns = {
    "GPT 5.4": "GPT_Score",
    "GPT 5.4 Mini": "GPT_Mini_Score",
    "Claude Sonnet 4.6": "Claude_Sonnet_Score"
}

# Columns flagging disagreement with the manual score (panel D)
disagree_columns = {
    "GPT 5.4": "GPT_disagree",
    "GPT 5.4 Mini": "GPT_Mini_disagree",
    "Claude Sonnet 4.6": "Claude_Sonnet_disagree"
}

color_map = {
    "Manual": "#D4B896",
    "GPT 5.4": "#4C9BE8",
    "GPT 5.4 Mini": "#57C4A7",
    "Claude Sonnet 4.6": "#E87B4C"
}

# Histogram panels (panel E), in display order
distribution_columns = {
    "Manual": "Manual_Score",
    "GPT 5.4": "GPT_Score",
    "GPT 5.4 Mini": "GPT_Mini_Score",
    "Claude Sonnet 4.6": "Claude_Sonnet_Score"
}

set_names = list(score_columns.keys())

panel_letters = "ABCDEFGH"

# ------------------------------------------------------------
# Confirm the required columns are present
# ------------------------------------------------------------

required_combined_columns = (
    [reference_column]
    + list(score_columns.values())
    + list(disagree_columns.values())
)

missing_columns = [
    column
    for column in required_combined_columns
    if column not in combined_df.columns
]

if missing_columns:
    raise KeyError(
        f"combined_df is missing required columns: {missing_columns}"
    )

required_full_info_columns = (
    ["Subject ID"]
    + list(distribution_columns.values())
)

missing_columns = [
    column
    for column in required_full_info_columns
    if column not in combined_df_full_info.columns
]

if missing_columns:
    raise KeyError(
        "combined_df_full_info is missing required columns: "
        f"{missing_columns}"
    )

# ============================================================
# PART 1: Venn calculations (panels A-C)
# Source: combined_df, positive calls vs Manual_Score
# ============================================================

manual_set = set(
    combined_df[combined_df[reference_column] == 1].index
)

results = {}

for method, score_column in score_columns.items():

    model_set = set(
        combined_df[combined_df[score_column] == 1].index
    )

    # Manual only = FN, model only = FP, both = TP
    false_negative = len(manual_set - model_set)
    false_positive = len(model_set - manual_set)
    true_positive = len(manual_set & model_set)

    true_negative = (
        len(combined_df)
        - true_positive
        - false_negative
        - false_positive
    )

    precision = (
        true_positive
        / (true_positive + false_positive)
        if true_positive + false_positive > 0
        else np.nan
    )

    recall = (
        true_positive
        / (true_positive + false_negative)
        if true_positive + false_negative > 0
        else np.nan
    )

    results[method] = {
        "score_column": score_column,
        "n_rows": len(combined_df),
        "manual_positives": (
            true_positive + false_negative
        ),
        "method_positives": (
            true_positive + false_positive
        ),
        "true_positive": true_positive,
        "false_positive": false_positive,
        "false_negative": false_negative,
        "true_negative": true_negative,
        "precision": precision,
        "recall": recall,
        "positive_union": (
            true_positive
            + false_positive
            + false_negative
        )
    }

# ------------------------------------------------------------
# Establish a shared Venn-diagram scale
# ------------------------------------------------------------

max_union = max(
    result["positive_union"]
    for result in results.values()
)

if max_union == 0:
    raise ValueError(
        "No positive calls were found in any scoring method."
    )


def draw_venn(ax, method, result):
    return venn2(
        subsets=(
            result["false_negative"],
            result["false_positive"],
            result["true_positive"]
        ),
        set_labels=(reference_name, method),
        set_colors=(reference_color, color_map[method]),
        alpha=0.5,
        normalize_to=result["positive_union"] / max_union,
        ax=ax
    )


# Measure the circles in every panel so all panels share one scale
circle_extents = []

for method, result in results.items():

    draft_fig, draft_ax = plt.subplots(figsize=(5, 5))
    draft_venn = draw_venn(draft_ax, method, result)

    for center, radius in zip(draft_venn.centers, draft_venn.radii):
        circle_extents.append({
            "left": center.x - radius,
            "right": center.x + radius,
            "bottom": center.y - radius,
            "top": center.y + radius
        })

    plt.close(draft_fig)

circles_left = min(item["left"] for item in circle_extents)
circles_right = max(item["right"] for item in circle_extents)
circles_bottom = min(item["bottom"] for item in circle_extents)
circles_top = max(item["top"] for item in circle_extents)

circles_width = circles_right - circles_left

# Room reserved outside the circles for the FN / FP callouts (left
# and right) and for the set labels (below)
side_margin = 0.34 * circles_width
bottom_margin = 0.12 * circles_width
top_margin = 0.03 * circles_width

shared_xlim = (
    circles_left - side_margin,
    circles_right + side_margin
)

shared_ylim = (
    circles_bottom - bottom_margin,
    circles_top + top_margin
)

# ============================================================
# PART 2: UpSet calculations (panel D)
# Source: combined_df, *_disagree columns
# ============================================================

disagreement_sets = {
    method: set(
        combined_df[combined_df[disagree_column] == 1].index
    )
    for method, disagree_column in disagree_columns.items()
}

# Count each exact combination of disagreeing methods
intersections = []

for combination_mask in range(1, 2 ** len(set_names)):

    membership = tuple(
        method
        for position, method in enumerate(set_names)
        if combination_mask & (1 << position)
    )

    members = set.intersection(
        *[disagreement_sets[method] for method in membership]
    )

    for method in set_names:
        if method not in membership:
            members = members - disagreement_sets[method]

    intersections.append(
        (membership, len(members))
    )

# Sort exact intersections by the number of disagreeing methods
# (single methods first, then pairs, then triples). Within each
# group, follow the method order in score_columns.
intersections = sorted(
    intersections,
    key=lambda item: (
        len(item[0]),
        tuple(
            set_names.index(method)
            for method in item[0]
        )
    )
)

memberships = [
    membership
    for membership, count in intersections
]

counts = [
    count
    for membership, count in intersections
]

# Total disagreements for each individual method
set_sizes = {
    method: len(disagreement_sets[method])
    for method in set_names
}

# Rows where every method agrees with the manual score
all_methods_agree_count = int(
    combined_df[list(disagree_columns.values())]
    .eq(0)
    .all(axis=1)
    .sum()
)

if max(counts) == 0:
    raise ValueError(
        "No disagreements with the manual score were found."
    )

# ============================================================
# PART 3: Phenotype distribution calculations (panel E)
# Source: combined_df_full_info, positive calls per subject
# ============================================================

distribution_score_columns = list(
    distribution_columns.values()
)

# Use a distinctive name so another cell does not overwrite it
positive_calls_by_subject = (
    combined_df_full_info
    .groupby(
        "Subject ID",
        as_index=False,
        observed=True
    )[distribution_score_columns]
    .sum()
)

expected_subjects = (
    combined_df_full_info["Subject ID"].nunique()
)

actual_subjects = (
    positive_calls_by_subject["Subject ID"].nunique()
)

print(f"Subjects expected: {expected_subjects:,}")
print(f"Subjects summarized: {actual_subjects:,}")

assert actual_subjects == expected_subjects, (
    f"Expected {expected_subjects} subjects but obtained "
    f"{actual_subjects}."
)

# One bin per integer value
all_values = pd.concat(
    [
        positive_calls_by_subject[column].dropna()
        for column in distribution_score_columns
    ]
)

bin_edges = (
    np.arange(
        all_values.min(),
        all_values.max() + 2
    )
    - 0.5
)

# ============================================================
# PART 4: Combined figure
# ============================================================

number_of_venn_rows = math.ceil(len(results) / 2)
number_of_distribution_rows = math.ceil(
    len(distribution_columns) / 2
)

upset_letter = panel_letters[len(results)]
distribution_letter = panel_letters[len(results) + 1]

# Vertical layout, measured in inches from the top of the figure.
# Adjust these if you change font sizes.
venn_row_height = 5.30
distribution_row_height = 4.20

venn_title_offset = 0.50
venn_subtitle_offset = 1.20
venn_top_offset = 2.30
venn_bottom_offset = (
    venn_top_offset
    + number_of_venn_rows * venn_row_height
)

upset_title_offset = venn_bottom_offset + 0.95
upset_subtitle_offset = upset_title_offset + 0.65
upset_top_offset = upset_subtitle_offset + 0.60
upset_bottom_offset = upset_top_offset + 6.30
footnote_offset = upset_bottom_offset + 1.15

distribution_title_offset = footnote_offset + 1.35
distribution_subtitle_offset = distribution_title_offset + 0.65
distribution_top_offset = distribution_subtitle_offset + 1.00
distribution_bottom_offset = (
    distribution_top_offset
    + number_of_distribution_rows * distribution_row_height
)
distribution_xlabel_offset = distribution_bottom_offset + 0.85

figure_width = 16
figure_height = distribution_xlabel_offset + 0.55

fig = plt.figure(
    figsize=(figure_width, figure_height)
)


def from_top(inches):
    """Convert a distance from the top edge (in inches) into a
    figure-fraction y coordinate."""
    return 1 - inches / figure_height


# ------------------------------------------------------------
# Panels A-C: Venn diagrams
# ------------------------------------------------------------

fig.text(
    0.5,
    from_top(venn_title_offset),
    "Positive Phenotype Calls Compared With Manual Scoring",
    ha="center",
    va="center",
    fontsize=26,
    fontweight="bold"
)

fig.text(
    0.5,
    from_top(venn_subtitle_offset),
    (
        "TP indicates positive agreement; FP and FN indicate "
        "disagreement."
    ),
    ha="center",
    va="center",
    fontsize=18,
    color="#52606D",
    linespacing=1.4
)

# Four half-width columns: each panel spans two of them, which lets
# a lone panel in the last row sit centered beneath the others
venn_grid = fig.add_gridspec(
    nrows=number_of_venn_rows,
    ncols=4,
    left=0.03,
    right=0.97,
    top=from_top(venn_top_offset),
    bottom=from_top(venn_bottom_offset),
    hspace=0.16,
    wspace=0.06
)

for index, (method, result) in enumerate(results.items()):

    panel_row = index // 2

    is_lone_panel = (
        index == len(results) - 1
        and len(results) % 2 == 1
    )

    if is_lone_panel:
        panel_columns = slice(1, 3)
    elif index % 2 == 0:
        panel_columns = slice(0, 2)
    else:
        panel_columns = slice(2, 4)

    ax = fig.add_subplot(venn_grid[panel_row, panel_columns])

    venn = draw_venn(ax, method, result)

    # --------------------------------------------------------
    # Correct the overlap color
    # --------------------------------------------------------

    overlap_patch = venn.get_patch_by_id("11")

    if overlap_patch is not None:

        # Average the manual and scoring-method colors
        overlap_color = (
            np.array(to_rgb(reference_color))
            + np.array(to_rgb(color_map[method]))
        ) / 2

        overlap_patch.set_facecolor(overlap_color)

        # Two layers at alpha=0.5 have an effective
        # combined opacity of 0.75
        overlap_patch.set_alpha(0.75)

    # Use the same scale across all panels
    ax.set_xlim(shared_xlim)
    ax.set_ylim(shared_ylim)
    ax.set_aspect("equal")

    # Format circle borders
    for patch in venn.patches:
        if patch is not None:
            patch.set_edgecolor("white")
            patch.set_linewidth(2)

    reference_center, method_center = venn.centers

    # --------------------------------------------------------
    # Region labels
    # --------------------------------------------------------

    # TP stays inside the overlap, which is always large enough
    tp_label = venn.get_label_by_id("11")

    if tp_label is not None:
        tp_label.set_text(
            f"TP\nn = {result['true_positive']:,}"
        )
        tp_label.set_fontsize(18)
        tp_label.set_fontweight("bold")

    # FN and FP sit in thin crescents, so their labels are moved
    # outside the circles and tied back with a leader line
    callouts = {
        "10": {
            "text": f"FN\nn = {result['false_negative']:,}",
            "x": circles_left - 0.06 * circles_width,
            "ha": "right"
        },
        "01": {
            "text": f"FP\nn = {result['false_positive']:,}",
            "x": circles_right + 0.06 * circles_width,
            "ha": "left"
        }
    }

    for region_id, callout in callouts.items():

        label = venn.get_label_by_id(region_id)

        if label is None:
            continue

        region_x, region_y = label.get_position()
        label.set_visible(False)

        ax.annotate(
            callout["text"],
            xy=(region_x, region_y),
            xytext=(callout["x"], region_y),
            ha=callout["ha"],
            va="center",
            fontsize=17,
            fontweight="bold",
            arrowprops={
                "arrowstyle": "-",
                "color": "#52606D",
                "linewidth": 1.4,
                "shrinkA": 6,
                "shrinkB": 0
            }
        )

        # Small dot marking the region the leader line points to
        ax.plot(
            region_x,
            region_y,
            marker="o",
            markersize=5,
            color="#52606D"
        )

    # --------------------------------------------------------
    # Set labels: anchored apart from each other below the circles
    # --------------------------------------------------------

    set_label_y = circles_bottom - 0.04 * circles_width
    set_label_gap = 0.03 * circles_width
    set_label_mid = (reference_center.x + method_center.x) / 2

    reference_label, method_label = venn.set_labels

    if reference_label is not None:
        reference_label.set_position(
            (set_label_mid - set_label_gap, set_label_y)
        )
        reference_label.set_ha("right")

    if method_label is not None:
        method_label.set_position(
            (set_label_mid + set_label_gap, set_label_y)
        )
        method_label.set_ha("left")

    for label in venn.set_labels:
        if label is not None:
            label.set_va("top")
            label.set_fontsize(14)
            label.set_fontweight("bold")

    ax.set_title(
        f"{reference_name} vs {method}",
        fontsize=20,
        fontweight="bold",
        pad=10
    )

    # Panel letter in the top-left corner of each panel
    ax.set_title(
        panel_letters[index],
        loc="left",
        fontsize=28,
        fontweight="bold",
        pad=10
    )

# ------------------------------------------------------------
# Panel D: UpSet plot
# ------------------------------------------------------------

number_of_intersections = len(intersections)

fig.text(
    0.03,
    from_top(upset_title_offset),
    upset_letter,
    ha="left",
    va="center",
    fontsize=28,
    fontweight="bold"
)

fig.text(
    0.5,
    from_top(upset_title_offset),
    "Overlap of Scoring-Method Disagreements "
    "With Manual Scoring",
    ha="center",
    va="center",
    fontsize=22,
    fontweight="bold"
)

fig.text(
    0.5,
    from_top(upset_subtitle_offset),
    (
        "Each bar represents one exact combination of methods "
        f"that disagreed with {reference_column}."
    ),
    ha="center",
    va="center",
    fontsize=15,
    color="#52606D"
)

# The wide left margin leaves room for the row labels
upset_grid = fig.add_gridspec(
    nrows=2,
    ncols=1,
    height_ratios=[3.0, 0.375 * len(set_names)],
    hspace=0.05,
    left=0.32,
    right=0.97,
    top=from_top(upset_top_offset),
    bottom=from_top(upset_bottom_offset)
)

# Top: exact intersection counts
ax_bar = fig.add_subplot(upset_grid[0])

bar_positions = np.arange(
    number_of_intersections
)

bars = ax_bar.bar(
    bar_positions,
    counts,
    color="#4C9BE8",
    width=0.65,
    edgecolor="white"
)

maximum_count = max(counts)

label_padding = max(
    maximum_count * 0.015,
    0.5
)

for position, count in zip(
    bar_positions,
    counts
):
    ax_bar.text(
        position,
        count + label_padding,
        f"{count:,}",
        ha="center",
        va="bottom",
        fontsize=14,
        fontweight="bold"
    )

ax_bar.set_xticks([])

ax_bar.set_ylabel(
    "Exact intersection count",
    fontsize=17,
    labelpad=10
)

ax_bar.tick_params(
    axis="y",
    labelsize=14
)

ax_bar.set_ylim(
    0,
    maximum_count * 1.15
)

ax_bar.spines["top"].set_visible(False)
ax_bar.spines["right"].set_visible(False)
ax_bar.spines["bottom"].set_visible(False)

ax_bar.grid(
    axis="y",
    color="#D5DDE5",
    linewidth=0.8,
    alpha=0.7
)

ax_bar.set_axisbelow(True)

# Bottom: membership matrix
ax_matrix = fig.add_subplot(
    upset_grid[1],
    sharex=ax_bar
)

for position, membership in zip(
    bar_positions,
    memberships
):
    active_indices = [
        set_names.index(method)
        for method in membership
    ]

    # Connecting line behind active dots
    if len(active_indices) > 1:
        ax_matrix.plot(
            [position, position],
            [
                min(active_indices),
                max(active_indices)
            ],
            color="#333333",
            linewidth=2.5,
            zorder=1
        )

    for set_index, method in enumerate(
        set_names
    ):
        is_active = (
            set_index in active_indices
        )

        dot_color = (
            color_map[method]
            if is_active
            else "#DDDDDD"
        )

        dot_size = (
            15 if is_active else 12
        )

        ax_matrix.plot(
            position,
            set_index,
            "o",
            color=dot_color,
            markersize=dot_size,
            markeredgecolor=(
                "#333333"
                if is_active
                else "#DDDDDD"
            ),
            markeredgewidth=0.8,
            zorder=2
        )

ax_matrix.set_yticks(
    range(len(set_names))
)

ax_matrix.set_yticklabels(
    [
        f"{method} (total disagreements = "
        f"{set_sizes[method]:,})"
        for method in set_names
    ],
    fontsize=15
)

ax_matrix.set_xticks([])

ax_matrix.set_xlim(
    -0.7,
    number_of_intersections - 0.3
)

ax_matrix.set_ylim(
    -0.5,
    len(set_names) - 0.5
)

ax_matrix.invert_yaxis()

for spine in ax_matrix.spines.values():
    spine.set_visible(False)

ax_matrix.tick_params(
    axis="y",
    length=0,
    pad=10
)

# Mark the groups: 1 method, 2 methods, 3 methods
group_sizes = [
    len(membership)
    for membership in memberships
]

for group_size in sorted(set(group_sizes)):

    group_positions = [
        position
        for position, size in zip(bar_positions, group_sizes)
        if size == group_size
    ]

    group_start = min(group_positions)
    group_end = max(group_positions)

    # Group label beneath the matrix
    ax_matrix.text(
        (group_start + group_end) / 2,
        -0.14,
        (
            "1 method"
            if group_size == 1
            else f"{group_size} methods"
        ),
        transform=ax_matrix.get_xaxis_transform(),
        ha="center",
        va="top",
        fontsize=15,
        fontweight="bold",
        color="#52606D"
    )

    # Bracket line spanning the group
    ax_matrix.plot(
        [group_start - 0.3, group_end + 0.3],
        [-0.07, -0.07],
        transform=ax_matrix.get_xaxis_transform(),
        color="#9AA5B1",
        linewidth=1.5,
        clip_on=False
    )

    # Divider between this group and the next one
    if group_end < number_of_intersections - 1:
        for axis in (ax_bar, ax_matrix):
            axis.axvline(
                group_end + 0.5,
                color="#C5CED8",
                linewidth=1.2,
                linestyle="--",
                zorder=0
            )

fig.text(
    0.5,
    from_top(footnote_offset),
    (
        f"Rows where all {len(set_names)} methods agree with the "
        f"manual score: n = {all_methods_agree_count:,}. "
        "Inactive dots indicate agreement, not missing data."
    ),
    ha="center",
    va="center",
    fontsize=14,
    color="#52606D"
)

# ------------------------------------------------------------
# Panel E: phenotype distributions
# ------------------------------------------------------------

fig.text(
    0.03,
    from_top(distribution_title_offset),
    distribution_letter,
    ha="left",
    va="center",
    fontsize=28,
    fontweight="bold"
)

fig.text(
    0.5,
    from_top(distribution_title_offset),
    "Distribution of Phenotypes Scored, by Scoring Method",
    ha="center",
    va="center",
    fontsize=22,
    fontweight="bold"
)

fig.text(
    0.5,
    from_top(distribution_subtitle_offset),
    (
        "Each histogram shows the number of positive phenotype "
        "calls per subject."
    ),
    ha="center",
    va="center",
    fontsize=15,
    color="#52606D"
)

distribution_left = 0.11
distribution_right = 0.97

distribution_grid = fig.add_gridspec(
    nrows=number_of_distribution_rows,
    ncols=2,
    left=distribution_left,
    right=distribution_right,
    top=from_top(distribution_top_offset),
    bottom=from_top(distribution_bottom_offset),
    hspace=0.38,
    wspace=0.10
)

first_distribution_ax = None

for index, (method, column) in enumerate(
    distribution_columns.items()
):

    ax = fig.add_subplot(
        distribution_grid[index // 2, index % 2],
        sharex=first_distribution_ax,
        sharey=first_distribution_ax
    )

    if first_distribution_ax is None:
        first_distribution_ax = ax

    ax.hist(
        positive_calls_by_subject[column].dropna(),
        bins=bin_edges,
        color=color_map[method],
        edgecolor="white"
    )

    ax.set_title(
        method,
        fontsize=20,
        fontweight="bold",
        pad=10
    )

    ax.set_ylim(bottom=0)

    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

    ax.tick_params(
        axis="both",
        labelsize=14,
        labelbottom=True
    )

    # Shared y axis: only the left column carries tick labels
    if index % 2 == 1:
        ax.tick_params(
            axis="y",
            labelleft=False
        )

    ax.grid(
        axis="y",
        color="#D5DDE5",
        linewidth=0.8,
        alpha=0.7
    )

    ax.set_axisbelow(True)

# Shared axis labels for the histogram block
fig.text(
    (distribution_left + distribution_right) / 2,
    from_top(distribution_xlabel_offset),
    "Phenotypes scored per subject",
    ha="center",
    va="center",
    fontsize=17
)

fig.text(
    0.045,
    (
        from_top(distribution_top_offset)
        + from_top(distribution_bottom_offset)
    ) / 2,
    "Number of subjects",
    ha="center",
    va="center",
    rotation="vertical",
    fontsize=17
)

# To save a publication-quality copy, uncomment:
# fig.savefig("manual_reference_figure.png", dpi=300, bbox_inches="tight")
# fig.savefig("manual_reference_figure.pdf", bbox_inches="tight")

plt.show()

# ============================================================
# PART 5: Summary tables
# ============================================================

# Performance of each method against the manual score
performance_summary = pd.DataFrame([
    {
        "Scoring Method": method,
        "Rows_n": result["n_rows"],
        "Manual_Positives": result["manual_positives"],
        "Method_Positives": result["method_positives"],
        "True_Positive": result["true_positive"],
        "False_Positive": result["false_positive"],
        "False_Negative": result["false_negative"],
        "True_Negative": result["true_negative"],
        "Precision": result["precision"],
        "Recall": result["recall"]
    }
    for method, result in results.items()
])

display(performance_summary)

# Exact intersection counts
upset_intersection_summary = pd.DataFrame([
    {
        "Disagreeing methods": ", ".join(membership),
        "Number of methods": len(membership),
        "Exact intersection count": count
    }
    for membership, count in intersections
])

display(upset_intersection_summary)

# Total disagreements by scoring method
upset_set_size_summary = pd.DataFrame([
    {
        "Scoring method": method,
        "Total disagreements with manual score": (
            set_sizes[method]
        )
    }
    for method in set_names
])

display(upset_set_size_summary)

# Positive calls per subject
display(
    positive_calls_by_subject[distribution_score_columns].describe()
)

In [0]:
import math

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib_venn import venn2
from matplotlib.colors import to_rgb

# ============================================================
# Configuration
# ============================================================

# Reference scoring that every model is compared against
reference_name = "Manual"
reference_column = "Manual_Score"
reference_color = "#D4B896"

# The order here sets the Venn panel order (A-C) and the ordering of
# rows and bars in the UpSet plot (panel D)
score_columns = {
    "GPT 5.4": "GPT_Score",
    "GPT 5.4 Mini": "GPT_Mini_Score",
    "Claude Sonnet 4.6": "Claude_Sonnet_Score"
}

# Columns flagging disagreement with the manual score (panel D)
disagree_columns = {
    "GPT 5.4": "GPT_disagree",
    "GPT 5.4 Mini": "GPT_Mini_disagree",
    "Claude Sonnet 4.6": "Claude_Sonnet_disagree"
}

color_map = {
    "Manual": "#D4B896",
    "GPT 5.4": "#4C9BE8",
    "GPT 5.4 Mini": "#57C4A7",
    "Claude Sonnet 4.6": "#E87B4C"
}

set_names = list(score_columns.keys())

panel_letters = "ABCDEFGH"

# ------------------------------------------------------------
# Confirm the required columns are present
# ------------------------------------------------------------

required_combined_columns = (
    [reference_column]
    + list(score_columns.values())
    + list(disagree_columns.values())
)

missing_columns = [
    column
    for column in required_combined_columns
    if column not in combined_df.columns
]

if missing_columns:
    raise KeyError(
        f"combined_df is missing required columns: {missing_columns}"
    )

# ============================================================
# PART 1: Venn calculations (panels A-C)
# Source: combined_df, positive calls vs Manual_Score
# ============================================================

manual_set = set(
    combined_df[combined_df[reference_column] == 1].index
)

results = {}

for method, score_column in score_columns.items():

    model_set = set(
        combined_df[combined_df[score_column] == 1].index
    )

    # Manual only = FN, model only = FP, both = TP
    false_negative = len(manual_set - model_set)
    false_positive = len(model_set - manual_set)
    true_positive = len(manual_set & model_set)

    true_negative = (
        len(combined_df)
        - true_positive
        - false_negative
        - false_positive
    )

    precision = (
        true_positive
        / (true_positive + false_positive)
        if true_positive + false_positive > 0
        else np.nan
    )

    recall = (
        true_positive
        / (true_positive + false_negative)
        if true_positive + false_negative > 0
        else np.nan
    )

    results[method] = {
        "score_column": score_column,
        "n_rows": len(combined_df),
        "manual_positives": (
            true_positive + false_negative
        ),
        "method_positives": (
            true_positive + false_positive
        ),
        "true_positive": true_positive,
        "false_positive": false_positive,
        "false_negative": false_negative,
        "true_negative": true_negative,
        "precision": precision,
        "recall": recall,
        "positive_union": (
            true_positive
            + false_positive
            + false_negative
        )
    }

# ------------------------------------------------------------
# Establish a shared Venn-diagram scale
# ------------------------------------------------------------

max_union = max(
    result["positive_union"]
    for result in results.values()
)

if max_union == 0:
    raise ValueError(
        "No positive calls were found in any scoring method."
    )


def draw_venn(ax, method, result):
    return venn2(
        subsets=(
            result["false_negative"],
            result["false_positive"],
            result["true_positive"]
        ),
        set_labels=(reference_name, method),
        set_colors=(reference_color, color_map[method]),
        alpha=0.5,
        normalize_to=result["positive_union"] / max_union,
        ax=ax
    )


# Measure the circles in every panel so all panels share one scale
circle_extents = []

for method, result in results.items():

    draft_fig, draft_ax = plt.subplots(figsize=(5, 5))
    draft_venn = draw_venn(draft_ax, method, result)

    for center, radius in zip(draft_venn.centers, draft_venn.radii):
        circle_extents.append({
            "left": center.x - radius,
            "right": center.x + radius,
            "bottom": center.y - radius,
            "top": center.y + radius
        })

    plt.close(draft_fig)

circles_left = min(item["left"] for item in circle_extents)
circles_right = max(item["right"] for item in circle_extents)
circles_bottom = min(item["bottom"] for item in circle_extents)
circles_top = max(item["top"] for item in circle_extents)

circles_width = circles_right - circles_left

# Room reserved outside the circles for the FN / FP callouts (left
# and right) and for the set labels (below)
side_margin = 0.34 * circles_width
bottom_margin = 0.12 * circles_width
top_margin = 0.03 * circles_width

shared_xlim = (
    circles_left - side_margin,
    circles_right + side_margin
)

shared_ylim = (
    circles_bottom - bottom_margin,
    circles_top + top_margin
)

# ============================================================
# PART 2: UpSet calculations (panel D)
# Source: combined_df, *_disagree columns
# ============================================================

disagreement_sets = {
    method: set(
        combined_df[combined_df[disagree_column] == 1].index
    )
    for method, disagree_column in disagree_columns.items()
}

# Count each exact combination of disagreeing methods
intersections = []

for combination_mask in range(1, 2 ** len(set_names)):

    membership = tuple(
        method
        for position, method in enumerate(set_names)
        if combination_mask & (1 << position)
    )

    members = set.intersection(
        *[disagreement_sets[method] for method in membership]
    )

    for method in set_names:
        if method not in membership:
            members = members - disagreement_sets[method]

    intersections.append(
        (membership, len(members))
    )

# Sort exact intersections by the number of disagreeing methods
# (single methods first, then pairs, then triples). Within each
# group, follow the method order in score_columns.
intersections = sorted(
    intersections,
    key=lambda item: (
        len(item[0]),
        tuple(
            set_names.index(method)
            for method in item[0]
        )
    )
)

memberships = [
    membership
    for membership, count in intersections
]

counts = [
    count
    for membership, count in intersections
]

# Total disagreements for each individual method
set_sizes = {
    method: len(disagreement_sets[method])
    for method in set_names
}

# Rows where every method agrees with the manual score
all_methods_agree_count = int(
    combined_df[list(disagree_columns.values())]
    .eq(0)
    .all(axis=1)
    .sum()
)

if max(counts) == 0:
    raise ValueError(
        "No disagreements with the manual score were found."
    )

# ============================================================
# PART 3: Combined figure
# ============================================================

number_of_venn_rows = math.ceil(len(results) / 2)

upset_letter = panel_letters[len(results)]

# Vertical layout, measured in inches from the top of the figure.
# Adjust these if you change font sizes.
venn_row_height = 5.30

venn_title_offset = 0.50
venn_subtitle_offset = 1.20
venn_top_offset = 2.30
venn_bottom_offset = (
    venn_top_offset
    + number_of_venn_rows * venn_row_height
)

upset_title_offset = venn_bottom_offset + 0.95
upset_subtitle_offset = upset_title_offset + 0.65
upset_top_offset = upset_subtitle_offset + 0.60
upset_bottom_offset = upset_top_offset + 6.30
footnote_offset = upset_bottom_offset + 1.15

figure_width = 16
figure_height = footnote_offset + 0.60

fig = plt.figure(
    figsize=(figure_width, figure_height)
)


def from_top(inches):
    """Convert a distance from the top edge (in inches) into a
    figure-fraction y coordinate."""
    return 1 - inches / figure_height


# ------------------------------------------------------------
# Panels A-C: Venn diagrams
# ------------------------------------------------------------

fig.text(
    0.5,
    from_top(venn_title_offset),
    "Positive Phenotype Calls Compared With Manual Scoring",
    ha="center",
    va="center",
    fontsize=26,
    fontweight="bold"
)

fig.text(
    0.5,
    from_top(venn_subtitle_offset),
    (
        "TP indicates positive agreement; FP and FN indicate "
        "disagreement."
    ),
    ha="center",
    va="center",
    fontsize=18,
    color="#52606D",
    linespacing=1.4
)

# Four half-width columns: each panel spans two of them, which lets
# a lone panel in the last row sit centered beneath the others
venn_grid = fig.add_gridspec(
    nrows=number_of_venn_rows,
    ncols=4,
    left=0.03,
    right=0.97,
    top=from_top(venn_top_offset),
    bottom=from_top(venn_bottom_offset),
    hspace=0.16,
    wspace=0.06
)

for index, (method, result) in enumerate(results.items()):

    panel_row = index // 2

    is_lone_panel = (
        index == len(results) - 1
        and len(results) % 2 == 1
    )

    if is_lone_panel:
        panel_columns = slice(1, 3)
    elif index % 2 == 0:
        panel_columns = slice(0, 2)
    else:
        panel_columns = slice(2, 4)

    ax = fig.add_subplot(venn_grid[panel_row, panel_columns])

    venn = draw_venn(ax, method, result)

    # --------------------------------------------------------
    # Correct the overlap color
    # --------------------------------------------------------

    overlap_patch = venn.get_patch_by_id("11")

    if overlap_patch is not None:

        # Average the manual and scoring-method colors
        overlap_color = (
            np.array(to_rgb(reference_color))
            + np.array(to_rgb(color_map[method]))
        ) / 2

        overlap_patch.set_facecolor(overlap_color)

        # Two layers at alpha=0.5 have an effective
        # combined opacity of 0.75
        overlap_patch.set_alpha(0.75)

    # Use the same scale across all panels
    ax.set_xlim(shared_xlim)
    ax.set_ylim(shared_ylim)
    ax.set_aspect("equal")

    # Format circle borders
    for patch in venn.patches:
        if patch is not None:
            patch.set_edgecolor("white")
            patch.set_linewidth(2)

    reference_center, method_center = venn.centers

    # --------------------------------------------------------
    # Region labels
    # --------------------------------------------------------

    # TP stays inside the overlap, which is always large enough
    tp_label = venn.get_label_by_id("11")

    if tp_label is not None:
        tp_label.set_text(
            f"TP\nn = {result['true_positive']:,}"
        )
        tp_label.set_fontsize(18)
        tp_label.set_fontweight("bold")

    # FN and FP sit in thin crescents, so their labels are moved
    # outside the circles and tied back with a leader line
    callouts = {
        "10": {
            "text": f"FN\nn = {result['false_negative']:,}",
            "x": circles_left - 0.06 * circles_width,
            "ha": "right"
        },
        "01": {
            "text": f"FP\nn = {result['false_positive']:,}",
            "x": circles_right + 0.06 * circles_width,
            "ha": "left"
        }
    }

    for region_id, callout in callouts.items():

        label = venn.get_label_by_id(region_id)

        if label is None:
            continue

        region_x, region_y = label.get_position()
        label.set_visible(False)

        ax.annotate(
            callout["text"],
            xy=(region_x, region_y),
            xytext=(callout["x"], region_y),
            ha=callout["ha"],
            va="center",
            fontsize=17,
            fontweight="bold",
            arrowprops={
                "arrowstyle": "-",
                "color": "#52606D",
                "linewidth": 1.4,
                "shrinkA": 6,
                "shrinkB": 0
            }
        )

        # Small dot marking the region the leader line points to
        ax.plot(
            region_x,
            region_y,
            marker="o",
            markersize=5,
            color="#52606D"
        )

    # --------------------------------------------------------
    # Set labels: anchored apart from each other below the circles
    # --------------------------------------------------------

    set_label_y = circles_bottom - 0.04 * circles_width
    set_label_gap = 0.03 * circles_width
    set_label_mid = (reference_center.x + method_center.x) / 2

    reference_label, method_label = venn.set_labels

    if reference_label is not None:
        reference_label.set_position(
            (set_label_mid - set_label_gap, set_label_y)
        )
        reference_label.set_ha("right")

    if method_label is not None:
        method_label.set_position(
            (set_label_mid + set_label_gap, set_label_y)
        )
        method_label.set_ha("left")

    for label in venn.set_labels:
        if label is not None:
            label.set_va("top")
            label.set_fontsize(14)
            label.set_fontweight("bold")

    ax.set_title(
        f"{reference_name} vs {method}",
        fontsize=20,
        fontweight="bold",
        pad=10
    )

    # Panel letter in the top-left corner of each panel
    ax.set_title(
        panel_letters[index],
        loc="left",
        fontsize=28,
        fontweight="bold",
        pad=10
    )

# ------------------------------------------------------------
# Panel D: UpSet plot
# ------------------------------------------------------------

number_of_intersections = len(intersections)

fig.text(
    0.03,
    from_top(upset_title_offset),
    upset_letter,
    ha="left",
    va="center",
    fontsize=28,
    fontweight="bold"
)

fig.text(
    0.5,
    from_top(upset_title_offset),
    "Overlap of Scoring-Method Disagreements "
    "With Manual Scoring",
    ha="center",
    va="center",
    fontsize=22,
    fontweight="bold"
)

fig.text(
    0.5,
    from_top(upset_subtitle_offset),
    (
        "Each bar represents one exact combination of methods "
        f"that disagreed with {reference_column}."
    ),
    ha="center",
    va="center",
    fontsize=15,
    color="#52606D"
)

# The wide left margin leaves room for the row labels
upset_grid = fig.add_gridspec(
    nrows=2,
    ncols=1,
    height_ratios=[3.0, 0.375 * len(set_names)],
    hspace=0.05,
    left=0.32,
    right=0.97,
    top=from_top(upset_top_offset),
    bottom=from_top(upset_bottom_offset)
)

# Top: exact intersection counts
ax_bar = fig.add_subplot(upset_grid[0])

bar_positions = np.arange(
    number_of_intersections
)

bars = ax_bar.bar(
    bar_positions,
    counts,
    color="#4C9BE8",
    width=0.65,
    edgecolor="white"
)

maximum_count = max(counts)

label_padding = max(
    maximum_count * 0.015,
    0.5
)

for position, count in zip(
    bar_positions,
    counts
):
    ax_bar.text(
        position,
        count + label_padding,
        f"{count:,}",
        ha="center",
        va="bottom",
        fontsize=14,
        fontweight="bold"
    )

ax_bar.set_xticks([])

ax_bar.set_ylabel(
    "Exact intersection count",
    fontsize=17,
    labelpad=10
)

ax_bar.tick_params(
    axis="y",
    labelsize=14
)

ax_bar.set_ylim(
    0,
    maximum_count * 1.15
)

ax_bar.spines["top"].set_visible(False)
ax_bar.spines["right"].set_visible(False)
ax_bar.spines["bottom"].set_visible(False)

ax_bar.grid(
    axis="y",
    color="#D5DDE5",
    linewidth=0.8,
    alpha=0.7
)

ax_bar.set_axisbelow(True)

# Bottom: membership matrix
ax_matrix = fig.add_subplot(
    upset_grid[1],
    sharex=ax_bar
)

for position, membership in zip(
    bar_positions,
    memberships
):
    active_indices = [
        set_names.index(method)
        for method in membership
    ]

    # Connecting line behind active dots
    if len(active_indices) > 1:
        ax_matrix.plot(
            [position, position],
            [
                min(active_indices),
                max(active_indices)
            ],
            color="#333333",
            linewidth=2.5,
            zorder=1
        )

    for set_index, method in enumerate(
        set_names
    ):
        is_active = (
            set_index in active_indices
        )

        dot_color = (
            color_map[method]
            if is_active
            else "#DDDDDD"
        )

        dot_size = (
            15 if is_active else 12
        )

        ax_matrix.plot(
            position,
            set_index,
            "o",
            color=dot_color,
            markersize=dot_size,
            markeredgecolor=(
                "#333333"
                if is_active
                else "#DDDDDD"
            ),
            markeredgewidth=0.8,
            zorder=2
        )

ax_matrix.set_yticks(
    range(len(set_names))
)

ax_matrix.set_yticklabels(
    [
        f"{method} (total disagreements = "
        f"{set_sizes[method]:,})"
        for method in set_names
    ],
    fontsize=15
)

ax_matrix.set_xticks([])

ax_matrix.set_xlim(
    -0.7,
    number_of_intersections - 0.3
)

ax_matrix.set_ylim(
    -0.5,
    len(set_names) - 0.5
)

ax_matrix.invert_yaxis()

for spine in ax_matrix.spines.values():
    spine.set_visible(False)

ax_matrix.tick_params(
    axis="y",
    length=0,
    pad=10
)

# Mark the groups: 1 method, 2 methods, 3 methods
group_sizes = [
    len(membership)
    for membership in memberships
]

for group_size in sorted(set(group_sizes)):

    group_positions = [
        position
        for position, size in zip(bar_positions, group_sizes)
        if size == group_size
    ]

    group_start = min(group_positions)
    group_end = max(group_positions)

    # Group label beneath the matrix
    ax_matrix.text(
        (group_start + group_end) / 2,
        -0.14,
        (
            "1 method"
            if group_size == 1
            else f"{group_size} methods"
        ),
        transform=ax_matrix.get_xaxis_transform(),
        ha="center",
        va="top",
        fontsize=15,
        fontweight="bold",
        color="#52606D"
    )

    # Bracket line spanning the group
    ax_matrix.plot(
        [group_start - 0.3, group_end + 0.3],
        [-0.07, -0.07],
        transform=ax_matrix.get_xaxis_transform(),
        color="#9AA5B1",
        linewidth=1.5,
        clip_on=False
    )

    # Divider between this group and the next one
    if group_end < number_of_intersections - 1:
        for axis in (ax_bar, ax_matrix):
            axis.axvline(
                group_end + 0.5,
                color="#C5CED8",
                linewidth=1.2,
                linestyle="--",
                zorder=0
            )

fig.text(
    0.5,
    from_top(footnote_offset),
    (
        f"Rows where all {len(set_names)} methods agree with the "
        f"manual score: n = {all_methods_agree_count:,}. "
        "Inactive dots indicate agreement, not missing data."
    ),
    ha="center",
    va="center",
    fontsize=14,
    color="#52606D"
)

# To save a publication-quality copy, uncomment:
# fig.savefig("manual_reference_figure.png", dpi=300, bbox_inches="tight")
# fig.savefig("manual_reference_figure.pdf", bbox_inches="tight")

plt.show()

# ============================================================
# PART 4: Summary tables
# ============================================================

# Performance of each method against the manual score
performance_summary = pd.DataFrame([
    {
        "Scoring Method": method,
        "Rows_n": result["n_rows"],
        "Manual_Positives": result["manual_positives"],
        "Method_Positives": result["method_positives"],
        "True_Positive": result["true_positive"],
        "False_Positive": result["false_positive"],
        "False_Negative": result["false_negative"],
        "True_Negative": result["true_negative"],
        "Precision": result["precision"],
        "Recall": result["recall"]
    }
    for method, result in results.items()
])

display(performance_summary)

# Exact intersection counts
upset_intersection_summary = pd.DataFrame([
    {
        "Disagreeing methods": ", ".join(membership),
        "Number of methods": len(membership),
        "Exact intersection count": count
    }
    for membership, count in intersections
])

display(upset_intersection_summary)

# Total disagreements by scoring method
upset_set_size_summary = pd.DataFrame([
    {
        "Scoring method": method,
        "Total disagreements with manual score": (
            set_sizes[method]
        )
    }
    for method in set_names
])

display(upset_set_size_summary)

In [0]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.gridspec import GridSpec
from matplotlib.ticker import MaxNLocator, MultipleLocator
from matplotlib_venn import venn3

# ------------------------------------------------------------
# Shared style (matches the other combined figures in the paper)
# ------------------------------------------------------------

score_cols = [
    "Manual_Score",
    "GPT_Score",
    "GPT_Mini_Score",
    "Claude_Sonnet_Score"
]

colors = {
    "Manual_Score": "#D4B896",
    "GPT_Score": "#4C9BE8",
    "GPT_Mini_Score": "#57C4A7",
    "Claude_Sonnet_Score": "#E87B4C"
}

method_labels = {
    "Manual_Score": "Manual",
    "GPT_Score": "GPT 5.4",
    "GPT_Mini_Score": "GPT 5.4 Mini",
    "Claude_Sonnet_Score": "Claude Sonnet 4.6"
}

title_color = "#000000"
subtitle_color = "#52606D"
grid_color = "#D5DDE5"

figure_title_size = 18
panel_letter_size = 18
panel_title_size = 14
subtitle_size = 11
footnote_size = 10

# ------------------------------------------------------------
# Panel A data: overlap of LLM disagreements with manual score
# ------------------------------------------------------------

gpt = set(combined_df[combined_df["GPT_disagree"] == 1].index)
mini = set(combined_df[combined_df["GPT_Mini_disagree"] == 1].index)
claude = set(
    combined_df[combined_df["Claude_Sonnet_disagree"] == 1].index
)

# Order required by venn3: (100, 010, 110, 001, 101, 011, 111)
venn_subsets = (
    len(gpt - mini - claude),
    len(mini - gpt - claude),
    len((gpt & mini) - claude),
    len(claude - gpt - mini),
    len((gpt & claude) - mini),
    len((mini & claude) - gpt),
    len(gpt & mini & claude),
)

n_any_disagreement = len(gpt | mini | claude)
n_all_agree = len(combined_df) - n_any_disagreement

print(f"Pairs with at least one LLM disagreement: {n_any_disagreement:,}")
print(f"Pairs where all three LLMs agree with manual: {n_all_agree:,}")

# ------------------------------------------------------------
# Panel B data: positive calls per participant
# ------------------------------------------------------------

positive_calls_by_subject = (
    combined_df_full_info
    .groupby(
        "Subject ID",
        as_index=False,
        observed=True
    )[score_cols]
    .sum()
)

expected_subjects = combined_df_full_info["Subject ID"].nunique()
actual_subjects = positive_calls_by_subject["Subject ID"].nunique()

assert actual_subjects == expected_subjects, (
    f"Expected {expected_subjects} subjects but obtained "
    f"{actual_subjects}."
)

print(f"Participants summarized: {actual_subjects:,}")

display(positive_calls_by_subject[score_cols].describe())

# One bin per integer value
all_vals = pd.concat(
    [
        positive_calls_by_subject[column].dropna()
        for column in score_cols
    ]
)

bin_edges = (
    np.arange(all_vals.min(), all_vals.max() + 2) - 0.5
)

# ------------------------------------------------------------
# Figure layout
# ------------------------------------------------------------

fig = plt.figure(figsize=(12, 16))

# Overall title and subtitle
fig.text(
    0.5, 0.985,
    "Disagreements With Manual Scoring and Phenotypes "
    "Scored per Participant",
    ha="center", va="top",
    fontsize=figure_title_size,
    fontweight="bold",
    color=title_color
)

fig.text(
    0.5, 0.958,
    "Benchmark 1, before adjudication. Counts are "
    "participant-phenotype pairs (A) and participants (B).",
    ha="center", va="top",
    fontsize=subtitle_size + 1,
    color=subtitle_color
)


def add_panel_header(letter, title, subtitle, y):
    """Bold letter at left, bold centered title, gray subtitle."""
    fig.text(
        0.06, y, letter,
        ha="left", va="top",
        fontsize=panel_letter_size,
        fontweight="bold",
        color=title_color
    )

    fig.text(
        0.5, y, title,
        ha="center", va="top",
        fontsize=panel_title_size,
        fontweight="bold",
        color=title_color
    )

    fig.text(
        0.5, y - 0.022, subtitle,
        ha="center", va="top",
        fontsize=subtitle_size,
        color=subtitle_color,
        linespacing=1.4
    )


# ------------------------------------------------------------
# Panel A: Venn diagram
# ------------------------------------------------------------

add_panel_header(
    "A",
    "Overlap of LLM Disagreements With Manual Scoring",
    (
        "Each region counts the pairs on which exactly that "
        "combination of LLMs disagreed with the manual score.\n"
        "Areas are approximately proportional to counts."
    ),
    y=0.915
)

ax_venn = fig.add_axes([0.18, 0.535, 0.64, 0.33])

venn = venn3(
    subsets=venn_subsets,
    set_labels=(
        method_labels["GPT_Score"],
        method_labels["GPT_Mini_Score"],
        method_labels["Claude_Sonnet_Score"]
    ),
    set_colors=(
        colors["GPT_Score"],
        colors["GPT_Mini_Score"],
        colors["Claude_Sonnet_Score"]
    ),
    alpha=0.5,
    subset_label_formatter=lambda count: f"{count:,}",
    ax=ax_venn
)

for label in venn.subset_labels:
    if label:
        label.set_fontsize(12)
        label.set_fontweight("bold")

for patch in venn.patches:
    if patch:
        patch.set_edgecolor("white")
        patch.set_linewidth(2)

for label in venn.set_labels:
    if label:
        label.set_fontsize(13)
        label.set_fontweight("bold")

fig.text(
    0.5, 0.515,
    (
        f"Pairs with at least one LLM disagreement: "
        f"n = {n_any_disagreement:,}. "
        f"Pairs where all 3 LLMs agree with the manual score: "
        f"n = {n_all_agree:,}."
    ),
    ha="center", va="top",
    fontsize=footnote_size,
    color=subtitle_color
)

# ------------------------------------------------------------
# Panel B: histograms of positive calls per participant
# ------------------------------------------------------------

add_panel_header(
    "B",
    "Distribution of Phenotypes Scored, by Scoring Method",
    (
        "Each histogram shows the number of the 124 checklist "
        "phenotypes scored as present per participant "
        f"(n = {actual_subjects:,})."
    ),
    y=0.47
)

grid = GridSpec(
    2, 2,
    figure=fig,
    left=0.11, right=0.97,
    bottom=0.065, top=0.40,
    hspace=0.38, wspace=0.12
)

hist_axes = []

for i, column in enumerate(score_cols):

    share_with = hist_axes[0] if hist_axes else None

    ax = fig.add_subplot(
        grid[i // 2, i % 2],
        sharex=share_with,
        sharey=share_with
    )

    ax.hist(
        positive_calls_by_subject[column].dropna(),
        bins=bin_edges,
        color=colors[column],
        edgecolor="white"
    )

    ax.set_title(
        method_labels[column],
        fontsize=13,
        fontweight="bold",
        pad=8
    )

    ax.set_ylim(bottom=0)

    ax.xaxis.set_major_locator(MultipleLocator(5))
    ax.yaxis.set_major_locator(MaxNLocator(integer=True))

    ax.grid(axis="y", color=grid_color, linewidth=0.8, alpha=0.8)
    ax.set_axisbelow(True)

    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

    ax.tick_params(axis="both", labelsize=11)
    ax.tick_params(axis="x", labelbottom=True)
    ax.tick_params(axis="y", labelleft=True)

    hist_axes.append(ax)

# Shared axis labels for the histogram block
fig.text(
    0.54, 0.025,
    "Phenotypes scored per participant",
    ha="center", va="center",
    fontsize=13
)

fig.text(
    0.035, 0.2325,
    "Number of participants",
    ha="center", va="center",
    rotation="vertical",
    fontsize=13
)

# ------------------------------------------------------------
# Export
# ------------------------------------------------------------

output_stem = "Supplementary_Figure_2"

fig.savefig(
    f"{output_stem}.png",
    dpi=300,
    bbox_inches="tight",
    facecolor="white"
)

fig.savefig(
    f"{output_stem}.pdf",
    bbox_inches="tight",
    facecolor="white"
)

print(f"Saved {output_stem}.png and {output_stem}.pdf")

plt.show()

In [0]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.gridspec import GridSpec
from matplotlib.ticker import MaxNLocator, MultipleLocator
from matplotlib_venn import venn3

# ------------------------------------------------------------
# Shared style (matches the other combined figures in the paper)
# ------------------------------------------------------------

score_cols = [
    "Manual_Score",
    "GPT_Score",
    "GPT_Mini_Score",
    "Claude_Sonnet_Score"
]

colors = {
    "Manual_Score": "#D4B896",
    "GPT_Score": "#4C9BE8",
    "GPT_Mini_Score": "#57C4A7",
    "Claude_Sonnet_Score": "#E87B4C"
}

method_labels = {
    "Manual_Score": "Manual",
    "GPT_Score": "GPT 5.4",
    "GPT_Mini_Score": "GPT 5.4 Mini",
    "Claude_Sonnet_Score": "Claude Sonnet 4.6"
}

title_color = "#000000"
subtitle_color = "#52606D"
grid_color = "#D5DDE5"

figure_title_size = 18
panel_letter_size = 18
panel_title_size = 14
subtitle_size = 11
footnote_size = 10

# ------------------------------------------------------------
# Panel A data: overlap of LLM disagreements with manual score
# ------------------------------------------------------------

gpt = set(combined_df[combined_df["GPT_disagree"] == 1].index)
mini = set(combined_df[combined_df["GPT_Mini_disagree"] == 1].index)
claude = set(
    combined_df[combined_df["Claude_Sonnet_disagree"] == 1].index
)

# Order required by venn3: (100, 010, 110, 001, 101, 011, 111)
venn_subsets = (
    len(gpt - mini - claude),
    len(mini - gpt - claude),
    len((gpt & mini) - claude),
    len(claude - gpt - mini),
    len((gpt & claude) - mini),
    len((mini & claude) - gpt),
    len(gpt & mini & claude),
)

n_any_disagreement = len(gpt | mini | claude)
n_all_agree = len(combined_df) - n_any_disagreement

print(f"Pairs with at least one LLM disagreement: {n_any_disagreement:,}")
print(f"Pairs where all three LLMs agree with manual: {n_all_agree:,}")

# ------------------------------------------------------------
# Panel B data: positive calls per participant
# ------------------------------------------------------------

positive_calls_by_subject = (
    combined_df_full_info
    .groupby(
        "Subject ID",
        as_index=False,
        observed=True
    )[score_cols]
    .sum()
)

expected_subjects = combined_df_full_info["Subject ID"].nunique()
actual_subjects = positive_calls_by_subject["Subject ID"].nunique()

assert actual_subjects == expected_subjects, (
    f"Expected {expected_subjects} subjects but obtained "
    f"{actual_subjects}."
)

print(f"Participants summarized: {actual_subjects:,}")

display(positive_calls_by_subject[score_cols].describe())

# One bin per integer value
all_vals = pd.concat(
    [
        positive_calls_by_subject[column].dropna()
        for column in score_cols
    ]
)

bin_edges = (
    np.arange(all_vals.min(), all_vals.max() + 2) - 0.5
)

# ------------------------------------------------------------
# Figure layout
# ------------------------------------------------------------

fig = plt.figure(figsize=(12, 16))

# Overall title and subtitle
fig.text(
    0.5, 0.985,
    "Disagreements With Manual Scoring and Phenotypes "
    "Scored per Participant",
    ha="center", va="top",
    fontsize=figure_title_size,
    fontweight="bold",
    color=title_color
)

fig.text(
    0.5, 0.958,
    "Benchmark 1, before adjudication. Counts are "
    "participant-phenotype pairs (A) and participants (B).",
    ha="center", va="top",
    fontsize=subtitle_size + 1,
    color=subtitle_color
)


def add_panel_header(letter, title, subtitle, y):
    """Bold letter at left, bold centered title, gray subtitle."""
    fig.text(
        0.06, y, letter,
        ha="left", va="top",
        fontsize=panel_letter_size,
        fontweight="bold",
        color=title_color
    )

    fig.text(
        0.5, y, title,
        ha="center", va="top",
        fontsize=panel_title_size,
        fontweight="bold",
        color=title_color
    )

    fig.text(
        0.5, y - 0.022, subtitle,
        ha="center", va="top",
        fontsize=subtitle_size,
        color=subtitle_color,
        linespacing=1.4
    )


# ------------------------------------------------------------
# Panel A: Venn diagram
# ------------------------------------------------------------

add_panel_header(
    "A",
    "Overlap of LLM Disagreements With Manual Scoring",
    (
        "Each region counts the pairs on which exactly that "
        "combination of LLMs disagreed with the manual score.\n"
        "Areas are proportional to counts."
    ),
    y=0.915
)

ax_venn = fig.add_axes([0.18, 0.535, 0.64, 0.33])

venn = venn3(
    subsets=venn_subsets,
    set_labels=(
        method_labels["GPT_Score"],
        method_labels["GPT_Mini_Score"],
        method_labels["Claude_Sonnet_Score"]
    ),
    set_colors=(
        colors["GPT_Score"],
        colors["GPT_Mini_Score"],
        colors["Claude_Sonnet_Score"]
    ),
    alpha=0.5,
    subset_label_formatter=lambda count: f"{count:,}",
    ax=ax_venn
)

for label in venn.subset_labels:
    if label:
        label.set_fontsize(12)
        label.set_fontweight("bold")

for patch in venn.patches:
    if patch:
        patch.set_edgecolor("white")
        patch.set_linewidth(2)

for label in venn.set_labels:
    if label:
        label.set_fontsize(13)
        label.set_fontweight("bold")

# ------------------------------------------------------------
# Move each count to the roomiest spot inside its own region
# ------------------------------------------------------------

# The library's default label spots can sit on a circle edge in the
# thin crescents. For each region, sample a fine grid of points that
# fall inside it and keep the point farthest from the region's
# outline. Horizontal distances are halved before measuring, because
# a number is wider than it is tall and needs more side room.
horizontal_weight = 0.5
grid_points_per_side = 300


def roomiest_point(path):
    """Point inside a region path that is farthest from its edges."""
    # Region outlines are stored as curves, so sample points along
    # each curve segment (not its control points, which lie off the
    # true edge) to get an accurate outline
    steps = np.linspace(0, 1, 40)
    dense_boundary = np.vstack([
        segment(steps)
        for segment, _ in path.iter_bezier()
    ])

    x_min, y_min = dense_boundary.min(axis=0)
    x_max, y_max = dense_boundary.max(axis=0)

    grid_x, grid_y = np.meshgrid(
        np.linspace(x_min, x_max, grid_points_per_side),
        np.linspace(y_min, y_max, grid_points_per_side)
    )
    candidates = np.column_stack([grid_x.ravel(), grid_y.ravel()])
    candidates = candidates[path.contains_points(candidates)]

    if len(candidates) == 0:
        return None

    scale = np.array([horizontal_weight, 1.0])
    scaled_boundary = dense_boundary * scale

    # Distance from each candidate to its nearest outline point,
    # computed in chunks to keep memory use small
    nearest = np.empty(len(candidates))
    for start in range(0, len(candidates), 2000):
        chunk = candidates[start:start + 2000] * scale
        differences = (
            chunk[:, None, :] - scaled_boundary[None, :, :]
        )
        nearest[start:start + 2000] = np.sqrt(
            (differences ** 2).sum(axis=2)
        ).min(axis=1)

    return candidates[np.argmax(nearest)]


for region_id in ["100", "010", "110", "001", "101", "011", "111"]:
    patch = venn.get_patch_by_id(region_id)
    label = venn.get_label_by_id(region_id)

    if patch is None or label is None:
        continue

    point = roomiest_point(patch.get_path())

    if point is not None:
        label.set_position(point)
        label.set_ha("center")
        label.set_va("center")

# ------------------------------------------------------------
# Shrink all counts together until each one clears its edges
# ------------------------------------------------------------

# Some crescents are thin, so every count is checked against its own
# region with clear space around it. If any count would touch an
# edge, all counts step down together by half a point until every
# one fits, so they stay the same size and inside their regions.

# Clear space required around each number, as a share of its height
label_side_padding = 0.30
label_top_padding = 0.20

count_max_font_size = 12
count_min_font_size = 7

fig.canvas.draw()
renderer = fig.canvas.get_renderer()
to_data = ax_venn.transData.inverted()


def fits_inside(label, path):
    """True if the label, plus padding, lies wholly in the region."""
    extent = label.get_window_extent(renderer)
    pad_x = label_side_padding * extent.height
    pad_y = label_top_padding * extent.height

    xs = np.linspace(extent.x0 - pad_x, extent.x1 + pad_x, 12)
    ys = np.linspace(extent.y0 - pad_y, extent.y1 + pad_y, 6)
    grid_x, grid_y = np.meshgrid(xs, ys)
    corners_px = np.column_stack([grid_x.ravel(), grid_y.ravel()])

    return path.contains_points(to_data.transform(corners_px)).all()


count_regions = []

for region_id in ["100", "010", "110", "001", "101", "011", "111"]:
    patch = venn.get_patch_by_id(region_id)
    label = venn.get_label_by_id(region_id)

    if patch is not None and label is not None and label.get_text():
        count_regions.append((label, patch.get_path()))

count_font_size = count_max_font_size

while True:
    for label, _ in count_regions:
        label.set_fontsize(count_font_size)

    all_fit = all(
        fits_inside(label, path)
        for label, path in count_regions
    )

    if all_fit or count_font_size <= count_min_font_size:
        break

    count_font_size -= 0.5

print(f"Venn counts share font size {count_font_size:g}")

if not all_fit:
    print(
        "WARNING: some Venn counts touch an edge even at "
        f"{count_min_font_size} pt."
    )

#fig.text(
#   0.5, 0.515,
#    (
#        f"Pairs with at least one LLM disagreement: "
#        f"n = {n_any_disagreement:,}. "
#        f"Pairs where all 3 LLMs agree with the manual score: "
#        f"n = {n_all_agree:,}."
#    ),
#    ha="center", va="top",
#    fontsize=footnote_size,
#    color=subtitle_color
#)

# ------------------------------------------------------------
# Panel B: histograms of positive calls per participant
# ------------------------------------------------------------

add_panel_header(
    "B",
    "Distribution of Phenotypes Scored, by Scoring Method",
    (
        "Each histogram shows the number of the 124 checklist "
        "phenotypes scored as present per participant "
        f"(n = {actual_subjects:,})."
    ),
    y=0.47
)

grid = GridSpec(
    2, 2,
    figure=fig,
    left=0.11, right=0.97,
    bottom=0.065, top=0.40,
    hspace=0.38, wspace=0.12
)

hist_axes = []

for i, column in enumerate(score_cols):

    share_with = hist_axes[0] if hist_axes else None

    ax = fig.add_subplot(
        grid[i // 2, i % 2],
        sharex=share_with,
        sharey=share_with
    )

    ax.hist(
        positive_calls_by_subject[column].dropna(),
        bins=bin_edges,
        color=colors[column],
        edgecolor="white"
    )

    ax.set_title(
        method_labels[column],
        fontsize=13,
        fontweight="bold",
        pad=8
    )

    ax.set_ylim(bottom=0)

    ax.xaxis.set_major_locator(MultipleLocator(5))
    ax.yaxis.set_major_locator(MaxNLocator(integer=True))

    ax.grid(axis="y", color=grid_color, linewidth=0.8, alpha=0.8)
    ax.set_axisbelow(True)

    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

    ax.tick_params(axis="both", labelsize=11)
    ax.tick_params(axis="x", labelbottom=True)
    ax.tick_params(axis="y", labelleft=True)

    hist_axes.append(ax)

# Shared axis labels for the histogram block
fig.text(
    0.54, 0.025,
    "Phenotypes scored per participant",
    ha="center", va="center",
    fontsize=13
)

fig.text(
    0.035, 0.2325,
    "Number of participants",
    ha="center", va="center",
    rotation="vertical",
    fontsize=13
)

# ------------------------------------------------------------
# Export
# ------------------------------------------------------------

output_stem = "Supplementary_Figure_2"

fig.savefig(
    f"{output_stem}.png",
    dpi=300,
    bbox_inches="tight",
    facecolor="white"
)

fig.savefig(
    f"{output_stem}.pdf",
    bbox_inches="tight",
    facecolor="white"
)

print(f"Saved {output_stem}.png and {output_stem}.pdf")

plt.show()

In [0]:
import matplotlib.pyplot as plt
import pandas as pd

# Labels used in the publication table
table_labels = {
    "Manual_Score": "Manual annotation",
    "GPT_Score": "GPT 5.4",
    "GPT_Mini_Score": "GPT 5.4 Mini",
    "Claude_Sonnet_Score": "Claude Sonnet 4.6"
}

# Calculate the table values from subject_scores
summary = subject_scores[score_cols].describe()

table_rows = []

for column in score_cols:
    table_rows.append({
        "Scoring method": table_labels[column],
        "Subjects, n": f"{int(summary.loc['count', column]):,}",
        "Mean (SD)": (
            f"{summary.loc['mean', column]:.2f} "
            f"({summary.loc['std', column]:.2f})"
        ),
        "Median (IQR)": (
            f"{summary.loc['50%', column]:.0f} "
            f"({summary.loc['25%', column]:.0f}–"
            f"{summary.loc['75%', column]:.0f})"
        ),
        "Range": (
            f"{summary.loc['min', column]:.0f}–"
            f"{summary.loc['max', column]:.0f}"
        )
    })

table_df = pd.DataFrame(table_rows)

# Calculate the number of checklist phenotypes for the footnote
n_phenotypes = df["Phenotype"].nunique()

# ------------------------------------------------------------
# Create formatted table
# ------------------------------------------------------------

fig, ax = plt.subplots(figsize=(12, 5))
ax.axis("off")

fig.text(
    0.06,
    0.91,
    "Distribution of positive phenotype calls per subject by scoring method",
    fontsize=17,
    fontweight="bold",
    color="#202833",
    ha="left"
)

table = ax.table(
    cellText=table_df.values,
    colLabels=table_df.columns,
    colLoc="center",
    cellLoc="center",
    colWidths=[0.43, 0.14, 0.18, 0.18, 0.10],
    bbox=[0.05, 0.26, 0.90, 0.52]
)

table.auto_set_font_size(False)
table.set_fontsize(13)

header_color = "#E5EBF2"
alternate_row_color = "#F4F6F8"
text_color = "#29313A"
line_color = "#D2D8DE"
border_color = "#7F8A94"

for (row, column), cell in table.get_celld().items():

    cell.set_text_props(color=text_color)
    cell.set_edgecolor(line_color)
    cell.set_linewidth(0.8)

    if row == 0:
        cell.set_facecolor(header_color)
        cell.set_text_props(fontweight="bold")
    elif row % 2 == 0:
        cell.set_facecolor(alternate_row_color)
    else:
        cell.set_facecolor("white")

    if column == 0:
        cell.get_text().set_ha("left")
        cell.PAD = 0.05

# Add darker horizontal borders
left, bottom, width, height = 0.05, 0.26, 0.90, 0.52
n_rows = len(table_df) + 1
row_height = height / n_rows

ax.plot(
    [left, left + width],
    [bottom + height, bottom + height],
    color=border_color,
    linewidth=1.5,
    transform=ax.transAxes
)

ax.plot(
    [left, left + width],
    [bottom + height - row_height, bottom + height - row_height],
    color=border_color,
    linewidth=1.2,
    transform=ax.transAxes
)

ax.plot(
    [left, left + width],
    [bottom, bottom],
    color=border_color,
    linewidth=1.5,
    transform=ax.transAxes
)

# Footnote
fig.text(
    0.06,
    0.12,
    f"Note: Values summarize the number of the {n_phenotypes} checklist phenotypes "
    "classified as present for each subject.\n"
    "IQR, interquartile range; SD, standard deviation.",
    fontsize=10.5,
    color="#59636F",
    ha="left",
    va="top"
)

plt.tight_layout()

# Optional high-resolution export
# plt.savefig(
#     "/Workspace/Users/user/user/"
#     "positive_phenotype_calls_summary.png",
#     dpi=300,
#     bbox_inches="tight"
# )

plt.show()

display(table_df)

In [0]:
!pip install openpyxl pandas



file_path = "/Workspace/Users/user/user/UMD Enrollment Log_11.7.2025_for_Gabriel 2.xlsx"

umd_enrollment_log = pd.read_excel(file_path, dtype=str)
display(umd_enrollment_log)

In [0]:
display(
    umd_enrollment_log[["Consented By:"]]
    .drop_duplicates()
    .reset_index(drop=True)
)

In [0]:
# Specify only the values that need to be rewrittendef remap_consented_by(value):

# The below just fixed some naming inconsistencies in spelling/capitalization/punctuation. Removed to protect clinical coordinator identities. 
consented_by_remapping = {
    "",
    "": pd.NA
}

umd_enrollment_log["Consented By:"] = (
    umd_enrollment_log["Consented By:"]
    .astype("string")
    # Remove everything through "reconsent" and the following whitespace
    .str.replace(
        r"(?is)^.*\breconsent\s+",
        "",
        regex=True
    )
    .str.strip()
    # Apply the remaining custom remappings
    .replace(consented_by_remapping)
)

# Display each unique value and its count after remapping
display(
    umd_enrollment_log["Consented By:"]
    .value_counts(dropna=False)
    .rename_axis("Consented By:")
    .reset_index(name="Count")
)

In [0]:
# Quality check: confirm each Subject ID maps to exactly one unique
# 'Consented By:' value after the merge (i.e. no subject has conflicting values)

consented_per_subject = umd_enrollment_log.dropna(subset=['Consented By:']).groupby('Subject ID')['Consented By:'].nunique()

n_multi = (consented_per_subject > 1).sum()
print(f"Subjects with more than one distinct 'Consented By:' value: {n_multi}")

if n_multi > 0:
    problem_subjects = consented_per_subject[consented_per_subject > 1].index
    print("Subjects with conflicting values:")
    display(
        umd_enrollment_log[umd_enrollment_log['Subject ID'].isin(problem_subjects)][['Subject ID', 'Consented By:']]
        .drop_duplicates()
        .sort_values('Subject ID')
    )
else:
    print("Every subject has exactly one unique 'Consented By:' value. Merge looks clean.")

# Subjects with no matching 'Consented By:' value after the merge (i.e. didn't match
# to anything in umd_enrollment_log, or matched but the field itself was blank)

subjects_no_consent = umd_enrollment_log[umd_enrollment_log['Consented By:'].isna()][['Subject ID']].drop_duplicates().sort_values('Subject ID')

print(f"Subjects with no 'Consented By:' value: {len(subjects_no_consent)}")
display(subjects_no_consent)

In [0]:
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

# Join 'Consented By:' from the enrollment log onto combined_df_full_info, keyed on Subject ID
df = combined_df_full_info.merge(
    umd_enrollment_log[['Subject ID', 'Consented By:']],
    on='Subject ID',
    how='left'
)

score_cols = ['Manual_Score', 'GPT_Score', 'GPT_Mini_Score', 'Claude_Sonnet_Score']
colors = {'Manual_Score': '#D4B896', 'GPT_Score': '#4C9BE8', 'GPT_Mini_Score': '#57C4A7', 'Claude_Sonnet_Score': '#E87B4C'}
labels = {'Manual_Score': 'Manual', 'GPT_Score': 'GPT 5.4', 'GPT_Mini_Score': 'GPT 5.4 Mini', 'Claude_Sonnet_Score': 'Claude Sonnet 4.6'}

print(f"Rows with no matching 'Consented By:' after join: {df['Consented By:'].isna().sum()} / {len(df)}")

subject_level = df.dropna(subset=['Consented By:']).drop_duplicates(subset='Subject ID').copy()
notes_by_cat = subject_level['Consented By:'].value_counts().sort_values(ascending=True)  # ascending for barh (largest on top)

fig1, ax1 = plt.subplots(figsize=(9, max(4, 0.4 * len(notes_by_cat))))
ax1.barh(notes_by_cat.index.astype(str), notes_by_cat.values, color='#4C9BE8', edgecolor='white')
ax1.set_ylabel('Consented by', fontsize=12)
ax1.set_xlabel('Number of notes', fontsize=12)
ax1.set_title('Distribution of notes by consented by', fontsize=15, pad=15)
ax1.set_xlim(left=0)
ax1.spines['top'].set_visible(False)
ax1.spines['right'].set_visible(False)
for i, v in enumerate(notes_by_cat.values):
    ax1.text(v + notes_by_cat.max() * 0.01, i, str(v), va='center', ha='left', fontsize=10)
plt.tight_layout()
plt.show()

subject_scores = df.dropna(subset=['Consented By:']).groupby('Subject ID').agg(
    **{'Consented By:': ('Consented By:', 'first')},
    Manual_Score=('Manual_Score', 'sum'), GPT_Score=('GPT_Score', 'sum'),
    GPT_Mini_Score=('GPT_Mini_Score', 'sum'), Claude_Sonnet_Score=('Claude_Sonnet_Score', 'sum'),
).reset_index()
avg_by_cat = subject_scores.groupby('Consented By:')[score_cols].mean().reindex(notes_by_cat.index)  # same sorted order

y = np.arange(len(avg_by_cat))
height = 0.2
fig2, ax2 = plt.subplots(figsize=(11, max(5, 0.6 * len(avg_by_cat))))
for i, col in enumerate(score_cols):
    ax2.barh(y + (i - 1.5) * height, avg_by_cat[col], height, label=labels[col], color=colors[col])

ax2.set_ylabel('Consented by', fontsize=12)
ax2.set_xlabel('Average phenotypes scored per note', fontsize=12)
ax2.set_title('Average phenotypes scored per note, by consented by', fontsize=15, pad=15)
ax2.set_yticks(y)
ax2.set_yticklabels(avg_by_cat.index.astype(str))
ax2.set_xlim(left=0)
ax2.legend(loc='upper center', bbox_to_anchor=(0.5, -0.08), ncol=4, frameon=False, fontsize=10)
ax2.spines['top'].set_visible(False)
ax2.spines['right'].set_visible(False)
plt.tight_layout()
plt.show()

display(notes_by_cat.sort_values(ascending=False))
display(avg_by_cat)

In [0]:
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

df = combined_df_full_info.copy()

model_cols = ['GPT_Score', 'GPT_Mini_Score', 'Claude_Sonnet_Score']
colors = {'GPT_Score': '#4C9BE8', 'GPT_Mini_Score': '#57C4A7', 'Claude_Sonnet_Score': '#E87B4C'}
labels = {'GPT_Score': 'GPT 5.4', 'GPT_Mini_Score': 'GPT 5.4 Mini', 'Claude_Sonnet_Score': 'Claude Sonnet 4.6'}

# For each row, flag disagreement between each model and Manual
for col in model_cols:
    df[f'{col}_disagree'] = (df[col] != df['Manual_Score']).astype(int)

# Disagreement rate per phenotype, per model
disagreement_by_phenotype = df.groupby('Phenotype')[[f'{col}_disagree' for col in model_cols]].mean()
disagreement_by_phenotype.columns = model_cols  # rename back for convenience

# Sort by average disagreement across the three models, descending (most-disagreed at top)
disagreement_by_phenotype['avg_disagree'] = disagreement_by_phenotype[model_cols].mean(axis=1)
disagreement_by_phenotype = disagreement_by_phenotype.sort_values('avg_disagree', ascending=True)  # ascending for barh (highest on top)
disagreement_by_phenotype = disagreement_by_phenotype.drop(columns='avg_disagree')

y = np.arange(len(disagreement_by_phenotype))
height = 0.25

fig, ax = plt.subplots(figsize=(11, max(6, 0.35 * len(disagreement_by_phenotype))))
for i, col in enumerate(model_cols):
    ax.barh(y + (i - 1) * height, disagreement_by_phenotype[col], height,
            label=labels[col], color=colors[col])

ax.set_yticks(y)
ax.set_yticklabels(disagreement_by_phenotype.index.astype(str))
ax.set_xlabel('Disagreement rate with Manual score', fontsize=12)
ax.set_ylabel('Phenotype', fontsize=12)
ax.set_title('Disagreement with Manual scoring, by phenotype', fontsize=15, pad=15)
ax.set_xlim(0, 1)
ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x:.0%}'))
ax.legend(loc='upper center', bbox_to_anchor=(0.5, -0.06), ncol=3, frameon=False, fontsize=10)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

plt.tight_layout()
plt.show()

display(disagreement_by_phenotype.sort_values(disagreement_by_phenotype.columns[0], ascending=False))

In [0]:
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

df = combined_df_full_info.copy()

score_cols = ['Manual_Score', 'GPT_Score', 'GPT_Mini_Score', 'Claude_Sonnet_Score']
model_cols = ['GPT_Score', 'GPT_Mini_Score', 'Claude_Sonnet_Score']
colors = {'Manual_Score': '#D4B896', 'GPT_Score': '#4C9BE8', 'GPT_Mini_Score': '#57C4A7', 'Claude_Sonnet_Score': '#E87B4C'}
labels = {'Manual_Score': 'Manual', 'GPT_Score': 'GPT 5.4', 'GPT_Mini_Score': 'GPT 5.4 Mini', 'Claude_Sonnet_Score': 'Claude Sonnet 4.6'}

# Row-level disagreement flags: each model vs Manual
for col in model_cols:
    df[f'{col}_disagree'] = (df[col] != df['Manual_Score']).astype(int)

# Average score (rate scored positive) per phenotype, per model -- for the bars
score_by_phenotype = df.groupby('Phenotype')[score_cols].mean()

# Disagreement rate per phenotype, per model -- used only for sorting
disagreement_by_phenotype = df.groupby('Phenotype')[[f'{col}_disagree' for col in model_cols]].mean()

# Sort by the single highest-disagreeing model per phenotype (max across the three models)
sort_key = disagreement_by_phenotype.max(axis=1)
score_by_phenotype = score_by_phenotype.reindex(sort_key.sort_values(ascending=True).index)  # ascending -> highest ends up at top of barh

y = np.arange(len(score_by_phenotype))
height = 0.2

fig, ax = plt.subplots(figsize=(11, max(6, 0.4 * len(score_by_phenotype))))
for i, col in enumerate(score_cols):
    ax.barh(y + (i - 1.5) * height, score_by_phenotype[col], height,
            label=labels[col], color=colors[col])

ax.set_yticks(y)
ax.set_yticklabels(score_by_phenotype.index.astype(str))
ax.set_xlabel('Rate scored positive', fontsize=12)
ax.set_ylabel('Phenotype', fontsize=12)
ax.set_title('Phenotype scoring rate, by model\n(sorted by highest-disagreeing model vs Manual)', fontsize=15, pad=15)
ax.set_xlim(0, 1)
ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x:.0%}'))
ax.legend(loc='upper center', bbox_to_anchor=(0.5, -0.06), ncol=4, frameon=False, fontsize=10)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

plt.tight_layout()
plt.show()

# Show scores alongside each model's disagreement rate, for reference
result = score_by_phenotype.copy()
for col in model_cols:
    result[f'{col}_disagreement'] = disagreement_by_phenotype.loc[result.index, f'{col}_disagree']
display(result.iloc[::-1])  # highest-disagreement phenotype first

In [0]:
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

df = combined_df_full_info.copy()

score_cols = ['Manual_Score', 'GPT_Score', 'GPT_Mini_Score', 'Claude_Sonnet_Score']
model_cols = ['GPT_Score', 'GPT_Mini_Score', 'Claude_Sonnet_Score']
colors = {'Manual_Score': '#D4B896', 'GPT_Score': '#4C9BE8', 'GPT_Mini_Score': '#57C4A7', 'Claude_Sonnet_Score': '#E87B4C'}
labels = {'Manual_Score': 'Manual', 'GPT_Score': 'GPT 5.4', 'GPT_Mini_Score': 'GPT 5.4 Mini', 'Claude_Sonnet_Score': 'Claude Sonnet 4.6'}

# Row-level disagreement flags: each model vs Manual
for col in model_cols:
    df[f'{col}_disagree'] = (df[col] != df['Manual_Score']).astype(int)

# Total count scored positive per phenotype, per model -- for the bars
score_by_phenotype = df.groupby('Phenotype')[score_cols].sum()

# Total disagreement count per phenotype, per model -- used only for sorting
disagreement_by_phenotype = df.groupby('Phenotype')[[f'{col}_disagree' for col in model_cols]].sum()

# Sort by the single highest-disagreeing model per phenotype (max across the three models)
sort_key = disagreement_by_phenotype.max(axis=1)
score_by_phenotype = score_by_phenotype.reindex(sort_key.sort_values(ascending=True).index)  # ascending -> highest ends up at top of barh

y = np.arange(len(score_by_phenotype))
height = 0.2

fig, ax = plt.subplots(figsize=(11, max(6, 0.4 * len(score_by_phenotype))))
for i, col in enumerate(score_cols):
    ax.barh(y + (i - 1.5) * height, score_by_phenotype[col], height,
            label=labels[col], color=colors[col])

ax.set_yticks(y)
ax.set_yticklabels(score_by_phenotype.index.astype(str))
ax.set_xlabel('Number of subjects scored positive', fontsize=12)
ax.set_ylabel('Phenotype', fontsize=12)
ax.set_title('Phenotype scoring counts, by model\n(sorted by highest-disagreeing model vs Manual)', fontsize=15, pad=15)
ax.set_xlim(left=0)
ax.legend(loc='upper center', bbox_to_anchor=(0.5, -0.06), ncol=4, frameon=False, fontsize=10)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

plt.tight_layout()
plt.show()

# Show scores alongside each model's disagreement count, for reference
result = score_by_phenotype.copy()
for col in model_cols:
    result[f'{col}_disagreement'] = disagreement_by_phenotype.loc[result.index, f'{col}_disagree']
display(result.iloc[::-1])  # highest-disagreement phenotype first

In [0]:
# Count of subjects where Manual scored 'Autism Spectrum Disorder (ASD)' as positive

asd_manual_count = df[
    (df['Phenotype'] == 'Autism Spectrum Disorder (ASD)') &
    (df['Manual_Score'] == 1)
]['Subject ID'].nunique()

total_asd_rows = (df['Phenotype'] == 'Autism Spectrum Disorder (ASD)').sum()

print(f"Subjects with 'Autism Spectrum Disorder (ASD)' scored positive by Manual: {asd_manual_count}")
print(f"Total subjects with an ASD phenotype row: {total_asd_rows}")

In [0]:
import requests
import json
import os

output_folder = "/Workspace/Users/user/user/"
hp_json_path = output_folder + "hp.json"

# One-time download of the full HPO ontology (official OBO Foundry release).
# Only run this once -- after this, everything is done locally from the saved file.
if not os.path.exists(hp_json_path):
    url = "http://purl.obolibrary.org/obo/hp.json"
    resp = requests.get(url, timeout=60)
    resp.raise_for_status()
    with open(hp_json_path, 'wb') as f:
        f.write(resp.content)
    print(f"Downloaded HPO ontology to {hp_json_path} ({len(resp.content)/1e6:.1f} MB)")
else:
    print(f"HPO ontology already downloaded at {hp_json_path}, skipping.")

In [0]:
# ------------------------------------------------------------
# Parse the local hp.json file -- no network calls below this point
# ------------------------------------------------------------
with open(hp_json_path, 'r') as f:
    hpo_data = json.load(f)

graph = hpo_data['graphs'][0]

def iri_to_short(iri):
    # e.g. "http://purl.obolibrary.org/obo/HP_0001250" -> "HP:0001250"
    frag = iri.rsplit('/', 1)[-1]
    return frag.replace('_', ':', 1)

def short_to_iri(short_id):
    # e.g. "HP:0001250" -> "http://purl.obolibrary.org/obo/HP_0001250"
    return "http://purl.obolibrary.org/obo/" + short_id.replace(':', '_', 1)

# id (short form) -> label
id_to_label = {
    iri_to_short(n['id']): n.get('lbl', '')
    for n in graph['nodes'] if 'lbl' in n
}

# child (short form) -> set of parent ids (short form), from is_a edges only
parent_map = {}
for e in graph['edges']:
    if e.get('pred') in ('is_a', 'http://www.w3.org/2000/01/rdf-schema#subClassOf'):
        child = iri_to_short(e['sub'])
        parent = iri_to_short(e['obj'])
        parent_map.setdefault(child, set()).add(parent)

# Top-level categories = direct children of HP:0000118 (Phenotypic abnormality)
top_level_ids = {
    child for child, parents in parent_map.items() if 'HP:0000118' in parents
}
print(f"Top-level HPO categories found: {len(top_level_ids)}")
for tid in sorted(top_level_ids):
    print(f"  {tid}: {id_to_label.get(tid)}")

def find_top_level_category(hpo_id, parent_map, top_level_ids, id_to_label):
    """BFS up the parent hierarchy until a top-level category is hit."""
    if hpo_id in top_level_ids:
        return id_to_label.get(hpo_id, hpo_id)
    visited = set()
    frontier = [hpo_id]
    while frontier:
        next_frontier = []
        for node in frontier:
            if node in visited:
                continue
            visited.add(node)
            for parent in parent_map.get(node, []):
                if parent in top_level_ids:
                    return id_to_label.get(parent, parent)
                next_frontier.append(parent)
        frontier = next_frontier
    return 'Unmapped / Other'

# Build the HPO ID -> category lookup for just the phenotypes you actually use
phenotype_to_hpo = (
    checklist_filtered[['Term from checklist', 'HPO term (ID)']]
    .dropna()
    .drop_duplicates(subset='Term from checklist')
    .set_index('Term from checklist')['HPO term (ID)']
    .to_dict()
)

hpo_to_category = {
    hpo_id: find_top_level_category(hpo_id, parent_map, top_level_ids, id_to_label)
    for hpo_id in set(phenotype_to_hpo.values())
}

print(f"\nMapped {len(hpo_to_category)} unique HPO IDs to categories.")
display(pd.DataFrame(hpo_to_category.items(), columns=['HPO_ID', 'HPO_Category']))

In [0]:
# ============================================================
# Chart 1: Distribution of phenotypes by parent (HPO) ontology category
# ============================================================
import matplotlib.pyplot as plt
import numpy as np

# Count distinct phenotypes per HPO category (using the mapping built earlier)
phenotype_category_df = pd.DataFrame([
    {'Phenotype': pheno, 'HPO_Category': hpo_to_category.get(hpo_id, 'Unmapped / Other')}
    for pheno, hpo_id in phenotype_to_hpo.items()
])
# Include any phenotypes that had no HPO ID match at all in checklist_filtered
unmapped_phenos = set(df['Phenotype'].unique()) - set(phenotype_to_hpo.keys())
if unmapped_phenos:
    phenotype_category_df = pd.concat([
        phenotype_category_df,
        pd.DataFrame({'Phenotype': list(unmapped_phenos), 'HPO_Category': 'Unmapped / Other'})
    ], ignore_index=True)

phenotypes_by_category = phenotype_category_df['HPO_Category'].value_counts().sort_values(ascending=True)

fig1, ax1 = plt.subplots(figsize=(10, max(5, 0.4 * len(phenotypes_by_category))))
ax1.barh(phenotypes_by_category.index.astype(str), phenotypes_by_category.values, color='#4C9BE8', edgecolor='white')
ax1.set_xlabel('Number of distinct phenotypes', fontsize=12)
ax1.set_ylabel('HPO parent category', fontsize=12)
ax1.set_title('Distribution of phenotypes by parent HPO ontology category', fontsize=15, pad=15)
ax1.set_xlim(left=0)
ax1.spines['top'].set_visible(False)
ax1.spines['right'].set_visible(False)
for i, v in enumerate(phenotypes_by_category.values):
    ax1.text(v + phenotypes_by_category.max() * 0.01, i, str(v), va='center', ha='left', fontsize=10)

fig1.text(0.5, -0.03, "Categories determined using HPO (Human Phenotype Ontology) codes: each phenotype's parent category "
                       "is its top-level ancestor under HP:0000118.",
          ha='center', va='top', fontsize=9, style='italic', color='#555555')

plt.tight_layout()
plt.show()

display(phenotypes_by_category.sort_values(ascending=False))

In [0]:
# ============================================================
# Chart 2: Phenotypes scored by Manual/each model, by parent HPO ontology category
# ============================================================
import matplotlib.pyplot as plt
import numpy as np

df = combined_df_full_info.copy()

score_cols = ['Manual_Score', 'GPT_Score', 'GPT_Mini_Score', 'Claude_Sonnet_Score']
colors = {'Manual_Score': '#D4B896', 'GPT_Score': '#4C9BE8', 'GPT_Mini_Score': '#57C4A7', 'Claude_Sonnet_Score': '#E87B4C'}
labels = {'Manual_Score': 'Manual', 'GPT_Score': 'GPT 5.4', 'GPT_Mini_Score': 'GPT 5.4 Mini', 'Claude_Sonnet_Score': 'Claude Sonnet 4.6'}

# Re-attach HPO_Category using the mappings built earlier (phenotype_to_hpo, hpo_to_category
# must still be in memory from the HPO offline-parsing step)
df['HPO_ID'] = df['Phenotype'].map(phenotype_to_hpo)
df['HPO_Category'] = df['HPO_ID'].map(hpo_to_category)
df['HPO_Category'] = df['HPO_Category'].fillna('Unmapped / Other')

scored_by_category = df.groupby('HPO_Category')[score_cols].sum()

# Order categories the same way as Chart 1 (by total phenotype count) for consistency
scored_by_category = scored_by_category.reindex(phenotypes_by_category.index).dropna(how='all')

y = np.arange(len(scored_by_category))
height = 0.2

fig2, ax2 = plt.subplots(figsize=(11, max(6, 0.45 * len(scored_by_category))))
for i, col in enumerate(score_cols):
    ax2.barh(y + (i - 1.5) * height, scored_by_category[col], height,
             label=labels[col], color=colors[col])

ax2.set_yticks(y)
ax2.set_yticklabels(scored_by_category.index.astype(str))
ax2.set_xlabel('Number of subjects scored positive', fontsize=12)
ax2.set_ylabel('HPO parent category', fontsize=12)
ax2.set_title('Phenotypes scored by Manual and each model, by parent HPO ontology category', fontsize=15, pad=15)
ax2.set_xlim(left=0)
ax2.legend(loc='upper center', bbox_to_anchor=(0.5, -0.05), ncol=4, frameon=False, fontsize=10)
ax2.spines['top'].set_visible(False)
ax2.spines['right'].set_visible(False)

fig2.text(0.5, -0.02, "Categories determined using HPO (Human Phenotype Ontology) codes: each phenotype's parent category "
                       "is its top-level ancestor under HP:0000118.",
          ha='center', va='top', fontsize=9, style='italic', color='#555555')

plt.tight_layout()
plt.show()

display(scored_by_category)

In [0]:
def get_all_top_level_categories(hpo_id, parent_map, top_level_ids):
    visited = set()
    found = set()
    frontier = [hpo_id]
    while frontier:
        next_frontier = []
        for node in frontier:
            if node in visited:
                continue
            visited.add(node)
            if node in top_level_ids:
                found.add(node)
                continue  # don't traverse past a top-level hit
            for parent in parent_map.get(node, []):
                next_frontier.append(parent)
        frontier = next_frontier
    return found

multi_parent_report = {
    hpo_id: get_all_top_level_categories(hpo_id, parent_map, top_level_ids)
    for hpo_id in set(phenotype_to_hpo.values())
}
multi_parent_terms = {k: v for k, v in multi_parent_report.items() if len(v) > 1}
print(f"Phenotypes with more than one top-level HPO category: {len(multi_parent_terms)}")
for hpo_id, cats in multi_parent_terms.items():
    print(f"  {hpo_id}: {[id_to_label.get(c, c) for c in cats]}")

In [0]:
# ============================================================
# Calculate minimum ontology depth from HP:0000118
# for every phenotype
# ============================================================

from collections import deque
import pandas as pd


def compute_min_depth(
    hpo_id,
    parent_map,
    root="HP:0000118"
):
    """
    Calculate the shortest number of is_a steps from an HPO
    term upward to HP:0000118.

    Returns None if no path to the root is found.
    """

    if hpo_id == root:
        return 0

    visited = {hpo_id}
    queue = deque([
        (hpo_id, 0)
    ])

    while queue:
        node, current_depth = queue.popleft()

        for parent in parent_map.get(node, []):

            if parent == root:
                return current_depth + 1

            if parent not in visited:
                visited.add(parent)
                queue.append(
                    (parent, current_depth + 1)
                )

    return None


# ------------------------------------------------------------
# Calculate minimum depth for every mapped phenotype
# ------------------------------------------------------------

depth_records = []

for phenotype, hpo_id in phenotype_to_hpo.items():

    minimum_depth = compute_min_depth(
        hpo_id=hpo_id,
        parent_map=parent_map
    )

    depth_records.append({
        "Phenotype": phenotype,
        "HPO_ID": hpo_id,
        "HPO_Label": id_to_label.get(hpo_id, ""),
        "HPO_Min_Depth": minimum_depth
    })


# ------------------------------------------------------------
# Create and display the results dataframe
# ------------------------------------------------------------

depth_df = (
    pd.DataFrame(depth_records)
    .sort_values(
        "HPO_Min_Depth",
        na_position="last"
    )
    .reset_index(drop=True)
)

print(
    f"Total phenotypes checked: "
    f"{len(depth_df):,}"
)

print(
    "Unique minimum-depth values found:",
    sorted(
        depth_df["HPO_Min_Depth"]
        .dropna()
        .unique()
    )
)

print("\nMinimum-depth distribution:")

print(
    depth_df["HPO_Min_Depth"]
    .value_counts()
    .sort_index()
)

# ------------------------------------------------------------
# Report phenotypes without a path to HP:0000118
# ------------------------------------------------------------

no_path = depth_df[
    depth_df["HPO_Min_Depth"].isna()
]

if not no_path.empty:
    print(
        f"\nWARNING: {len(no_path):,} phenotype(s) had no "
        "traceable path to HP:0000118. Check these IDs:"
    )

    display(no_path)
else:
    print(
        "\nAll mapped phenotypes have a traceable path "
        "to HP:0000118."
    )

display(depth_df)

In [0]:
# ============================================================
# Visualize the minimum ontology-depth distribution
# ============================================================

import matplotlib.pyplot as plt

depth_counts = (
    depth_df["HPO_Min_Depth"]
    .dropna()
    .astype(int)
    .value_counts()
    .sort_index()
)

if depth_counts.empty:
    raise ValueError(
        "No valid HPO minimum-depth values are available to plot."
    )

fig, ax = plt.subplots(figsize=(9, 5))

ax.bar(
    depth_counts.index.astype(str),
    depth_counts.values,
    color="#4C9BE8",
    edgecolor="white"
)

ax.set_xlabel(
    "Minimum ontology depth (steps from HP:0000118)",
    fontsize=12
)

ax.set_ylabel(
    "Number of phenotypes",
    fontsize=12
)

ax.set_title(
    "Distribution of HPO Minimum Ontology Depth",
    fontsize=15,
    pad=15
)

ax.set_ylim(bottom=0)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

label_padding = max(
    depth_counts.max() * 0.01,
    0.1
)

for position, count in enumerate(depth_counts.values):
    ax.text(
        position,
        count + label_padding,
        f"{int(count):,}",
        ha="center",
        va="bottom",
        fontsize=10
    )

plt.tight_layout()
plt.show()

display(
    depth_counts.to_frame(
        name="Number of phenotypes"
    )
)

In [0]:
# ============================================================
# Calculate minimum ontology depth from HP:0000118
# for every phenotype
# ============================================================

from collections import deque
import pandas as pd


def compute_min_depth(
    hpo_id,
    parent_map,
    root="HP:0000118"
):
    """
    Calculate the shortest number of is_a steps from an HPO
    term upward to HP:0000118.

    Returns None if no path to the root is found.
    """

    if hpo_id == root:
        return 0

    visited = {hpo_id}
    queue = deque([
        (hpo_id, 0)
    ])

    while queue:
        node, current_depth = queue.popleft()

        for parent in parent_map.get(node, []):

            if parent == root:
                return current_depth + 1

            if parent not in visited:
                visited.add(parent)
                queue.append(
                    (parent, current_depth + 1)
                )

    return None


# ------------------------------------------------------------
# Calculate minimum depth for every mapped phenotype
# ------------------------------------------------------------

depth_records = []

for phenotype, hpo_id in phenotype_to_hpo.items():

    minimum_depth = compute_min_depth(
        hpo_id=hpo_id,
        parent_map=parent_map
    )

    depth_records.append({
        "Phenotype": phenotype,
        "HPO_ID": hpo_id,
        "HPO_Label": id_to_label.get(hpo_id, ""),
        "HPO_Min_Depth": minimum_depth
    })


# ------------------------------------------------------------
# Create and display the results dataframe
# ------------------------------------------------------------

depth_df = (
    pd.DataFrame(depth_records)
    .sort_values(
        "HPO_Min_Depth",
        na_position="last"
    )
    .reset_index(drop=True)
)

print(
    f"Total phenotypes checked: "
    f"{len(depth_df):,}"
)

print(
    "Unique minimum-depth values found:",
    sorted(
        depth_df["HPO_Min_Depth"]
        .dropna()
        .unique()
    )
)

print("\nMinimum-depth distribution:")

print(
    depth_df["HPO_Min_Depth"]
    .value_counts()
    .sort_index()
)

# ------------------------------------------------------------
# Report phenotypes without a path to HP:0000118
# ------------------------------------------------------------

no_path = depth_df[
    depth_df["HPO_Min_Depth"].isna()
]

if not no_path.empty:
    print(
        f"\nWARNING: {len(no_path):,} phenotype(s) had no "
        "traceable path to HP:0000118. Check these IDs:"
    )

    display(no_path)
else:
    print(
        "\nAll mapped phenotypes have a traceable path "
        "to HP:0000118."
    )

display(depth_df)

In [0]:
# ============================================================
# Add depth-2 HPO ancestors to depth_df
# ============================================================

from functools import lru_cache
import pandas as pd


@lru_cache(maxsize=None)
def cached_min_depth(hpo_id):
    return compute_min_depth(
        hpo_id=hpo_id,
        parent_map=parent_map,
        root="HP:0000118"
    )


def find_depth2_ancestors(hpo_id):
    """
    Find every ancestor whose minimum depth from HP:0000118
    equals 2.

    Multiple ancestors are retained because HPO is a DAG and
    terms can have multiple parents.
    """

    visited = set()
    frontier = [hpo_id]
    depth2_ancestors = set()

    while frontier:
        node = frontier.pop()

        if node in visited:
            continue

        visited.add(node)

        node_depth = cached_min_depth(node)

        if node_depth == 2:
            depth2_ancestors.add(node)
            continue

        frontier.extend(
            parent_map.get(node, [])
        )

    return sorted(depth2_ancestors)


depth2_records = []

for _, row in depth_df.iterrows():

    phenotype = row["Phenotype"]
    hpo_id = row["HPO_ID"]

    depth2_ids = find_depth2_ancestors(hpo_id)

    depth2_labels = [
        id_to_label.get(ancestor_id, ancestor_id)
        for ancestor_id in depth2_ids
    ]

    if len(depth2_ids) == 0:
        ancestor_value = None
        label_value = "Unmapped / Depth below 2"

    elif len(depth2_ids) == 1:
        ancestor_value = depth2_ids[0]
        label_value = depth2_labels[0]

    else:
        # Retain all valid depth-2 ancestors instead of
        # selecting one arbitrarily
        ancestor_value = " | ".join(depth2_ids)
        label_value = " / ".join(depth2_labels)

    depth2_records.append({
        "Phenotype": phenotype,
        "Depth2_Ancestor": ancestor_value,
        "Depth2_Label": label_value,
        "Depth2_Ancestor_Count": len(depth2_ids)
    })

depth2_mapping_df = pd.DataFrame(
    depth2_records
)

# Remove prior versions if this cell is rerun
depth_df = depth_df.drop(
    columns=[
        "Depth2_Ancestor",
        "Depth2_Label",
        "Depth2_Ancestor_Count"
    ],
    errors="ignore"
)

depth_df = depth_df.merge(
    depth2_mapping_df,
    on="Phenotype",
    how="left",
    validate="one_to_one"
)

print(
    "Phenotypes with one depth-2 ancestor:",
    f"{(depth_df['Depth2_Ancestor_Count'] == 1).sum():,}"
)

print(
    "Phenotypes with multiple depth-2 ancestors:",
    f"{(depth_df['Depth2_Ancestor_Count'] > 1).sum():,}"
)

print(
    "Phenotypes without a depth-2 ancestor:",
    f"{(depth_df['Depth2_Ancestor_Count'] == 0).sum():,}"
)

display(depth_df)

In [0]:
# ============================================================
# Distribution of phenotypes by depth-2 HPO ontology bucket
# ============================================================

import matplotlib.pyplot as plt
import pandas as pd

phenotype_depth2_df = (
    depth_df[
        [
            "Phenotype",
            "HPO_ID",
            "Depth2_Ancestor",
            "Depth2_Label",
            "Depth2_Ancestor_Count"
        ]
    ]
    .drop_duplicates(subset="Phenotype")
    .copy()
)

phenotypes_by_depth2 = (
    phenotype_depth2_df["Depth2_Label"]
    .fillna("Unmapped / Depth below 2")
    .value_counts()
    .sort_values(ascending=True)
)

if phenotypes_by_depth2.empty:
    raise ValueError(
        "No depth-2 HPO categories were available to plot."
    )

fig1, ax1 = plt.subplots(
    figsize=(
        10,
        max(
            6,
            0.3 * len(phenotypes_by_depth2)
        )
    )
)

ax1.barh(
    phenotypes_by_depth2.index.astype(str),
    phenotypes_by_depth2.values,
    color="#4C9BE8",
    edgecolor="white"
)

ax1.set_xlabel(
    "Number of distinct phenotypes",
    fontsize=12
)

ax1.set_ylabel(
    "Depth-2 HPO ontology bucket",
    fontsize=12
)

ax1.set_title(
    "Distribution of Phenotypes by Depth-2 HPO Ontology Category",
    fontsize=15,
    pad=15
)

ax1.set_xlim(left=0)
ax1.spines["top"].set_visible(False)
ax1.spines["right"].set_visible(False)

label_padding = max(
    phenotypes_by_depth2.max() * 0.01,
    0.1
)

for position, count in enumerate(
    phenotypes_by_depth2.values
):
    ax1.text(
        count + label_padding,
        position,
        f"{int(count):,}",
        va="center",
        ha="left",
        fontsize=9
    )

fig1.text(
    0.5,
    -0.03,
    (
        "Each phenotype was rolled up to its ancestor at "
        "minimum depth 2 from HP:0000118. Phenotypes with "
        "multiple valid depth-2 ancestors are shown as combined "
        "categories rather than assigned arbitrarily."
    ),
    ha="center",
    va="top",
    fontsize=9,
    style="italic",
    color="#555555"
)

plt.tight_layout()
plt.show()

display(
    phenotypes_by_depth2
    .sort_values(ascending=False)
    .to_frame(name="Number of phenotypes")
)

In [0]:
# ============================================================
# Chart 1: Distribution of phenotypes by depth-2 HPO ontology bucket
# ============================================================
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# Build phenotype -> depth-2 ancestor mapping (reuses depth_df from the previous step)
phenotype_depth2_df = depth_df[['Phenotype', 'HPO_ID', 'Depth2_Ancestor', 'Depth2_Label']].copy()

phenotypes_by_depth2 = phenotype_depth2_df['Depth2_Label'].value_counts().sort_values(ascending=True)

fig1, ax1 = plt.subplots(figsize=(10, max(6, 0.3 * len(phenotypes_by_depth2))))
ax1.barh(phenotypes_by_depth2.index.astype(str), phenotypes_by_depth2.values, color='#4C9BE8', edgecolor='white')
ax1.set_xlabel('Number of distinct phenotypes', fontsize=12)
ax1.set_ylabel('Depth-2 HPO ontology bucket', fontsize=12)
ax1.set_title('Distribution of phenotypes by depth-2 HPO ontology category', fontsize=15, pad=15)
ax1.set_xlim(left=0)
ax1.spines['top'].set_visible(False)
ax1.spines['right'].set_visible(False)
for i, v in enumerate(phenotypes_by_depth2.values):
    ax1.text(v + phenotypes_by_depth2.max() * 0.01, i, str(v), va='center', ha='left', fontsize=9)

fig1.text(0.5, -0.02, "Categories determined using HPO (Human Phenotype Ontology) codes: each phenotype was rolled up "
                       "to its ancestor at depth 2 from HP:0000118, so all phenotypes are compared at the same ontology level.",
          ha='center', va='top', fontsize=9, style='italic', color='#555555')

plt.tight_layout()
plt.show()

display(phenotypes_by_depth2.sort_values(ascending=False))

In [0]:
# ============================================================
# Chart 2: Phenotypes scored by Manual/each model, by depth-2 HPO ontology bucket
# ============================================================
import matplotlib.pyplot as plt
import numpy as np

df = combined_df_full_info.copy()

score_cols = ['Manual_Score', 'GPT_Score', 'GPT_Mini_Score', 'Claude_Sonnet_Score']
colors = {'Manual_Score': '#D4B896', 'GPT_Score': '#4C9BE8', 'GPT_Mini_Score': '#57C4A7', 'Claude_Sonnet_Score': '#E87B4C'}
labels = {'Manual_Score': 'Manual', 'GPT_Score': 'GPT 5.4', 'GPT_Mini_Score': 'GPT 5.4 Mini', 'Claude_Sonnet_Score': 'Claude Sonnet 4.6'}

# Map each row's Phenotype -> depth-2 ancestor label, using the lookup built earlier
phenotype_to_depth2_label = dict(zip(phenotype_depth2_df['Phenotype'], phenotype_depth2_df['Depth2_Label']))
df['Depth2_Category'] = df['Phenotype'].map(phenotype_to_depth2_label)
df['Depth2_Category'] = df['Depth2_Category'].fillna('Unmapped / Other')

scored_by_depth2 = df.groupby('Depth2_Category')[score_cols].sum()

# Order to match Chart 1 for consistency
scored_by_depth2 = scored_by_depth2.reindex(phenotypes_by_depth2.index).dropna(how='all')

y = np.arange(len(scored_by_depth2))
height = 0.2

fig2, ax2 = plt.subplots(figsize=(11, max(8, 0.35 * len(scored_by_depth2))))
for i, col in enumerate(score_cols):
    ax2.barh(y + (i - 1.5) * height, scored_by_depth2[col], height,
             label=labels[col], color=colors[col])

ax2.set_yticks(y)
ax2.set_yticklabels(scored_by_depth2.index.astype(str))
ax2.set_xlabel('Number of subjects scored positive', fontsize=12)
ax2.set_ylabel('Depth-2 HPO ontology bucket', fontsize=12)
ax2.set_title('Phenotypes scored by Manual and each model, by depth-2 HPO ontology category', fontsize=15, pad=15)
ax2.set_xlim(left=0)
ax2.legend(loc='upper center', bbox_to_anchor=(0.5, -0.04), ncol=4, frameon=False, fontsize=10)
ax2.spines['top'].set_visible(False)
ax2.spines['right'].set_visible(False)

fig2.text(0.5, -0.01, "Categories determined using HPO (Human Phenotype Ontology) codes: each phenotype was rolled up "
                       "to its ancestor at depth 2 from HP:0000118, so all phenotypes are compared at the same ontology level.",
          ha='center', va='top', fontsize=9, style='italic', color='#555555')

plt.tight_layout()
plt.show()

display(scored_by_depth2)

In [0]:
seven_imputed_ids = [7 WES IDs]
set(seven_imputed_ids) & set(results_GPT_full_df['Subject ID'])

In [0]:
combined_df_full_info.columns

In [0]:
group0 = pd.read_csv("/Workspace/Users/user/user/Kappa_Adjudications_Unscored.csv")
group1 = pd.read_csv("/Workspace/Users/user/user/Burak_Adjudications_Unscored.csv")
group2 = pd.read_csv("/Workspace/Users/user/user/Angie_Adjudications_Unscored.csv")
group3 = pd.read_csv("/Workspace/Users/user/user/Dustin_Adjudications_Unscored_updated.csv")

all_adjudications = pd.concat([group0, group1, group2, group3], ignore_index=True)
adjudicated_pairs = set(zip(all_adjudications['Subject ID'], all_adjudications['Phenotype']))

disagree_mask = combined_df_full_info['any_disagree'] == 1
disagree_pairs = set(zip(
    combined_df_full_info.loc[disagree_mask, 'Subject ID'],
    combined_df_full_info.loc[disagree_mask, 'Phenotype']
))

print(f"Rows loaded per file: {len(group0)}, {len(group1)}, {len(group2)}, {len(group3)}")
print(f"Total concatenated: {len(all_adjudications)}")
print()
print(f"Pairs where any_disagree == 1 in combined_df_full_info: {len(disagree_pairs)}")
print(f"Pairs in concatenated adjudication files: {len(adjudicated_pairs)}")
print()
print(f"In disagree set but NOT in adjudication files: {len(disagree_pairs - adjudicated_pairs)}")
print(f"In adjudication files but NOT in disagree set: {len(adjudicated_pairs - disagree_pairs)}")
print(f"In both: {len(disagree_pairs & adjudicated_pairs)}")

In [0]:
extra_pairs = adjudicated_pairs - disagree_pairs
extra_df = pd.DataFrame(sorted(extra_pairs), columns=['Subject ID', 'Phenotype'])

print(f"Pairs in adjudication files but not flagged as any_disagree: {len(extra_df)}")
print()
print(extra_df.groupby('Phenotype').size().sort_values(ascending=False))

In [0]:
manual_score = pd.read_csv("/Workspace/Users/user/user/manual_score.csv")

remap = {
    'Double Outlet Right Ventricle (DORV': 'Double Outlet Right Ventricle (DORV)',
    'Hypertrophic cardiomyopathy (HCM': 'Hypertrophic cardiomyopathy (HCM)',
    'Tetralogy of Fallot (TOF': 'Tetralogy of Fallot (TOF)',
    'CPEO (opthalmoplegia': 'CPEO (opthalmoplegia)',
    'Autism Spectrum Disorder (ASD': 'Autism Spectrum Disorder (ASD)',
    'Diabetes Mellitus Type I (insulin dependent': 'Diabetes Mellitus Type I (insulin dependent)',
    'Diabetes Mellitus Type II (non-insulin dependent': 'Diabetes Mellitus Type II (non-insulin dependent)',
    'Chiari I malformation': 'Chiari malformation',
}

phen_normalized = manual_score['Phenotype'].str.strip().map(lambda x: remap.get(x, x))
old_pairs_normalized = set(zip(manual_score['Subject ID'], phen_normalized))
new_pairs = set(zip(manual_score_fixed['Subject ID'], manual_score_fixed['Phenotype']))

lost = old_pairs_normalized - new_pairs
recovered = new_pairs - old_pairs_normalized

print(f"manual_score.csv on disk: {len(manual_score)} rows")
print(f"old_pairs, normalized:     {len(old_pairs_normalized)}")
print(f"new_pairs (fixed rebuild): {len(new_pairs)}")
print()
print(f"Recovered (new, not in old normalized): {len(recovered)}")
print(f"Lost (old normalized, not in new):      {len(lost)}")
print(f"In both:                                {len(old_pairs_normalized & new_pairs)}")

if len(lost) > 0:
    lost_df2 = pd.DataFrame(sorted(lost), columns=['Subject ID', 'Phenotype'])
    print()
    print("Lost, by phenotype:")
    print(lost_df2.groupby('Phenotype').size().sort_values(ascending=False))

In [0]:
print(f"len(old_pairs) -- current manual_score, deduplicated: {len(old_pairs)}")
print(f"len(new_pairs) -- manual_score_fixed, deduplicated:   {len(new_pairs)}")
print()
print(f"In new_pairs but not old_pairs (recovered):   {len(new_pairs - old_pairs)}")
print(f"In old_pairs but not new_pairs (LOST):         {len(old_pairs - new_pairs)}")
print(f"In both:                                       {len(old_pairs & new_pairs)}")

In [0]:
lost_pairs = old_pairs - new_pairs
lost_df = pd.DataFrame(sorted(lost_pairs), columns=['Subject ID', 'Phenotype'])

bug_terms = [
    'Autism Spectrum Disorder (ASD)', 'Chronic diarrhea', 'Diabetes Mellitus Type I (insulin dependent)',
    'Diabetes Mellitus Type II (non-insulin dependent)', 'Double Outlet Right Ventricle (DORV)',
    'Hypertrophic cardiomyopathy (HCM)', 'Ketosis', 'Tetralogy of Fallot (TOF)', 'Undescended testis'
]

print(f"Total lost pairs: {len(lost_df)}")
print()
print("Lost pairs by phenotype:")
print(lost_df.groupby('Phenotype').size().sort_values(ascending=False))
print()
print(f"Of the lost pairs, in the 9 known bug-affected terms: {lost_df['Phenotype'].isin(bug_terms).sum()}")
print(f"Of the lost pairs, NOT in the 9 known bug-affected terms: {(~lost_df['Phenotype'].isin(bug_terms)).sum()}")

In [0]:
remap = {
    'Double Outlet Right Ventricle (DORV': 'Double Outlet Right Ventricle (DORV)',
    'Hypertrophic cardiomyopathy (HCM': 'Hypertrophic cardiomyopathy (HCM)',
    'Tetralogy of Fallot (TOF': 'Tetralogy of Fallot (TOF)',
    'CPEO (opthalmoplegia': 'CPEO (opthalmoplegia)',
    'Autism Spectrum Disorder (ASD': 'Autism Spectrum Disorder (ASD)',
    'Diabetes Mellitus Type I (insulin dependent': 'Diabetes Mellitus Type I (insulin dependent)',
    'Diabetes Mellitus Type II (non-insulin dependent': 'Diabetes Mellitus Type II (non-insulin dependent)',
    'Chiari I malformation': 'Chiari malformation',
}

lost_df['Phenotype_corrected'] = lost_df['Phenotype'].map(lambda x: remap.get(x, x))
lost_df['pair_corrected'] = list(zip(lost_df['Subject ID'], lost_df['Phenotype_corrected']))
lost_df['found_under_corrected_spelling'] = lost_df['pair_corrected'].isin(new_pairs)

print(f"Total lost pairs: {len(lost_df)}")
print(f"Found in manual_score_fixed once typo'd spelling is corrected: {lost_df['found_under_corrected_spelling'].sum()}")
print(f"Still missing after correcting spelling -- real loss: {(~lost_df['found_under_corrected_spelling']).sum()}")
print()
print("Any real losses, by phenotype:")
print(lost_df.loc[~lost_df['found_under_corrected_spelling']].groupby('Phenotype').size().sort_values(ascending=False))

In [0]:
lost_subject_ids = set(lost_df['Subject ID'])
manual_df_subject_ids = set(manual_df['Subject ID'])

print(f"Distinct subjects among the 355 lost pairs: {len(lost_subject_ids)}")
print(f"Present as a Subject ID anywhere in CURRENT manual_df: {len(lost_subject_ids & manual_df_subject_ids)}")
print(f"Absent from CURRENT manual_df entirely: {len(lost_subject_ids - manual_df_subject_ids)}")

In [0]:
target_terms = [
    ('Autism Spectrum Disorder (ASD)', 'Autism Spectrum Disorder (ASD'),
    ('Chronic diarrhea', None),
    ('Undescended testis', None),
    ('Diabetes Mellitus Type I (insulin dependent)', 'Diabetes Mellitus Type I (insulin dependent'),
    ('Diabetes Mellitus Type II (non-insulin dependent)', 'Diabetes Mellitus Type II (non-insulin dependent'),
    ('Hypertrophic cardiomyopathy (HCM)', 'Hypertrophic cardiomyopathy (HCM'),
    ('Double Outlet Right Ventricle (DORV)', 'Double Outlet Right Ventricle (DORV'),
    ('Ketosis', None),
    ('Tetralogy of Fallot (TOF)', 'Tetralogy of Fallot (TOF'),
]

print(f"REDCap_mapping (== data_labels_filtered4.csv as loaded): {REDCap_mapping.shape}")
print()
for closed, unclosed in target_terms:
    for form in [closed, unclosed]:
        if form is None:
            continue
        if form in REDCap_mapping.columns:
            n = REDCap_mapping[form].notna().sum()
            print(f"PRESENT  {n:>4d} non-null   {form!r}")
        else:
            print(f"ABSENT               {form!r}")

In [0]:
six_terms = [
    'Autism Spectrum Disorder (ASD)', 'Diabetes Mellitus Type I (insulin dependent)',
    'Diabetes Mellitus Type II (non-insulin dependent)', 'Hypertrophic cardiomyopathy (HCM)',
    'Double Outlet Right Ventricle (DORV)', 'Tetralogy of Fallot (TOF)'
]
three_terms = ['Chronic diarrhea', 'Undescended testis', 'Ketosis']

new_pairs_by_term = pd.DataFrame(sorted(new_pairs), columns=['Subject ID', 'Phenotype'])

print("Rows in manual_score_fixed for the 6 'should already work' terms:")
print(new_pairs_by_term[new_pairs_by_term['Phenotype'].isin(six_terms)].groupby('Phenotype').size())
print()
print("Rows in manual_score_fixed for the 3 genuinely-absent-from-source terms:")
print(new_pairs_by_term[new_pairs_by_term['Phenotype'].isin(three_terms)].groupby('Phenotype').size())
print()
print(f"Columns in REDCap_mapping containing 'diarrhea', 'testis', or 'ketosis' (case-insensitive):")
for kw in ['diarrhea', 'testis', 'ketosis']:
    matches = [c for c in REDCap_mapping.columns if kw in c.lower()]
    print(f"  {kw}: {matches if matches else 'none'}")

In [0]:
fragments = ['diarrhea', 'testis', 'ketosis', 'autism', 'diabetes', 'dorv', 'cardiomyopathy', 'fallot']

found = {}
for item in manual_list:
    low = item.lower()
    for frag in fragments:
        if frag in low:
            found.setdefault(frag, set()).add(item)

for frag in fragments:
    vals = found.get(frag, set())
    print(f"'{frag}': {len(vals)} distinct raw value(s) in manual_list")
    for v in vals:
        print(f"    {v!r}")
    print()

In [0]:
import pandas as pd

remap = {
    'Double Outlet Right Ventricle (DORV': 'Double Outlet Right Ventricle (DORV)',
    'Hypertrophic cardiomyopathy (HCM': 'Hypertrophic cardiomyopathy (HCM)',
    'Tetralogy of Fallot (TOF': 'Tetralogy of Fallot (TOF)',
    'CPEO (opthalmoplegia': 'CPEO (opthalmoplegia)',
    'Autism Spectrum Disorder (ASD': 'Autism Spectrum Disorder (ASD)',
    'Diabetes Mellitus Type I (insulin dependent': 'Diabetes Mellitus Type I (insulin dependent)',
    'Diabetes Mellitus Type II (non-insulin dependent': 'Diabetes Mellitus Type II (non-insulin dependent)',
    'Chiari I malformation': 'Chiari malformation',
}

# checklist
checklist_full = pd.read_csv("/Workspace/Users/user/user/hposetta_stone2.csv")
checklist_filtered = checklist_full[checklist_full['Ignore or not'] != 'ignore'].copy()
checklist_filtered.rename(columns={'HPO version: hpo-web@1.7.0 - hpo-obo@2020-08-11': 'Text from Checklist'}, inplace=True)
checklist_filtered.drop(columns=['Ignore or not'], inplace=True)
checklist_filtered['Term from checklist'] = checklist_filtered['Term from checklist'].str.strip().map(lambda x: remap.get(x, x))

# REDCap -> records_formatted -> manual_list
REDCap_mapping = pd.read_csv("/Workspace/Users/user/user/data_labels_filtered4.csv")
records = []
for i in range(0, len(REDCap_mapping)):
    phen = []
    for j in range(0, len(REDCap_mapping.columns)):
        if pd.isnull(REDCap_mapping.iloc[i][j]) == False:
            phen.append(REDCap_mapping.iloc[i][j])
    records.append(phen)
records_formatted = pd.DataFrame(records)

manual_list = []
for j in range(0, len(records_formatted)):
    for i in range(0, len(records_formatted.columns)):
        manual_list.append(records_formatted[i][j])
manual_list = [x for x in manual_list if x is not None]

# manual_df, normalized in this SAME cell — nothing between build and normalize
rows = []
current_id = None
for item in manual_list:
    if item.startswith('WES'):
        current_id = item
    else:
        rows.append((current_id, item))
manual_df = pd.DataFrame(rows, columns=['Subject ID', 'Phenotype'])
manual_df['Phenotype'] = manual_df['Phenotype'].str.strip().map(lambda x: remap.get(x, x))

# join
manual_score_fixed = manual_df.merge(
    checklist_filtered[['Term from checklist', 'HPO term (ID)']],
    left_on='Phenotype', right_on='Term from checklist', how='inner'
).drop(columns=['Term from checklist'])
manual_score_fixed['Phenotype'] = manual_score_fixed['Phenotype'].str.strip()

# verify immediately, before anything else can run
nine_terms = [
    'Autism Spectrum Disorder (ASD)', 'Chronic diarrhea', 'Undescended testis',
    'Diabetes Mellitus Type I (insulin dependent)', 'Diabetes Mellitus Type II (non-insulin dependent)',
    'Hypertrophic cardiomyopathy (HCM)', 'Double Outlet Right Ventricle (DORV)', 'Ketosis',
    'Tetralogy of Fallot (TOF)'
]
print(f"manual_df: {manual_df.shape}")
print(f"checklist_filtered unique terms: {checklist_filtered['Term from checklist'].nunique()}")
print(f"manual_score_fixed: {manual_score_fixed.shape}")
print()
print("Rows for the 9 bug-affected terms (expect ~231/48/47/7/6/6/4/3/3):")
print(manual_score_fixed[manual_score_fixed['Phenotype'].isin(nine_terms)].groupby('Phenotype').size().sort_values(ascending=False))

In [0]:
dup_mask = manual_score_fixed.duplicated(subset=['Subject ID', 'Phenotype'], keep=False)
dup_rows = manual_score_fixed[dup_mask].sort_values(['Subject ID', 'Phenotype'])

# are the duplicate rows fully identical (same HPO term too), or do they conflict?
conflict_check = dup_rows.groupby(['Subject ID', 'Phenotype'])['HPO term (ID)'].nunique()

print(f"Distinct Subject ID x Phenotype pairs with duplicate rows: {conflict_check.shape[0]}")
print(f"Of those, pairs where the duplicate rows carry DIFFERENT HPO term (ID): {(conflict_check > 1).sum()}")
print()
print("Which phenotypes account for the duplication:")
print(dup_rows.groupby('Phenotype').size().sort_values(ascending=False).head(15))
print()
print("Example — one duplicated subject/phenotype pair, full rows:")
example_pair = conflict_check.index[0]
display(manual_df[(manual_df['Subject ID'] == example_pair[0]) & (manual_df['Phenotype'] == example_pair[1])])

In [0]:
dup_check = manual_score_fixed.duplicated(subset=['Subject ID', 'Phenotype']).sum()
print(f"Duplicate Subject ID x Phenotype pairs in manual_score_fixed: {dup_check}")

In [0]:
print(f"Keys in phenotype_to_hpo: {len(phenotype_to_hpo)}")
print(f"Unique phenotype terms in checklist_filtered: {checklist_filtered['Term from checklist'].nunique()}")
print()

multi_code_terms = checklist_filtered['Term from checklist'].value_counts()
multi_code_terms = multi_code_terms[multi_code_terms > 1].index.tolist()
print(f"Phenotypes with multiple HPO codes in checklist_filtered: {len(multi_code_terms)}")
print()

for term in multi_code_terms:
    all_codes = checklist_filtered.loc[checklist_filtered['Term from checklist'] == term, 'HPO term (ID)'].tolist()
    kept = phenotype_to_hpo.get(term)
    match = 'one of the real codes, other silently dropped' if kept in all_codes else 'MISMATCH -- investigate'
    print(f"{term!r}")
    print(f"   all codes: {all_codes}")
    print(f"   kept in phenotype_to_hpo: {kept!r}  ({match})")

In [0]:
group0 = pd.read_csv("/Workspace/Users/user/user/Kappa_Adjudications_Unscored.csv")
group1 = pd.read_csv("/Workspace/Users/user/user/Burak_Adjudications_Unscored.csv")
group2 = pd.read_csv("/Workspace/Users/user/user/Angie_Adjudications_Unscored.csv")
group3 = pd.read_csv("/Workspace/Users/user/user/Dustin_Adjudications_Unscored_updated.csv")
all_adjudications = pd.concat([group0, group1, group2, group3], ignore_index=True)
adjudicated_pairs = set(zip(all_adjudications['Subject ID'], all_adjudications['Phenotype']))

cohort_ids = set(combined_df_full_info['Subject ID'])

fixed_positive_pairs = set(zip(manual_score_fixed['Subject ID'], manual_score_fixed['Phenotype']))
fixed_positive_pairs_cohort = {(s, p) for (s, p) in fixed_positive_pairs if s in cohort_ids}

current_positive_mask = combined_df_full_info['Manual_Score'] == 1
current_positive_pairs = set(zip(
    combined_df_full_info.loc[current_positive_mask, 'Subject ID'],
    combined_df_full_info.loc[current_positive_mask, 'Phenotype']
))

disagree_mask = combined_df_full_info['any_disagree'] == 1
disagree_pairs = set(zip(
    combined_df_full_info.loc[disagree_mask, 'Subject ID'],
    combined_df_full_info.loc[disagree_mask, 'Phenotype']
))

flips_0_to_1 = fixed_positive_pairs_cohort - current_positive_pairs
flips_1_to_0 = current_positive_pairs - fixed_positive_pairs_cohort

newly_created_0_to_1 = flips_0_to_1 - disagree_pairs
newly_created_1_to_0 = flips_1_to_0 - disagree_pairs
needing_adjudication = newly_created_0_to_1 | newly_created_1_to_0
still_missing = needing_adjudication - adjudicated_pairs

print(f"Rows loaded per file: {len(group0)}, {len(group1)}, {len(group2)}, {len(group3)}")
print()
print(f"Flips 0 -> 1 (correction adds a positive):    {len(flips_0_to_1)}")
print(f"Flips 1 -> 0 (correction removes a positive): {len(flips_1_to_0)}")
print()
print(f"0->1 flips not already caught by any_disagree: {len(newly_created_0_to_1)}")
print(f"1->0 flips not already caught by any_disagree: {len(newly_created_1_to_0)}")
print()
print(f"Total pairs needing adjudication under the corrected standard: {len(needing_adjudication)}")
print(f"Already present in the adjudication files: {len(needing_adjudication & adjudicated_pairs)}")
print(f"STILL missing from adjudication files: {len(still_missing)}")

if len(still_missing) > 0:
    missing_df = pd.DataFrame(sorted(still_missing), columns=['Subject ID', 'Phenotype'])
    print()
    print(missing_df.groupby('Phenotype').size().sort_values(ascending=False))

In [0]:
print(f"cohort_ids: {len(cohort_ids)} subjects")
print(f"combined_df_full_info rows: {len(combined_df_full_info)}")
print(f"Expected if fully dense (subjects x 125): {len(cohort_ids) * 125}")
print()
missing_phenos = set(manual_score_fixed['Phenotype'].unique()) - set(combined_df_full_info['Phenotype'].unique())
print(f"Phenotype strings in manual_score_fixed but absent from combined_df_full_info: {missing_phenos}")
print()

# does combined_df_full_info's Manual_Score actually match manual_score_fixed now?
check_pairs = set(zip(combined_df_full_info['Subject ID'], combined_df_full_info['Phenotype']))
fixed_pairs_all = set(zip(manual_score_fixed['Subject ID'], manual_score_fixed['Phenotype']))

cdf_says_positive = set(zip(
    combined_df_full_info.loc[combined_df_full_info['Manual_Score'] == 1, 'Subject ID'],
    combined_df_full_info.loc[combined_df_full_info['Manual_Score'] == 1, 'Phenotype']
))
should_be_positive = {(s, p) for (s, p) in fixed_pairs_all if s in cohort_ids}

mismatch = cdf_says_positive.symmetric_difference(should_be_positive)
print(f"combined_df_full_info Manual_Score==1 pairs: {len(cdf_says_positive)}")
print(f"manual_score_fixed positive pairs (cohort):   {len(should_be_positive)}")
print(f"Mismatch between the two (should be 0 if your edit is correct): {len(mismatch)}")

In [0]:
print(f"combined_df_full_info rows: {len(combined_df_full_info)}")
print(f"Duplicate (Subject ID, Phenotype) rows: {combined_df_full_info.duplicated(subset=['Subject ID','Phenotype']).sum()}")
print(f"Null Manual_Score values: {combined_df_full_info['Manual_Score'].isna().sum()}")
print()

cohort_ids_check = set(combined_df_full_info['Subject ID'])

fixed_pairs_all_check = set(zip(manual_score_fixed['Subject ID'], manual_score_fixed['Phenotype']))
should_be_positive_check = {(s, p) for (s, p) in fixed_pairs_all_check if s in cohort_ids_check}

cdf_says_positive_check = set(zip(
    combined_df_full_info.loc[combined_df_full_info['Manual_Score'] == 1, 'Subject ID'],
    combined_df_full_info.loc[combined_df_full_info['Manual_Score'] == 1, 'Phenotype']
))

mismatch_check_fresh = cdf_says_positive_check.symmetric_difference(should_be_positive_check)

print(f"cohort_ids (recomputed fresh): {len(cohort_ids_check)}")
print(f"combined_df_full_info Manual_Score==1 pairs: {len(cdf_says_positive_check)}")
print(f"manual_score_fixed positive pairs (cohort): {len(should_be_positive_check)}")
print(f"Mismatch (recomputed fresh, in this same cell): {len(mismatch_check_fresh)}")

# Adjudication Figures

In [0]:
import pandas as pd

burak_note_path = "/Workspace/Users/user/user/Burak_Note_Adjudications_Scored.csv"
dustin_note_path1 = "/Workspace/Users/user/user/Dustin_Note_Adjudications_Scored.csv"
dustin_note_path2 ="/Workspace/Users/user/user/Dustin_Note_Adjudications_Extras_Scored.csv"
angie_note_path = "/Workspace/Users/user/user/Angie_Note_Adjudications_Scored.csv"
burak_kappa_path = "/Workspace/Users/user/user/Burak_Kappa_Adjudications_Scored.csv"
angie_kappa_path = "/Workspace/Users/user/user/Angie_Kappa_Adjudications_Scored.csv"
dustin_kappa_path = "/Workspace/Users/user/user/Dustin_Kappa_Adjudications_Scored.csv"

burak_note_adjudications = pd.read_csv(burak_note_path)

dustin_note_adjudications = pd.concat(
    [pd.read_csv(dustin_note_path1), pd.read_csv(dustin_note_path2)], 
    ignore_index=True
)

angie_note_adjudications = pd.read_csv(angie_note_path)
burak_kappa_adjudications = pd.read_csv(burak_kappa_path)
angie_kappa_adjudications = pd.read_csv(angie_kappa_path)
dustin_kappa_adjudications = pd.read_csv(dustin_kappa_path)

print("Burak Note Adjudications columns:")
print(burak_note_adjudications.columns.tolist())
print("Number of Rows:", len(burak_note_adjudications))
print("Number of Unique Pairs:", len(set(zip(burak_note_adjudications['Subject ID'], burak_note_adjudications['Phenotype']))))

print("Dustin Note Adjudications columns:")
print(dustin_note_adjudications.columns.tolist())
print("Number of Rows:", len(dustin_note_adjudications))
print("Number of Unique Pairs:", len(set(zip(dustin_note_adjudications['Subject ID'], dustin_note_adjudications['Phenotype']))))

print("Angie Note Adjudications columns:")
print(angie_note_adjudications.columns.tolist())
print("Number of Rows:", len(angie_note_adjudications))
print("Number of Unique Pairs:", len(set(zip(angie_note_adjudications['Subject ID'], angie_note_adjudications['Phenotype']))))
print("\n")

print("\nBurak Kappa Adjudications columns:")
print(burak_kappa_adjudications.columns.tolist())
print("Number of Rows", len(burak_kappa_adjudications))
print("Number of Unique Pairs:", len(set(zip(burak_kappa_adjudications['Subject ID'], burak_kappa_adjudications['Phenotype']))))


print("\nAngie Kappa Adjudications columns:")
print(angie_kappa_adjudications.columns.tolist())
print("Number of Rows", len(angie_kappa_adjudications))
print("Number of Unique Pairs:", len(set(zip(angie_kappa_adjudications['Subject ID'], angie_kappa_adjudications['Phenotype']))))



print("\nDustin Kappa Adjudications columns:")
print(dustin_kappa_adjudications.columns.tolist())
print("Number of Rows", len(dustin_kappa_adjudications))
print("Number of Unique Pairs:", len(set(zip(dustin_kappa_adjudications['Subject ID'], dustin_kappa_adjudications['Phenotype']))))

In [0]:
import pandas as pd
import numpy as np
from sklearn.metrics import cohen_kappa_score


def calculate_fleiss_kappa(ratings):
    """
    Calculate Fleiss's kappa.

    Rows represent subject–phenotype pairs.
    Columns represent adjudicators.
    """

    ratings = np.asarray(ratings)

    number_of_items, number_of_raters = ratings.shape

    if number_of_items == 0:
        raise ValueError("No overlapping ratings were available.")

    if number_of_raters < 2:
        raise ValueError(
            "Fleiss's kappa requires at least two adjudicators."
        )

    categories = np.unique(ratings)

    # Count how many adjudicators assigned each category
    # to each subject–phenotype pair
    category_counts = np.column_stack([
        (ratings == category).sum(axis=1)
        for category in categories
    ])

    # Observed agreement for each subject–phenotype pair
    item_agreement = (
        (category_counts ** 2).sum(axis=1) - number_of_raters
    ) / (
        number_of_raters * (number_of_raters - 1)
    )

    observed_agreement = item_agreement.mean()

    # Overall prevalence of each score category
    category_proportions = category_counts.sum(axis=0) / (
        number_of_items * number_of_raters
    )

    expected_agreement = (category_proportions ** 2).sum()

    if np.isclose(1 - expected_agreement, 0):
        raise ValueError(
            "Fleiss's kappa is undefined because all ratings "
            "belong to one score category."
        )

    kappa = (
        observed_agreement - expected_agreement
    ) / (
        1 - expected_agreement
    )

    return kappa, categories


# --------------------------------------------------
# Prepare the three adjudication dataframes
# --------------------------------------------------

keys = ["Subject ID", "Phenotype"]

burak_kappa = burak_kappa_adjudications[
    keys + ["adjudicated_score"]
].copy().rename(
    columns={
        "adjudicated_score": "adjudicated_score_burak"
    }
)

angie_kappa = angie_kappa_adjudications[
    keys + ["adjudicated_score"]
].copy().rename(
    columns={
        "adjudicated_score": "adjudicated_score_angie"
    }
)

dustin_kappa = dustin_kappa_adjudications[
    keys + ["adjudicated_score"]
].copy().rename(
    columns={
        "adjudicated_score": "adjudicated_score_dustin"
    }
)


# --------------------------------------------------
# Check for duplicate subject–phenotype pairs
# --------------------------------------------------

adjudicator_dataframes = {
    "Burak": burak_kappa,
    "Angie": angie_kappa,
    "Dustin": dustin_kappa
}

for name, dataframe in adjudicator_dataframes.items():
    duplicate_count = dataframe.duplicated(
        subset=keys,
        keep=False
    ).sum()

    if duplicate_count > 0:
        raise ValueError(
            f"{name}'s dataframe contains {duplicate_count} rows "
            "with duplicate Subject ID + Phenotype pairs."
        )


# --------------------------------------------------
# Merge the three adjudication dataframes
# --------------------------------------------------

merged_kappa = (
    burak_kappa
    .merge(
        angie_kappa,
        on=keys,
        how="inner",
        validate="one_to_one"
    )
    .merge(
        dustin_kappa,
        on=keys,
        how="inner",
        validate="one_to_one"
    )
)

print(f"Rows in Burak's file: {len(burak_kappa):,}")
print(f"Rows in Angie's file: {len(angie_kappa):,}")
print(f"Rows in Dustin's file: {len(dustin_kappa):,}")
print(
    "Rows scored by all three adjudicators: "
    f"{len(merged_kappa):,}"
)


# --------------------------------------------------
# Confirm all three files contain the same pairs
# --------------------------------------------------

all_input_lengths = {
    len(burak_kappa),
    len(angie_kappa),
    len(dustin_kappa)
}

if (
    len(all_input_lengths) != 1
    or len(merged_kappa) != len(burak_kappa)
):
    raise ValueError(
        "The three files do not contain exactly the same "
        "Subject ID + Phenotype pairs. Stop and inspect the "
        "unmatched rows before trusting the agreement statistics."
    )


# --------------------------------------------------
# Clean and validate the adjudicated scores
# --------------------------------------------------

score_columns = [
    "adjudicated_score_burak",
    "adjudicated_score_angie",
    "adjudicated_score_dustin"
]

merged_kappa[score_columns] = merged_kappa[
    score_columns
].apply(
    pd.to_numeric,
    errors="coerce"
)

missing_scores = merged_kappa[score_columns].isna().sum()

if missing_scores.any():
    raise ValueError(
        "Missing or nonnumeric adjudicated scores were found:\n"
        f"{missing_scores[missing_scores > 0]}"
    )


# --------------------------------------------------
# Calculate complete three-way agreement
# --------------------------------------------------

complete_agreement = (
    merged_kappa[score_columns].nunique(axis=1) == 1
)

agree_count = complete_agreement.sum()

percent_complete_agreement = (
    agree_count / len(merged_kappa)
)


# --------------------------------------------------
# Calculate Fleiss's kappa
# --------------------------------------------------

ratings = merged_kappa[score_columns].to_numpy()

fleiss_score, score_categories = (
    calculate_fleiss_kappa(ratings)
)

print()
print("Three-adjudicator agreement")
print("----------------------------")
print(
    "Complete three-way agreement: "
    f"{percent_complete_agreement:.4f} "
    f"({agree_count:,}/{len(merged_kappa):,})"
)
print(f"Fleiss's kappa: {fleiss_score:.4f}")
print(
    "Score categories included: "
    f"{score_categories.tolist()}"
)


# --------------------------------------------------
# Calculate pairwise Cohen's kappas
# --------------------------------------------------

adjudicator_pairs = [
    ("Burak", "Angie"),
    ("Burak", "Dustin"),
    ("Angie", "Dustin")
]

print()
print("Pairwise agreement statistics")
print("-----------------------------")

for first_name, second_name in adjudicator_pairs:
    first_column = (
        f"adjudicated_score_{first_name.lower()}"
    )
    second_column = (
        f"adjudicated_score_{second_name.lower()}"
    )

    pair_agreement = (
        merged_kappa[first_column]
        == merged_kappa[second_column]
    )

    pair_agree_count = pair_agreement.sum()

    pair_percent_agreement = (
        pair_agree_count / len(merged_kappa)
    )

    pair_kappa = cohen_kappa_score(
        merged_kappa[first_column],
        merged_kappa[second_column]
    )

    print()
    print(f"{first_name} vs. {second_name}")
    print(
        "Percent agreement: "
        f"{pair_percent_agreement:.4f} "
        f"({pair_agree_count:,}/{len(merged_kappa):,})"
    )
    print(f"Cohen's kappa: {pair_kappa:.4f}")
    print(
        f"Confusion matrix "
        f"(rows={first_name}, columns={second_name}):"
    )

    print(
        pd.crosstab(
            merged_kappa[first_column],
            merged_kappa[second_column],
            rownames=[first_name],
            colnames=[second_name]
        )
    )

In [0]:
import pandas as pd
import numpy as np

# --------------------------------------------------
# Load the three scored kappa files
# --------------------------------------------------

burak_kappa_path = (
    "/Workspace/Users/user/user/"
    "Burak_Kappa_Adjudications_Scored.csv"
)

angie_kappa_path = (
    "/Workspace/Users/user/user/"
    "Angie_Kappa_Adjudications_Scored.csv"
)

dustin_kappa_path = (
    "/Workspace/Users/user/user/"
    "Dustin_Kappa_Adjudications_Scored.csv"
)

burak_kappa_adjudications = pd.read_csv(
    burak_kappa_path
)

angie_kappa_adjudications = pd.read_csv(
    angie_kappa_path
)

dustin_kappa_adjudications = pd.read_csv(
    dustin_kappa_path
)

key_columns = ["Subject ID", "Phenotype"]


# --------------------------------------------------
# Prepare and validate each dataframe
# --------------------------------------------------

kappa_dataframes = {
    "Burak": burak_kappa_adjudications,
    "Angie": angie_kappa_adjudications,
    "Dustin": dustin_kappa_adjudications
}

for adjudicator, dataframe in kappa_dataframes.items():
    dataframe.columns = (
        dataframe.columns.astype(str).str.strip()
    )

    required_columns = key_columns + ["adjudicated_score"]

    missing_columns = [
        column
        for column in required_columns
        if column not in dataframe.columns
    ]

    if missing_columns:
        raise KeyError(
            f"{adjudicator}'s file is missing columns: "
            f"{missing_columns}"
        )

    if dataframe[key_columns].isna().any().any():
        raise ValueError(
            f"{adjudicator}'s file contains a missing "
            "Subject ID or Phenotype."
        )

    duplicate_count = dataframe.duplicated(
        subset=key_columns
    ).sum()

    if duplicate_count > 0:
        raise ValueError(
            f"{adjudicator}'s file contains "
            f"{duplicate_count:,} duplicate "
            "Subject ID–Phenotype pairs."
        )


# --------------------------------------------------
# Merge scores by Subject ID and Phenotype
# --------------------------------------------------

burak_scores = (
    burak_kappa_adjudications[
        key_columns + ["adjudicated_score"]
    ]
    .rename(
        columns={
            "adjudicated_score": "score_burak"
        }
    )
)

angie_scores = (
    angie_kappa_adjudications[
        key_columns + ["adjudicated_score"]
    ]
    .rename(
        columns={
            "adjudicated_score": "score_angie"
        }
    )
)

dustin_scores = (
    dustin_kappa_adjudications[
        key_columns + ["adjudicated_score"]
    ]
    .rename(
        columns={
            "adjudicated_score": "score_dustin"
        }
    )
)

all_kappa_scores = (
    burak_scores
    .merge(
        angie_scores,
        on=key_columns,
        how="inner",
        validate="one_to_one"
    )
    .merge(
        dustin_scores,
        on=key_columns,
        how="inner",
        validate="one_to_one"
    )
)


# Confirm that all three files contain the same pairs
input_lengths = {
    "Burak": len(burak_scores),
    "Angie": len(angie_scores),
    "Dustin": len(dustin_scores)
}

if any(
    len(all_kappa_scores) != input_length
    for input_length in input_lengths.values()
):
    raise ValueError(
        "The three files do not contain exactly the same "
        "Subject ID–Phenotype pairs."
    )

score_columns = [
    "score_burak",
    "score_angie",
    "score_dustin"
]

all_kappa_scores[score_columns] = (
    all_kappa_scores[score_columns].apply(
        pd.to_numeric,
        errors="coerce"
    )
)

if all_kappa_scores[score_columns].isna().any().any():
    raise ValueError(
        "At least one adjudicated score is missing or nonnumeric."
    )


# --------------------------------------------------
# Create balanced, randomized adjudicator assignments
# --------------------------------------------------

RANDOM_SEED = 42
rng = np.random.default_rng(RANDOM_SEED)

adjudicators = np.array([
    "Burak",
    "Angie",
    "Dustin"
])

number_of_pairs = len(all_kappa_scores)

# Randomize which adjudicator receives any remainder when
# the number of pairs is not divisible exactly by three
randomized_adjudicator_order = rng.permutation(
    adjudicators
)

selected_adjudicators = np.resize(
    randomized_adjudicator_order,
    number_of_pairs
)

# Randomize assignments across subject–phenotype pairs
rng.shuffle(selected_adjudicators)

score_column_lookup = {
    "Burak": "score_burak",
    "Angie": "score_angie",
    "Dustin": "score_dustin"
}

all_kappa_scores["score_source"] = (
    selected_adjudicators
)

all_kappa_scores["adjudicated_score"] = [
    row[score_column_lookup[row["score_source"]]]
    for _, row in all_kappa_scores.iterrows()
]

all_kappa_scores["adjudicated_score"] = (
    all_kappa_scores["adjudicated_score"].astype(int)
)


# --------------------------------------------------
# Create and save the final scored file
# --------------------------------------------------

kappa_adjudications_scored = (
    all_kappa_scores[
        key_columns + [
            "adjudicated_score",
            "score_source"
        ]
    ]
    .copy()
)

output_path = (
    "/Workspace/Users/user/user/"
    "Kappa_Adjudications_Scored_RandomSplit.csv"
)

#kappa_adjudications_scored.to_csv(
#    output_path,
#    index=False
#)


# --------------------------------------------------
# Display the contribution from each adjudicator
# --------------------------------------------------

print(f"Random seed: {RANDOM_SEED}")
print(f"Total rows in output: {len(kappa_adjudications_scored):,}")
print(f"Saved to: {output_path}")

for adjudicator in adjudicators:
    adjudicator_output = (
        all_kappa_scores.loc[
            all_kappa_scores["score_source"].eq(adjudicator),
            key_columns + [
                "adjudicated_score",
                "score_source"
            ]
        ]
        .reset_index(drop=True)
    )

    print()
    print(
        f"{adjudicator} rows selected: "
        f"{len(adjudicator_output):,}"
    )

    display(adjudicator_output)


random_kappa_path = (
    "/Workspace/Users/user/user/"
    "Kappa_Adjudications_Scored_RandomSplit.csv"
)

random_kappa_adjudications = pd.read_csv(
    random_kappa_path
)

display(random_kappa_adjudications)

In [0]:
disagreement_adjudications = pd.concat([random_kappa_adjudications, dustin_note_adjudications, burak_note_adjudications, angie_note_adjudications])



# Inner join adds adjudicated_score and retains only matching rows
combined_df_full_info_adjudications = combined_df_full_info.merge(
   disagreement_adjudications[
        ["Subject ID", "Phenotype", "adjudicated_score"]
    ],
    on=["Subject ID", "Phenotype"],
    how="inner"
)

print("adjudications scores shape:",disagreement_adjudications.shape)
print("Joined dataframe shape:", combined_df_full_info_adjudications.shape)
display(combined_df_full_info_adjudications.head())


In [0]:
# ------------------------------------------------------------
# Remove adjudication rows where all four scoring methods agree
# ------------------------------------------------------------

score_cols = [
    "Manual_Score",
    "GPT_Score",
    "GPT_Mini_Score",
    "Claude_Sonnet_Score"
]

# Ensure scores are numeric
combined_df_full_info_adjudications[score_cols] = (
    combined_df_full_info_adjudications[score_cols]
    .apply(pd.to_numeric, errors="coerce")
)

# Only call a row unanimous when all four scores are present
all_scores_present = (
    combined_df_full_info_adjudications[score_cols]
    .notna()
    .all(axis=1)
)

all_four_agree = (
    all_scores_present
    & combined_df_full_info_adjudications[score_cols]
      .nunique(axis=1)
      .eq(1)
)

# Save removed rows for inspection
removed_unanimous_adjudications = (
    combined_df_full_info_adjudications[
        all_four_agree
    ].copy()
)

# Retain only actual scoring disagreements
combined_df_full_info_adjudications = (
    combined_df_full_info_adjudications[
        ~all_four_agree
    ]
    .copy()
    .reset_index(drop=True)
)

print(
    "Unanimous adjudication rows removed:",
    f"{len(removed_unanimous_adjudications):,}"
)

print(
    "Adjudication rows remaining:",
    f"{len(combined_df_full_info_adjudications):,}"
)

display(removed_unanimous_adjudications)

In [0]:
## Concatenate Burak's two adjudication datasets
#burak_scores = pd.concat(
#    [burak_note_adjudications, burak_kappa_adjudications],
#    ignore_index=True
#)
#
## Inner join adds adjudicated_score and retains only matching rows
#combined_df_full_info_burak = combined_df_full_info.merge(
#    burak_scores[
#        ["Subject ID", "Phenotype", "adjudicated_score"]
#    ],
#    on=["Subject ID", "Phenotype"],
#    how="inner"
#)
#
#print("Burak scores shape:", burak_scores.shape)
#print("Joined dataframe shape:", combined_df_full_info_burak.shape)
#display(combined_df_full_info_burak.head())

In [0]:
disagreement_adjudications.columns

In [0]:
# Remove accidental whitespace from column names
disagreement_adjudications.columns = [
    str(column).strip()
    for column in disagreement_adjudications.columns
]

adjudicated = pd.to_numeric(
    disagreement_adjudications['adjudicated_score'],
    errors="coerce"
)

In [0]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
from matplotlib.colors import to_rgb

# Display order: Manual, GPT 5.4, GPT 5.4 Mini, Claude Sonnet 4.6
score_columns = {
    "Manual_Score": "Manual",
    "GPT_Score": "GPT 5.4",
    "GPT_Mini_Score": "GPT 5.4 Mini",
    "Claude_Sonnet_Score": "Claude Sonnet 4.6"
}

# Same method colors as the other figures
color_map = {
    "Manual": "#D4B896",
    "GPT 5.4": "#4C9BE8",
    "GPT 5.4 Mini": "#57C4A7",
    "Claude Sonnet 4.6": "#E87B4C"
}


def lighten(color, amount=0.62):
    """Blend a color toward white; amount=0 keeps it, 1 gives white."""
    rgb = np.array(to_rgb(color))
    return tuple(rgb + (1 - rgb) * amount)


adjudicated = pd.to_numeric(
    combined_df_full_info_adjudications[
        "adjudicated_score"
    ],
    errors="coerce"
)

summary = []

for column, method in score_columns.items():
    scores = pd.to_numeric(
        combined_df_full_info_adjudications[column],
        errors="coerce"
    )

    total_ones = scores.eq(1).sum()

    agreeing_ones = (
        scores.eq(1)
        & adjudicated.eq(1)
    ).sum()

    disagreeing_ones = (
        total_ones - agreeing_ones
    )

    agreement_percent = (
        agreeing_ones / total_ones * 100
        if total_ones > 0
        else 0
    )

    summary.append({
        "Score": column,
        "Method": method,
        "Total_ones": total_ones,
        "Agreeing_ones": agreeing_ones,
        "Disagreeing_ones": disagreeing_ones,
        "Agreement_percent": agreement_percent
    })

agreement_summary = pd.DataFrame(summary)

# Create vertical stacked bars
x = np.arange(len(agreement_summary))
max_total = agreement_summary["Total_ones"].max()
label_padding = max(max_total * 0.02, 0.1)

bar_colors = [
    color_map[method]
    for method in agreement_summary["Method"]
]

light_bar_colors = [
    lighten(color)
    for color in bar_colors
]

fig, ax = plt.subplots(figsize=(11, 7))

# Agreeing scores: the method's full color
ax.bar(
    x,
    agreement_summary["Agreeing_ones"],
    color=bar_colors,
    edgecolor=bar_colors,
    linewidth=1.2
)

# Disagreeing scores: a pale tint of the same color, outlined in the
# full color so the segment stays visible
ax.bar(
    x,
    agreement_summary["Disagreeing_ones"],
    bottom=agreement_summary["Agreeing_ones"],
    color=light_bar_colors,
    edgecolor=bar_colors,
    linewidth=1.2
)

for i, row in agreement_summary.iterrows():
    ax.text(
        i,
        row["Total_ones"] + label_padding,
        (
            f'n = {int(row["Total_ones"]):,}\n'
            f'{row["Agreement_percent"]:.1f}% agree'
        ),
        ha="center",
        va="bottom",
        fontweight="bold"
    )

ax.set_xticks(x)

ax.set_xticklabels(
    agreement_summary["Method"]
)

ax.set_xlabel("Scoring Method")
ax.set_ylabel("Count of Scores Equal to 1")

ax.set_title(
    "Count of Positive Scores and Agreement "
    "with Adjudicated Scores"
)

# The bars are colored by method, so the legend uses neutral gray
# to explain the dark and pale shades
legend_gray = "#7B8794"

ax.legend(
    handles=[
        mpatches.Patch(
            facecolor=legend_gray,
            edgecolor=legend_gray,
            label="Agrees with adjudicated score"
        ),
        mpatches.Patch(
            facecolor=lighten(legend_gray),
            edgecolor=legend_gray,
            label="Does not agree"
        )
    ],
    frameon=False
)

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

ax.set_ylim(
    0,
    max(max_total * 1.18, 1)
)

plt.tight_layout()
plt.show()

display(agreement_summary)

In [0]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
from matplotlib.colors import to_rgb

analysis_df = combined_df_full_info_adjudications

# --------------------------------------------------
# Settings
# --------------------------------------------------

# Display order: Manual, GPT 5.4, GPT 5.4 Mini, Claude Sonnet 4.6
score_columns = {
    "Manual_Score": "Manual",
    "GPT_Score": "GPT 5.4",
    "GPT_Mini_Score": "GPT 5.4 Mini",
    "Claude_Sonnet_Score": "Claude Sonnet 4.6"
}

# Same method colors as the other figures
color_map = {
    "Manual": "#D4B896",
    "GPT 5.4": "#4C9BE8",
    "GPT 5.4 Mini": "#57C4A7",
    "Claude Sonnet 4.6": "#E87B4C"
}


def lighten(color, amount=0.62):
    """Blend a color toward white; amount=0 keeps it, 1 gives white."""
    rgb = np.array(to_rgb(color))
    return tuple(rgb + (1 - rgb) * amount)


# --------------------------------------------------
# Calculate positive-score agreement
# --------------------------------------------------

required_columns = (
    list(score_columns) + ["adjudicated_score"]
)

missing_columns = [
    column
    for column in required_columns
    if column not in analysis_df.columns
]

if missing_columns:
    raise KeyError(
        f"Missing required columns: {missing_columns}"
    )

adjudicated = pd.to_numeric(
    analysis_df["adjudicated_score"],
    errors="coerce"
)

summary = []

for column, method in score_columns.items():
    scores = pd.to_numeric(
        analysis_df[column],
        errors="coerce"
    )

    total_ones = scores.eq(1).sum()

    agreeing_ones = (
        scores.eq(1)
        & adjudicated.eq(1)
    ).sum()

    disagreeing_ones = (
        total_ones - agreeing_ones
    )

    agreement_percent = (
        agreeing_ones / total_ones * 100
        if total_ones > 0
        else 0
    )

    summary.append({
        "Score": column,
        "Method": method,
        "Total_ones": total_ones,
        "Agreeing_ones": agreeing_ones,
        "Disagreeing_ones": disagreeing_ones,
        "Agreement_percent": agreement_percent
    })

agreement_summary = pd.DataFrame(summary)

# --------------------------------------------------
# Create vertical stacked bars
# --------------------------------------------------

x = np.arange(len(agreement_summary))
max_total = agreement_summary["Total_ones"].max()
label_padding = max(max_total * 0.02, 1)

bar_colors = [
    color_map[method]
    for method in agreement_summary["Method"]
]

light_bar_colors = [
    lighten(color)
    for color in bar_colors
]

fig, ax = plt.subplots(figsize=(11, 7))

# Agreeing scores: the method's full color
ax.bar(
    x,
    agreement_summary["Agreeing_ones"],
    color=bar_colors,
    edgecolor=bar_colors,
    linewidth=1.2
)

# Disagreeing scores: a pale tint of the same color, outlined in the
# full color so the segment stays visible
ax.bar(
    x,
    agreement_summary["Disagreeing_ones"],
    bottom=agreement_summary["Agreeing_ones"],
    color=light_bar_colors,
    edgecolor=bar_colors,
    linewidth=1.2
)

for i, row in agreement_summary.iterrows():
    ax.text(
        i,
        row["Total_ones"] + label_padding,
        (
            f'n = {int(row["Total_ones"]):,}\n'
            f'{row["Agreement_percent"]:.1f}% agree'
        ),
        ha="center",
        va="bottom",
        fontweight="bold"
    )

ax.set_xticks(x)

ax.set_xticklabels(
    agreement_summary["Method"]
)

ax.set_xlabel("Scoring Method")
ax.set_ylabel("Count of Scores Equal to 1")

ax.set_title(
    "Count of Positive Scores and Agreement "
    "with Adjudicated Scores"
)

# The bars are colored by method, so the legend uses neutral gray
# to explain the dark and pale shades
legend_gray = "#7B8794"

ax.legend(
    handles=[
        mpatches.Patch(
            facecolor=legend_gray,
            edgecolor=legend_gray,
            label="Agrees with adjudicated score"
        ),
        mpatches.Patch(
            facecolor=lighten(legend_gray),
            edgecolor=legend_gray,
            label="Does not agree"
        )
    ],
    frameon=False
)

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

ax.set_ylim(
    0,
    max(
        max_total * 1.18,
        max_total + label_padding * 3,
        1
    )
)

plt.tight_layout()
plt.show()

display(agreement_summary)

In [0]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
from matplotlib.colors import to_rgb

analysis_df = combined_df_full_info_adjudications

# --------------------------------------------------
# Settings
# --------------------------------------------------

# Display order: Manual, GPT 5.4, GPT 5.4 Mini, Claude Sonnet 4.6
score_columns = {
    "Manual_Score": "Manual",
    "GPT_Score": "GPT 5.4",
    "GPT_Mini_Score": "GPT 5.4 Mini",
    "Claude_Sonnet_Score": "Claude Sonnet 4.6"
}

# Same method colors as the other figures
color_map = {
    "Manual": "#D4B896",
    "GPT 5.4": "#4C9BE8",
    "GPT 5.4 Mini": "#57C4A7",
    "Claude Sonnet 4.6": "#E87B4C"
}


def lighten(color, amount=0.62):
    """Blend a color toward white; amount=0 keeps it, 1 gives white."""
    rgb = np.array(to_rgb(color))
    return tuple(rgb + (1 - rgb) * amount)


# --------------------------------------------------
# Calculate agreement for scores equal to 0
# --------------------------------------------------

adjudicated = pd.to_numeric(
    analysis_df["adjudicated_score"],
    errors="coerce"
)

zero_summary = []

for column, method in score_columns.items():
    scores = pd.to_numeric(
        analysis_df[column],
        errors="coerce"
    )

    total_zeros = scores.eq(0).sum()

    agreeing_zeros = (
        scores.eq(0)
        & adjudicated.eq(0)
    ).sum()

    disagreeing_zeros = (
        total_zeros - agreeing_zeros
    )

    agreement_percent = (
        agreeing_zeros / total_zeros * 100
        if total_zeros > 0
        else 0
    )

    zero_summary.append({
        "Score": column,
        "Method": method,
        "Total_zeros": total_zeros,
        "Agreeing_zeros": agreeing_zeros,
        "Disagreeing_zeros": disagreeing_zeros,
        "Agreement_percent": agreement_percent
    })

zero_agreement_summary = pd.DataFrame(
    zero_summary
)

# --------------------------------------------------
# Create vertical stacked bars
# --------------------------------------------------

x = np.arange(len(zero_agreement_summary))
max_total = zero_agreement_summary["Total_zeros"].max()
label_padding = max(max_total * 0.02, 1)

bar_colors = [
    color_map[method]
    for method in zero_agreement_summary["Method"]
]

light_bar_colors = [
    lighten(color)
    for color in bar_colors
]

fig, ax = plt.subplots(figsize=(11, 7))

# Agreeing scores: the method's full color
ax.bar(
    x,
    zero_agreement_summary["Agreeing_zeros"],
    color=bar_colors,
    edgecolor=bar_colors,
    linewidth=1.2
)

# Disagreeing scores: a pale tint of the same color, outlined in the
# full color so the segment stays visible
ax.bar(
    x,
    zero_agreement_summary["Disagreeing_zeros"],
    bottom=zero_agreement_summary["Agreeing_zeros"],
    color=light_bar_colors,
    edgecolor=bar_colors,
    linewidth=1.2
)

for i, row in zero_agreement_summary.iterrows():
    ax.text(
        i,
        row["Total_zeros"] + label_padding,
        (
            f'n = {int(row["Total_zeros"]):,}\n'
            f'{row["Agreement_percent"]:.1f}% agree'
        ),
        ha="center",
        va="bottom",
        fontweight="bold"
    )

ax.set_xticks(x)

ax.set_xticklabels(
    zero_agreement_summary["Method"]
)

ax.set_xlabel("Scoring Method")
ax.set_ylabel("Count of Scores Equal to 0")

ax.set_title(
    "Count of Negative Scores and Agreement "
    "with Adjudicated Scores"
)

# The bars are colored by method, so the legend uses neutral gray
# to explain the dark and pale shades
legend_gray = "#7B8794"

ax.legend(
    handles=[
        mpatches.Patch(
            facecolor=legend_gray,
            edgecolor=legend_gray,
            label="Agrees with adjudicated score"
        ),
        mpatches.Patch(
            facecolor=lighten(legend_gray),
            edgecolor=legend_gray,
            label="Does not agree"
        )
    ],
    frameon=False
)

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

ax.set_ylim(
    0,
    max(
        max_total * 1.18,
        max_total + label_padding * 3,
        1
    )
)

plt.tight_layout()
plt.show()

display(zero_agreement_summary)

In [0]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
from matplotlib.colors import to_rgb

analysis_df = (
    combined_df_full_info_adjudications.copy()
)

# Display order: Manual, GPT 5.4, GPT 5.4 Mini, Claude Sonnet 4.6
score_columns = [
    "Manual_Score",
    "GPT_Score",
    "GPT_Mini_Score",
    "Claude_Sonnet_Score"
]

labels = {
    "Manual_Score": "Manual",
    "GPT_Score": "GPT 5.4",
    "GPT_Mini_Score": "GPT 5.4 Mini",
    "Claude_Sonnet_Score": "Claude Sonnet 4.6"
}

# Same method colors as the other figures
color_map = {
    "Manual": "#D4B896",
    "GPT 5.4": "#4C9BE8",
    "GPT 5.4 Mini": "#57C4A7",
    "Claude Sonnet 4.6": "#E87B4C"
}


def lighten(color, amount=0.62):
    """Blend a color toward white; amount=0 keeps it, 1 gives white."""
    rgb = np.array(to_rgb(color))
    return tuple(rgb + (1 - rgb) * amount)


required_columns = (
    score_columns + ["adjudicated_score"]
)

missing_columns = [
    column
    for column in required_columns
    if column not in analysis_df.columns
]

if missing_columns:
    raise KeyError(
        f"Missing required columns: {missing_columns}"
    )

# Convert all scoring columns to numeric
analysis_df[required_columns] = (
    analysis_df[required_columns]
    .apply(
        pd.to_numeric,
        errors="coerce"
    )
)

# Report missing or nonnumeric values
for column in required_columns:
    n_null = analysis_df[column].isna().sum()

    if n_null > 0:
        print(
            f"WARNING: {n_null:,} missing or nonnumeric "
            f"values in {column}"
        )

# Confirm that all nonmissing scores are binary
for column in required_columns:
    unexpected_values = (
        analysis_df.loc[
            analysis_df[column].notna()
            & ~analysis_df[column].isin([0, 1]),
            column
        ]
        .unique()
        .tolist()
    )

    if unexpected_values:
        raise ValueError(
            f"{column} contains values other than 0 and 1: "
            f"{sorted(unexpected_values)}"
        )

adjudicated = analysis_df["adjudicated_score"]


# --------------------------------------------------
# Build class-specific accuracy summaries
# --------------------------------------------------

def build_summary(mask, correct_value):
    rows = []

    for column in score_columns:
        scores = analysis_df.loc[
            mask,
            column
        ]

        # Denominator includes only nonmissing method scores
        n_total = scores.notna().sum()

        n_correct = (
            scores.eq(correct_value)
            .sum()
        )

        n_incorrect = (
            n_total - n_correct
        )

        accuracy_percent = (
            n_correct / n_total * 100
            if n_total > 0
            else 0
        )

        rows.append({
            "Score": labels[column],
            "Total_n": n_total,
            "Correct": n_correct,
            "Incorrect": n_incorrect,
            "Accuracy_percent": accuracy_percent
        })

    return pd.DataFrame(rows)


neg_mask = adjudicated.eq(0)
pos_mask = adjudicated.eq(1)

specificity_summary = build_summary(
    neg_mask,
    correct_value=0
)

sensitivity_summary = build_summary(
    pos_mask,
    correct_value=1
)

print(
    "\nAdjudicated negative rows: "
    f"{neg_mask.sum():,}"
)

print(
    "Adjudicated positive rows: "
    f"{pos_mask.sum():,}"
)

print(
    "Rows with missing adjudicated scores: "
    f"{adjudicated.isna().sum():,}"
)


# --------------------------------------------------
# Plotting helper
# --------------------------------------------------

def plot_accuracy_bars(
    summary_df,
    title,
    correct_label
):
    x = np.arange(len(summary_df))

    max_total = summary_df["Total_n"].max()

    label_padding = max(
        max_total * 0.02,
        1
    )

    bar_colors = [
        color_map[method]
        for method in summary_df["Score"]
    ]

    light_bar_colors = [
        lighten(color)
        for color in bar_colors
    ]

    fig, ax = plt.subplots(figsize=(9, 7))

    # Correct scores: the method's full color
    ax.bar(
        x,
        summary_df["Correct"],
        color=bar_colors,
        edgecolor=bar_colors,
        linewidth=1.2
    )

    # Incorrect scores: a pale tint of the same color, outlined in
    # the full color so the segment stays visible
    ax.bar(
        x,
        summary_df["Incorrect"],
        bottom=summary_df["Correct"],
        color=light_bar_colors,
        edgecolor=bar_colors,
        linewidth=1.2
    )

    for i, row in summary_df.iterrows():
        ax.text(
            i,
            row["Total_n"] + label_padding,
            (
                f'n = {int(row["Total_n"]):,}\n'
                f'{row["Accuracy_percent"]:.1f}% correct'
            ),
            ha="center",
            va="bottom",
            fontweight="bold"
        )

    ax.set_xticks(x)
    ax.set_xticklabels(summary_df["Score"])
    ax.set_xlabel("Scoring Method")
    ax.set_ylabel("Count of Adjudicated Rows")
    ax.set_title(title)

    # The bars are colored by method, so the legend uses neutral
    # gray to explain the dark and pale shades
    legend_gray = "#7B8794"

    ax.legend(
        handles=[
            mpatches.Patch(
                facecolor=legend_gray,
                edgecolor=legend_gray,
                label=correct_label
            ),
            mpatches.Patch(
                facecolor=lighten(legend_gray),
                edgecolor=legend_gray,
                label="Does not match"
            )
        ],
        # Below the axis: every bar is the same height here, so a
        # legend inside the plot would cover the bar labels
        loc="upper center",
        bbox_to_anchor=(0.5, -0.10),
        ncol=2,
        frameon=False
    )

    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

    ax.set_ylim(
        0,
        max(
            max_total * 1.18,
            max_total + label_padding * 3,
            1
        )
    )

    plt.tight_layout()
    plt.show()


plot_accuracy_bars(
    specificity_summary,
    (
        "Performance on Adjudicated Negatives\n"
        "(Specificity Within the Adjudicated Subset)"
    ),
    correct_label="Correctly scored 0"
)

display(specificity_summary)

plot_accuracy_bars(
    sensitivity_summary,
    (
        "Performance on Adjudicated Positives\n"
        "(Recall Within the Adjudicated Subset)"
    ),
    correct_label="Correctly scored 1"
)

display(sensitivity_summary)

In [0]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

analysis_df = (
    combined_df_full_info_adjudications.copy()
)

score_cols = [
    "GPT_Score",
    "GPT_Mini_Score",
    "Claude_Sonnet_Score",
    "Manual_Score"
]

labels = {
    "GPT_Score": "GPT",
    "GPT_Mini_Score": "GPT Mini",
    "Claude_Sonnet_Score": "Claude Sonnet",
    "Manual_Score": "Manual"
}

colors = {
    "GPT_Score": "#4C9BE8",
    "GPT_Mini_Score": "#57C4A7",
    "Claude_Sonnet_Score": "#E87B4C",
    "Manual_Score": "#D4B896"
}

incorrect_color = "#D9D9D9"

required_columns = (
    ["Phenotype", "adjudicated_score"]
    + score_cols
)

missing_columns = [
    column
    for column in required_columns
    if column not in analysis_df.columns
]

if missing_columns:
    raise KeyError(
        f"Missing required columns: {missing_columns}"
    )

# Convert all scores to numeric
numeric_columns = (
    score_cols + ["adjudicated_score"]
)

analysis_df[numeric_columns] = (
    analysis_df[numeric_columns]
    .apply(
        pd.to_numeric,
        errors="coerce"
    )
)

# Report missing or nonnumeric values
for column in numeric_columns:
    n_null = analysis_df[column].isna().sum()

    if n_null > 0:
        print(
            f"WARNING: {n_null:,} missing or nonnumeric "
            f"values in {column}"
        )

# Confirm scores are binary
for column in numeric_columns:
    unexpected_values = (
        analysis_df.loc[
            analysis_df[column].notna()
            & ~analysis_df[column].isin([0, 1]),
            column
        ]
        .unique()
        .tolist()
    )

    if unexpected_values:
        raise ValueError(
            f"{column} contains values other than 0 and 1: "
            f"{sorted(unexpected_values)}"
        )


# --------------------------------------------------
# Build phenotype-level summary table
# --------------------------------------------------

def build_subset_table(dataframe, value):
    subset = dataframe[
        dataframe["adjudicated_score"].eq(value)
    ]

    rows = []

    for phenotype, group in subset.groupby(
        "Phenotype"
    ):
        row = {
            "Phenotype": phenotype,
            "n": len(group)
        }

        for column in score_cols:
            scores = group[column]

            # Exclude missing method scores from that
            # method's denominator
            n_scored = scores.notna().sum()

            n_correct = (
                scores.eq(value).sum()
            )

            n_incorrect = (
                n_scored - n_correct
            )

            accuracy_percent = (
                n_correct / n_scored * 100
                if n_scored > 0
                else np.nan
            )

            row[f"{column}_total"] = n_scored
            row[f"{column}_correct"] = n_correct
            row[f"{column}_incorrect"] = n_incorrect
            row[
                f"{column}_accuracy_pct"
            ] = round(
                accuracy_percent,
                1
            )

        rows.append(row)

    if not rows:
        return pd.DataFrame()

    return (
        pd.DataFrame(rows)
        .set_index("Phenotype")
    )


# --------------------------------------------------
# Plot phenotype-level results
# --------------------------------------------------

def plot_subset(table, value, label_text):

    if table.empty:
        print(
            f"No phenotypes were adjudicated as {value}."
        )
        return table

    accuracy_columns = [
        f"{column}_accuracy_pct"
        for column in score_cols
    ]

    mean_accuracy = (
        table[accuracy_columns]
        .mean(axis=1)
    )

    # Best accuracy is placed at the bottom and
    # worst accuracy at the top of the horizontal plot
    table_sorted = table.loc[
        mean_accuracy
        .sort_values(
            ascending=False,
            na_position="first"
        )
        .index
    ]

    number_of_phenotypes = len(table_sorted)
    y = np.arange(number_of_phenotypes)
    height = 0.2

    fig, ax = plt.subplots(
        figsize=(
            11,
            max(8, 0.22 * number_of_phenotypes)
        )
    )

    legend_handles = [
        mpatches.Patch(
            color=colors[column],
            label=labels[column]
        )
        for column in score_cols
    ]

    legend_handles.append(
        mpatches.Patch(
            color=incorrect_color,
            label="Does not match adjudication"
        )
    )

    for i, column in enumerate(score_cols):
        correct = table_sorted[
            f"{column}_correct"
        ]

        incorrect = table_sorted[
            f"{column}_incorrect"
        ]

        y_position = (
            y + (i - 1.5) * height
        )

        ax.barh(
            y_position,
            correct,
            height,
            color=colors[column],
            edgecolor="none"
        )

        ax.barh(
            y_position,
            incorrect,
            height,
            left=correct,
            color=incorrect_color,
            edgecolor="none"
        )

    ax.set_yticks(y)

    ax.set_yticklabels(
        [
            f"{phenotype} (n={int(n):,})"
            for phenotype, n in zip(
                table_sorted.index,
                table_sorted["n"]
            )
        ],
        fontsize=7
    )

    ax.set_xlabel(
        "Number of scored adjudicated cells",
        fontsize=12
    )

    ax.set_title(
        "Scoring Accuracy by Phenotype — "
        f"Cells Adjudicated as {label_text}\n"
        "(colored = correct; gray = incorrect; "
        "worst mean accuracy at top)",
        fontsize=13,
        pad=14
    )

    ax.legend(
        handles=legend_handles,
        loc="upper center",
        bbox_to_anchor=(0.5, -0.02),
        ncol=5,
        frameon=False,
        fontsize=9
    )

    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.set_xlim(left=0)

    plt.tight_layout()
    plt.show()

    return table_sorted


# --------------------------------------------------
# Create and display results
# --------------------------------------------------

table_0 = build_subset_table(
    analysis_df,
    0
)

table_1 = build_subset_table(
    analysis_df,
    1
)

total_phenotypes = (
    analysis_df["Phenotype"]
    .nunique(dropna=True)
)

print(
    "Phenotypes represented in the adjudicated-negative "
    f"subset: {len(table_0):,} of {total_phenotypes:,}"
)

print(
    "Phenotypes represented in the adjudicated-positive "
    f"subset: {len(table_1):,} of {total_phenotypes:,}"
)

table_0_sorted = plot_subset(
    table_0,
    0,
    "0 (Negative)"
)

display(
    table_0_sorted.iloc[::-1]
)

table_1_sorted = plot_subset(
    table_1,
    1,
    "1 (Positive)"
)

display(
    table_1_sorted.iloc[::-1]
)

In [0]:
import pandas as pd

# Record the original counts
rows_before = len(notes_enrolled4)
subjects_before = (
    notes_enrolled4["Subject ID"]
    .nunique(dropna=True)
)

# Use Subject IDs represented in the adjudicated subset
matching_subject_ids = (
    combined_df_full_info_adjudications[
        "Subject ID"
    ]
    .dropna()
    .unique()
)

notes_enrolled4 = notes_enrolled4[
    notes_enrolled4["Subject ID"].isin(
        matching_subject_ids
    )
].copy()

# Convert note timestamp to datetime
original_timestamps = (
    notes_enrolled4["NOTE_FILE_TIME_DTTM"]
    .copy()
)

notes_enrolled4["NOTE_FILE_TIME_DTTM"] = (
    pd.to_datetime(
        original_timestamps,
        errors="coerce"
    )
)

n_unparsed = (
    original_timestamps.notna().sum()
    - notes_enrolled4[
        "NOTE_FILE_TIME_DTTM"
    ].notna().sum()
)

# Print results
print(f"Rows before filtering: {rows_before:,}")
print(f"Rows remaining: {len(notes_enrolled4):,}")
print()

print(
    "Unique Subject IDs before filtering: "
    f"{subjects_before:,}"
)

print(
    "Unique Subject IDs remaining: "
    f"{notes_enrolled4['Subject ID'].nunique():,}"
)

print(
    "Subject IDs expected from adjudicated subset: "
    f"{len(matching_subject_ids):,}"
)

print()
print(
    "Dates that failed to parse: "
    f"{n_unparsed:,}"
)

print(
    "Earliest note date:",
    notes_enrolled4[
        "NOTE_FILE_TIME_DTTM"
    ].min()
)

print(
    "Latest note date:",
    notes_enrolled4[
        "NOTE_FILE_TIME_DTTM"
    ].max()
)

In [0]:
#import pandas as pd
#
#updated_path = (
#    "/Workspace/Users/user/user/Dustin_Adjudications_Unscored_updated.csv"
#)
#
#original_path = (
#    "/Workspace/Users/user/user/Dustin_Adjudications_Unscored.csv"
#)
#
#dustin_updated_unscored = pd.read_csv(updated_path)
#dustin_original_unscored = pd.read_csv(original_path)
#
#key_columns = ["Subject ID", "Phenotype"]
#
## Confirm the comparison columns exist in both files
#for column in key_columns:
#    if column not in dustin_updated_unscored.columns:
#        raise KeyError(f"{column!r} is missing from the updated file.")
#
#    if column not in dustin_original_unscored.columns:
#        raise KeyError(f"{column!r} is missing from the original file.")
#
## Identify rows in the updated file that are absent from the original
#original_pairs = (
#    dustin_original_unscored[key_columns]
#    .drop_duplicates()
#    .assign(_present_in_original=True)
#)
#
#dustin_added_rows = dustin_updated_unscored.merge(
#    original_pairs,
#    on=key_columns,
#    how="left",
#    validate="many_to_one"
#)
#
#dustin_added_rows = (
#    dustin_added_rows[
#        dustin_added_rows["_present_in_original"].isna()
#    ]
#    .drop(columns="_present_in_original")
#    .reset_index(drop=True)
#)
#
#print(
#    "Rows in updated file but not original file:",
#    f"{len(dustin_added_rows):,}"
#)
#
#display(dustin_added_rows)

In [0]:
#extras_path = (
#    "/Workspace/Users/user/user/"
#    "Dustin_Adjudications_Extras_Unscored.csv"
#)
#
#dustin_added_rows.to_csv(
#    extras_path,
#    index=False
#)
#
#print(f"Saved {len(dustin_added_rows):,} rows to:")
#print(extras_path)

In [0]:
disagreement_adjudications.columns

In [0]:
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

df = combined_df_full_info_adjudications.copy()

score_cols = [
    "Manual_Score",
    "GPT_Score",
    "GPT_Mini_Score",
    "Claude_Sonnet_Score"
]

required_columns = [
    "Subject ID",
    "Date of last chart review:",
    "adjudicated_score"
] + score_cols

missing_columns = [
    column
    for column in required_columns
    if column not in df.columns
]

if missing_columns:
    raise KeyError(
        f"Missing required columns: {missing_columns}"
    )

# Ensure all scores are numeric
numeric_columns = (
    score_cols + ["adjudicated_score"]
)

df[numeric_columns] = (
    df[numeric_columns]
    .apply(
        pd.to_numeric,
        errors="coerce"
    )
)

# Confirm all nonmissing scores are binary
for column in numeric_columns:
    unexpected_values = (
        df.loc[
            df[column].notna()
            & ~df[column].isin([0, 1]),
            column
        ]
        .unique()
        .tolist()
    )

    if unexpected_values:
        raise ValueError(
            f"{column} contains values other than 0 and 1: "
            f"{sorted(unexpected_values)}"
        )

# Convert chart-review dates to datetime
df["review_date"] = pd.to_datetime(
    df["Date of last chart review:"],
    errors="coerce"
)

n_unparsed = (
    df["Date of last chart review:"].notna().sum()
    - df["review_date"].notna().sum()
)

print(
    "Rows with a date that failed to parse: "
    f"{n_unparsed:,}"
)

# Choose "Q" for quarterly or "M" for monthly
period_freq = "Q"

period_name = (
    "quarter" if period_freq == "Q" else "month"
)

df["review_period"] = (
    df["review_date"].dt.to_period(period_freq)
)

valid_periods = df["review_period"].dropna()

if valid_periods.empty:
    raise ValueError(
        "No valid chart-review dates were available."
    )

# Use the same period range in every graph
all_periods = pd.period_range(
    start=valid_periods.min(),
    end=valid_periods.max(),
    freq=period_freq
)

x_positions = np.arange(len(all_periods))
period_labels = all_periods.astype(str)

colors = {
    "Manual_Score": "#D4B896",
    "GPT_Score": "#4C9BE8",
    "GPT_Mini_Score": "#57C4A7",
    "Claude_Sonnet_Score": "#E87B4C"
}

labels = {
    "Manual_Score": "Manual",
    "GPT_Score": "GPT 5.4",
    "GPT_Mini_Score": "GPT 5.4 Mini",
    "Claude_Sonnet_Score": "Claude Sonnet 4.6"
}

notes_by_period_results = {}
avg_by_period_results = {}


# ============================================================
# Produce graphs for adjudicated scores of 1 and 0
# ============================================================

for adjudicated_value in [1, 0]:

    filtered_df = df[
        df["adjudicated_score"].eq(
            adjudicated_value
        )
    ].copy()

    print()
    print(f"Adjudicated score = {adjudicated_value}")
    print("-" * 35)
    print(f"Adjudicated rows: {len(filtered_df):,}")
    print(
        "Unique notes represented: "
        f"{filtered_df['Subject ID'].nunique():,}"
    )

    # --------------------------------------------------------
    # Figure 1: Distribution of notes by review period
    # --------------------------------------------------------

    subject_level = (
        filtered_df
        .dropna(subset=["review_period"])
        .drop_duplicates(subset="Subject ID")
    )

    notes_by_period = (
        subject_level["review_period"]
        .value_counts()
        .sort_index()
        .reindex(all_periods, fill_value=0)
    )

    notes_by_period.index.name = "review_period"

    notes_by_period_results[
        adjudicated_value
    ] = notes_by_period

    fig1, ax1 = plt.subplots(figsize=(11, 5))

    ax1.bar(
        x_positions,
        notes_by_period.values,
        color="#4C9BE8",
        edgecolor="white"
    )

    ax1.set_xticks(x_positions)
    ax1.set_xticklabels(
        period_labels,
        rotation=45,
        ha="right"
    )

    ax1.set_xlabel(
        f"Review date period ({period_name})",
        fontsize=12
    )

    ax1.set_ylabel(
        "Number of notes",
        fontsize=12
    )

    ax1.set_title(
        "Distribution of Notes by Review Date\n"
        f"(Adjudicated Score = {adjudicated_value})",
        fontsize=15,
        pad=15
    )

    ax1.set_ylim(bottom=0)
    ax1.spines["top"].set_visible(False)
    ax1.spines["right"].set_visible(False)

    if len(notes_by_period) > 0:
        label_padding = max(
            notes_by_period.max() * 0.01,
            0.1
        )

        for i, value in enumerate(
            notes_by_period.values
        ):
            ax1.text(
                i,
                value + label_padding,
                f"{int(value):,}",
                ha="center",
                va="bottom",
                fontsize=10
            )

    plt.tight_layout()
    plt.show()

    # --------------------------------------------------------
    # Figures 2 and 3:
    # Average number of exact 1s and 0s per note
    # --------------------------------------------------------

    for method_score_value in [1, 0]:

        score_indicator_df = filtered_df[
            [
                "Subject ID",
                "review_period"
            ] + score_cols
        ].copy()

        # Count only exact matches to the requested score.
        # Missing scores remain uncounted.
        for column in score_cols:
            score_indicator_df[column] = (
                score_indicator_df[column]
                .eq(method_score_value)
                .astype(int)
            )

        subject_score_counts = (
            score_indicator_df
            .dropna(subset=["review_period"])
            .groupby("Subject ID")
            .agg(
                review_period=(
                    "review_period",
                    "first"
                ),
                Manual_Score=(
                    "Manual_Score",
                    "sum"
                ),
                GPT_Score=(
                    "GPT_Score",
                    "sum"
                ),
                GPT_Mini_Score=(
                    "GPT_Mini_Score",
                    "sum"
                ),
                Claude_Sonnet_Score=(
                    "Claude_Sonnet_Score",
                    "sum"
                )
            )
            .reset_index()
        )

        avg_by_period = (
            subject_score_counts
            .groupby("review_period")[score_cols]
            .mean()
            .reindex(all_periods)
        )

        avg_by_period.index.name = "review_period"

        avg_by_period_results[
            (adjudicated_value, method_score_value)
        ] = avg_by_period

        fig, ax = plt.subplots(figsize=(11, 6))

        for column in score_cols:
            ax.plot(
                x_positions,
                avg_by_period[column].values,
                marker="o",
                label=labels[column],
                color=colors[column],
                linewidth=2
            )

        ax.set_xticks(x_positions)
        ax.set_xticklabels(
            period_labels,
            rotation=45,
            ha="right"
        )

        ax.set_xlabel(
            f"Review period ({period_name})",
            fontsize=12
        )

        ax.set_ylabel(
            f"Average phenotypes scored "
            f"{method_score_value} per note",
            fontsize=12
        )

        ax.set_title(
            f"Average Phenotypes Scored {method_score_value} "
            "per Note by Review Date\n"
            f"(Adjudicated Score = {adjudicated_value})",
            fontsize=15,
            pad=15
        )

        ax.legend(frameon=False, fontsize=11)
        ax.set_ylim(bottom=0)
        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)

        plt.tight_layout()
        plt.show()

        print(
            f"Average number scored {method_score_value} "
            f"when adjudicated score = {adjudicated_value}:"
        )

        display(avg_by_period)

    print(
        "Notes by period when adjudicated score "
        f"= {adjudicated_value}:"
    )

    display(notes_by_period)

In [0]:
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

df = combined_df_full_info_adjudications.copy()

score_cols = [
    "Manual_Score",
    "GPT_Score",
    "GPT_Mini_Score",
    "Claude_Sonnet_Score"
]

required_columns = [
    "Subject ID",
    "NOTE_TEXT",
    "adjudicated_score"
] + score_cols

missing_columns = [
    column
    for column in required_columns
    if column not in df.columns
]

if missing_columns:
    raise KeyError(
        f"Missing required columns: {missing_columns}"
    )

# ------------------------------------------------------------
# Clean and validate scores
# ------------------------------------------------------------

numeric_columns = (
    score_cols + ["adjudicated_score"]
)

df[numeric_columns] = (
    df[numeric_columns]
    .apply(
        pd.to_numeric,
        errors="coerce"
    )
)

for column in numeric_columns:
    unexpected_values = (
        df.loc[
            df[column].notna()
            & ~df[column].isin([0, 1]),
            column
        ]
        .unique()
        .tolist()
    )

    if unexpected_values:
        raise ValueError(
            f"{column} contains values other than 0 and 1: "
            f"{sorted(unexpected_values)}"
        )

# Calculate character count for each row
df["char_count"] = (
    df["NOTE_TEXT"]
    .astype("string")
    .str.len()
)

# Create one character count per unique note/Subject ID
unique_notes_adjudication = (
    df[
        ["Subject ID", "NOTE_TEXT", "char_count"]
    ]
    .dropna(
        subset=[
            "Subject ID",
            "NOTE_TEXT",
            "char_count"
        ]
    )
    .drop_duplicates(subset="Subject ID")
)

char_counts_adjudication = (
    unique_notes_adjudication["char_count"]
    .astype(int)
)

print(
    "Unique notes with nonmissing text: "
    f"{len(char_counts_adjudication):,}"
)

if char_counts_adjudication.empty:
    raise ValueError(
        "No nonmissing note text was available."
    )

# ------------------------------------------------------------
# Create common note-length bins
# ------------------------------------------------------------

number_of_bins = 10

minimum_length = (
    char_counts_adjudication.min()
)

maximum_length = (
    char_counts_adjudication.max()
)

if minimum_length == maximum_length:
    raise ValueError(
        "All notes have the same character count, so "
        "note-length bins cannot be created."
    )

bin_edges = np.linspace(
    minimum_length,
    maximum_length,
    number_of_bins + 1
)

bin_labels = [
    f"{int(round(left)):,}–{int(round(right)):,}"
    for left, right in zip(
        bin_edges[:-1],
        bin_edges[1:]
    )
]

df["char_count_bin"] = pd.cut(
    df["char_count"],
    bins=bin_edges,
    labels=bin_labels,
    include_lowest=True
)

colors = {
    "Manual_Score": "#D4B896",
    "GPT_Score": "#4C9BE8",
    "GPT_Mini_Score": "#57C4A7",
    "Claude_Sonnet_Score": "#E87B4C"
}

labels = {
    "Manual_Score": "Manual",
    "GPT_Score": "GPT 5.4",
    "GPT_Mini_Score": "GPT 5.4 Mini",
    "Claude_Sonnet_Score": "Claude Sonnet 4.6"
}

character_distribution_results = {}
average_score_results = {}


# ============================================================
# Produce graphs for adjudicated scores of 1 and 0
# ============================================================

for adjudicated_value in [1, 0]:

    filtered_df = df[
        df["adjudicated_score"].eq(
            adjudicated_value
        )
    ].copy()

    print()
    print(f"Adjudicated score = {adjudicated_value}")
    print("-" * 35)
    print(f"Adjudicated rows: {len(filtered_df):,}")
    print(
        "Unique notes represented: "
        f"{filtered_df['Subject ID'].nunique():,}"
    )

    # --------------------------------------------------------
    # Figure 1: Distribution of note character counts
    # --------------------------------------------------------

    unique_note_lengths = (
        filtered_df[
            ["Subject ID", "NOTE_TEXT", "char_count"]
        ]
        .dropna(
            subset=[
                "Subject ID",
                "NOTE_TEXT",
                "char_count"
            ]
        )
        .drop_duplicates(subset="Subject ID")
    )

    character_distribution_results[
        adjudicated_value
    ] = unique_note_lengths["char_count"]

    fig1, ax1 = plt.subplots(figsize=(11, 5))

    counts, _, patches = ax1.hist(
        unique_note_lengths["char_count"],
        bins=bin_edges,
        color="#4C9BE8",
        edgecolor="white"
    )

    ax1.set_xlabel(
        "Note length in characters",
        fontsize=12
    )

    ax1.set_ylabel(
        "Number of notes",
        fontsize=12
    )

    ax1.set_title(
        "Distribution of Note Character Counts\n"
        f"(Adjudicated Score = {adjudicated_value})",
        fontsize=15,
        pad=15
    )

    ax1.set_ylim(bottom=0)
    ax1.spines["top"].set_visible(False)
    ax1.spines["right"].set_visible(False)

    if len(counts) > 0:
        label_padding = max(
            counts.max() * 0.01,
            0.1
        )

        for count, patch in zip(
            counts,
            patches
        ):
            if count > 0:
                ax1.text(
                    (
                        patch.get_x()
                        + patch.get_width() / 2
                    ),
                    count + label_padding,
                    f"{int(count):,}",
                    ha="center",
                    va="bottom",
                    fontsize=10
                )

    plt.tight_layout()
    plt.show()

    # --------------------------------------------------------
    # Figures 2 and 3:
    # Average number of exact 1s and exact 0s per note,
    # grouped by note-length bin
    # --------------------------------------------------------

    for method_score_value in [1, 0]:

        score_indicator_df = filtered_df[
            [
                "Subject ID",
                "char_count_bin"
            ] + score_cols
        ].copy()

        for column in score_cols:
            score_indicator_df[column] = (
                score_indicator_df[column]
                .eq(method_score_value)
                .astype(int)
            )

        # Count the requested score for each note
        subject_score_counts = (
            score_indicator_df
            .dropna(subset=["char_count_bin"])
            .groupby(
                "Subject ID",
                observed=True
            )
            .agg(
                char_count_bin=(
                    "char_count_bin",
                    "first"
                ),
                Manual_Score=(
                    "Manual_Score",
                    "sum"
                ),
                GPT_Score=(
                    "GPT_Score",
                    "sum"
                ),
                GPT_Mini_Score=(
                    "GPT_Mini_Score",
                    "sum"
                ),
                Claude_Sonnet_Score=(
                    "Claude_Sonnet_Score",
                    "sum"
                )
            )
            .reset_index()
        )

        avg_by_length = (
            subject_score_counts
            .groupby(
                "char_count_bin",
                observed=True
            )[score_cols]
            .mean()
            .reindex(bin_labels)
        )

        avg_by_length.index.name = (
            "char_count_bin"
        )

        average_score_results[
            (
                adjudicated_value,
                method_score_value
            )
        ] = avg_by_length

        x_positions = np.arange(
            len(bin_labels)
        )

        fig, ax = plt.subplots(figsize=(12, 6))

        for column in score_cols:
            ax.plot(
                x_positions,
                avg_by_length[column].values,
                marker="o",
                label=labels[column],
                color=colors[column],
                linewidth=2
            )

        ax.set_xticks(x_positions)

        ax.set_xticklabels(
            bin_labels,
            rotation=45,
            ha="right"
        )

        ax.set_xlabel(
            "Note-length range in characters",
            fontsize=12
        )

        ax.set_ylabel(
            f"Average phenotypes scored "
            f"{method_score_value} per note",
            fontsize=12
        )

        ax.set_title(
            f"Average Phenotypes Scored {method_score_value} "
            "per Note by Note Length\n"
            f"(Adjudicated Score = {adjudicated_value})",
            fontsize=15,
            pad=15
        )

        ax.legend(frameon=False, fontsize=11)
        ax.set_ylim(bottom=0)
        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)

        plt.tight_layout()
        plt.show()

        print(
            f"Average number scored {method_score_value} "
            f"when adjudicated score = {adjudicated_value}:"
        )

        display(avg_by_length)

# Summary of overall unique-note character counts
display(
    char_counts_adjudication
    .describe()
    .to_frame(name="Character count")
)

In [0]:
# ============================================================
# Variable-width age bins:
# 1-year bins through 14
# 2-year bins through 20
# 5-year bins through 30
# 10-year bins through 70
# One 70+ overflow bin
# ============================================================

import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

df = combined_df_full_info_adjudications.copy()

score_cols = [
    "Manual_Score",
    "GPT_Score",
    "GPT_Mini_Score",
    "Claude_Sonnet_Score"
]

colors = {
    "Manual_Score": "#D4B896",
    "GPT_Score": "#4C9BE8",
    "GPT_Mini_Score": "#57C4A7",
    "Claude_Sonnet_Score": "#E87B4C"
}

labels = {
    "Manual_Score": "Manual",
    "GPT_Score": "GPT 5.4",
    "GPT_Mini_Score": "GPT 5.4 Mini",
    "Claude_Sonnet_Score": "Claude Sonnet 4.6"
}

required_columns = [
    "Subject ID",
    "NOTE_FILE_TIME_DTTM",
    "D.O.B.",
    "adjudicated_score"
] + score_cols

missing_columns = [
    column
    for column in required_columns
    if column not in df.columns
]

if missing_columns:
    raise KeyError(
        f"Missing required columns: {missing_columns}"
    )

# ------------------------------------------------------------
# Clean and validate scores
# ------------------------------------------------------------

numeric_columns = (
    score_cols + ["adjudicated_score"]
)

df[numeric_columns] = (
    df[numeric_columns]
    .apply(
        pd.to_numeric,
        errors="coerce"
    )
)

for column in numeric_columns:
    unexpected_values = (
        df.loc[
            df[column].notna()
            & ~df[column].isin([0, 1]),
            column
        ]
        .unique()
        .tolist()
    )

    if unexpected_values:
        raise ValueError(
            f"{column} contains values other than 0 and 1: "
            f"{sorted(unexpected_values)}"
        )

# ------------------------------------------------------------
# Calculate age at note
# ------------------------------------------------------------

df["note_date"] = pd.to_datetime(
    df["NOTE_FILE_TIME_DTTM"],
    errors="coerce"
)

df["dob"] = pd.to_datetime(
    df["D.O.B."],
    errors="coerce"
)

df["age_at_note_years"] = (
    (df["note_date"] - df["dob"]).dt.days
    / 365.25
)

n_parsed = (
    df[["note_date", "dob"]]
    .notna()
    .all(axis=1)
    .sum()
)

print(
    "Rows with both dates successfully parsed: "
    f"{n_parsed:,} / {len(df):,}"
)

valid_ages = (
    df["age_at_note_years"]
    .dropna()
)

if valid_ages.empty:
    raise ValueError(
        "No valid ages could be calculated from the note date "
        "and date of birth."
    )

negative_age_count = (
    df["age_at_note_years"] < 0
).sum()

if negative_age_count > 0:
    print(
        f"WARNING: {negative_age_count:,} rows have "
        "negative calculated ages and will fall outside "
        "the defined bins."
    )

# ------------------------------------------------------------
# Build common variable-width age bins
# ------------------------------------------------------------

max_age = valid_ages.max()

edges = list(range(0, 15, 1))
edges += list(range(16, 21, 2))
edges += [25, 30]
edges += list(range(40, 71, 10))
edges.append(
    max(max_age + 1, 71)
)

edges = sorted(set(edges))

print(f"Bin edges: {edges}")

df["age_bin"] = pd.cut(
    df["age_at_note_years"],
    bins=edges,
    right=False,
    include_lowest=True
)

age_categories = (
    df["age_bin"].cat.categories
)

bin_positions = np.arange(
    len(age_categories)
)

tick_labels = []

for interval in age_categories:
    width = interval.right - interval.left

    if interval.left >= 70:
        tick_labels.append("70+")
    elif width == 1:
        tick_labels.append(
            f"{int(interval.right)}"
        )
    else:
        tick_labels.append(
            f"{int(interval.left)}–"
            f"{int(interval.right)}"
        )

bin_explanation = (
    "X-axis bins: 1-year intervals up to 14, "
    "2-year intervals up to 20, 5-year intervals up to 30, "
    "10-year intervals up to 70, then a single 70+ bin. "
    "Bins are evenly spaced categorically, not scaled to age."
)

notes_by_bin_results = {}
average_score_results = {}

# Use the same y-axis for all four phenotype-per-note graphs
phenotype_y_min = 0
phenotype_y_max = 5
phenotype_y_ticks = np.arange(
    phenotype_y_min,
    phenotype_y_max + 1,
    1
)


# ============================================================
# Produce graphs for adjudicated scores of 1 and 0
# ============================================================

for adjudicated_value in [1, 0]:

    filtered_df = df[
        df["adjudicated_score"].eq(
            adjudicated_value
        )
    ].copy()

    print()
    print(f"Adjudicated score = {adjudicated_value}")
    print("-" * 35)
    print(f"Adjudicated rows: {len(filtered_df):,}")
    print(
        "Unique notes represented: "
        f"{filtered_df['Subject ID'].nunique():,}"
    )

    # --------------------------------------------------------
    # Figure 1: Distribution of notes by age bin
    # --------------------------------------------------------

    subject_level = (
        filtered_df
        .dropna(subset=["age_bin"])
        .drop_duplicates(subset="Subject ID")
        .copy()
    )

    notes_by_bin = (
        subject_level["age_bin"]
        .value_counts(sort=False)
        .reindex(
            age_categories,
            fill_value=0
        )
    )

    notes_by_bin.index.name = "age_bin"

    notes_by_bin_results[
        adjudicated_value
    ] = notes_by_bin

    fig1, ax1 = plt.subplots(figsize=(12, 5))

    ax1.plot(
        bin_positions,
        notes_by_bin.values,
        marker="o",
        color="#4C9BE8",
        linewidth=2
    )

    ax1.set_xticks(bin_positions)

    ax1.set_xticklabels(
        tick_labels,
        rotation=45,
        ha="right"
    )

    ax1.set_xlabel(
        "Age at note (years, binned)",
        fontsize=12
    )

    ax1.set_ylabel(
        "Number of notes",
        fontsize=12
    )

    ax1.set_title(
        "Distribution of Notes by Age at Note\n"
        f"(Adjudicated Score = {adjudicated_value})",
        fontsize=15,
        pad=15
    )

    ax1.set_ylim(bottom=0)
    ax1.spines["top"].set_visible(False)
    ax1.spines["right"].set_visible(False)

    fig1.text(
        0.5,
        0.01,
        bin_explanation,
        ha="center",
        va="bottom",
        fontsize=10,
        style="italic",
        color="#555555"
    )

    plt.tight_layout(
        rect=[0, 0.12, 1, 1]
    )

    plt.show()

    # --------------------------------------------------------
    # Figures 2 and 3:
    # Average number of exact 1s and exact 0s assigned
    # per note, by age bin
    # --------------------------------------------------------

    for method_score_value in [1, 0]:

        score_indicator_df = filtered_df[
            [
                "Subject ID",
                "age_bin"
            ] + score_cols
        ].copy()

        # Indicate whether each method assigned the
        # requested score
        for column in score_cols:
            score_indicator_df[column] = (
                score_indicator_df[column]
                .eq(method_score_value)
                .astype(int)
            )

        # Count the requested score for each note
        subject_score_counts = (
            score_indicator_df
            .dropna(subset=["age_bin"])
            .groupby(
                "Subject ID",
                observed=True
            )
            .agg(
                age_bin=(
                    "age_bin",
                    "first"
                ),
                Manual_Score=(
                    "Manual_Score",
                    "sum"
                ),
                GPT_Score=(
                    "GPT_Score",
                    "sum"
                ),
                GPT_Mini_Score=(
                    "GPT_Mini_Score",
                    "sum"
                ),
                Claude_Sonnet_Score=(
                    "Claude_Sonnet_Score",
                    "sum"
                )
            )
            .reset_index()
        )

        avg_by_bin = (
            subject_score_counts
            .groupby(
                "age_bin",
                observed=True
            )[score_cols]
            .mean()
            .reindex(age_categories)
        )

        avg_by_bin.index.name = "age_bin"

        average_score_results[
            (
                adjudicated_value,
                method_score_value
            )
        ] = avg_by_bin

        fig, ax = plt.subplots(figsize=(12, 6))

        for column in score_cols:
            ax.plot(
                bin_positions,
                avg_by_bin[column].values,
                marker="o",
                label=labels[column],
                color=colors[column],
                linewidth=2
            )

        ax.set_xticks(bin_positions)

        ax.set_xticklabels(
            tick_labels,
            rotation=45,
            ha="right"
        )

        ax.set_xlabel(
            "Age at note (years, binned)",
            fontsize=12
        )

        ax.set_ylabel(
            f"Average phenotypes scored "
            f"{method_score_value} per note",
            fontsize=12
        )

        ax.set_title(
            f"Average Phenotypes Scored {method_score_value} "
            "per Note by Age at Note\n"
            f"(Adjudicated Score = {adjudicated_value})",
            fontsize=15,
            pad=15
        )

        # Identical y-axis across all four graphs
        ax.set_ylim(
            phenotype_y_min,
            phenotype_y_max
        )

        ax.set_yticks(
            phenotype_y_ticks
        )

        ax.legend(
            loc="center left",
            bbox_to_anchor=(1.02, 0.5),
            frameon=False,
            fontsize=10
        )

        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)

        fig.text(
            0.41,
            0.01,
            bin_explanation,
            ha="center",
            va="bottom",
            fontsize=10,
            style="italic",
            color="#555555"
        )

        plt.tight_layout(
            rect=[0, 0.12, 0.82, 1]
        )

        plt.show()

        print(
            f"Average number scored {method_score_value} "
            f"when adjudicated score = {adjudicated_value}:"
        )

        display(avg_by_bin)

    print(
        "Notes by age bin when adjudicated score "
        f"= {adjudicated_value}:"
    )

    display(notes_by_bin)

In [0]:
# ============================================================
# Variable-width age bins:
# 1-year bins through 14
# 2-year bins through 20
# 5-year bins through 30
# 10-year bins through 70
# One 70+ overflow bin
#
# Output: one 3 x 2 multi-panel figure
#   Left column  (A, B, C): adjudicated score = 1
#   Right column (D, E, F): adjudicated score = 0
#   Row 1: distribution of notes by age
#   Row 2: average phenotypes scored 1 per note
#   Row 3: average phenotypes scored 0 per note
# ============================================================

import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

df = combined_df_full_info_adjudications.copy()

score_cols = [
    "Manual_Score",
    "GPT_Score",
    "GPT_Mini_Score",
    "Claude_Sonnet_Score"
]

colors = {
    "Manual_Score": "#D4B896",
    "GPT_Score": "#4C9BE8",
    "GPT_Mini_Score": "#57C4A7",
    "Claude_Sonnet_Score": "#E87B4C"
}

labels = {
    "Manual_Score": "Manual",
    "GPT_Score": "GPT 5.4",
    "GPT_Mini_Score": "GPT 5.4 Mini",
    "Claude_Sonnet_Score": "Claude Sonnet 4.6"
}

required_columns = [
    "Subject ID",
    "NOTE_FILE_TIME_DTTM",
    "D.O.B.",
    "adjudicated_score"
] + score_cols

missing_columns = [c for c in required_columns if c not in df.columns]

if missing_columns:
    raise KeyError(f"Missing required columns: {missing_columns}")

# ------------------------------------------------------------
# Clean and validate scores
# ------------------------------------------------------------

numeric_columns = score_cols + ["adjudicated_score"]

df[numeric_columns] = df[numeric_columns].apply(pd.to_numeric, errors="coerce")

for column in numeric_columns:
    unexpected_values = (
        df.loc[df[column].notna() & ~df[column].isin([0, 1]), column]
        .unique()
        .tolist()
    )
    if unexpected_values:
        raise ValueError(
            f"{column} contains values other than 0 and 1: "
            f"{sorted(unexpected_values)}"
        )

# ------------------------------------------------------------
# Calculate age at note
# ------------------------------------------------------------

df["note_date"] = pd.to_datetime(df["NOTE_FILE_TIME_DTTM"], errors="coerce")
df["dob"] = pd.to_datetime(df["D.O.B."], errors="coerce")
df["age_at_note_years"] = (df["note_date"] - df["dob"]).dt.days / 365.25

n_parsed = df[["note_date", "dob"]].notna().all(axis=1).sum()
print(f"Rows with both dates successfully parsed: {n_parsed:,} / {len(df):,}")

valid_ages = df["age_at_note_years"].dropna()

if valid_ages.empty:
    raise ValueError(
        "No valid ages could be calculated from the note date "
        "and date of birth."
    )

negative_age_count = (df["age_at_note_years"] < 0).sum()

if negative_age_count > 0:
    print(
        f"WARNING: {negative_age_count:,} rows have negative calculated "
        "ages and will fall outside the defined bins."
    )

# ------------------------------------------------------------
# Build common variable-width age bins
# ------------------------------------------------------------

max_age = valid_ages.max()

edges = list(range(0, 15, 1))
edges += list(range(16, 21, 2))
edges += [25, 30]
edges += list(range(40, 71, 10))
edges.append(max(max_age + 1, 71))
edges = sorted(set(edges))

print(f"Bin edges: {edges}")

df["age_bin"] = pd.cut(
    df["age_at_note_years"],
    bins=edges,
    right=False,
    include_lowest=True
)

age_categories = df["age_bin"].cat.categories
bin_positions = np.arange(len(age_categories))

tick_labels = []
for interval in age_categories:
    width = interval.right - interval.left
    if interval.left >= 70:
        tick_labels.append("70+")
    elif width == 1:
        tick_labels.append(f"{int(interval.right)}")
    else:
        tick_labels.append(f"{int(interval.left)}–{int(interval.right)}")

bin_explanation = (
    "X-axis bins: 1-year intervals up to 14, "
    "2-year intervals up to 20, 5-year intervals up to 30, "
    "10-year intervals up to 70, then a single 70+ bin. "
    "Bins are evenly spaced categorically, not scaled to age."
)

# ============================================================
# Step 1: Compute all results for adjudicated scores of 1 and 0
# ============================================================

notes_by_bin_results = {}
average_score_results = {}

for adjudicated_value in [1, 0]:

    filtered_df = df[df["adjudicated_score"].eq(adjudicated_value)].copy()

    print()
    print(f"Adjudicated score = {adjudicated_value}")
    print("-" * 35)
    print(f"Adjudicated rows: {len(filtered_df):,}")
    print(f"Unique notes represented: {filtered_df['Subject ID'].nunique():,}")

    # Distribution of notes by age bin
    subject_level = (
        filtered_df
        .dropna(subset=["age_bin"])
        .drop_duplicates(subset="Subject ID")
    )

    notes_by_bin = (
        subject_level["age_bin"]
        .value_counts(sort=False)
        .reindex(age_categories, fill_value=0)
    )
    notes_by_bin.index.name = "age_bin"
    notes_by_bin_results[adjudicated_value] = notes_by_bin

    # Average number of exact 1s and exact 0s assigned per note
    for method_score_value in [1, 0]:

        score_indicator_df = filtered_df[["Subject ID", "age_bin"] + score_cols].copy()

        # Indicate whether each method assigned the requested score
        for column in score_cols:
            score_indicator_df[column] = (
                score_indicator_df[column].eq(method_score_value).astype(int)
            )

        # Count the requested score for each note
        subject_score_counts = (
            score_indicator_df
            .dropna(subset=["age_bin"])
            .groupby("Subject ID", observed=True)
            .agg(
                age_bin=("age_bin", "first"),
                **{column: (column, "sum") for column in score_cols}
            )
            .reset_index()
        )

        avg_by_bin = (
            subject_score_counts
            .groupby("age_bin", observed=True)[score_cols]
            .mean()
            .reindex(age_categories)
        )
        avg_by_bin.index.name = "age_bin"
        average_score_results[(adjudicated_value, method_score_value)] = avg_by_bin

# ============================================================
# Step 2: Plot everything as one 3 x 2 figure
# ============================================================

# Panel layout: (row, column) -> (panel letter, adjudicated value, method score)
# method score of None means the note-distribution panel
panel_layout = {
    (0, 0): ("A", 1, None),
    (1, 0): ("B", 1, 1),
    (2, 0): ("C", 1, 0),
    (0, 1): ("D", 0, None),
    (1, 1): ("E", 0, 1),
    (2, 1): ("F", 0, 0),
}

# Same y-axis for the two note-distribution panels (A and D)
count_y_max = max(s.max() for s in notes_by_bin_results.values())
count_y_max = int(np.ceil(count_y_max * 1.05 / 10.0) * 10) if count_y_max > 0 else 1

# Same y-axis for all four phenotype-per-note panels (B, C, E, F)
phenotype_y_min = 0
phenotype_y_max = 5
phenotype_y_ticks = np.arange(phenotype_y_min, phenotype_y_max + 1, 1)

# Set to False to show the bin explanation once at the bottom of the
# figure instead of under every panel
footnote_under_each_panel = True

fig, axes = plt.subplots(
    nrows=3,
    ncols=2,
    figsize=(24, 20),
    layout="constrained"
)


def style_x_axis(ax):
    ax.set_xticks(bin_positions)
    ax.set_xticklabels(tick_labels, rotation=45, ha="right", fontsize=9)
    ax.set_xlabel("Age at note (years, binned)", fontsize=11)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)


def add_panel_letter(ax, letter):
    ax.annotate(
        letter,
        xy=(0, 1),
        xycoords="axes fraction",
        xytext=(-45, 18),
        textcoords="offset points",
        fontsize=22,
        fontweight="bold",
        ha="right",
        va="bottom"
    )


def add_panel_footnote(ax):
    ax.annotate(
        bin_explanation,
        xy=(0, 0),
        xycoords="axes fraction",
        xytext=(-45, -75),
        textcoords="offset points",
        ha="left",
        va="top",
        fontsize=8,
        style="italic",
        color="#555555"
    )


for (row, col), (letter, adjudicated_value, method_score_value) in panel_layout.items():
    ax = axes[row, col]

    if method_score_value is None:
        # Distribution of notes by age at note
        notes_by_bin = notes_by_bin_results[adjudicated_value]

        ax.plot(
            bin_positions,
            notes_by_bin.values,
            marker="o",
            color="#4C9BE8",
            linewidth=2
        )
        ax.set_ylabel("Number of notes", fontsize=11)
        ax.set_title(
            "Distribution of Notes by Age at Note\n"
            f"(Adjudicated Score = {adjudicated_value})",
            fontsize=13
        )
        ax.set_ylim(0, count_y_max)

    else:
        # Average phenotypes scored 1 or 0 per note
        avg_by_bin = average_score_results[(adjudicated_value, method_score_value)]

        for column in score_cols:
            ax.plot(
                bin_positions,
                avg_by_bin[column].values,
                marker="o",
                label=labels[column],
                color=colors[column],
                linewidth=2
            )

        ax.set_ylabel(
            f"Average phenotypes scored {method_score_value} per note",
            fontsize=11
        )
        ax.set_title(
            f"Average Phenotypes Scored {method_score_value} "
            "per Note by Age at Note\n"
            f"(Adjudicated Score = {adjudicated_value})",
            fontsize=13
        )
        ax.set_ylim(phenotype_y_min, phenotype_y_max)
        ax.set_yticks(phenotype_y_ticks)
        ax.legend(
            loc="center left",
            bbox_to_anchor=(1.02, 0.5),
            frameon=False,
            fontsize=10
        )

    style_x_axis(ax)
    add_panel_letter(ax, letter)

    if footnote_under_each_panel:
        add_panel_footnote(ax)

if not footnote_under_each_panel:
    fig.supxlabel(
        bin_explanation,
        fontsize=10,
        style="italic",
        color="#555555"
    )

fig.get_layout_engine().set(h_pad=0.25, w_pad=0.3, hspace=0.08, wspace=0.08)

fig.savefig("age_bin_panels_A-F.png", dpi=300, bbox_inches="tight")
fig.savefig("age_bin_panels_A-F.pdf", bbox_inches="tight")
plt.show()

# ============================================================
# Step 3: Display the underlying tables
# ============================================================

for adjudicated_value in [1, 0]:
    for method_score_value in [1, 0]:
        print(
            f"Average number scored {method_score_value} "
            f"when adjudicated score = {adjudicated_value}:"
        )
        display(average_score_results[(adjudicated_value, method_score_value)])

    print(f"Notes by age bin when adjudicated score = {adjudicated_value}:")
    display(notes_by_bin_results[adjudicated_value])

In [0]:
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

df = combined_df_full_info_adjudications.copy()

score_cols = [
    "Manual_Score",
    "GPT_Score",
    "GPT_Mini_Score",
    "Claude_Sonnet_Score"
]

colors = {
    "Manual_Score": "#D4B896",
    "GPT_Score": "#4C9BE8",
    "GPT_Mini_Score": "#57C4A7",
    "Claude_Sonnet_Score": "#E87B4C"
}

labels = {
    "Manual_Score": "Manual",
    "GPT_Score": "GPT 5.4",
    "GPT_Mini_Score": "GPT 5.4 Mini",
    "Claude_Sonnet_Score": "Claude Sonnet 4.6"
}

# Ensure scores are numeric
df["adjudicated_score"] = pd.to_numeric(
    df["adjudicated_score"],
    errors="coerce"
)

df[score_cols] = df[score_cols].apply(
    pd.to_numeric,
    errors="coerce"
)

# Convert note-file timestamps to datetime
df["review_date"] = pd.to_datetime(
    df["NOTE_FILE_TIME_DTTM"],
    errors="coerce"
)

n_unparsed = (
    df["NOTE_FILE_TIME_DTTM"].notna().sum()
    - df["review_date"].notna().sum()
)

print(
    f"Rows with a date that failed to parse: "
    f"{n_unparsed:,}"
)

# Choose "Q" for quarterly or "M" for monthly
period_freq = "Q"

period_name = (
    "quarter" if period_freq == "Q" else "month"
)

df["review_period"] = (
    df["review_date"].dt.to_period(period_freq)
)

valid_periods = df["review_period"].dropna()

if valid_periods.empty:
    raise ValueError(
        "No valid note-file timestamps were available."
    )

# Use the same complete period range for every graph
all_periods = pd.period_range(
    start=valid_periods.min(),
    end=valid_periods.max(),
    freq=period_freq
)

x_positions = np.arange(len(all_periods))
x_labels = all_periods.astype(str)

# Store output tables
notes_by_period_results = {}
average_score_results = {}


# ============================================================
# Produce graphs for adjudicated scores of 1 and 0
# ============================================================

for adjudicated_value in [1, 0]:

    filtered_df = df[
        df["adjudicated_score"].eq(adjudicated_value)
    ].copy()

    print()
    print(f"Adjudicated score = {adjudicated_value}")
    print("-" * 35)
    print(f"Phenotype rows: {len(filtered_df):,}")
    print(
        "Unique notes: "
        f"{filtered_df['Subject ID'].nunique():,}"
    )

    # --------------------------------------------------------
    # Figure 1: Distribution of notes by period
    # --------------------------------------------------------

    subject_level = (
        filtered_df
        .dropna(subset=["review_period"])
        .drop_duplicates(subset="Subject ID")
    )

    notes_by_period = (
        subject_level["review_period"]
        .value_counts()
        .sort_index()
        .reindex(all_periods, fill_value=0)
    )

    notes_by_period.index.name = "review_period"

    notes_by_period_results[
        adjudicated_value
    ] = notes_by_period

    fig1, ax1 = plt.subplots(figsize=(11, 5))

    ax1.bar(
        x_positions,
        notes_by_period.values,
        color="#4C9BE8",
        edgecolor="white"
    )

    ax1.set_xticks(x_positions)

    ax1.set_xticklabels(
        x_labels,
        rotation=45,
        ha="right"
    )

    ax1.set_xlabel(
        f"Note file time period ({period_name})",
        fontsize=12
    )

    ax1.set_ylabel(
        "Number of notes",
        fontsize=12
    )

    ax1.set_title(
        "Distribution of Notes by Period of Note File Time\n"
        f"(Adjudicated Score = {adjudicated_value})",
        fontsize=15,
        pad=15
    )

    ax1.set_ylim(bottom=0)
    ax1.spines["top"].set_visible(False)
    ax1.spines["right"].set_visible(False)

    if len(notes_by_period) > 0:
        label_padding = max(
            notes_by_period.max() * 0.01,
            0.1
        )

        for i, value in enumerate(
            notes_by_period.values
        ):
            ax1.text(
                i,
                value + label_padding,
                f"{int(value):,}",
                ha="center",
                va="bottom",
                fontsize=10
            )

    plt.tight_layout()
    plt.show()

    # --------------------------------------------------------
    # Figures 2 and 3:
    # Average number of exact 1s and exact 0s assigned
    # per note, by note file time
    # --------------------------------------------------------

    for method_score_value in [1, 0]:

        score_indicator_df = filtered_df[
            [
                "Subject ID",
                "review_period"
            ] + score_cols
        ].copy()

        # Convert each scoring column into an indicator for
        # whether it equals the requested score
        for column in score_cols:
            score_indicator_df[column] = (
                score_indicator_df[column]
                .eq(method_score_value)
                .astype(int)
            )

        # Count exact 1s or exact 0s for each note
        subject_score_counts = (
            score_indicator_df
            .dropna(subset=["review_period"])
            .groupby("Subject ID")
            .agg(
                review_period=(
                    "review_period",
                    "first"
                ),
                Manual_Score=(
                    "Manual_Score",
                    "sum"
                ),
                GPT_Score=(
                    "GPT_Score",
                    "sum"
                ),
                GPT_Mini_Score=(
                    "GPT_Mini_Score",
                    "sum"
                ),
                Claude_Sonnet_Score=(
                    "Claude_Sonnet_Score",
                    "sum"
                )
            )
            .reset_index()
        )

        # Calculate average counts per note in each period
        avg_by_period = (
            subject_score_counts
            .groupby("review_period")[score_cols]
            .mean()
            .reindex(all_periods)
        )

        avg_by_period.index.name = "review_period"

        average_score_results[
            (adjudicated_value, method_score_value)
        ] = avg_by_period

        fig, ax = plt.subplots(figsize=(11, 6))

        for column in score_cols:
            ax.plot(
                x_positions,
                avg_by_period[column].values,
                marker="o",
                label=labels[column],
                color=colors[column],
                linewidth=2
            )

        ax.set_xticks(x_positions)

        ax.set_xticklabels(
            x_labels,
            rotation=45,
            ha="right"
        )

        ax.set_xlabel(
            f"Note file time period ({period_name})",
            fontsize=12
        )

        ax.set_ylabel(
            f"Average phenotypes scored "
            f"{method_score_value} per note",
            fontsize=12
        )

        ax.set_title(
            f"Average Phenotypes Scored {method_score_value} "
            "per Note by Period of Note File Time\n"
            f"(Adjudicated Score = {adjudicated_value})",
            fontsize=15,
            pad=15
        )

        ax.legend(
            frameon=False,
            fontsize=11
        )

        ax.set_ylim(bottom=0)
        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)

        plt.tight_layout()
        plt.show()

        print(
            f"Average number scored {method_score_value} "
            f"when adjudicated score = {adjudicated_value}:"
        )

        display(avg_by_period)

    print(
        f"Notes by period when adjudicated score "
        f"= {adjudicated_value}:"
    )

    display(notes_by_period)

In [0]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.ticker import MaxNLocator

df = combined_df_full_info_adjudications.copy()

manual_column = "Manual_Score"

model_columns = {
    "GPT_Score": "GPT 5.4",
    "GPT_Mini_Score": "GPT 5.4 Mini",
    "Claude_Sonnet_Score": "Claude Sonnet 4.6"
}

required_columns = [
    manual_column,
    "adjudicated_score"
] + list(model_columns.keys())

missing_columns = [
    column
    for column in required_columns
    if column not in df.columns
]

if missing_columns:
    raise KeyError(
        f"Missing required columns: {missing_columns}"
    )

# ------------------------------------------------------------
# Clean and validate scores
# ------------------------------------------------------------

df[required_columns] = (
    df[required_columns]
    .apply(
        pd.to_numeric,
        errors="coerce"
    )
)

for column in required_columns:
    n_missing = df[column].isna().sum()

    if n_missing > 0:
        print(
            f"WARNING: {n_missing:,} missing or nonnumeric "
            f"values in {column}"
        )

    unexpected_values = (
        df.loc[
            df[column].notna()
            & ~df[column].isin([0, 1]),
            column
        ]
        .unique()
        .tolist()
    )

    if unexpected_values:
        raise ValueError(
            f"{column} contains values other than 0 and 1: "
            f"{sorted(unexpected_values)}"
        )

# ------------------------------------------------------------
# Construct the two disagreement-direction panels
# ------------------------------------------------------------

panel_a_rows = []
panel_b_rows = []

for model_column, model_label in model_columns.items():

    complete_rows = df[
        [
            manual_column,
            model_column,
            "adjudicated_score"
        ]
    ].dropna()

    # Panel A: Manual = 0 and AI = 1
    panel_a_subset = complete_rows[
        complete_rows[manual_column].eq(0)
        & complete_rows[model_column].eq(1)
    ]

    # AI upheld means adjudicated score = 1
    panel_a_ai_upheld = (
        panel_a_subset["adjudicated_score"]
        .eq(1)
        .sum()
    )

    # Manual upheld means adjudicated score = 0
    panel_a_manual_upheld = (
        panel_a_subset["adjudicated_score"]
        .eq(0)
        .sum()
    )

    panel_a_rows.append({
        "Model": model_label,
        "AI_upheld": panel_a_ai_upheld,
        "Manual_upheld": panel_a_manual_upheld,
        "Total_n": len(panel_a_subset)
    })

    # Panel B: Manual = 1 and AI = 0
    panel_b_subset = complete_rows[
        complete_rows[manual_column].eq(1)
        & complete_rows[model_column].eq(0)
    ]

    # AI upheld means adjudicated score = 0
    panel_b_ai_upheld = (
        panel_b_subset["adjudicated_score"]
        .eq(0)
        .sum()
    )

    # Manual upheld means adjudicated score = 1
    panel_b_manual_upheld = (
        panel_b_subset["adjudicated_score"]
        .eq(1)
        .sum()
    )

    panel_b_rows.append({
        "Model": model_label,
        "AI_upheld": panel_b_ai_upheld,
        "Manual_upheld": panel_b_manual_upheld,
        "Total_n": len(panel_b_subset)
    })

panel_a = (
    pd.DataFrame(panel_a_rows)
    .set_index("Model")
)

panel_b = (
    pd.DataFrame(panel_b_rows)
    .set_index("Model")
)

# Confirm stacked counts equal total n
for panel_name, panel_table in {
    "Panel A": panel_a,
    "Panel B": panel_b
}.items():

    calculated_totals = (
        panel_table["AI_upheld"]
        + panel_table["Manual_upheld"]
    )

    if not calculated_totals.equals(
        panel_table["Total_n"]
    ):
        raise ValueError(
            f"{panel_name} contains totals that do not match "
            "the sum of its stacked sections."
        )

# ------------------------------------------------------------
# Color scheme
# ------------------------------------------------------------

# Blue always means AI score upheld
ai_colors = {
    0: "#79A9D8",  # Lighter blue: adjudicated score = 0
    1: "#2F6FB3"   # Darker blue: adjudicated score = 1
}

# Orange always means manual score upheld
manual_colors = {
    0: "#F2B36F",  # Lighter orange: adjudicated score = 0
    1: "#D97820"   # Darker orange: adjudicated score = 1
}

# Text colors selected for contrast
ai_text_colors = {
    0: "#17212B",
    1: "white"
}

manual_text_colors = {
    0: "#17212B",
    1: "white"
}

# ------------------------------------------------------------
# Plot settings
# ------------------------------------------------------------

maximum_total = max(
    panel_a["Total_n"].max(),
    panel_b["Total_n"].max()
)

if maximum_total == 0:
    raise ValueError(
        "No model–manual disagreements were found."
    )

label_padding = max(
    maximum_total * 0.02,
    1
)

y_positions = np.arange(len(panel_a))

fig, axes = plt.subplots(
    nrows=2,
    ncols=1,
    figsize=(15, 13.5),
    sharex=True
)

# ------------------------------------------------------------
# Add only counts inside the boxes
# ------------------------------------------------------------

# Font sizes tried, largest first, for the counts inside the boxes.
# A count that does not fit its box even at the smallest size is
# moved next to the "Total n" label instead (see the fitting step
# after the layout is finalized).
count_label_font_sizes = [14, 12, 10]

count_labels = []


def add_count_label(
    ax,
    left,
    width,
    y_position,
    text_color,
    segment_name,
    total_label
):
    if width == 0:
        return

    segment_center = (
        left + width / 2
    )

    label = ax.text(
        segment_center,
        y_position,
        f"n = {int(width):,}",
        ha="center",
        va="center",
        color=text_color,
        fontsize=count_label_font_sizes[0],
        fontweight="bold"
    )

    count_labels.append({
        "ax": ax,
        "label": label,
        "left": left,
        "width": width,
        "segment_name": segment_name,
        "total_label": total_label
    })

# ------------------------------------------------------------
# Plotting function
# ------------------------------------------------------------

def plot_panel(
    ax,
    table,
    panel_title,
    ai_adjudicated_value,
    manual_adjudicated_value
):
    ai_counts = (
        table["AI_upheld"]
        .to_numpy()
    )

    manual_counts = (
        table["Manual_upheld"]
        .to_numpy()
    )

    totals = (
        table["Total_n"]
        .to_numpy()
    )

    ai_color = ai_colors[
        ai_adjudicated_value
    ]

    manual_color = manual_colors[
        manual_adjudicated_value
    ]

    # AI-upheld segment is displayed first on the left
    ax.barh(
        y_positions,
        ai_counts,
        height=0.66,
        color=ai_color,
        edgecolor="white"
    )

    # Manual-upheld segment is displayed second on the right
    ax.barh(
        y_positions,
        manual_counts,
        left=ai_counts,
        height=0.66,
        color=manual_color,
        edgecolor="white"
    )

    for i in range(len(table)):

        total_label = ax.text(
            totals[i] + label_padding,
            y_positions[i],
            f"Total n = {int(totals[i]):,}",
            ha="left",
            va="center",
            fontsize=14,
            color="#1F2933",
            linespacing=1.35
        )

        # AI-upheld count on the left
        add_count_label(
            ax=ax,
            left=0,
            width=ai_counts[i],
            y_position=y_positions[i],
            text_color=ai_text_colors[
                ai_adjudicated_value
            ],
            segment_name="AI upheld",
            total_label=total_label
        )

        # Manual-upheld count on the right
        add_count_label(
            ax=ax,
            left=ai_counts[i],
            width=manual_counts[i],
            y_position=y_positions[i],
            text_color=manual_text_colors[
                manual_adjudicated_value
            ],
            segment_name="manual upheld",
            total_label=total_label
        )

    ax.set_yticks(y_positions)

    ax.set_yticklabels(
        table.index,
        fontsize=16
    )

    ax.tick_params(
        axis="x",
        labelsize=14
    )

    # Place GPT first at the top
    ax.invert_yaxis()

    ax.set_title(
        panel_title,
        loc="left",
        fontsize=19,
        fontweight="bold",
        pad=72
    )

    ax.set_ylabel("")

    # Use the same raw-count scale for both panels
    ax.set_xlim(
        0,
        maximum_total * 1.27
    )

    ax.xaxis.set_major_locator(
        MaxNLocator(integer=True)
    )

    ax.grid(
        axis="x",
        color="#D5DDE5",
        linewidth=0.8,
        alpha=0.8
    )

    ax.set_axisbelow(True)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.spines["left"].set_visible(False)

    # Each panel receives its own legend.
    # AI is listed first to match the bar order.
    legend_handles = [
        mpatches.Patch(
            color=ai_color,
            label=(
                "AI score upheld "
                f"(adjudicated score = "
                f"{ai_adjudicated_value})"
            )
        ),
        mpatches.Patch(
            color=manual_color,
            label=(
                "Manual score upheld "
                f"(adjudicated score = "
                f"{manual_adjudicated_value})"
            )
        )
    ]

    ax.legend(
        handles=legend_handles,
        loc="lower center",
        bbox_to_anchor=(0.5, 1.01),
        ncol=2,
        frameon=False,
        fontsize=14
    )

# ------------------------------------------------------------
# Panel A above Panel B
# ------------------------------------------------------------

plot_panel(
    axes[0],
    panel_a,
    (
        "A. Manual = 0 and AI = 1\n"
        "Which score did adjudication support?"
    ),
    ai_adjudicated_value=1,
    manual_adjudicated_value=0
)

plot_panel(
    axes[1],
    panel_b,
    (
        "B. Manual = 1 and AI = 0\n"
        "Which score did adjudication support?"
    ),
    ai_adjudicated_value=0,
    manual_adjudicated_value=1
)

axes[1].set_xlabel(
    "Raw count of adjudicated model–manual disagreements",
    fontsize=16,
    labelpad=10
)

# ------------------------------------------------------------
# Overall figure title and subtitle
# ------------------------------------------------------------

fig.suptitle(
    "Adjudicated Outcomes of Model–Manual "
    "Phenotype Disagreements",
    fontsize=25,
    fontweight="bold",
    y=0.985
)

fig.text(
    0.5,
    0.94,
    (
        "AI-upheld outcomes are shown first in blue; "
        "manual-upheld outcomes are shown second in orange."
    ),
    ha="center",
    va="center",
    fontsize=16,
    color="#52606D"
)

plt.tight_layout(
    rect=[0.03, 0.03, 0.98, 0.93],
    h_pad=4
)

# ------------------------------------------------------------
# Fit the counts inside their boxes now that the layout is final
# ------------------------------------------------------------

fig.canvas.draw()
renderer = fig.canvas.get_renderer()

for item in count_labels:

    ax = item["ax"]
    label = item["label"]

    # Width of the box in pixels
    box_left = ax.transData.transform((item["left"], 0))[0]
    box_right = ax.transData.transform(
        (item["left"] + item["width"], 0)
    )[0]
    available_width = 0.90 * (box_right - box_left)

    fits = False

    for font_size in count_label_font_sizes:

        label.set_fontsize(font_size)

        if (
            label.get_window_extent(renderer).width
            <= available_width
        ):
            fits = True
            break

    if not fits:
        # Too narrow for a readable label: report the count on a
        # second line beside the total instead
        label.set_visible(False)

        item["total_label"].set_text(
            item["total_label"].get_text()
            + f"\n({item['segment_name']} n = "
            + f"{int(item['width']):,})"
        )

plt.show()

# ------------------------------------------------------------
# Display underlying raw-count table
# ------------------------------------------------------------

raw_count_summary = pd.concat(
    [
        panel_a.reset_index().assign(
            Panel="A",
            Disagreement_direction="Manual 0, AI 1",
            AI_upheld_adjudicated_score=1,
            Manual_upheld_adjudicated_score=0
        ),
        panel_b.reset_index().assign(
            Panel="B",
            Disagreement_direction="Manual 1, AI 0",
            AI_upheld_adjudicated_score=0,
            Manual_upheld_adjudicated_score=1
        )
    ],
    ignore_index=True
)

raw_count_summary = raw_count_summary.rename(
    columns={
        "AI_upheld": "AI_upheld_n",
        "Manual_upheld": "Manual_upheld_n"
    }
)

raw_count_summary = raw_count_summary[
    [
        "Panel",
        "Disagreement_direction",
        "Model",
        "AI_upheld_adjudicated_score",
        "AI_upheld_n",
        "Manual_upheld_adjudicated_score",
        "Manual_upheld_n",
        "Total_n"
    ]
]

display(raw_count_summary)

In [0]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.legend_handler import HandlerTuple
from matplotlib.ticker import MaxNLocator

df = combined_df_full_info_adjudications.copy()

manual_column = "Manual_Score"

model_columns = {
    "GPT_Score": "GPT 5.4",
    "GPT_Mini_Score": "GPT 5.4 Mini",
    "Claude_Sonnet_Score": "Claude Sonnet 4.6"
}

required_columns = [
    manual_column,
    "adjudicated_score"
] + list(model_columns.keys())

missing_columns = [
    column
    for column in required_columns
    if column not in df.columns
]

if missing_columns:
    raise KeyError(
        f"Missing required columns: {missing_columns}"
    )

# ------------------------------------------------------------
# Clean and validate scores
# ------------------------------------------------------------

df[required_columns] = (
    df[required_columns]
    .apply(
        pd.to_numeric,
        errors="coerce"
    )
)

for column in required_columns:
    n_missing = df[column].isna().sum()

    if n_missing > 0:
        print(
            f"WARNING: {n_missing:,} missing or nonnumeric "
            f"values in {column}"
        )

    unexpected_values = (
        df.loc[
            df[column].notna()
            & ~df[column].isin([0, 1]),
            column
        ]
        .unique()
        .tolist()
    )

    if unexpected_values:
        raise ValueError(
            f"{column} contains values other than 0 and 1: "
            f"{sorted(unexpected_values)}"
        )

# ------------------------------------------------------------
# Construct the two disagreement-direction panels
# ------------------------------------------------------------

panel_a_rows = []
panel_b_rows = []

for model_column, model_label in model_columns.items():

    complete_rows = df[
        [
            manual_column,
            model_column,
            "adjudicated_score"
        ]
    ].dropna()

    # Panel A: Manual = 0 and AI = 1
    panel_a_subset = complete_rows[
        complete_rows[manual_column].eq(0)
        & complete_rows[model_column].eq(1)
    ]

    # AI upheld means adjudicated score = 1
    panel_a_ai_upheld = (
        panel_a_subset["adjudicated_score"]
        .eq(1)
        .sum()
    )

    # Manual upheld means adjudicated score = 0
    panel_a_manual_upheld = (
        panel_a_subset["adjudicated_score"]
        .eq(0)
        .sum()
    )

    panel_a_rows.append({
        "Model": model_label,
        "AI_upheld": panel_a_ai_upheld,
        "Manual_upheld": panel_a_manual_upheld,
        "Total_n": len(panel_a_subset)
    })

    # Panel B: Manual = 1 and AI = 0
    panel_b_subset = complete_rows[
        complete_rows[manual_column].eq(1)
        & complete_rows[model_column].eq(0)
    ]

    # AI upheld means adjudicated score = 0
    panel_b_ai_upheld = (
        panel_b_subset["adjudicated_score"]
        .eq(0)
        .sum()
    )

    # Manual upheld means adjudicated score = 1
    panel_b_manual_upheld = (
        panel_b_subset["adjudicated_score"]
        .eq(1)
        .sum()
    )

    panel_b_rows.append({
        "Model": model_label,
        "AI_upheld": panel_b_ai_upheld,
        "Manual_upheld": panel_b_manual_upheld,
        "Total_n": len(panel_b_subset)
    })

panel_a = (
    pd.DataFrame(panel_a_rows)
    .set_index("Model")
)

panel_b = (
    pd.DataFrame(panel_b_rows)
    .set_index("Model")
)

# Confirm stacked counts equal total n
for panel_name, panel_table in {
    "Panel A": panel_a,
    "Panel B": panel_b
}.items():

    calculated_totals = (
        panel_table["AI_upheld"]
        + panel_table["Manual_upheld"]
    )

    if not calculated_totals.equals(
        panel_table["Total_n"]
    ):
        raise ValueError(
            f"{panel_name} contains totals that do not match "
            "the sum of its stacked sections."
        )

# ------------------------------------------------------------
# Color scheme
# ------------------------------------------------------------

# Same method colors as the other figures.
# The AI-upheld segment takes its model's color, and the
# manual-upheld segment is always the Manual color.
color_map = {
    "Manual": "#D4B896",
    "GPT 5.4": "#4C9BE8",
    "GPT 5.4 Mini": "#57C4A7",
    "Claude Sonnet 4.6": "#E87B4C"
}

manual_color = color_map["Manual"]

# Dark text is readable on all four method colors
count_text_color = "#17212B"

# ------------------------------------------------------------
# Plot settings
# ------------------------------------------------------------

maximum_total = max(
    panel_a["Total_n"].max(),
    panel_b["Total_n"].max()
)

if maximum_total == 0:
    raise ValueError(
        "No model–manual disagreements were found."
    )

label_padding = max(
    maximum_total * 0.02,
    1
)

y_positions = np.arange(len(panel_a))

fig, axes = plt.subplots(
    nrows=2,
    ncols=1,
    figsize=(15, 13.5),
    sharex=True
)

# ------------------------------------------------------------
# Add only counts inside the boxes
# ------------------------------------------------------------

# Font sizes tried, largest first, for the counts inside the boxes.
# A count that does not fit its box even at the smallest size is
# moved next to the "Total n" label instead (see the fitting step
# after the layout is finalized).
count_label_font_sizes = [14, 12, 10]

count_labels = []


def add_count_label(
    ax,
    left,
    width,
    y_position,
    text_color,
    segment_name,
    total_label
):
    if width == 0:
        return

    segment_center = (
        left + width / 2
    )

    label = ax.text(
        segment_center,
        y_position,
        f"n = {int(width):,}",
        ha="center",
        va="center",
        color=text_color,
        fontsize=count_label_font_sizes[0],
        fontweight="bold"
    )

    count_labels.append({
        "ax": ax,
        "label": label,
        "left": left,
        "width": width,
        "segment_name": segment_name,
        "total_label": total_label
    })

# ------------------------------------------------------------
# Plotting function
# ------------------------------------------------------------

def plot_panel(
    ax,
    table,
    panel_title,
    ai_adjudicated_value,
    manual_adjudicated_value
):
    ai_counts = (
        table["AI_upheld"]
        .to_numpy()
    )

    manual_counts = (
        table["Manual_upheld"]
        .to_numpy()
    )

    totals = (
        table["Total_n"]
        .to_numpy()
    )

    ai_segment_colors = [
        color_map[model]
        for model in table.index
    ]

    # AI-upheld segment is displayed first on the left
    ax.barh(
        y_positions,
        ai_counts,
        height=0.66,
        color=ai_segment_colors,
        edgecolor="white"
    )

    # Manual-upheld segment is displayed second on the right
    ax.barh(
        y_positions,
        manual_counts,
        left=ai_counts,
        height=0.66,
        color=manual_color,
        edgecolor="white"
    )

    for i in range(len(table)):

        total_label = ax.text(
            totals[i] + label_padding,
            y_positions[i],
            f"Total n = {int(totals[i]):,}",
            ha="left",
            va="center",
            fontsize=14,
            color="#1F2933",
            linespacing=1.35
        )

        # AI-upheld count on the left
        add_count_label(
            ax=ax,
            left=0,
            width=ai_counts[i],
            y_position=y_positions[i],
            text_color=count_text_color,
            segment_name="AI upheld",
            total_label=total_label
        )

        # Manual-upheld count on the right
        add_count_label(
            ax=ax,
            left=ai_counts[i],
            width=manual_counts[i],
            y_position=y_positions[i],
            text_color=count_text_color,
            segment_name="manual upheld",
            total_label=total_label
        )

    ax.set_yticks(y_positions)

    ax.set_yticklabels(
        table.index,
        fontsize=16
    )

    ax.tick_params(
        axis="x",
        labelsize=14
    )

    # Place GPT first at the top
    ax.invert_yaxis()

    ax.set_title(
        panel_title,
        loc="left",
        fontsize=19,
        fontweight="bold",
        pad=72
    )

    ax.set_ylabel("")

    # Use the same raw-count scale for both panels
    ax.set_xlim(
        0,
        maximum_total * 1.27
    )

    ax.xaxis.set_major_locator(
        MaxNLocator(integer=True)
    )

    ax.grid(
        axis="x",
        color="#D5DDE5",
        linewidth=0.8,
        alpha=0.8
    )

    ax.set_axisbelow(True)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.spines["left"].set_visible(False)

    # Each panel receives its own legend.
    # AI is listed first to match the bar order. Its swatch shows
    # all of the model colors side by side.
    ai_handle = tuple(
        mpatches.Patch(color=color)
        for color in ai_segment_colors
    )

    manual_handle = mpatches.Patch(
        color=manual_color
    )

    ax.legend(
        handles=[ai_handle, manual_handle],
        labels=[
            (
                "AI score upheld "
                f"(adjudicated score = "
                f"{ai_adjudicated_value})"
            ),
            (
                "Manual score upheld "
                f"(adjudicated score = "
                f"{manual_adjudicated_value})"
            )
        ],
        handler_map={
            tuple: HandlerTuple(ndivide=None, pad=0)
        },
        handlelength=3,
        loc="lower center",
        bbox_to_anchor=(0.5, 1.01),
        ncol=2,
        frameon=False,
        fontsize=14
    )

# ------------------------------------------------------------
# Panel A above Panel B
# ------------------------------------------------------------

plot_panel(
    axes[0],
    panel_a,
    (
        "A. Manual = 0 and AI = 1\n"
        "Which score did adjudication support?"
    ),
    ai_adjudicated_value=1,
    manual_adjudicated_value=0
)

plot_panel(
    axes[1],
    panel_b,
    (
        "B. Manual = 1 and AI = 0\n"
        "Which score did adjudication support?"
    ),
    ai_adjudicated_value=0,
    manual_adjudicated_value=1
)

axes[1].set_xlabel(
    "Raw count of adjudicated model–manual disagreements",
    fontsize=16,
    labelpad=10
)

# ------------------------------------------------------------
# Overall figure title and subtitle
# ------------------------------------------------------------

fig.suptitle(
    "Adjudicated Outcomes of Model–Manual "
    "Phenotype Disagreements",
    fontsize=25,
    fontweight="bold",
    y=0.985
)

fig.text(
    0.5,
    0.94,
    (
        "AI-upheld outcomes are shown first in each model's color; "
        "manual-upheld outcomes are shown second in tan."
    ),
    ha="center",
    va="center",
    fontsize=16,
    color="#52606D"
)

plt.tight_layout(
    rect=[0.03, 0.03, 0.98, 0.93],
    h_pad=4
)

# ------------------------------------------------------------
# Fit the counts inside their boxes now that the layout is final
# ------------------------------------------------------------

fig.canvas.draw()
renderer = fig.canvas.get_renderer()

for item in count_labels:

    ax = item["ax"]
    label = item["label"]

    # Width of the box in pixels
    box_left = ax.transData.transform((item["left"], 0))[0]
    box_right = ax.transData.transform(
        (item["left"] + item["width"], 0)
    )[0]
    available_width = 0.90 * (box_right - box_left)

    fits = False

    for font_size in count_label_font_sizes:

        label.set_fontsize(font_size)

        if (
            label.get_window_extent(renderer).width
            <= available_width
        ):
            fits = True
            break

    if not fits:
        # Too narrow for a readable label: report the count on a
        # second line beside the total instead
        label.set_visible(False)

        item["total_label"].set_text(
            item["total_label"].get_text()
            + f"\n({item['segment_name']} n = "
            + f"{int(item['width']):,})"
        )

plt.show()

# ------------------------------------------------------------
# Display underlying raw-count table
# ------------------------------------------------------------

raw_count_summary = pd.concat(
    [
        panel_a.reset_index().assign(
            Panel="A",
            Disagreement_direction="Manual 0, AI 1",
            AI_upheld_adjudicated_score=1,
            Manual_upheld_adjudicated_score=0
        ),
        panel_b.reset_index().assign(
            Panel="B",
            Disagreement_direction="Manual 1, AI 0",
            AI_upheld_adjudicated_score=0,
            Manual_upheld_adjudicated_score=1
        )
    ],
    ignore_index=True
)

raw_count_summary = raw_count_summary.rename(
    columns={
        "AI_upheld": "AI_upheld_n",
        "Manual_upheld": "Manual_upheld_n"
    }
)

raw_count_summary = raw_count_summary[
    [
        "Panel",
        "Disagreement_direction",
        "Model",
        "AI_upheld_adjudicated_score",
        "AI_upheld_n",
        "Manual_upheld_adjudicated_score",
        "Manual_upheld_n",
        "Total_n"
    ]
]

display(raw_count_summary)

In [0]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.legend_handler import HandlerTuple
from matplotlib.ticker import MaxNLocator

df = combined_df_full_info_adjudications.copy()

manual_column = "Manual_Score"

model_columns = {
    "GPT_Score": "GPT 5.4",
    "GPT_Mini_Score": "GPT 5.4 Mini",
    "Claude_Sonnet_Score": "Claude Sonnet 4.6"
}

required_columns = [
    manual_column,
    "adjudicated_score"
] + list(model_columns.keys())

missing_columns = [
    column
    for column in required_columns
    if column not in df.columns
]

if missing_columns:
    raise KeyError(
        f"Missing required columns: {missing_columns}"
    )

# ------------------------------------------------------------
# Clean and validate scores
# ------------------------------------------------------------

df[required_columns] = (
    df[required_columns]
    .apply(
        pd.to_numeric,
        errors="coerce"
    )
)

for column in required_columns:
    n_missing = df[column].isna().sum()

    if n_missing > 0:
        print(
            f"WARNING: {n_missing:,} missing or nonnumeric "
            f"values in {column}"
        )

    unexpected_values = (
        df.loc[
            df[column].notna()
            & ~df[column].isin([0, 1]),
            column
        ]
        .unique()
        .tolist()
    )

    if unexpected_values:
        raise ValueError(
            f"{column} contains values other than 0 and 1: "
            f"{sorted(unexpected_values)}"
        )

# ------------------------------------------------------------
# Construct the two disagreement-direction panels
# ------------------------------------------------------------

panel_a_rows = []
panel_b_rows = []

for model_column, model_label in model_columns.items():

    complete_rows = df[
        [
            manual_column,
            model_column,
            "adjudicated_score"
        ]
    ].dropna()

    # Panel A: Manual = 0 and AI = 1
    panel_a_subset = complete_rows[
        complete_rows[manual_column].eq(0)
        & complete_rows[model_column].eq(1)
    ]

    # AI upheld means adjudicated score = 1
    panel_a_ai_upheld = (
        panel_a_subset["adjudicated_score"]
        .eq(1)
        .sum()
    )

    # Manual upheld means adjudicated score = 0
    panel_a_manual_upheld = (
        panel_a_subset["adjudicated_score"]
        .eq(0)
        .sum()
    )

    panel_a_rows.append({
        "Model": model_label,
        "AI_upheld": panel_a_ai_upheld,
        "Manual_upheld": panel_a_manual_upheld,
        "Total_n": len(panel_a_subset)
    })

    # Panel B: Manual = 1 and AI = 0
    panel_b_subset = complete_rows[
        complete_rows[manual_column].eq(1)
        & complete_rows[model_column].eq(0)
    ]

    # AI upheld means adjudicated score = 0
    panel_b_ai_upheld = (
        panel_b_subset["adjudicated_score"]
        .eq(0)
        .sum()
    )

    # Manual upheld means adjudicated score = 1
    panel_b_manual_upheld = (
        panel_b_subset["adjudicated_score"]
        .eq(1)
        .sum()
    )

    panel_b_rows.append({
        "Model": model_label,
        "AI_upheld": panel_b_ai_upheld,
        "Manual_upheld": panel_b_manual_upheld,
        "Total_n": len(panel_b_subset)
    })

panel_a = (
    pd.DataFrame(panel_a_rows)
    .set_index("Model")
)

panel_b = (
    pd.DataFrame(panel_b_rows)
    .set_index("Model")
)

# Confirm stacked counts equal total n
for panel_name, panel_table in {
    "Panel A": panel_a,
    "Panel B": panel_b
}.items():

    calculated_totals = (
        panel_table["AI_upheld"]
        + panel_table["Manual_upheld"]
    )

    if not calculated_totals.equals(
        panel_table["Total_n"]
    ):
        raise ValueError(
            f"{panel_name} contains totals that do not match "
            "the sum of its stacked sections."
        )

# ------------------------------------------------------------
# Percentages
# ------------------------------------------------------------

# Each segment's percentage is its share of that model's total
# disagreements in the panel, so the two segments of a bar add up
# to 100%.


def segment_percent(count, total):
    return 100 * count / total if total > 0 else np.nan


def format_percent(percent):
    return "NA" if np.isnan(percent) else f"{percent:.1f}%"


for panel_table in [panel_a, panel_b]:
    for column in ["AI_upheld", "Manual_upheld"]:
        panel_table[f"{column}_pct"] = [
            segment_percent(count, total)
            for count, total in zip(
                panel_table[column],
                panel_table["Total_n"]
            )
        ]

# ------------------------------------------------------------
# Color scheme
# ------------------------------------------------------------

# Same method colors as the other figures.
# The AI-upheld segment takes its model's color, and the
# manual-upheld segment is always the Manual color.
color_map = {
    "Manual": "#D4B896",
    "GPT 5.4": "#4C9BE8",
    "GPT 5.4 Mini": "#57C4A7",
    "Claude Sonnet 4.6": "#E87B4C"
}

manual_color = color_map["Manual"]

# Dark text is readable on all four method colors
count_text_color = "#17212B"

# ------------------------------------------------------------
# Plot settings
# ------------------------------------------------------------

maximum_total = max(
    panel_a["Total_n"].max(),
    panel_b["Total_n"].max()
)

if maximum_total == 0:
    raise ValueError(
        "No model–manual disagreements were found."
    )

label_padding = max(
    maximum_total * 0.02,
    1
)

bar_height = 0.66

y_positions = np.arange(len(panel_a))

fig, axes = plt.subplots(
    nrows=2,
    ncols=1,
    figsize=(15, 13.5),
    sharex=True
)

# ------------------------------------------------------------
# Add counts and percentages inside the boxes
# ------------------------------------------------------------

# Font sizes tried, largest first, for the labels inside the boxes.
# Each label is first tried on one line ("n = 12 (40.0%)"), then
# split over two lines ("n = 12" above "40.0%") for narrower boxes.
# A label that does not fit its box either way, even at the smallest
# size, is moved next to the "Total n" label instead (see the fitting
# step after the layout is finalized).
count_label_font_sizes = [14, 12, 10]

count_labels = []


def add_count_label(
    ax,
    left,
    width,
    total,
    y_position,
    text_color,
    segment_name,
    total_label
):
    if width == 0:
        return

    segment_center = (
        left + width / 2
    )

    count_text = f"n = {int(width):,}"
    percent_text = format_percent(
        segment_percent(width, total)
    )

    one_line_text = f"{count_text} ({percent_text})"
    two_line_text = f"{count_text}\n{percent_text}"

    label = ax.text(
        segment_center,
        y_position,
        one_line_text,
        ha="center",
        va="center",
        color=text_color,
        fontsize=count_label_font_sizes[0],
        fontweight="bold",
        linespacing=1.15
    )

    count_labels.append({
        "ax": ax,
        "label": label,
        "left": left,
        "width": width,
        "y_position": y_position,
        "one_line_text": one_line_text,
        "two_line_text": two_line_text,
        "count_text": count_text,
        "percent_text": percent_text,
        "segment_name": segment_name,
        "total_label": total_label
    })

# ------------------------------------------------------------
# Plotting function
# ------------------------------------------------------------

def plot_panel(
    ax,
    table,
    panel_title,
    ai_adjudicated_value,
    manual_adjudicated_value
):
    ai_counts = (
        table["AI_upheld"]
        .to_numpy()
    )

    manual_counts = (
        table["Manual_upheld"]
        .to_numpy()
    )

    totals = (
        table["Total_n"]
        .to_numpy()
    )

    ai_segment_colors = [
        color_map[model]
        for model in table.index
    ]

    # AI-upheld segment is displayed first on the left
    ax.barh(
        y_positions,
        ai_counts,
        height=bar_height,
        color=ai_segment_colors,
        edgecolor="white"
    )

    # Manual-upheld segment is displayed second on the right
    ax.barh(
        y_positions,
        manual_counts,
        left=ai_counts,
        height=bar_height,
        color=manual_color,
        edgecolor="white"
    )

    for i in range(len(table)):

        total_label = ax.text(
            totals[i] + label_padding,
            y_positions[i],
            f"Total n = {int(totals[i]):,}",
            ha="left",
            va="center",
            fontsize=14,
            color="#1F2933",
            linespacing=1.35
        )

        # AI-upheld count on the left
        add_count_label(
            ax=ax,
            left=0,
            width=ai_counts[i],
            total=totals[i],
            y_position=y_positions[i],
            text_color=count_text_color,
            segment_name="AI upheld",
            total_label=total_label
        )

        # Manual-upheld count on the right
        add_count_label(
            ax=ax,
            left=ai_counts[i],
            width=manual_counts[i],
            total=totals[i],
            y_position=y_positions[i],
            text_color=count_text_color,
            segment_name="manual upheld",
            total_label=total_label
        )

    ax.set_yticks(y_positions)

    ax.set_yticklabels(
        table.index,
        fontsize=16
    )

    ax.tick_params(
        axis="x",
        labelsize=14
    )

    # Place GPT first at the top
    ax.invert_yaxis()

    ax.set_title(
        panel_title,
        loc="left",
        fontsize=19,
        fontweight="bold",
        pad=72
    )

    ax.set_ylabel("")

    # Use the same raw-count scale for both panels. Extra room on
    # the right leaves space for a "Total n" label that also carries
    # a moved count and percentage.
    ax.set_xlim(
        0,
        maximum_total * 1.35
    )

    ax.xaxis.set_major_locator(
        MaxNLocator(integer=True)
    )

    ax.grid(
        axis="x",
        color="#D5DDE5",
        linewidth=0.8,
        alpha=0.8
    )

    ax.set_axisbelow(True)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.spines["left"].set_visible(False)

    # Each panel receives its own legend.
    # AI is listed first to match the bar order. Its swatch shows
    # all of the model colors side by side.
    ai_handle = tuple(
        mpatches.Patch(color=color)
        for color in ai_segment_colors
    )

    manual_handle = mpatches.Patch(
        color=manual_color
    )

    ax.legend(
        handles=[ai_handle, manual_handle],
        labels=[
            (
                "AI score upheld "
                f"(adjudicated score = "
                f"{ai_adjudicated_value})"
            ),
            (
                "Manual score upheld "
                f"(adjudicated score = "
                f"{manual_adjudicated_value})"
            )
        ],
        handler_map={
            tuple: HandlerTuple(ndivide=None, pad=0)
        },
        handlelength=3,
        loc="lower center",
        bbox_to_anchor=(0.5, 1.01),
        ncol=2,
        frameon=False,
        fontsize=14
    )

# ------------------------------------------------------------
# Panel A above Panel B
# ------------------------------------------------------------

plot_panel(
    axes[0],
    panel_a,
    (
        "A. Manual = 0 and AI = 1\n"
        "Which score did adjudication support?"
    ),
    ai_adjudicated_value=1,
    manual_adjudicated_value=0
)

plot_panel(
    axes[1],
    panel_b,
    (
        "B. Manual = 1 and AI = 0\n"
        "Which score did adjudication support?"
    ),
    ai_adjudicated_value=0,
    manual_adjudicated_value=1
)

axes[1].set_xlabel(
    "Raw count of adjudicated model–manual disagreements",
    fontsize=16,
    labelpad=10
)

# ------------------------------------------------------------
# Overall figure title and subtitle
# ------------------------------------------------------------

fig.suptitle(
    "Adjudicated Outcomes of Model–Manual "
    "Phenotype Disagreements",
    fontsize=25,
    fontweight="bold",
    y=0.985
)

fig.text(
    0.5,
    0.925,
    (
        "AI-upheld outcomes are shown first in each model's color; "
        "manual-upheld outcomes are shown second in tan.\n"
        "Percentages are shares of each model's total "
        "disagreements in that panel."
    ),
    ha="center",
    va="center",
    fontsize=16,
    color="#52606D",
    linespacing=1.4
)

plt.tight_layout(
    rect=[0.03, 0.03, 0.98, 0.895],
    h_pad=4
)

# ------------------------------------------------------------
# Fit the labels inside their boxes now that the layout is final
# ------------------------------------------------------------

fig.canvas.draw()
renderer = fig.canvas.get_renderer()

for item in count_labels:

    ax = item["ax"]
    label = item["label"]

    # Size of the box in pixels
    box_left = ax.transData.transform((item["left"], 0))[0]
    box_right = ax.transData.transform(
        (item["left"] + item["width"], 0)
    )[0]
    available_width = 0.90 * (box_right - box_left)

    box_top = ax.transData.transform(
        (0, item["y_position"] - bar_height / 2)
    )[1]
    box_bottom = ax.transData.transform(
        (0, item["y_position"] + bar_height / 2)
    )[1]
    available_height = 0.90 * abs(box_top - box_bottom)

    # Try every font size on one line first, then on two lines
    candidates = (
        [
            (item["one_line_text"], font_size)
            for font_size in count_label_font_sizes
        ]
        + [
            (item["two_line_text"], font_size)
            for font_size in count_label_font_sizes
        ]
    )

    fits = False

    for text, font_size in candidates:

        label.set_text(text)
        label.set_fontsize(font_size)

        extent = label.get_window_extent(renderer)

        if (
            extent.width <= available_width
            and extent.height <= available_height
        ):
            fits = True
            break

    if not fits:
        # Too narrow for a readable label: report the count and
        # percentage on a new line beside the total instead
        label.set_visible(False)

        item["total_label"].set_text(
            item["total_label"].get_text()
            + f"\n({item['segment_name']} "
            + f"{item['count_text']}, {item['percent_text']})"
        )

plt.show()

# ------------------------------------------------------------
# Display underlying raw-count table
# ------------------------------------------------------------

raw_count_summary = pd.concat(
    [
        panel_a.reset_index().assign(
            Panel="A",
            Disagreement_direction="Manual 0, AI 1",
            AI_upheld_adjudicated_score=1,
            Manual_upheld_adjudicated_score=0
        ),
        panel_b.reset_index().assign(
            Panel="B",
            Disagreement_direction="Manual 1, AI 0",
            AI_upheld_adjudicated_score=0,
            Manual_upheld_adjudicated_score=1
        )
    ],
    ignore_index=True
)

raw_count_summary = raw_count_summary.rename(
    columns={
        "AI_upheld": "AI_upheld_n",
        "Manual_upheld": "Manual_upheld_n"
    }
)

raw_count_summary = raw_count_summary[
    [
        "Panel",
        "Disagreement_direction",
        "Model",
        "AI_upheld_adjudicated_score",
        "AI_upheld_n",
        "AI_upheld_pct",
        "Manual_upheld_adjudicated_score",
        "Manual_upheld_n",
        "Manual_upheld_pct",
        "Total_n"
    ]
]

raw_count_summary[["AI_upheld_pct", "Manual_upheld_pct"]] = (
    raw_count_summary[["AI_upheld_pct", "Manual_upheld_pct"]]
    .round(1)
)

display(raw_count_summary)

In [0]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.legend_handler import HandlerTuple
from matplotlib.ticker import MaxNLocator

df = combined_df_full_info_adjudications.copy()

manual_column = "Manual_Score"

model_columns = {
    "GPT_Score": "GPT 5.4",
    "GPT_Mini_Score": "GPT 5.4 Mini",
    "Claude_Sonnet_Score": "Claude Sonnet 4.6"
}

required_columns = [
    manual_column,
    "adjudicated_score"
] + list(model_columns.keys())

missing_columns = [
    column
    for column in required_columns
    if column not in df.columns
]

if missing_columns:
    raise KeyError(
        f"Missing required columns: {missing_columns}"
    )

# ------------------------------------------------------------
# Clean and validate scores
# ------------------------------------------------------------

df[required_columns] = (
    df[required_columns]
    .apply(
        pd.to_numeric,
        errors="coerce"
    )
)

for column in required_columns:
    n_missing = df[column].isna().sum()

    if n_missing > 0:
        print(
            f"WARNING: {n_missing:,} missing or nonnumeric "
            f"values in {column}"
        )

    unexpected_values = (
        df.loc[
            df[column].notna()
            & ~df[column].isin([0, 1]),
            column
        ]
        .unique()
        .tolist()
    )

    if unexpected_values:
        raise ValueError(
            f"{column} contains values other than 0 and 1: "
            f"{sorted(unexpected_values)}"
        )

# ------------------------------------------------------------
# Construct the two disagreement-direction panels
# ------------------------------------------------------------

panel_a_rows = []
panel_b_rows = []

for model_column, model_label in model_columns.items():

    complete_rows = df[
        [
            manual_column,
            model_column,
            "adjudicated_score"
        ]
    ].dropna()

    # Panel A: Manual = 0 and AI = 1
    panel_a_subset = complete_rows[
        complete_rows[manual_column].eq(0)
        & complete_rows[model_column].eq(1)
    ]

    # AI upheld means adjudicated score = 1
    panel_a_ai_upheld = (
        panel_a_subset["adjudicated_score"]
        .eq(1)
        .sum()
    )

    # Manual upheld means adjudicated score = 0
    panel_a_manual_upheld = (
        panel_a_subset["adjudicated_score"]
        .eq(0)
        .sum()
    )

    panel_a_rows.append({
        "Model": model_label,
        "AI_upheld": panel_a_ai_upheld,
        "Manual_upheld": panel_a_manual_upheld,
        "Total_n": len(panel_a_subset)
    })

    # Panel B: Manual = 1 and AI = 0
    panel_b_subset = complete_rows[
        complete_rows[manual_column].eq(1)
        & complete_rows[model_column].eq(0)
    ]

    # AI upheld means adjudicated score = 0
    panel_b_ai_upheld = (
        panel_b_subset["adjudicated_score"]
        .eq(0)
        .sum()
    )

    # Manual upheld means adjudicated score = 1
    panel_b_manual_upheld = (
        panel_b_subset["adjudicated_score"]
        .eq(1)
        .sum()
    )

    panel_b_rows.append({
        "Model": model_label,
        "AI_upheld": panel_b_ai_upheld,
        "Manual_upheld": panel_b_manual_upheld,
        "Total_n": len(panel_b_subset)
    })

panel_a = (
    pd.DataFrame(panel_a_rows)
    .set_index("Model")
)

panel_b = (
    pd.DataFrame(panel_b_rows)
    .set_index("Model")
)

# Confirm stacked counts equal total n
for panel_name, panel_table in {
    "Panel A": panel_a,
    "Panel B": panel_b
}.items():

    calculated_totals = (
        panel_table["AI_upheld"]
        + panel_table["Manual_upheld"]
    )

    if not calculated_totals.equals(
        panel_table["Total_n"]
    ):
        raise ValueError(
            f"{panel_name} contains totals that do not match "
            "the sum of its stacked sections."
        )

# ------------------------------------------------------------
# Percentages
# ------------------------------------------------------------

# Each segment's percentage is its share of that model's total
# disagreements in the panel, so the two segments of a bar add up
# to 100%. Only the AI-upheld segments show a percentage on the
# plot; both percentages remain in the summary table below.


def segment_percent(count, total):
    return 100 * count / total if total > 0 else np.nan


def format_percent(percent):
    return "NA" if np.isnan(percent) else f"{percent:.1f}%"


for panel_table in [panel_a, panel_b]:
    for column in ["AI_upheld", "Manual_upheld"]:
        panel_table[f"{column}_pct"] = [
            segment_percent(count, total)
            for count, total in zip(
                panel_table[column],
                panel_table["Total_n"]
            )
        ]

# ------------------------------------------------------------
# Color scheme
# ------------------------------------------------------------

# Same method colors as the other figures.
# The AI-upheld segment takes its model's color, and the
# manual-upheld segment is always the Manual color.
color_map = {
    "Manual": "#D4B896",
    "GPT 5.4": "#4C9BE8",
    "GPT 5.4 Mini": "#57C4A7",
    "Claude Sonnet 4.6": "#E87B4C"
}

manual_color = color_map["Manual"]

# Dark text is readable on all four method colors
count_text_color = "#17212B"

# ------------------------------------------------------------
# Plot settings
# ------------------------------------------------------------

maximum_total = max(
    panel_a["Total_n"].max(),
    panel_b["Total_n"].max()
)

if maximum_total == 0:
    raise ValueError(
        "No model–manual disagreements were found."
    )

label_padding = max(
    maximum_total * 0.02,
    1
)

bar_height = 0.66

y_positions = np.arange(len(panel_a))

fig, axes = plt.subplots(
    nrows=2,
    ncols=1,
    figsize=(15, 13.5),
    sharex=True
)

# ------------------------------------------------------------
# Label settings for the counts inside the boxes
# ------------------------------------------------------------

# Every count label in the figure (AI-upheld and manual-upheld,
# both panels) uses one shared font size, and every label stays
# inside its own box. The shared size is set by the most limited
# box, measured with comfortable room left around the text.

# Largest size used if every box has room for it
label_max_font_size = 14

# Share of a box's width and height the text may use. The rest is
# clear space around the label, so even the most limited box is
# not a tight fit. (Buffer is 15% of the width and 22.5% of the
# height, half of the earlier 30% and 45%.)
label_width_fill = 0.85
label_height_fill = 0.775

# Below this size a warning is printed, since the labels may be
# hard to read. The labels still stay inside their boxes.
label_readable_font_size = 8

count_labels = []
total_labels = []


def add_count_label(
    ax,
    left,
    width,
    total,
    y_position,
    text_color,
    show_percent
):
    if width == 0:
        return

    segment_center = (
        left + width / 2
    )

    count_text = f"n={int(width):,}"

    # AI-upheld labels try one line first, then two lines.
    # Manual-upheld labels show the count only.
    if show_percent:
        percent_text = format_percent(
            segment_percent(width, total)
        )
        candidate_texts = [
            f"{count_text} ({percent_text})",
            f"{count_text}\n{percent_text}"
        ]
    else:
        candidate_texts = [count_text]

    label = ax.text(
        segment_center,
        y_position,
        candidate_texts[0],
        ha="center",
        va="center",
        color=text_color,
        fontsize=label_max_font_size,
        fontweight="bold",
        linespacing=1.15
    )

    count_labels.append({
        "ax": ax,
        "label": label,
        "left": left,
        "width": width,
        "y_position": y_position,
        "candidate_texts": candidate_texts
    })

# ------------------------------------------------------------
# Plotting function
# ------------------------------------------------------------

def plot_panel(
    ax,
    table,
    panel_title,
    ai_adjudicated_value,
    manual_adjudicated_value
):
    ai_counts = (
        table["AI_upheld"]
        .to_numpy()
    )

    manual_counts = (
        table["Manual_upheld"]
        .to_numpy()
    )

    totals = (
        table["Total_n"]
        .to_numpy()
    )

    ai_segment_colors = [
        color_map[model]
        for model in table.index
    ]

    # AI-upheld segment is displayed first on the left
    ax.barh(
        y_positions,
        ai_counts,
        height=bar_height,
        color=ai_segment_colors,
        edgecolor="white"
    )

    # Manual-upheld segment is displayed second on the right
    ax.barh(
        y_positions,
        manual_counts,
        left=ai_counts,
        height=bar_height,
        color=manual_color,
        edgecolor="white"
    )

    for i in range(len(table)):

        total_label = ax.text(
            totals[i] + label_padding,
            y_positions[i],
            f"Total n = {int(totals[i]):,}",
            ha="left",
            va="center",
            fontsize=14,
            color="#1F2933"
        )

        total_labels.append({
            "label": total_label,
            "total": totals[i]
        })

        # AI-upheld count and percentage on the left
        add_count_label(
            ax=ax,
            left=0,
            width=ai_counts[i],
            total=totals[i],
            y_position=y_positions[i],
            text_color=count_text_color,
            show_percent=True
        )

        # Manual-upheld count only on the right
        add_count_label(
            ax=ax,
            left=ai_counts[i],
            width=manual_counts[i],
            total=totals[i],
            y_position=y_positions[i],
            text_color=count_text_color,
            show_percent=False
        )

    ax.set_yticks(y_positions)

    ax.set_yticklabels(
        table.index,
        fontsize=16
    )

    ax.tick_params(
        axis="x",
        labelsize=14
    )

    # Place GPT first at the top
    ax.invert_yaxis()

    ax.set_title(
        panel_title,
        loc="left",
        fontsize=19,
        fontweight="bold",
        pad=72
    )

    ax.set_ylabel("")

    # Starting x range. It is narrowed after the layout is final so
    # the axis ends just past the widest "Total n" label, which
    # gives the bars (and the labels inside them) as much width as
    # possible.
    ax.set_xlim(
        0,
        maximum_total * 1.35
    )

    ax.xaxis.set_major_locator(
        MaxNLocator(integer=True)
    )

    ax.grid(
        axis="x",
        color="#D5DDE5",
        linewidth=0.8,
        alpha=0.8
    )

    ax.set_axisbelow(True)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.spines["left"].set_visible(False)

    # Each panel receives its own legend.
    # AI is listed first to match the bar order. Its swatch shows
    # all of the model colors side by side.
    ai_handle = tuple(
        mpatches.Patch(color=color)
        for color in ai_segment_colors
    )

    manual_handle = mpatches.Patch(
        color=manual_color
    )

    ax.legend(
        handles=[ai_handle, manual_handle],
        labels=[
            (
                "AI score upheld "
                f"(adjudicated score = "
                f"{ai_adjudicated_value})"
            ),
            (
                "Manual score upheld "
                f"(adjudicated score = "
                f"{manual_adjudicated_value})"
            )
        ],
        handler_map={
            tuple: HandlerTuple(ndivide=None, pad=0)
        },
        handlelength=3,
        loc="lower center",
        bbox_to_anchor=(0.5, 1.01),
        ncol=2,
        frameon=False,
        fontsize=14
    )

# ------------------------------------------------------------
# Panel A above Panel B
# ------------------------------------------------------------

plot_panel(
    axes[0],
    panel_a,
    (
        "A. Manual = 0 and AI = 1\n"
        "Which score did adjudication support?"
    ),
    ai_adjudicated_value=1,
    manual_adjudicated_value=0
)

plot_panel(
    axes[1],
    panel_b,
    (
        "B. Manual = 1 and AI = 0\n"
        "Which score did adjudication support?"
    ),
    ai_adjudicated_value=0,
    manual_adjudicated_value=1
)

axes[1].set_xlabel(
    "Raw count of adjudicated model–manual disagreements",
    fontsize=16,
    labelpad=10
)

# ------------------------------------------------------------
# Overall figure title and subtitle
# ------------------------------------------------------------

fig.suptitle(
    "Adjudicated Outcomes of Model–Manual "
    "Phenotype Disagreements",
    fontsize=25,
    fontweight="bold",
    y=0.985
)

fig.text(
    0.5,
    0.925,
    (
        "AI-upheld outcomes are shown first in each model's color; "
        "manual-upheld outcomes are shown second in tan.\n"
        "Percentages on the AI-upheld segments are shares of each "
        "model's total disagreements in that panel."
    ),
    ha="center",
    va="center",
    fontsize=16,
    color="#52606D",
    linespacing=1.4
)

# ------------------------------------------------------------
# Finalize the layout and narrow the x range
# ------------------------------------------------------------

# The axis is narrowed so it ends just past the widest "Total n"
# label. Done twice because the layout and the x range affect
# each other slightly.
for _ in range(2):

    plt.tight_layout(
        rect=[0.03, 0.03, 0.98, 0.895],
        h_pad=4
    )

    fig.canvas.draw()
    renderer = fig.canvas.get_renderer()

    axes_width_px = (
        axes[0].get_window_extent(renderer).width
    )
    usable_width_px = 0.98 * axes_width_px

    required_xmax = maximum_total

    for item in total_labels:
        label_width_px = (
            item["label"]
            .get_window_extent(renderer)
            .width
        )

        label_start = item["total"] + label_padding

        required_xmax = max(
            required_xmax,
            label_start * axes_width_px
            / (usable_width_px - label_width_px)
        )

    # Both panels share the x axis, so this sets both
    axes[0].set_xlim(0, required_xmax)

fig.canvas.draw()
renderer = fig.canvas.get_renderer()

# ------------------------------------------------------------
# Fit every count label inside its box at one shared size
# ------------------------------------------------------------


def box_size_pixels(item):
    """Width and height of a bar segment in display pixels."""
    ax = item["ax"]

    x_left, y_low = ax.transData.transform(
        (item["left"], item["y_position"] - bar_height / 2)
    )
    x_right, y_high = ax.transData.transform(
        (item["left"] + item["width"],
         item["y_position"] + bar_height / 2)
    )

    return abs(x_right - x_left), abs(y_high - y_low)


def comfortable_room(item):
    """Width and height the text may use inside its box."""
    box_width, box_height = box_size_pixels(item)

    return (
        label_width_fill * box_width,
        label_height_fill * box_height
    )


def fits_comfortably(item):
    """Whether the label's current text and size fit its box."""
    available_width, available_height = comfortable_room(item)

    extent = item["label"].get_window_extent(renderer)

    return (
        extent.width <= available_width
        and extent.height <= available_height
    )


# Step 1: find the largest comfortable size for each box. Text
# size scales in proportion to font size, so one measurement at
# the maximum size gives the answer for each layout.
for item in count_labels:

    label = item["label"]
    available_width, available_height = comfortable_room(item)

    largest_size = 0

    for text in item["candidate_texts"]:

        label.set_text(text)
        label.set_fontsize(label_max_font_size)

        extent = label.get_window_extent(renderer)

        scale = min(
            available_width / extent.width,
            available_height / extent.height
        )

        largest_size = max(
            largest_size,
            label_max_font_size * scale
        )

    item["largest_comfortable_size"] = largest_size

# Step 2: the most limited box sets the size for every label,
# rounded down to the nearest half point
shared_font_size = min(
    label_max_font_size,
    min(
        item["largest_comfortable_size"]
        for item in count_labels
    )
)

shared_font_size = np.floor(shared_font_size * 2) / 2


def apply_shared_size(font_size):
    """Set every label to font_size, choosing one line where it
    fits and two lines otherwise. Returns True if all fit."""
    all_fit = True

    for item in count_labels:

        label = item["label"]
        label.set_fontsize(font_size)

        item_fits = False

        for text in item["candidate_texts"]:
            label.set_text(text)

            if fits_comfortably(item):
                item_fits = True
                break

        all_fit = all_fit and item_fits

    return all_fit


# Step 3: apply it, and step down by half a point in the rare case
# a measurement rounds the wrong way
while (
    not apply_shared_size(shared_font_size)
    and shared_font_size > 1
):
    shared_font_size -= 0.5

print(
    "All count labels share font size "
    f"{shared_font_size:g}"
)

if shared_font_size < label_readable_font_size:
    print(
        "WARNING: the narrowest box forces the count labels "
        f"below {label_readable_font_size} pt. Consider a wider "
        "figure or a smaller label_width_fill margin."
    )

plt.show()

# ------------------------------------------------------------
# Display underlying raw-count table
# ------------------------------------------------------------

raw_count_summary = pd.concat(
    [
        panel_a.reset_index().assign(
            Panel="A",
            Disagreement_direction="Manual 0, AI 1",
            AI_upheld_adjudicated_score=1,
            Manual_upheld_adjudicated_score=0
        ),
        panel_b.reset_index().assign(
            Panel="B",
            Disagreement_direction="Manual 1, AI 0",
            AI_upheld_adjudicated_score=0,
            Manual_upheld_adjudicated_score=1
        )
    ],
    ignore_index=True
)

raw_count_summary = raw_count_summary.rename(
    columns={
        "AI_upheld": "AI_upheld_n",
        "Manual_upheld": "Manual_upheld_n"
    }
)

raw_count_summary = raw_count_summary[
    [
        "Panel",
        "Disagreement_direction",
        "Model",
        "AI_upheld_adjudicated_score",
        "AI_upheld_n",
        "AI_upheld_pct",
        "Manual_upheld_adjudicated_score",
        "Manual_upheld_n",
        "Manual_upheld_pct",
        "Total_n"
    ]
]

raw_count_summary[["AI_upheld_pct", "Manual_upheld_pct"]] = (
    raw_count_summary[["AI_upheld_pct", "Manual_upheld_pct"]]
    .round(1)
)

display(raw_count_summary)

In [0]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.ticker import MaxNLocator

df = combined_df_full_info_adjudications.copy()

manual_column = "Manual_Score"

model_columns = {
    "GPT_Score": "GPT 5.4",
    "GPT_Mini_Score": "GPT 5.4 Mini",
    "Claude_Sonnet_Score": "Claude Sonnet 4.6"
}

required_columns = [
    manual_column,
    "adjudicated_score"
] + list(model_columns.keys())

missing_columns = [
    column
    for column in required_columns
    if column not in df.columns
]

if missing_columns:
    raise KeyError(
        f"Missing required columns: {missing_columns}"
    )

# ------------------------------------------------------------
# Clean and validate scores
# ------------------------------------------------------------

df[required_columns] = df[required_columns].apply(
    pd.to_numeric,
    errors="coerce"
)

for column in required_columns:
    n_missing = df[column].isna().sum()

    if n_missing > 0:
        print(
            f"WARNING: {n_missing:,} missing or nonnumeric "
            f"values in {column}"
        )

    unexpected_values = (
        df.loc[
            df[column].notna()
            & ~df[column].isin([0, 1]),
            column
        ]
        .unique()
        .tolist()
    )

    if unexpected_values:
        raise ValueError(
            f"{column} contains values other than 0 and 1: "
            f"{sorted(unexpected_values)}"
        )

# ------------------------------------------------------------
# Construct the two disagreement-direction panels
# ------------------------------------------------------------

panel_a_rows = []
panel_b_rows = []

for model_column, model_label in model_columns.items():

    complete_rows = df[
        [
            manual_column,
            model_column,
            "adjudicated_score"
        ]
    ].dropna()

    # Panel A: Manual = 0 and AI = 1
    panel_a_subset = complete_rows[
        complete_rows[manual_column].eq(0)
        & complete_rows[model_column].eq(1)
    ]

    # AI upheld means adjudicated score = 1
    panel_a_ai_upheld = (
        panel_a_subset["adjudicated_score"]
        .eq(1)
        .sum()
    )

    # Manual upheld means adjudicated score = 0
    panel_a_manual_upheld = (
        panel_a_subset["adjudicated_score"]
        .eq(0)
        .sum()
    )

    panel_a_rows.append({
        "Model": model_label,
        "AI_upheld": panel_a_ai_upheld,
        "Manual_upheld": panel_a_manual_upheld,
        "Total_n": len(panel_a_subset)
    })

    # Panel B: Manual = 1 and AI = 0
    panel_b_subset = complete_rows[
        complete_rows[manual_column].eq(1)
        & complete_rows[model_column].eq(0)
    ]

    # AI upheld means adjudicated score = 0
    panel_b_ai_upheld = (
        panel_b_subset["adjudicated_score"]
        .eq(0)
        .sum()
    )

    # Manual upheld means adjudicated score = 1
    panel_b_manual_upheld = (
        panel_b_subset["adjudicated_score"]
        .eq(1)
        .sum()
    )

    panel_b_rows.append({
        "Model": model_label,
        "AI_upheld": panel_b_ai_upheld,
        "Manual_upheld": panel_b_manual_upheld,
        "Total_n": len(panel_b_subset)
    })

panel_a = (
    pd.DataFrame(panel_a_rows)
    .set_index("Model")
)

panel_b = (
    pd.DataFrame(panel_b_rows)
    .set_index("Model")
)

# Confirm that stacked counts equal total n
for panel_name, panel_table in {
    "Panel A": panel_a,
    "Panel B": panel_b
}.items():

    calculated_totals = (
        panel_table["AI_upheld"]
        + panel_table["Manual_upheld"]
    )

    if not calculated_totals.equals(
        panel_table["Total_n"]
    ):
        raise ValueError(
            f"{panel_name} contains totals that do not match "
            "the sum of its stacked sections."
        )

# ------------------------------------------------------------
# Color scheme
# ------------------------------------------------------------

# Blue always means AI score upheld
ai_colors = {
    0: "#79A9D8",  # Lighter blue: adjudicated score = 0
    1: "#2F6FB3"   # Darker blue: adjudicated score = 1
}

# Orange always means manual score upheld
manual_colors = {
    0: "#F2B36F",  # Lighter orange: adjudicated score = 0
    1: "#D97820"   # Darker orange: adjudicated score = 1
}

# Text colors selected for contrast
ai_text_colors = {
    0: "#17212B",
    1: "white"
}

manual_text_colors = {
    0: "#17212B",
    1: "white"
}

# ------------------------------------------------------------
# Plot settings
# ------------------------------------------------------------

maximum_total = max(
    panel_a["Total_n"].max(),
    panel_b["Total_n"].max()
)

if maximum_total == 0:
    raise ValueError(
        "No model–manual disagreements were found."
    )

label_padding = max(
    maximum_total * 0.02,
    1
)

small_segment_threshold = max(
    maximum_total * 0.08,
    2
)

y_positions = np.arange(len(panel_a))

fig, axes = plt.subplots(
    nrows=2,
    ncols=1,
    figsize=(13, 11.5),
    sharex=True
)

# ------------------------------------------------------------
# Add only counts inside the boxes
# ------------------------------------------------------------

def add_count_label(
    ax,
    left,
    width,
    y_position,
    text_color
):
    if width == 0:
        return

    segment_center = (
        left + width / 2
    )

    font_size = (
        10
        if width >= small_segment_threshold
        else 7
    )

    ax.text(
        segment_center,
        y_position,
        f"n = {int(width):,}",
        ha="center",
        va="center",
        color=text_color,
        fontsize=font_size,
        fontweight="bold"
    )

# ------------------------------------------------------------
# Plotting function
# ------------------------------------------------------------

def plot_panel(
    ax,
    table,
    panel_title,
    ai_adjudicated_value,
    manual_adjudicated_value,
    first_segment
):
    ai_counts = (
        table["AI_upheld"]
        .to_numpy()
    )

    manual_counts = (
        table["Manual_upheld"]
        .to_numpy()
    )

    totals = (
        table["Total_n"]
        .to_numpy()
    )

    ai_color = ai_colors[
        ai_adjudicated_value
    ]

    manual_color = manual_colors[
        manual_adjudicated_value
    ]

    ai_legend_label = (
        "AI score upheld "
        f"(adjudicated score = {ai_adjudicated_value})"
    )

    manual_legend_label = (
        "Manual score upheld "
        f"(adjudicated score = {manual_adjudicated_value})"
    )

    # Determine the order of the two stacked sections
    if first_segment == "AI":

        first_counts = ai_counts
        first_color = ai_color
        first_text_color = ai_text_colors[
            ai_adjudicated_value
        ]
        first_legend_label = ai_legend_label

        second_counts = manual_counts
        second_color = manual_color
        second_text_color = manual_text_colors[
            manual_adjudicated_value
        ]
        second_legend_label = manual_legend_label

    elif first_segment == "Manual":

        first_counts = manual_counts
        first_color = manual_color
        first_text_color = manual_text_colors[
            manual_adjudicated_value
        ]
        first_legend_label = manual_legend_label

        second_counts = ai_counts
        second_color = ai_color
        second_text_color = ai_text_colors[
            ai_adjudicated_value
        ]
        second_legend_label = ai_legend_label

    else:
        raise ValueError(
            "first_segment must be either 'AI' or 'Manual'."
        )

    # First segment is displayed on the left
    ax.barh(
        y_positions,
        first_counts,
        height=0.66,
        color=first_color,
        edgecolor="white"
    )

    # Second segment is displayed on the right
    ax.barh(
        y_positions,
        second_counts,
        left=first_counts,
        height=0.66,
        color=second_color,
        edgecolor="white"
    )

    for i in range(len(table)):

        # Count inside the left segment
        add_count_label(
            ax=ax,
            left=0,
            width=first_counts[i],
            y_position=y_positions[i],
            text_color=first_text_color
        )

        # Count inside the right segment
        add_count_label(
            ax=ax,
            left=first_counts[i],
            width=second_counts[i],
            y_position=y_positions[i],
            text_color=second_text_color
        )

        ax.text(
            totals[i] + label_padding,
            y_positions[i],
            f"Total n = {int(totals[i]):,}",
            ha="left",
            va="center",
            fontsize=10,
            color="#1F2933"
        )

    ax.set_yticks(y_positions)

    ax.set_yticklabels(
        table.index,
        fontsize=11
    )

    # Place GPT first at the top
    ax.invert_yaxis()

    ax.set_title(
        panel_title,
        loc="left",
        fontsize=14,
        fontweight="bold",
        pad=62
    )

    ax.set_ylabel("")

    # Use the same raw-count scale for both panels
    ax.set_xlim(
        0,
        maximum_total * 1.22
    )

    ax.xaxis.set_major_locator(
        MaxNLocator(integer=True)
    )

    ax.grid(
        axis="x",
        color="#D5DDE5",
        linewidth=0.8,
        alpha=0.8
    )

    ax.set_axisbelow(True)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.spines["left"].set_visible(False)

    # Each panel's legend follows its bar order
    legend_handles = [
        mpatches.Patch(
            color=first_color,
            label=first_legend_label
        ),
        mpatches.Patch(
            color=second_color,
            label=second_legend_label
        )
    ]

    ax.legend(
        handles=legend_handles,
        loc="lower center",
        bbox_to_anchor=(0.5, 1.01),
        ncol=2,
        frameon=False,
        fontsize=10
    )

# ------------------------------------------------------------
# Panel A above Panel B
# ------------------------------------------------------------

plot_panel(
    axes[0],
    panel_a,
    (
        "A. Manual = 0 and AI = 1\n"
        "Which score did adjudication support?"
    ),
    ai_adjudicated_value=1,
    manual_adjudicated_value=0,
    first_segment="AI"
)

plot_panel(
    axes[1],
    panel_b,
    (
        "B. Manual = 1 and AI = 0\n"
        "Which score did adjudication support?"
    ),
    ai_adjudicated_value=0,
    manual_adjudicated_value=1,
    first_segment="Manual"
)

axes[1].set_xlabel(
    "Raw count of adjudicated model–manual disagreements",
    fontsize=12
)

# ------------------------------------------------------------
# Overall figure title and subtitle
# ------------------------------------------------------------

fig.suptitle(
    "Adjudicated Outcomes of Model–Manual "
    "Phenotype Disagreements",
    fontsize=21,
    fontweight="bold",
    y=0.99
)

fig.text(
    0.5,
    0.955,
    (
        "Panel A shows AI-upheld outcomes first; "
        "Panel B shows manual-upheld outcomes first."
    ),
    ha="center",
    fontsize=12,
    color="#52606D"
)

plt.tight_layout(
    rect=[0.04, 0.05, 0.98, 0.87],
    h_pad=4
)

plt.show()

# ------------------------------------------------------------
# Display underlying raw-count table
# ------------------------------------------------------------

raw_count_summary = pd.concat(
    [
        panel_a.reset_index().assign(
            Panel="A",
            Disagreement_direction="Manual 0, AI 1",
            AI_upheld_adjudicated_score=1,
            Manual_upheld_adjudicated_score=0
        ),
        panel_b.reset_index().assign(
            Panel="B",
            Disagreement_direction="Manual 1, AI 0",
            AI_upheld_adjudicated_score=0,
            Manual_upheld_adjudicated_score=1
        )
    ],
    ignore_index=True
)

raw_count_summary = raw_count_summary.rename(
    columns={
        "AI_upheld": "AI_upheld_n",
        "Manual_upheld": "Manual_upheld_n"
    }
)

raw_count_summary = raw_count_summary[
    [
        "Panel",
        "Disagreement_direction",
        "Model",
        "AI_upheld_adjudicated_score",
        "AI_upheld_n",
        "Manual_upheld_adjudicated_score",
        "Manual_upheld_n",
        "Total_n"
    ]
]

display(raw_count_summary)

In [0]:
len(combined_df_full_info_adjudications)

# Full Gold Standard Analysis

In [0]:
import pandas as pd
import numpy as np

keys = [
    "Subject ID",
    "Phenotype"
]

score_columns = [
    "Manual_Score",
    "GPT_Score",
    "GPT_Mini_Score",
    "Claude_Sonnet_Score"
]

base = combined_df_full_info.copy()
adjudicated = (
    combined_df_full_info_adjudications.copy()
)

# Confirm each dataframe has one row per Subject ID–Phenotype pair
if base.duplicated(keys).any():
    raise ValueError(
        "combined_df_full_info contains duplicate pairs."
    )

if adjudicated.duplicated(keys).any():
    raise ValueError(
        "combined_df_full_info_adjudications contains "
        "duplicate pairs."
    )

# ------------------------------------------------------------
# Retain only adjudication rows matching the base dataframe
# ------------------------------------------------------------

base["_base_order"] = np.arange(len(base))

adjudicated_matched = adjudicated.merge(
    base[keys + ["_base_order"]],
    on=keys,
    how="inner",
    validate="one_to_one"
)

print(
    "Adjudication rows without a matching base row: "
    f"{len(adjudicated) - len(adjudicated_matched):,}"
)

# Mark which version of each row is being used
base["has_adjudication"] = False
adjudicated_matched["has_adjudication"] = True

# ------------------------------------------------------------
# Replace matching base rows with adjudicated rows
# ------------------------------------------------------------

gold_standard_full_info = (
    pd.concat(
        [base, adjudicated_matched],
        ignore_index=True,
        sort=False
    )
    # Adjudication rows were concatenated second, so keep them
    .drop_duplicates(
        subset=keys,
        keep="last"
    )
    .sort_values("_base_order")
    .drop(columns="_base_order")
    .reset_index(drop=True)
)

# ------------------------------------------------------------
# Identify unanimous agreement and disagreement
# ------------------------------------------------------------

gold_standard_full_info[score_columns] = (
    gold_standard_full_info[score_columns]
    .apply(
        pd.to_numeric,
        errors="coerce"
    )
)

scores = gold_standard_full_info[score_columns]

complete_scores = scores.notna().all(axis=1)

unanimous_agreement = (
    complete_scores
    & scores.eq(
        scores["Manual_Score"],
        axis=0
    ).all(axis=1)
)

scoring_disagreement = (
    complete_scores
    & ~unanimous_agreement
)

has_adjudication = (
    gold_standard_full_info[
        "has_adjudication"
    ]
)

# ------------------------------------------------------------
# Create the final gold-standard score
# ------------------------------------------------------------

gold_standard_full_info[
    "adjudicated_score"
] = pd.to_numeric(
    gold_standard_full_info[
        "adjudicated_score"
    ],
    errors="coerce"
)

gold_score = pd.Series(
    pd.NA,
    index=gold_standard_full_info.index,
    dtype="Int64"
)

# Use unanimous score when no adjudication is available
gold_score.loc[unanimous_agreement] = (
    gold_standard_full_info.loc[
        unanimous_agreement,
        "Manual_Score"
    ].astype("Int64")
)

# Adjudication always takes precedence when available
gold_score.loc[has_adjudication] = (
    gold_standard_full_info.loc[
        has_adjudication,
        "adjudicated_score"
    ].astype("Int64")
)

gold_standard_full_info[
    "gold_standard_score"
] = gold_score

# Record the source of each result
gold_standard_full_info[
    "gold_standard_source"
] = "Incomplete scoring"

gold_standard_full_info.loc[
    scoring_disagreement,
    "gold_standard_source"
] = "Unresolved disagreement"

gold_standard_full_info.loc[
    unanimous_agreement,
    "gold_standard_source"
] = "Unanimous agreement"

gold_standard_full_info.loc[
    has_adjudication,
    "gold_standard_source"
] = "Adjudication"

# ------------------------------------------------------------
# Create and count the two requested subsets
# ------------------------------------------------------------

disagreement_without_adjudication = (
    gold_standard_full_info.loc[
        scoring_disagreement
        & ~has_adjudication
    ].copy()
)

agreement_with_adjudication = (
    gold_standard_full_info.loc[
        unanimous_agreement
        & has_adjudication
    ].copy()
)

print(
    "Rows in gold_standard_full_info: "
    f"{len(gold_standard_full_info):,}"
)

print(
    "Disagreements without adjudication: "
    f"{len(disagreement_without_adjudication):,}"
)

print(
    "Unanimous rows with adjudication: "
    f"{len(agreement_with_adjudication):,}"
)

print(
    "Rows without a resolved gold-standard score: "
    f"{gold_standard_full_info['gold_standard_score'].isna().sum():,}"
)

display(
    gold_standard_full_info[
        "gold_standard_source"
    ]
    .value_counts(dropna=False)
    .rename_axis("Gold-standard source")
    .reset_index(name="Count")
)

display(gold_standard_full_info.head())

In [0]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib_venn import venn2
from matplotlib.colors import to_rgb

df = gold_standard_full_info.copy()

key_columns = [
    "Subject ID",
    "Phenotype"
]

score_columns = {
    "Manual": "Manual_Score",
    "GPT 5.4": "GPT_Score",
    "GPT 5.4 Mini": "GPT_Mini_Score",
    "Claude Sonnet 4.6": "Claude_Sonnet_Score"
}

# Panel labels
panel_labels = {
    "Manual": "A",
    "GPT 5.4": "B",
    "GPT 5.4 Mini": "C",
    "Claude Sonnet 4.6": "D"
}

# Preserve the original scoring-method colors
color_map = {
    "Manual": "#D4B896",
    "GPT 5.4": "#4C9BE8",
    "GPT 5.4 Mini": "#57C4A7",
    "Claude Sonnet 4.6": "#E87B4C"
}

# Soft pearl/off-white for the gold standard
gold_standard_color = "#D6DCE3"

required_columns = (
    key_columns
    + ["gold_standard_score"]
    + list(score_columns.values())
)

missing_columns = [
    column
    for column in required_columns
    if column not in df.columns
]

if missing_columns:
    raise KeyError(
        f"Missing required columns: {missing_columns}"
    )

# Each Subject ID–Phenotype pair should be one observation
if df.duplicated(key_columns).any():
    raise ValueError(
        "gold_standard_full_info contains duplicate "
        "Subject ID–Phenotype pairs."
    )

# ------------------------------------------------------------
# Convert scores to numeric
# ------------------------------------------------------------

numeric_columns = (
    ["gold_standard_score"]
    + list(score_columns.values())
)

df[numeric_columns] = (
    df[numeric_columns]
    .apply(pd.to_numeric, errors="coerce")
)

# Confirm that all nonmissing scores are binary
for column in numeric_columns:

    unexpected_values = (
        df.loc[
            df[column].notna()
            & ~df[column].isin([0, 1]),
            column
        ]
        .unique()
        .tolist()
    )

    if unexpected_values:
        raise ValueError(
            f"{column} contains values other than 0 and 1: "
            f"{sorted(unexpected_values)}"
        )

print(
    "Rows excluded because the gold-standard score is unresolved: "
    f"{df['gold_standard_score'].isna().sum():,}"
)

# ------------------------------------------------------------
# Calculate TP, FP, FN, TN, precision, and recall
# ------------------------------------------------------------

results = {}

for method, score_column in score_columns.items():

    # Evaluate rows where both scores are available
    evaluable = df.loc[
        df["gold_standard_score"].notna()
        & df[score_column].notna(),
        key_columns
        + ["gold_standard_score", score_column]
    ].copy()

    evaluable["gold_standard_score"] = (
        evaluable["gold_standard_score"]
        .astype(int)
    )

    evaluable[score_column] = (
        evaluable[score_column]
        .astype(int)
    )

    # Positive Subject ID–Phenotype pairs
    gold_positive_set = set(
        evaluable.loc[
            evaluable["gold_standard_score"].eq(1),
            key_columns
        ]
        .itertuples(
            index=False,
            name=None
        )
    )

    method_positive_set = set(
        evaluable.loc[
            evaluable[score_column].eq(1),
            key_columns
        ]
        .itertuples(
            index=False,
            name=None
        )
    )

    true_positive = len(
        gold_positive_set
        & method_positive_set
    )

    false_negative = len(
        gold_positive_set
        - method_positive_set
    )

    false_positive = len(
        method_positive_set
        - gold_positive_set
    )

    true_negative = (
        len(evaluable)
        - true_positive
        - false_negative
        - false_positive
    )

    precision = (
        true_positive
        / (true_positive + false_positive)
        if true_positive + false_positive > 0
        else np.nan
    )

    recall = (
        true_positive
        / (true_positive + false_negative)
        if true_positive + false_negative > 0
        else np.nan
    )

    results[method] = {
        "score_column": score_column,
        "n_evaluable": len(evaluable),
        "gold_positives": (
            true_positive + false_negative
        ),
        "method_positives": (
            true_positive + false_positive
        ),
        "true_positive": true_positive,
        "false_positive": false_positive,
        "false_negative": false_negative,
        "true_negative": true_negative,
        "precision": precision,
        "recall": recall,
        "positive_union": (
            true_positive
            + false_positive
            + false_negative
        )
    }

# ------------------------------------------------------------
# Establish a shared Venn-diagram scale
# ------------------------------------------------------------

max_union = max(
    result["positive_union"]
    for result in results.values()
)

if max_union == 0:
    raise ValueError(
        "No positive calls were found in any scoring method."
    )

draft_limits = []

for method, result in results.items():

    normalize_to = (
        result["positive_union"] / max_union
    )

    draft_fig, draft_ax = plt.subplots(
        figsize=(5, 5)
    )

    venn2(
        subsets=(
            result["false_negative"],
            result["false_positive"],
            result["true_positive"]
        ),
        set_labels=(
            "Gold standard",
            method
        ),
        set_colors=(
            gold_standard_color,
            color_map[method]
        ),
        alpha=0.5,
        normalize_to=normalize_to,
        ax=draft_ax
    )

    draft_limits.append({
        "xlim": draft_ax.get_xlim(),
        "ylim": draft_ax.get_ylim()
    })

    plt.close(draft_fig)

shared_xlim = (
    min(item["xlim"][0] for item in draft_limits),
    max(item["xlim"][1] for item in draft_limits)
)

shared_ylim = (
    min(item["ylim"][0] for item in draft_limits),
    max(item["ylim"][1] for item in draft_limits)
)

# ------------------------------------------------------------
# Draw the four-panel Venn figure
# ------------------------------------------------------------

fig, axes = plt.subplots(
    nrows=2,
    ncols=2,
    figsize=(15, 12)
)

axes = axes.flatten()

for ax, (method, result) in zip(
    axes,
    results.items()
):

    normalize_to = (
        result["positive_union"] / max_union
    )

    venn = venn2(
        subsets=(
            result["false_negative"],
            result["false_positive"],
            result["true_positive"]
        ),
        set_labels=(
            "Gold standard",
            method
        ),
        set_colors=(
            gold_standard_color,
            color_map[method]
        ),
        alpha=0.5,
        normalize_to=normalize_to,
        ax=ax
    )

    # --------------------------------------------------------
    # Correct the overlap color
    # --------------------------------------------------------

    overlap_patch = venn.get_patch_by_id("11")

    if overlap_patch is not None:

        overlap_color = (
            np.array(to_rgb(gold_standard_color))
            + np.array(to_rgb(color_map[method]))
        ) / 2

        overlap_patch.set_facecolor(
            overlap_color
        )

        overlap_patch.set_alpha(0.75)

    # Use the same scale across all four panels
    ax.set_xlim(shared_xlim)
    ax.set_ylim(shared_ylim)
    ax.set_aspect("equal")

    # Label the three positive-call regions
    region_labels = {
        "10": (
            "FN\n"
            f"n = {result['false_negative']:,}"
        ),
        "01": (
            "FP\n"
            f"n = {result['false_positive']:,}"
        ),
        "11": (
            "TP\n"
            f"n = {result['true_positive']:,}"
        )
    }

    for region_id, label_text in region_labels.items():

        label = venn.get_label_by_id(region_id)

        if label is not None:
            label.set_text(label_text)
            label.set_fontsize(11)
            label.set_fontweight("bold")

    # Format circle borders
    for patch in venn.patches:
        if patch is not None:
            patch.set_edgecolor("white")
            patch.set_linewidth(2)

    # Format set labels
    for label in venn.set_labels:
        if label is not None:
            label.set_fontsize(12)
            label.set_fontweight("bold")

    ax.set_title(
        f"Gold Standard vs {method}",
        fontsize=15,
        fontweight="bold",
        pad=18
    )

    # Add publication-style panel label
    ax.text(
        -0.02,
        1.04,
        panel_labels[method],
        transform=ax.transAxes,
        ha="left",
        va="top",
        fontsize=19,
        fontweight="bold",
        clip_on=False
    )

fig.suptitle(
    "Positive Phenotype Calls Compared With the Gold Standard",
    fontsize=21,
    fontweight="bold",
    y=0.98
)

fig.text(
    0.5,
    0.945,
    (
        "TP indicates positive agreement; FP and FN indicate "
        "disagreement. Precision and recall are reported in "
        "the accompanying table."
    ),
    ha="center",
    fontsize=11,
    color="#52606D"
)

plt.subplots_adjust(
    top=0.89,
    bottom=0.05,
    hspace=0.25,
    wspace=0.18
)

plt.show()

# ------------------------------------------------------------
# Create the performance table
# ------------------------------------------------------------

performance_summary = pd.DataFrame([
    {
        "Panel": panel_labels[method],
        "Scoring Method": method,
        "Evaluable n": result["n_evaluable"],
        "True Positive": result["true_positive"],
        "False Positive": result["false_positive"],
        "False Negative": result["false_negative"],
        "True Negative": result["true_negative"],
        "Precision": result["precision"],
        "Recall": result["recall"]
    }
    for method, result in results.items()
])

# Create a formatted display version while preserving the
# numeric values in performance_summary for later analyses
performance_table = performance_summary.copy()

performance_table["Precision"] = (
    performance_table["Precision"]
    .map(
        lambda value:
            "NA"
            if pd.isna(value)
            else f"{value:.3f}"
    )
)

performance_table["Recall"] = (
    performance_table["Recall"]
    .map(
        lambda value:
            "NA"
            if pd.isna(value)
            else f"{value:.3f}"
    )
)

print(
    "Precision = TP / (TP + FP); "
    "Recall = TP / (TP + FN)"
)

display(performance_table)

In [0]:
# ------------------------------------------------------------
# Create publication-ready performance table
# ------------------------------------------------------------

performance_summary = pd.DataFrame([
    {
        "Panel": panel_labels[method],
        "Scoring method": method,
        "Evaluable, n": result["n_evaluable"],
        "TP": result["true_positive"],
        "FP": result["false_positive"],
        "FN": result["false_negative"],
        "TN": result["true_negative"],
        "Precision": result["precision"],
        "Recall": result["recall"]
    }
    for method, result in results.items()
])

# Preserve performance_summary as numeric data and create a
# separately formatted dataframe for display
publication_table = performance_summary.copy()

count_columns = [
    "Evaluable, n",
    "TP",
    "FP",
    "FN",
    "TN"
]

for column in count_columns:
    publication_table[column] = (
        publication_table[column]
        .map(lambda value: f"{int(value):,}")
    )

for column in ["Precision", "Recall"]:
    publication_table[column] = (
        publication_table[column]
        .map(
            lambda value:
                "NA"
                if pd.isna(value)
                else f"{value:.3f}"
        )
    )

# ------------------------------------------------------------
# Render the table as a publication-ready figure
# ------------------------------------------------------------

fig_table, ax_table = plt.subplots(
    figsize=(14, 4.8)
)

ax_table.axis("off")

fig_table.text(
    0.05,
    0.92,
    "Performance of Phenotype Scoring Methods Against the Gold Standard",
    fontsize=17,
    fontweight="bold",
    color="#202833",
    ha="left"
)

table_bbox = [
    0.05,   # left
    0.28,   # bottom
    0.90,   # width
    0.50    # height
]

column_widths = [
    0.07,   # Panel
    0.24,   # Scoring method
    0.13,   # Evaluable n
    0.09,   # TP
    0.09,   # FP
    0.09,   # FN
    0.09,   # TN
    0.10,   # Precision
    0.10    # Recall
]

table = ax_table.table(
    cellText=publication_table.values,
    colLabels=publication_table.columns,
    colLoc="center",
    cellLoc="center",
    colWidths=column_widths,
    bbox=table_bbox
)

table.auto_set_font_size(False)
table.set_fontsize(12)

header_color = "#E5EBF2"
alternate_row_color = "#F4F6F8"
text_color = "#29313A"
line_color = "#D2D8DE"
border_color = "#7F8A94"

n_table_rows = len(publication_table) + 1
n_table_columns = len(publication_table.columns)

for (row, column), cell in table.get_celld().items():

    cell.set_text_props(
        color=text_color
    )

    cell.set_edgecolor(line_color)
    cell.set_linewidth(0.7)

    if row == 0:
        cell.set_facecolor(header_color)
        cell.set_text_props(
            fontweight="bold",
            color="#202833"
        )
    elif row % 2 == 0:
        cell.set_facecolor(alternate_row_color)
    else:
        cell.set_facecolor("white")

    # Left-align scoring-method names
    if column == 1:
        cell.get_text().set_ha("left")
        cell.PAD = 0.05

    # Emphasize panel labels
    if column == 0 and row > 0:
        cell.set_text_props(
            fontweight="bold"
        )

# Add darker horizontal borders above the header,
# below the header, and below the table
left, bottom, width, height = table_bbox
row_height = height / n_table_rows

ax_table.plot(
    [left, left + width],
    [bottom + height, bottom + height],
    color=border_color,
    linewidth=1.5,
    transform=ax_table.transAxes
)

ax_table.plot(
    [left, left + width],
    [
        bottom + height - row_height,
        bottom + height - row_height
    ],
    color=border_color,
    linewidth=1.2,
    transform=ax_table.transAxes
)

ax_table.plot(
    [left, left + width],
    [bottom, bottom],
    color=border_color,
    linewidth=1.5,
    transform=ax_table.transAxes
)

# Table note
fig_table.text(
    0.05,
    0.15,
    (
        "Note: Each observation represents one evaluable "
        "Subject ID–Phenotype pair. Precision = TP/(TP + FP); "
        "recall = TP/(TP + FN).\n"
        "TP, true positive; FP, false positive; "
        "FN, false negative; TN, true negative."
    ),
    fontsize=10.5,
    color="#59636F",
    ha="left",
    va="top"
)

plt.tight_layout()
plt.show()

# Optional high-resolution export
# fig_table.savefig(
#     "/Workspace/Users/user/user/"
#     "gold_standard_performance_table.png",
#     dpi=300,
#     bbox_inches="tight"
# )

# Numeric version retained for subsequent analyses
display(performance_summary)

In [0]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib_venn import venn2
from matplotlib.colors import to_rgb

df = gold_standard_full_info.copy()

key_columns = [
    "Subject ID",
    "Phenotype"
]

score_columns = {
    "Manual": "Manual_Score",
    "GPT 5.4": "GPT_Score",
    "GPT 5.4 Mini": "GPT_Mini_Score",
    "Claude Sonnet 4.6": "Claude_Sonnet_Score"
}

# Preserve the colors from the original reference cell
color_map = {
    "Manual": "#D4B896",
    "GPT 5.4": "#4C9BE8",
    "GPT 5.4 Mini": "#57C4A7",
    "Claude Sonnet 4.6": "#E87B4C"
}

# Soft pearl/off-white for the gold standard
gold_standard_color = "#D6DCE3"

required_columns = (
    key_columns
    + ["gold_standard_score"]
    + list(score_columns.values())
)

missing_columns = [
    column
    for column in required_columns
    if column not in df.columns
]

if missing_columns:
    raise KeyError(
        f"Missing required columns: {missing_columns}"
    )

# Each Subject ID–Phenotype pair should represent one observation
if df.duplicated(key_columns).any():
    raise ValueError(
        "gold_standard_full_info contains duplicate "
        "Subject ID–Phenotype pairs."
    )

# ------------------------------------------------------------
# Convert scores to numeric
# ------------------------------------------------------------

numeric_columns = (
    ["gold_standard_score"]
    + list(score_columns.values())
)

df[numeric_columns] = df[numeric_columns].apply(
    pd.to_numeric,
    errors="coerce"
)

# Confirm that all nonmissing scores are binary
for column in numeric_columns:

    unexpected_values = (
        df.loc[
            df[column].notna()
            & ~df[column].isin([0, 1]),
            column
        ]
        .unique()
        .tolist()
    )

    if unexpected_values:
        raise ValueError(
            f"{column} contains values other than 0 and 1: "
            f"{sorted(unexpected_values)}"
        )

print(
    "Rows excluded because the gold-standard score is unresolved: "
    f"{df['gold_standard_score'].isna().sum():,}"
)

# ------------------------------------------------------------
# Calculate TP, FP, FN, TN, precision, and recall
# ------------------------------------------------------------

results = {}

for method, score_column in score_columns.items():

    # Evaluate rows where both scores are available
    evaluable = df.loc[
        df["gold_standard_score"].notna()
        & df[score_column].notna(),
        key_columns
        + ["gold_standard_score", score_column]
    ].copy()

    evaluable["gold_standard_score"] = (
        evaluable["gold_standard_score"].astype(int)
    )

    evaluable[score_column] = (
        evaluable[score_column].astype(int)
    )

    # Identify positive Subject ID–Phenotype pairs
    gold_positive_set = set(
        evaluable.loc[
            evaluable["gold_standard_score"].eq(1),
            key_columns
        ].itertuples(
            index=False,
            name=None
        )
    )

    method_positive_set = set(
        evaluable.loc[
            evaluable[score_column].eq(1),
            key_columns
        ].itertuples(
            index=False,
            name=None
        )
    )

    true_positive = len(
        gold_positive_set
        & method_positive_set
    )

    false_negative = len(
        gold_positive_set
        - method_positive_set
    )

    false_positive = len(
        method_positive_set
        - gold_positive_set
    )

    true_negative = (
        len(evaluable)
        - true_positive
        - false_negative
        - false_positive
    )

    precision = (
        true_positive
        / (true_positive + false_positive)
        if true_positive + false_positive > 0
        else np.nan
    )

    recall = (
        true_positive
        / (true_positive + false_negative)
        if true_positive + false_negative > 0
        else np.nan
    )

    results[method] = {
        "score_column": score_column,
        "n_evaluable": len(evaluable),
        "gold_positives": (
            true_positive + false_negative
        ),
        "method_positives": (
            true_positive + false_positive
        ),
        "true_positive": true_positive,
        "false_positive": false_positive,
        "false_negative": false_negative,
        "true_negative": true_negative,
        "precision": precision,
        "recall": recall,
        "positive_union": (
            true_positive
            + false_positive
            + false_negative
        )
    }

# ------------------------------------------------------------
# Establish a shared Venn-diagram scale
# ------------------------------------------------------------

max_union = max(
    result["positive_union"]
    for result in results.values()
)

if max_union == 0:
    raise ValueError(
        "No positive calls were found in any scoring method."
    )


def draw_venn(ax, method, result):
    return venn2(
        subsets=(
            result["false_negative"],
            result["false_positive"],
            result["true_positive"]
        ),
        set_labels=("Gold standard", method),
        set_colors=(gold_standard_color, color_map[method]),
        alpha=0.5,
        normalize_to=result["positive_union"] / max_union,
        ax=ax
    )


# Measure the circles in every panel so all four share one scale
circle_extents = []

for method, result in results.items():

    draft_fig, draft_ax = plt.subplots(figsize=(5, 5))
    draft_venn = draw_venn(draft_ax, method, result)

    for center, radius in zip(draft_venn.centers, draft_venn.radii):
        circle_extents.append({
            "left": center.x - radius,
            "right": center.x + radius,
            "bottom": center.y - radius,
            "top": center.y + radius
        })

    plt.close(draft_fig)

circles_left = min(item["left"] for item in circle_extents)
circles_right = max(item["right"] for item in circle_extents)
circles_bottom = min(item["bottom"] for item in circle_extents)
circles_top = max(item["top"] for item in circle_extents)

circles_width = circles_right - circles_left

# Room reserved outside the circles for the FN / FP callouts (left
# and right) and for the set labels (below)
side_margin = 0.34 * circles_width
bottom_margin = 0.12 * circles_width
top_margin = 0.03 * circles_width

shared_xlim = (
    circles_left - side_margin,
    circles_right + side_margin
)

shared_ylim = (
    circles_bottom - bottom_margin,
    circles_top + top_margin
)

# ------------------------------------------------------------
# Draw the four-panel figure
# ------------------------------------------------------------

fig = plt.figure(figsize=(16, 17))

# Each panel gets a Venn row and its own metrics row, so the metrics
# text can never run into the panel below it
grid = fig.add_gridspec(
    nrows=4,
    ncols=2,
    height_ratios=[1, 0.50, 1, 0.50],
    left=0.03,
    right=0.97,
    top=0.865,
    bottom=0.02,
    hspace=0.10,
    wspace=0.06
)

panel_letters = "ABCDEFGH"

for index, (method, result) in enumerate(results.items()):

    panel_row = 2 * (index // 2)
    panel_col = index % 2

    ax = fig.add_subplot(grid[panel_row, panel_col])
    metrics_ax = fig.add_subplot(grid[panel_row + 1, panel_col])
    metrics_ax.axis("off")

    venn = draw_venn(ax, method, result)

    # --------------------------------------------------------
    # Correct the overlap color
    # --------------------------------------------------------

    overlap_patch = venn.get_patch_by_id("11")

    if overlap_patch is not None:

        # Average the gold-standard and scoring-method colors
        overlap_color = (
            np.array(to_rgb(gold_standard_color))
            + np.array(to_rgb(color_map[method]))
        ) / 2

        overlap_patch.set_facecolor(overlap_color)

        # Two layers at alpha=0.5 have an effective
        # combined opacity of 0.75
        overlap_patch.set_alpha(0.75)

    # Use the same scale across all four panels
    ax.set_xlim(shared_xlim)
    ax.set_ylim(shared_ylim)
    ax.set_aspect("equal")

    # Format circle borders
    for patch in venn.patches:
        if patch is not None:
            patch.set_edgecolor("white")
            patch.set_linewidth(2)

    gold_center, method_center = venn.centers
    gold_radius, method_radius = venn.radii

    # --------------------------------------------------------
    # Region labels
    # --------------------------------------------------------

    # TP stays inside the overlap, which is always large enough
    tp_label = venn.get_label_by_id("11")

    if tp_label is not None:
        tp_label.set_text(
            f"TP\nn = {result['true_positive']:,}"
        )
        tp_label.set_fontsize(18)
        tp_label.set_fontweight("bold")

    # FN and FP sit in thin crescents, so their labels are moved
    # outside the circles and tied back with a leader line
    callouts = {
        "10": {
            "text": f"FN\nn = {result['false_negative']:,}",
            "x": circles_left - 0.06 * circles_width,
            "ha": "right"
        },
        "01": {
            "text": f"FP\nn = {result['false_positive']:,}",
            "x": circles_right + 0.06 * circles_width,
            "ha": "left"
        }
    }

    for region_id, callout in callouts.items():

        label = venn.get_label_by_id(region_id)

        if label is None:
            continue

        region_x, region_y = label.get_position()
        label.set_visible(False)

        ax.annotate(
            callout["text"],
            xy=(region_x, region_y),
            xytext=(callout["x"], region_y),
            ha=callout["ha"],
            va="center",
            fontsize=17,
            fontweight="bold",
            arrowprops={
                "arrowstyle": "-",
                "color": "#52606D",
                "linewidth": 1.4,
                "shrinkA": 6,
                "shrinkB": 0
            }
        )

        # Small dot marking the region the leader line points to
        ax.plot(
            region_x,
            region_y,
            marker="o",
            markersize=5,
            color="#52606D"
        )

    # --------------------------------------------------------
    # Set labels: anchored apart from each other below the circles
    # --------------------------------------------------------

    set_label_y = circles_bottom - 0.04 * circles_width
    set_label_gap = 0.03 * circles_width
    set_label_mid = (gold_center.x + method_center.x) / 2

    gold_label, method_label = venn.set_labels

    if gold_label is not None:
        gold_label.set_position(
            (set_label_mid - set_label_gap, set_label_y)
        )
        gold_label.set_ha("right")

    if method_label is not None:
        method_label.set_position(
            (set_label_mid + set_label_gap, set_label_y)
        )
        method_label.set_ha("left")

    for label in venn.set_labels:
        if label is not None:
            label.set_va("top")
            label.set_fontsize(14)
            label.set_fontweight("bold")

    ax.set_title(
        f"Gold Standard vs {method}",
        fontsize=20,
        fontweight="bold",
        pad=10
    )

    # Panel letter (A-D) in the top-left corner of each panel
    ax.set_title(
        panel_letters[index],
        loc="left",
        fontsize=28,
        fontweight="bold",
        pad=10
    )

    precision_text = (
        "NA"
        if pd.isna(result["precision"])
        else f"{result['precision']:.3f}"
    )

    recall_text = (
        "NA"
        if pd.isna(result["recall"])
        else f"{result['recall']:.3f}"
    )

    metrics_text = (
        f"True Positives: {result['true_positive']:,}\n"
        f"False Positives: {result['false_positive']:,}\n"
        f"False Negatives: {result['false_negative']:,}\n"
        f"Precision = {precision_text}\n"
        f"Recall = {recall_text}"
    )

    #metrics_ax.text(
    #    0.5,
    #    1.0,
    #    metrics_text,
    #    transform=metrics_ax.transAxes,
    #    ha="center",
    #    va="top",
    #    fontsize=16,
    #    fontfamily="monospace",
    #    linespacing=1.4
    #)

fig.suptitle(
    "Positive Phenotype Calls Compared With the Gold Standard",
    fontsize=26,
    fontweight="bold",
    y=0.975
)

fig.text(
    0.5,
    0.925,
    (
        "TP indicates positive agreement; FP and FN indicate "
        "disagreement."
    ),
    ha="center",
    va="center",
    fontsize=18,
    color="#52606D",
    linespacing=1.4
)

plt.show()

# ------------------------------------------------------------
# Display the complete performance table
# ------------------------------------------------------------

performance_summary = pd.DataFrame([
    {
        "Scoring Method": method,
        "Evaluable_n": result["n_evaluable"],
        "Gold_Standard_Positives": result["gold_positives"],
        "Method_Positives": result["method_positives"],
        "True_Positive": result["true_positive"],
        "False_Positive": result["false_positive"],
        "False_Negative": result["false_negative"],
        "True_Negative": result["true_negative"],
        "Precision": result["precision"],
        "Recall": result["recall"]
    }
    for method, result in results.items()
])

display(performance_summary)

In [0]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib_venn import venn2

df = gold_standard_full_info.copy()

key_columns = [
    "Subject ID",
    "Phenotype"
]

score_columns = {
    "Manual": "Manual_Score",
    "GPT 5.4": "GPT_Score",
    "GPT 5.4 Mini": "GPT_Mini_Score",
    "Claude Sonnet 4.6": "Claude_Sonnet_Score"
}

# Preserve the colors from the original reference cell
color_map = {
    "Manual": "#D4B896",
    "GPT 5.4": "#4C9BE8",
    "GPT 5.4 Mini": "#57C4A7",
    "Claude Sonnet 4.6": "#E87B4C"
}

# Soft pearl/off-white for the gold standard
gold_standard_color = "#D6DCE3"

required_columns = (
    key_columns
    + ["gold_standard_score"]
    + list(score_columns.values())
)

missing_columns = [
    column
    for column in required_columns
    if column not in df.columns
]

if missing_columns:
    raise KeyError(
        f"Missing required columns: {missing_columns}"
    )

# Each Subject ID–Phenotype pair should represent one observation
if df.duplicated(key_columns).any():
    raise ValueError(
        "gold_standard_full_info contains duplicate "
        "Subject ID–Phenotype pairs."
    )

# ------------------------------------------------------------
# Convert scores to numeric
# ------------------------------------------------------------

numeric_columns = (
    ["gold_standard_score"]
    + list(score_columns.values())
)

df[numeric_columns] = df[numeric_columns].apply(
    pd.to_numeric,
    errors="coerce"
)

# Confirm that all nonmissing scores are binary
for column in numeric_columns:

    unexpected_values = (
        df.loc[
            df[column].notna()
            & ~df[column].isin([0, 1]),
            column
        ]
        .unique()
        .tolist()
    )

    if unexpected_values:
        raise ValueError(
            f"{column} contains values other than 0 and 1: "
            f"{sorted(unexpected_values)}"
        )

print(
    "Rows excluded because the gold-standard score is unresolved: "
    f"{df['gold_standard_score'].isna().sum():,}"
)

# ------------------------------------------------------------
# Calculate TP, FP, FN, TN, precision, and recall
# ------------------------------------------------------------

results = {}

for method, score_column in score_columns.items():

    # Evaluate rows where both scores are available
    evaluable = df.loc[
        df["gold_standard_score"].notna()
        & df[score_column].notna(),
        key_columns
        + ["gold_standard_score", score_column]
    ].copy()

    evaluable["gold_standard_score"] = (
        evaluable["gold_standard_score"].astype(int)
    )

    evaluable[score_column] = (
        evaluable[score_column].astype(int)
    )

    # Use Subject ID–Phenotype pairs instead of dataframe indices
    gold_positive_set = set(
        evaluable.loc[
            evaluable["gold_standard_score"].eq(1),
            key_columns
        ].itertuples(
            index=False,
            name=None
        )
    )

    method_positive_set = set(
        evaluable.loc[
            evaluable[score_column].eq(1),
            key_columns
        ].itertuples(
            index=False,
            name=None
        )
    )

    true_positive = len(
        gold_positive_set
        & method_positive_set
    )

    false_negative = len(
        gold_positive_set
        - method_positive_set
    )

    false_positive = len(
        method_positive_set
        - gold_positive_set
    )

    true_negative = (
        len(evaluable)
        - true_positive
        - false_negative
        - false_positive
    )

    precision = (
        true_positive
        / (true_positive + false_positive)
        if true_positive + false_positive > 0
        else np.nan
    )

    recall = (
        true_positive
        / (true_positive + false_negative)
        if true_positive + false_negative > 0
        else np.nan
    )

    results[method] = {
        "score_column": score_column,
        "n_evaluable": len(evaluable),
        "gold_positives": (
            true_positive + false_negative
        ),
        "method_positives": (
            true_positive + false_positive
        ),
        "true_positive": true_positive,
        "false_positive": false_positive,
        "false_negative": false_negative,
        "true_negative": true_negative,
        "precision": precision,
        "recall": recall,
        "positive_union": (
            true_positive
            + false_positive
            + false_negative
        )
    }

# ------------------------------------------------------------
# Establish a shared Venn-diagram scale
# ------------------------------------------------------------

max_union = max(
    result["positive_union"]
    for result in results.values()
)

if max_union == 0:
    raise ValueError(
        "No positive calls were found in any scoring method."
    )

draft_limits = []

for method, result in results.items():

    normalize_to = (
        result["positive_union"] / max_union
    )

    draft_fig, draft_ax = plt.subplots(
        figsize=(5, 5)
    )

    venn2(
        subsets=(
            result["false_negative"],
            result["false_positive"],
            result["true_positive"]
        ),
        set_labels=(
            "Gold standard",
            method
        ),
        set_colors=(
            gold_standard_color,
            color_map[method]
        ),
        alpha=0.5,
        normalize_to=normalize_to,
        ax=draft_ax
    )

    draft_limits.append({
        "xlim": draft_ax.get_xlim(),
        "ylim": draft_ax.get_ylim()
    })

    plt.close(draft_fig)

shared_xlim = (
    min(item["xlim"][0] for item in draft_limits),
    max(item["xlim"][1] for item in draft_limits)
)

shared_ylim = (
    min(item["ylim"][0] for item in draft_limits),
    max(item["ylim"][1] for item in draft_limits)
)

# ------------------------------------------------------------
# Draw the four-panel figure
# ------------------------------------------------------------

fig, axes = plt.subplots(
    nrows=2,
    ncols=2,
    figsize=(15, 14)
)

axes = axes.flatten()

for ax, (method, result) in zip(
    axes,
    results.items()
):

    normalize_to = (
        result["positive_union"] / max_union
    )

    venn = venn2(
        subsets=(
            result["false_negative"],
            result["false_positive"],
            result["true_positive"]
        ),
        set_labels=(
            "Gold standard",
            method
        ),
        set_colors=(
            gold_standard_color,
            color_map[method]
        ),
        alpha=0.5,
        normalize_to=normalize_to,
        ax=ax
    )

    # Use the same scale across all four panels
    ax.set_xlim(shared_xlim)
    ax.set_ylim(shared_ylim)
    ax.set_aspect("equal")

    # Label the three positive-call regions
    region_labels = {
        "10": (
            "FN\n"
            f"n = {result['false_negative']:,}"
        ),
        "01": (
            "FP\n"
            f"n = {result['false_positive']:,}"
        ),
        "11": (
            "TP\n"
            f"n = {result['true_positive']:,}"
        )
    }

    for region_id, label_text in region_labels.items():

        label = venn.get_label_by_id(region_id)

        if label is not None:
            label.set_text(label_text)
            label.set_fontsize(11)
            label.set_fontweight("bold")

    # Format circle borders
    for patch in venn.patches:
        if patch is not None:
            patch.set_edgecolor("white")
            patch.set_linewidth(2)

    # Format set labels
    for label in venn.set_labels:
        if label is not None:
            label.set_fontsize(12)
            label.set_fontweight("bold")

    ax.set_title(
        f"Gold Standard vs {method}",
        fontsize=15,
        fontweight="bold",
        pad=18
    )

    precision_text = (
        "NA"
        if pd.isna(result["precision"])
        else f"{result['precision']:.3f}"
    )

    recall_text = (
        "NA"
        if pd.isna(result["recall"])
        else f"{result['recall']:.3f}"
    )

    metrics_text = (
        "Precision = TP / (TP + FP)\n"
        f"          = {result['true_positive']:,} / "
        f"({result['true_positive']:,} + "
        f"{result['false_positive']:,}) "
        f"= {precision_text}\n"
        "Recall    = TP / (TP + FN)\n"
        f"          = {result['true_positive']:,} / "
        f"({result['true_positive']:,} + "
        f"{result['false_negative']:,}) "
        f"= {recall_text}\n"
        f"Both negative (TN) = "
        f"{result['true_negative']:,}    |    "
        f"Evaluable n = {result['n_evaluable']:,}"
    )

    ax.text(
        0.5,
        -0.15,
        metrics_text,
        transform=ax.transAxes,
        ha="center",
        va="top",
        fontsize=10,
        fontfamily="monospace",
        linespacing=1.45
    )

fig.suptitle(
    "Positive Phenotype Calls Compared With the Gold Standard",
    fontsize=21,
    fontweight="bold",
    y=0.98
)

fig.text(
    0.5,
    0.95,
    (
        "TP indicates positive agreement; FP and FN indicate "
        "disagreement. Negative agreement is reported as TN "
        "beneath each panel."
    ),
    ha="center",
    fontsize=11,
    color="#52606D"
)

plt.subplots_adjust(
    top=0.90,
    bottom=0.08,
    hspace=0.55,
    wspace=0.18
)

plt.show()

# ------------------------------------------------------------
# Display the complete performance table
# ------------------------------------------------------------

performance_summary = pd.DataFrame([
    {
        "Scoring Method": method,
        "Evaluable_n": result["n_evaluable"],
        "Gold_Standard_Positives": result["gold_positives"],
        "Method_Positives": result["method_positives"],
        "True_Positive": result["true_positive"],
        "False_Positive": result["false_positive"],
        "False_Negative": result["false_negative"],
        "True_Negative": result["true_negative"],
        "Precision": result["precision"],
        "Recall": result["recall"]
    }
    for method, result in results.items()
])

display(performance_summary)

In [0]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib_venn import venn2
from matplotlib.colors import to_rgb
from collections import Counter

df = gold_standard_full_info.copy()

key_columns = [
    "Subject ID",
    "Phenotype"
]

# The order here sets the panel order (A-D) and the ordering of
# rows and bars in the UpSet plot (panel E)
score_columns = {
    "Manual": "Manual_Score",
    "GPT 5.4": "GPT_Score",
    "GPT 5.4 Mini": "GPT_Mini_Score",
    "Claude Sonnet 4.6": "Claude_Sonnet_Score"
}

color_map = {
    "Manual": "#D4B896",
    "GPT 5.4": "#4C9BE8",
    "GPT 5.4 Mini": "#57C4A7",
    "Claude Sonnet 4.6": "#E87B4C"
}

# Soft pearl/off-white for the gold standard
gold_standard_color = "#D6DCE3"

set_names = list(score_columns.keys())

required_columns = (
    key_columns
    + ["gold_standard_score"]
    + list(score_columns.values())
)

missing_columns = [
    column
    for column in required_columns
    if column not in df.columns
]

if missing_columns:
    raise KeyError(
        f"Missing required columns: {missing_columns}"
    )

# Each Subject ID-Phenotype pair should represent one observation
if df.duplicated(key_columns).any():
    raise ValueError(
        "gold_standard_full_info contains duplicate "
        "Subject ID-Phenotype pairs."
    )

# ------------------------------------------------------------
# Convert scores to numeric and confirm they are binary
# ------------------------------------------------------------

numeric_columns = (
    ["gold_standard_score"]
    + list(score_columns.values())
)

df[numeric_columns] = df[numeric_columns].apply(
    pd.to_numeric,
    errors="coerce"
)

for column in numeric_columns:

    unexpected_values = (
        df.loc[
            df[column].notna()
            & ~df[column].isin([0, 1]),
            column
        ]
        .unique()
        .tolist()
    )

    if unexpected_values:
        raise ValueError(
            f"{column} contains values other than 0 and 1: "
            f"{sorted(unexpected_values)}"
        )

# ============================================================
# PART 1: Venn calculations (panels A-D)
# ============================================================

# ------------------------------------------------------------
# Calculate TP, FP, FN, TN, precision, and recall
# ------------------------------------------------------------

results = {}

for method, score_column in score_columns.items():

    # Evaluate rows where both scores are available
    evaluable = df.loc[
        df["gold_standard_score"].notna()
        & df[score_column].notna(),
        key_columns
        + ["gold_standard_score", score_column]
    ].copy()

    evaluable["gold_standard_score"] = (
        evaluable["gold_standard_score"].astype(int)
    )

    evaluable[score_column] = (
        evaluable[score_column].astype(int)
    )

    # Identify positive Subject ID–Phenotype pairs
    gold_positive_set = set(
        evaluable.loc[
            evaluable["gold_standard_score"].eq(1),
            key_columns
        ].itertuples(
            index=False,
            name=None
        )
    )

    method_positive_set = set(
        evaluable.loc[
            evaluable[score_column].eq(1),
            key_columns
        ].itertuples(
            index=False,
            name=None
        )
    )

    true_positive = len(
        gold_positive_set
        & method_positive_set
    )

    false_negative = len(
        gold_positive_set
        - method_positive_set
    )

    false_positive = len(
        method_positive_set
        - gold_positive_set
    )

    true_negative = (
        len(evaluable)
        - true_positive
        - false_negative
        - false_positive
    )

    precision = (
        true_positive
        / (true_positive + false_positive)
        if true_positive + false_positive > 0
        else np.nan
    )

    recall = (
        true_positive
        / (true_positive + false_negative)
        if true_positive + false_negative > 0
        else np.nan
    )

    results[method] = {
        "score_column": score_column,
        "n_evaluable": len(evaluable),
        "gold_positives": (
            true_positive + false_negative
        ),
        "method_positives": (
            true_positive + false_positive
        ),
        "true_positive": true_positive,
        "false_positive": false_positive,
        "false_negative": false_negative,
        "true_negative": true_negative,
        "precision": precision,
        "recall": recall,
        "positive_union": (
            true_positive
            + false_positive
            + false_negative
        )
    }

# ------------------------------------------------------------
# Establish a shared Venn-diagram scale
# ------------------------------------------------------------

max_union = max(
    result["positive_union"]
    for result in results.values()
)

if max_union == 0:
    raise ValueError(
        "No positive calls were found in any scoring method."
    )


def draw_venn(ax, method, result):
    return venn2(
        subsets=(
            result["false_negative"],
            result["false_positive"],
            result["true_positive"]
        ),
        set_labels=("Gold standard", method),
        set_colors=(gold_standard_color, color_map[method]),
        alpha=0.5,
        normalize_to=result["positive_union"] / max_union,
        ax=ax
    )


# Measure the circles in every panel so all four share one scale
circle_extents = []

for method, result in results.items():

    draft_fig, draft_ax = plt.subplots(figsize=(5, 5))
    draft_venn = draw_venn(draft_ax, method, result)

    for center, radius in zip(draft_venn.centers, draft_venn.radii):
        circle_extents.append({
            "left": center.x - radius,
            "right": center.x + radius,
            "bottom": center.y - radius,
            "top": center.y + radius
        })

    plt.close(draft_fig)

circles_left = min(item["left"] for item in circle_extents)
circles_right = max(item["right"] for item in circle_extents)
circles_bottom = min(item["bottom"] for item in circle_extents)
circles_top = max(item["top"] for item in circle_extents)

circles_width = circles_right - circles_left

# Room reserved outside the circles for the FN / FP callouts (left
# and right) and for the set labels (below)
side_margin = 0.34 * circles_width
bottom_margin = 0.12 * circles_width
top_margin = 0.03 * circles_width

shared_xlim = (
    circles_left - side_margin,
    circles_right + side_margin
)

shared_ylim = (
    circles_bottom - bottom_margin,
    circles_top + top_margin
)

# ============================================================
# PART 2: UpSet calculations (panel E)
# ============================================================

# ------------------------------------------------------------
# Create a common evaluable cohort
# ------------------------------------------------------------

resolved_gold = (
    df["gold_standard_score"].notna()
)

complete_method_scores = (
    df[list(score_columns.values())]
    .notna()
    .all(axis=1)
)

evaluable_mask = (
    resolved_gold
    & complete_method_scores
)

upset_df = (
    df.loc[evaluable_mask].copy()
)

print(f"Total rows: {len(df):,}")

print(
    "Rows excluded because gold standard is unresolved: "
    f"{(~resolved_gold).sum():,}"
)

print(
    "Rows excluded because at least one method score is "
    f"missing: "
    f"{(resolved_gold & ~complete_method_scores).sum():,}"
)

print(
    "Rows included in the UpSet analysis: "
    f"{len(upset_df):,}"
)

if upset_df.empty:
    raise ValueError(
        "No rows have both a resolved gold standard and "
        "complete scores from all four methods."
    )

# ------------------------------------------------------------
# Identify disagreement membership for every row
# ------------------------------------------------------------

disagreement_matrix = pd.DataFrame(
    {
        method: (
            upset_df[score_column]
            .ne(upset_df["gold_standard_score"])
        )
        for method, score_column
        in score_columns.items()
    },
    index=upset_df.index
)

# Count each exact combination of disagreements
membership_counter = Counter()

for disagreement_pattern in (
    disagreement_matrix.itertuples(
        index=False,
        name=None
    )
):
    membership = tuple(
        method
        for method, disagrees
        in zip(
            set_names,
            disagreement_pattern
        )
        if disagrees
    )

    membership_counter[membership] += 1

# Rows where every method agrees with the gold standard
all_methods_agree_count = (
    membership_counter.get((), 0)
)

# Remove the empty intersection from the UpSet bars
membership_counter.pop((), None)

if not membership_counter:
    raise ValueError(
        "All four scoring methods agree with the gold standard "
        "on every evaluable row."
    )

# Sort exact intersections by the number of disagreeing methods
# (single methods first, then pairs, then triples, ...). Within each
# group, follow the method order in score_columns: Manual, GPT 5.4,
# GPT 5.4 Mini, Claude Sonnet 4.6.
intersections = sorted(
    membership_counter.items(),
    key=lambda item: (
        len(item[0]),
        tuple(
            set_names.index(method)
            for method in item[0]
        )
    )
)

memberships = [
    membership
    for membership, count in intersections
]

counts = [
    count
    for membership, count in intersections
]

# Total disagreements for each individual method
set_sizes = {
    method: int(
        disagreement_matrix[method].sum()
    )
    for method in set_names
}

# ============================================================
# PART 3: Combined figure
# ============================================================

figure_width = 16
figure_height = 23.5

fig = plt.figure(
    figsize=(figure_width, figure_height)
)


def from_top(inches):
    """Convert a distance from the top edge (in inches) into a
    figure-fraction y coordinate."""
    return 1 - inches / figure_height


# Vertical layout, measured in inches from the top of the figure.
# Adjust these if you change font sizes or the figure height.
venn_title_y = from_top(0.50)
venn_subtitle_y = from_top(1.20)
venn_top = from_top(2.30)
venn_bottom = from_top(12.90)
upset_title_y = from_top(13.85)
upset_subtitle_y = from_top(14.50)
upset_top = from_top(15.10)
upset_bottom = from_top(21.90)
footnote_y = from_top(23.05)

# ------------------------------------------------------------
# Panels A-D: Venn diagrams
# ------------------------------------------------------------

fig.text(
    0.5,
    venn_title_y,
    "Positive Phenotype Calls Compared With the Gold Standard",
    ha="center",
    va="center",
    fontsize=26,
    fontweight="bold"
)

fig.text(
    0.5,
    venn_subtitle_y,
    (
        "TP indicates positive agreement; FP and FN indicate "
        "disagreement."
    ),
    ha="center",
    va="center",
    fontsize=18,
    color="#52606D",
    linespacing=1.4
)

venn_grid = fig.add_gridspec(
    nrows=2,
    ncols=2,
    left=0.03,
    right=0.97,
    top=venn_top,
    bottom=venn_bottom,
    hspace=0.16,
    wspace=0.06
)

panel_letters = "ABCDEFGH"

for index, (method, result) in enumerate(results.items()):

    panel_row = index // 2
    panel_col = index % 2

    ax = fig.add_subplot(venn_grid[panel_row, panel_col])

    venn = draw_venn(ax, method, result)

    # --------------------------------------------------------
    # Correct the overlap color
    # --------------------------------------------------------

    overlap_patch = venn.get_patch_by_id("11")

    if overlap_patch is not None:

        # Average the gold-standard and scoring-method colors
        overlap_color = (
            np.array(to_rgb(gold_standard_color))
            + np.array(to_rgb(color_map[method]))
        ) / 2

        overlap_patch.set_facecolor(overlap_color)

        # Two layers at alpha=0.5 have an effective
        # combined opacity of 0.75
        overlap_patch.set_alpha(0.75)

    # Use the same scale across all four panels
    ax.set_xlim(shared_xlim)
    ax.set_ylim(shared_ylim)
    ax.set_aspect("equal")

    # Format circle borders
    for patch in venn.patches:
        if patch is not None:
            patch.set_edgecolor("white")
            patch.set_linewidth(2)

    gold_center, method_center = venn.centers
    gold_radius, method_radius = venn.radii

    # --------------------------------------------------------
    # Region labels
    # --------------------------------------------------------

    # TP stays inside the overlap, which is always large enough
    tp_label = venn.get_label_by_id("11")

    if tp_label is not None:
        tp_label.set_text(
            f"TP\nn = {result['true_positive']:,}"
        )
        tp_label.set_fontsize(18)
        tp_label.set_fontweight("bold")

    # FN and FP sit in thin crescents, so their labels are moved
    # outside the circles and tied back with a leader line
    callouts = {
        "10": {
            "text": f"FN\nn = {result['false_negative']:,}",
            "x": circles_left - 0.06 * circles_width,
            "ha": "right"
        },
        "01": {
            "text": f"FP\nn = {result['false_positive']:,}",
            "x": circles_right + 0.06 * circles_width,
            "ha": "left"
        }
    }

    for region_id, callout in callouts.items():

        label = venn.get_label_by_id(region_id)

        if label is None:
            continue

        region_x, region_y = label.get_position()
        label.set_visible(False)

        ax.annotate(
            callout["text"],
            xy=(region_x, region_y),
            xytext=(callout["x"], region_y),
            ha=callout["ha"],
            va="center",
            fontsize=17,
            fontweight="bold",
            arrowprops={
                "arrowstyle": "-",
                "color": "#52606D",
                "linewidth": 1.4,
                "shrinkA": 6,
                "shrinkB": 0
            }
        )

        # Small dot marking the region the leader line points to
        ax.plot(
            region_x,
            region_y,
            marker="o",
            markersize=5,
            color="#52606D"
        )

    # --------------------------------------------------------
    # Set labels: anchored apart from each other below the circles
    # --------------------------------------------------------

    set_label_y = circles_bottom - 0.04 * circles_width
    set_label_gap = 0.03 * circles_width
    set_label_mid = (gold_center.x + method_center.x) / 2

    gold_label, method_label = venn.set_labels

    if gold_label is not None:
        gold_label.set_position(
            (set_label_mid - set_label_gap, set_label_y)
        )
        gold_label.set_ha("right")

    if method_label is not None:
        method_label.set_position(
            (set_label_mid + set_label_gap, set_label_y)
        )
        method_label.set_ha("left")

    for label in venn.set_labels:
        if label is not None:
            label.set_va("top")
            label.set_fontsize(14)
            label.set_fontweight("bold")

    ax.set_title(
        f"Gold Standard vs {method}",
        fontsize=20,
        fontweight="bold",
        pad=10
    )

    # Panel letter (A-D) in the top-left corner of each panel
    ax.set_title(
        panel_letters[index],
        loc="left",
        fontsize=28,
        fontweight="bold",
        pad=10
    )

# ------------------------------------------------------------
# Panel E: UpSet plot
# ------------------------------------------------------------

number_of_intersections = len(intersections)

# Panel letter, aligned with the A and C letters above
fig.text(
    0.03,
    upset_title_y,
    panel_letters[len(results)],
    ha="left",
    va="center",
    fontsize=28,
    fontweight="bold"
)

fig.text(
    0.5,
    upset_title_y,
    "Overlap of Scoring-Method Disagreements "
    "With the Gold Standard",
    ha="center",
    va="center",
    fontsize=22,
    fontweight="bold"
)

fig.text(
    0.5,
    upset_subtitle_y,
    (
        "Each bar represents one exact combination of methods "
        "that disagreed with gold_standard_score."
    ),
    ha="center",
    va="center",
    fontsize=15,
    color="#52606D"
)

# The wide left margin leaves room for the row labels
upset_grid = fig.add_gridspec(
    nrows=2,
    ncols=1,
    height_ratios=[3.0, 1.5],
    hspace=0.05,
    left=0.32,
    right=0.97,
    top=upset_top,
    bottom=upset_bottom
)

# ------------------------------------------------------------
# Top: exact intersection counts
# ------------------------------------------------------------

ax_bar = fig.add_subplot(upset_grid[0])

bar_positions = np.arange(
    number_of_intersections
)

bars = ax_bar.bar(
    bar_positions,
    counts,
    color="#4C9BE8",
    width=0.65,
    edgecolor="white"
)

maximum_count = max(counts)

label_padding = max(
    maximum_count * 0.015,
    0.5
)

for position, count in zip(
    bar_positions,
    counts
):
    ax_bar.text(
        position,
        count + label_padding,
        f"{count:,}",
        ha="center",
        va="bottom",
        fontsize=14,
        fontweight="bold"
    )

ax_bar.set_xticks([])

ax_bar.set_ylabel(
    "Exact intersection count",
    fontsize=17,
    labelpad=10
)

ax_bar.tick_params(
    axis="y",
    labelsize=14
)

ax_bar.set_ylim(
    0,
    maximum_count * 1.15
)

ax_bar.spines["top"].set_visible(False)
ax_bar.spines["right"].set_visible(False)
ax_bar.spines["bottom"].set_visible(False)

ax_bar.grid(
    axis="y",
    color="#D5DDE5",
    linewidth=0.8,
    alpha=0.7
)

ax_bar.set_axisbelow(True)

# ------------------------------------------------------------
# Bottom: membership matrix
# ------------------------------------------------------------

ax_matrix = fig.add_subplot(
    upset_grid[1],
    sharex=ax_bar
)

for position, membership in zip(
    bar_positions,
    memberships
):
    active_indices = [
        set_names.index(method)
        for method in membership
    ]

    # Connecting line behind active dots
    if len(active_indices) > 1:
        ax_matrix.plot(
            [position, position],
            [
                min(active_indices),
                max(active_indices)
            ],
            color="#333333",
            linewidth=2.5,
            zorder=1
        )

    for set_index, method in enumerate(
        set_names
    ):
        is_active = (
            set_index in active_indices
        )

        dot_color = (
            color_map[method]
            if is_active
            else "#DDDDDD"
        )

        dot_size = (
            15 if is_active else 12
        )

        ax_matrix.plot(
            position,
            set_index,
            "o",
            color=dot_color,
            markersize=dot_size,
            markeredgecolor=(
                "#333333"
                if is_active
                else "#DDDDDD"
            ),
            markeredgewidth=0.8,
            zorder=2
        )

ax_matrix.set_yticks(
    range(len(set_names))
)

ax_matrix.set_yticklabels(
    [
        f"{method} (total disagreements = "
        f"{set_sizes[method]:,})"
        for method in set_names
    ],
    fontsize=15
)

ax_matrix.set_xticks([])

ax_matrix.set_xlim(
    -0.7,
    number_of_intersections - 0.3
)

ax_matrix.set_ylim(
    -0.5,
    len(set_names) - 0.5
)

ax_matrix.invert_yaxis()

for spine in ax_matrix.spines.values():
    spine.set_visible(False)

ax_matrix.tick_params(
    axis="y",
    length=0,
    pad=10
)

# ------------------------------------------------------------
# Mark the groups: 1 method, 2 methods, 3 methods, ...
# ------------------------------------------------------------

group_sizes = [
    len(membership)
    for membership in memberships
]

for group_size in sorted(set(group_sizes)):

    group_positions = [
        position
        for position, size in zip(bar_positions, group_sizes)
        if size == group_size
    ]

    group_start = min(group_positions)
    group_end = max(group_positions)

    # Group label beneath the matrix
    ax_matrix.text(
        (group_start + group_end) / 2,
        -0.10,
        (
            "1 method"
            if group_size == 1
            else f"{group_size} methods"
        ),
        transform=ax_matrix.get_xaxis_transform(),
        ha="center",
        va="top",
        fontsize=15,
        fontweight="bold",
        color="#52606D"
    )

    # Bracket line spanning the group
    ax_matrix.plot(
        [group_start - 0.3, group_end + 0.3],
        [-0.05, -0.05],
        transform=ax_matrix.get_xaxis_transform(),
        color="#9AA5B1",
        linewidth=1.5,
        clip_on=False
    )

    # Divider between this group and the next one
    if group_end < number_of_intersections - 1:
        for axis in (ax_bar, ax_matrix):
            axis.axvline(
                group_end + 0.5,
                color="#C5CED8",
                linewidth=1.2,
                linestyle="--",
                zorder=0
            )

fig.text(
    0.5,
    footnote_y,
    (
        f"Rows where all four methods agree with the gold "
        f"standard: n = {all_methods_agree_count:,}. "
        "Inactive dots indicate agreement, not missing data."
    ),
    ha="center",
    va="center",
    fontsize=14,
    color="#52606D"
)

# To save a publication-quality copy, uncomment:
# fig.savefig("combined_figure.png", dpi=300, bbox_inches="tight")
# fig.savefig("combined_figure.pdf", bbox_inches="tight")

plt.show()

# ============================================================
# PART 4: Summary tables
# ============================================================

# ------------------------------------------------------------
# Display the complete performance table
# ------------------------------------------------------------

performance_summary = pd.DataFrame([
    {
        "Scoring Method": method,
        "Evaluable_n": result["n_evaluable"],
        "Gold_Standard_Positives": result["gold_positives"],
        "Method_Positives": result["method_positives"],
        "True_Positive": result["true_positive"],
        "False_Positive": result["false_positive"],
        "False_Negative": result["false_negative"],
        "True_Negative": result["true_negative"],
        "Precision": result["precision"],
        "Recall": result["recall"]
    }
    for method, result in results.items()
])

display(performance_summary)

# ------------------------------------------------------------
# Display exact intersection counts
# ------------------------------------------------------------

upset_intersection_summary = pd.DataFrame([
    {
        "Disagreeing methods": ", ".join(membership),
        "Number of methods": len(membership),
        "Exact intersection count": count
    }
    for membership, count in intersections
])

display(upset_intersection_summary)

# Display total disagreements by scoring method
upset_set_size_summary = pd.DataFrame([
    {
        "Scoring method": method,
        "Total disagreements with gold standard": (
            set_sizes[method]
        )
    }
    for method in set_names
])

display(upset_set_size_summary)

In [0]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib_venn import venn2
from matplotlib.colors import to_rgb
from collections import Counter

df = gold_standard_full_info.copy()

key_columns = [
    "Subject ID",
    "Phenotype"
]

# The order here sets the panel order (A-D) and the ordering of
# rows and bars in the UpSet plot (panel E)
score_columns = {
    "Manual": "Manual_Score",
    "GPT 5.4": "GPT_Score",
    "GPT 5.4 Mini": "GPT_Mini_Score",
    "Claude Sonnet 4.6": "Claude_Sonnet_Score"
}

color_map = {
    "Manual": "#D4B896",
    "GPT 5.4": "#4C9BE8",
    "GPT 5.4 Mini": "#57C4A7",
    "Claude Sonnet 4.6": "#E87B4C"
}

# Soft pearl/off-white for the gold standard
gold_standard_color = "#D6DCE3"

set_names = list(score_columns.keys())

required_columns = (
    key_columns
    + ["gold_standard_score"]
    + list(score_columns.values())
)

missing_columns = [
    column
    for column in required_columns
    if column not in df.columns
]

if missing_columns:
    raise KeyError(
        f"Missing required columns: {missing_columns}"
    )

# Each Subject ID-Phenotype pair should represent one observation
if df.duplicated(key_columns).any():
    raise ValueError(
        "gold_standard_full_info contains duplicate "
        "Subject ID-Phenotype pairs."
    )

# ------------------------------------------------------------
# Convert scores to numeric and confirm they are binary
# ------------------------------------------------------------

numeric_columns = (
    ["gold_standard_score"]
    + list(score_columns.values())
)

df[numeric_columns] = df[numeric_columns].apply(
    pd.to_numeric,
    errors="coerce"
)

for column in numeric_columns:

    unexpected_values = (
        df.loc[
            df[column].notna()
            & ~df[column].isin([0, 1]),
            column
        ]
        .unique()
        .tolist()
    )

    if unexpected_values:
        raise ValueError(
            f"{column} contains values other than 0 and 1: "
            f"{sorted(unexpected_values)}"
        )

# ============================================================
# PART 1: Venn calculations (panels A-D)
# ============================================================

# ------------------------------------------------------------
# Calculate TP, FP, FN, TN, precision, and recall
# ------------------------------------------------------------

results = {}

for method, score_column in score_columns.items():

    # Evaluate rows where both scores are available
    evaluable = df.loc[
        df["gold_standard_score"].notna()
        & df[score_column].notna(),
        key_columns
        + ["gold_standard_score", score_column]
    ].copy()

    evaluable["gold_standard_score"] = (
        evaluable["gold_standard_score"].astype(int)
    )

    evaluable[score_column] = (
        evaluable[score_column].astype(int)
    )

    # Identify positive Subject ID–Phenotype pairs
    gold_positive_set = set(
        evaluable.loc[
            evaluable["gold_standard_score"].eq(1),
            key_columns
        ].itertuples(
            index=False,
            name=None
        )
    )

    method_positive_set = set(
        evaluable.loc[
            evaluable[score_column].eq(1),
            key_columns
        ].itertuples(
            index=False,
            name=None
        )
    )

    true_positive = len(
        gold_positive_set
        & method_positive_set
    )

    false_negative = len(
        gold_positive_set
        - method_positive_set
    )

    false_positive = len(
        method_positive_set
        - gold_positive_set
    )

    true_negative = (
        len(evaluable)
        - true_positive
        - false_negative
        - false_positive
    )

    precision = (
        true_positive
        / (true_positive + false_positive)
        if true_positive + false_positive > 0
        else np.nan
    )

    recall = (
        true_positive
        / (true_positive + false_negative)
        if true_positive + false_negative > 0
        else np.nan
    )

    results[method] = {
        "score_column": score_column,
        "n_evaluable": len(evaluable),
        "gold_positives": (
            true_positive + false_negative
        ),
        "method_positives": (
            true_positive + false_positive
        ),
        "true_positive": true_positive,
        "false_positive": false_positive,
        "false_negative": false_negative,
        "true_negative": true_negative,
        "precision": precision,
        "recall": recall,
        "positive_union": (
            true_positive
            + false_positive
            + false_negative
        )
    }

# ------------------------------------------------------------
# Establish a shared Venn-diagram scale
# ------------------------------------------------------------

max_union = max(
    result["positive_union"]
    for result in results.values()
)

if max_union == 0:
    raise ValueError(
        "No positive calls were found in any scoring method."
    )


def draw_venn(ax, method, result):
    return venn2(
        subsets=(
            result["false_negative"],
            result["false_positive"],
            result["true_positive"]
        ),
        set_labels=("Gold standard", method),
        set_colors=(gold_standard_color, color_map[method]),
        alpha=0.5,
        normalize_to=result["positive_union"] / max_union,
        ax=ax
    )


# Measure the circles in every panel so all four share one scale
circle_extents = []

for method, result in results.items():

    draft_fig, draft_ax = plt.subplots(figsize=(5, 5))
    draft_venn = draw_venn(draft_ax, method, result)

    for center, radius in zip(draft_venn.centers, draft_venn.radii):
        circle_extents.append({
            "left": center.x - radius,
            "right": center.x + radius,
            "bottom": center.y - radius,
            "top": center.y + radius
        })

    plt.close(draft_fig)

circles_left = min(item["left"] for item in circle_extents)
circles_right = max(item["right"] for item in circle_extents)
circles_bottom = min(item["bottom"] for item in circle_extents)
circles_top = max(item["top"] for item in circle_extents)

circles_width = circles_right - circles_left

# Room reserved outside the circles for the FN / FP callouts (left
# and right) and for the set labels (below)
side_margin = 0.34 * circles_width
bottom_margin = 0.12 * circles_width
top_margin = 0.03 * circles_width

shared_xlim = (
    circles_left - side_margin,
    circles_right + side_margin
)

shared_ylim = (
    circles_bottom - bottom_margin,
    circles_top + top_margin
)

# ============================================================
# PART 2: UpSet calculations (panel E)
# ============================================================

# ------------------------------------------------------------
# Create a common evaluable cohort
# ------------------------------------------------------------

resolved_gold = (
    df["gold_standard_score"].notna()
)

complete_method_scores = (
    df[list(score_columns.values())]
    .notna()
    .all(axis=1)
)

evaluable_mask = (
    resolved_gold
    & complete_method_scores
)

upset_df = (
    df.loc[evaluable_mask].copy()
)

print(f"Total rows: {len(df):,}")

print(
    "Rows excluded because gold standard is unresolved: "
    f"{(~resolved_gold).sum():,}"
)

print(
    "Rows excluded because at least one method score is "
    f"missing: "
    f"{(resolved_gold & ~complete_method_scores).sum():,}"
)

print(
    "Rows included in the UpSet analysis: "
    f"{len(upset_df):,}"
)

if upset_df.empty:
    raise ValueError(
        "No rows have both a resolved gold standard and "
        "complete scores from all four methods."
    )

# ------------------------------------------------------------
# Identify disagreement membership for every row
# ------------------------------------------------------------

disagreement_matrix = pd.DataFrame(
    {
        method: (
            upset_df[score_column]
            .ne(upset_df["gold_standard_score"])
        )
        for method, score_column
        in score_columns.items()
    },
    index=upset_df.index
)

# Count each exact combination of disagreements
membership_counter = Counter()

for disagreement_pattern in (
    disagreement_matrix.itertuples(
        index=False,
        name=None
    )
):
    membership = tuple(
        method
        for method, disagrees
        in zip(
            set_names,
            disagreement_pattern
        )
        if disagrees
    )

    membership_counter[membership] += 1

# Rows where every method agrees with the gold standard
all_methods_agree_count = (
    membership_counter.get((), 0)
)

# Remove the empty intersection from the UpSet bars
membership_counter.pop((), None)

if not membership_counter:
    raise ValueError(
        "All four scoring methods agree with the gold standard "
        "on every evaluable row."
    )

# Sort exact intersections by the number of disagreeing methods
# (single methods first, then pairs, then triples, ...). Within each
# group, follow the method order in score_columns: Manual, GPT 5.4,
# GPT 5.4 Mini, Claude Sonnet 4.6.
intersections = sorted(
    membership_counter.items(),
    key=lambda item: (
        len(item[0]),
        tuple(
            set_names.index(method)
            for method in item[0]
        )
    )
)

memberships = [
    membership
    for membership, count in intersections
]

counts = [
    count
    for membership, count in intersections
]

# Total disagreements for each individual method
set_sizes = {
    method: int(
        disagreement_matrix[method].sum()
    )
    for method in set_names
}

# ============================================================
# PART 3: Combined figure
# ============================================================

figure_width = 16
figure_height = 23.5

fig = plt.figure(
    figsize=(figure_width, figure_height)
)


def from_top(inches):
    """Convert a distance from the top edge (in inches) into a
    figure-fraction y coordinate."""
    return 1 - inches / figure_height


# Vertical layout, measured in inches from the top of the figure.
# Adjust these if you change font sizes or the figure height.
venn_title_y = from_top(0.50)
venn_subtitle_y = from_top(1.20)
venn_top = from_top(2.30)
venn_bottom = from_top(12.90)
upset_title_y = from_top(13.85)
upset_subtitle_y = from_top(14.50)
upset_top = from_top(15.10)
upset_bottom = from_top(21.90)
footnote_y = from_top(23.05)

# ------------------------------------------------------------
# Panels A-D: Venn diagrams
# ------------------------------------------------------------

fig.text(
    0.5,
    venn_title_y,
    "Positive Phenotype Calls Compared With the Gold Standard",
    ha="center",
    va="center",
    fontsize=26,
    fontweight="bold"
)

fig.text(
    0.5,
    venn_subtitle_y,
    (
        "TP indicates positive agreement; FP and FN indicate "
        "disagreement."
    ),
    ha="center",
    va="center",
    fontsize=18,
    color="#52606D",
    linespacing=1.4
)

venn_grid = fig.add_gridspec(
    nrows=2,
    ncols=2,
    left=0.03,
    right=0.97,
    top=venn_top,
    bottom=venn_bottom,
    hspace=0.16,
    wspace=0.06
)

panel_letters = "ABCDEFGH"

for index, (method, result) in enumerate(results.items()):

    panel_row = index // 2
    panel_col = index % 2

    ax = fig.add_subplot(venn_grid[panel_row, panel_col])

    venn = draw_venn(ax, method, result)

    # --------------------------------------------------------
    # Correct the overlap color
    # --------------------------------------------------------

    overlap_patch = venn.get_patch_by_id("11")

    if overlap_patch is not None:

        # Average the gold-standard and scoring-method colors
        overlap_color = (
            np.array(to_rgb(gold_standard_color))
            + np.array(to_rgb(color_map[method]))
        ) / 2

        overlap_patch.set_facecolor(overlap_color)

        # Two layers at alpha=0.5 have an effective
        # combined opacity of 0.75
        overlap_patch.set_alpha(0.75)

    # Use the same scale across all four panels
    ax.set_xlim(shared_xlim)
    ax.set_ylim(shared_ylim)
    ax.set_aspect("equal")

    # Format circle borders
    for patch in venn.patches:
        if patch is not None:
            patch.set_edgecolor("white")
            patch.set_linewidth(2)

    gold_center, method_center = venn.centers
    gold_radius, method_radius = venn.radii

    # --------------------------------------------------------
    # Region labels
    # --------------------------------------------------------

    # TP stays inside the overlap, which is always large enough
    tp_label = venn.get_label_by_id("11")

    if tp_label is not None:
        tp_label.set_text(
            f"TP\nn = {result['true_positive']:,}"
        )
        tp_label.set_fontsize(18)
        tp_label.set_fontweight("bold")

    # FN and FP sit in thin crescents, so their labels are moved
    # outside the circles and tied back with a leader line
    callouts = {
        "10": {
            "text": f"FN\nn = {result['false_negative']:,}",
            "x": circles_left - 0.06 * circles_width,
            "ha": "right"
        },
        "01": {
            "text": f"FP\nn = {result['false_positive']:,}",
            "x": circles_right + 0.06 * circles_width,
            "ha": "left"
        }
    }

    for region_id, callout in callouts.items():

        label = venn.get_label_by_id(region_id)

        if label is None:
            continue

        region_x, region_y = label.get_position()
        label.set_visible(False)

        ax.annotate(
            callout["text"],
            xy=(region_x, region_y),
            xytext=(callout["x"], region_y),
            ha=callout["ha"],
            va="center",
            fontsize=17,
            fontweight="bold",
            arrowprops={
                "arrowstyle": "-",
                "color": "#52606D",
                "linewidth": 1.4,
                "shrinkA": 6,
                "shrinkB": 0
            }
        )

        # Small dot marking the region the leader line points to
        ax.plot(
            region_x,
            region_y,
            marker="o",
            markersize=5,
            color="#52606D"
        )

    # --------------------------------------------------------
    # Set labels: anchored apart from each other below the circles
    # --------------------------------------------------------

    set_label_y = circles_bottom - 0.04 * circles_width
    set_label_gap = 0.03 * circles_width
    set_label_mid = (gold_center.x + method_center.x) / 2

    gold_label, method_label = venn.set_labels

    if gold_label is not None:
        gold_label.set_position(
            (set_label_mid - set_label_gap, set_label_y)
        )
        gold_label.set_ha("right")

    if method_label is not None:
        method_label.set_position(
            (set_label_mid + set_label_gap, set_label_y)
        )
        method_label.set_ha("left")

    for label in venn.set_labels:
        if label is not None:
            label.set_va("top")
            label.set_fontsize(14)
            label.set_fontweight("bold")

    ax.set_title(
        f"Gold Standard vs {method}",
        fontsize=20,
        fontweight="bold",
        pad=10
    )

    # Panel letter (A-D) in the top-left corner of each panel
    ax.set_title(
        panel_letters[index],
        loc="left",
        fontsize=28,
        fontweight="bold",
        pad=10
    )

# ------------------------------------------------------------
# Panel E: UpSet plot
# ------------------------------------------------------------

number_of_intersections = len(intersections)

# Panel letter, aligned with the A and C letters above
fig.text(
    0.03,
    upset_title_y,
    panel_letters[len(results)],
    ha="left",
    va="center",
    fontsize=28,
    fontweight="bold"
)

fig.text(
    0.5,
    upset_title_y,
    "Overlap of Scoring-Method Disagreements "
    "With the Gold Standard",
    ha="center",
    va="center",
    fontsize=22,
    fontweight="bold"
)

fig.text(
    0.5,
    upset_subtitle_y,
    (
        "Each bar represents one exact combination of methods "
        "that disagreed with gold_standard_score."
    ),
    ha="center",
    va="center",
    fontsize=15,
    color="#52606D"
)

# The wide left margin leaves room for the row labels
upset_grid = fig.add_gridspec(
    nrows=2,
    ncols=1,
    height_ratios=[3.0, 1.5],
    hspace=0.05,
    left=0.32,
    right=0.97,
    top=upset_top,
    bottom=upset_bottom
)

# ------------------------------------------------------------
# Top: exact intersection counts
# ------------------------------------------------------------

ax_bar = fig.add_subplot(upset_grid[0])

bar_positions = np.arange(
    number_of_intersections
)

bars = ax_bar.bar(
    bar_positions,
    counts,
    color="#4C9BE8",
    width=0.65,
    edgecolor="white"
)

maximum_count = max(counts)

label_padding = max(
    maximum_count * 0.015,
    0.5
)

for position, count in zip(
    bar_positions,
    counts
):
    ax_bar.text(
        position,
        count + label_padding,
        f"{count:,}",
        ha="center",
        va="bottom",
        fontsize=14,
        fontweight="bold"
    )

ax_bar.set_xticks([])

ax_bar.set_ylabel(
    "Exact intersection count",
    fontsize=17,
    labelpad=10
)

ax_bar.tick_params(
    axis="y",
    labelsize=14
)

ax_bar.set_ylim(
    0,
    maximum_count * 1.15
)

ax_bar.spines["top"].set_visible(False)
ax_bar.spines["right"].set_visible(False)
ax_bar.spines["bottom"].set_visible(False)

ax_bar.grid(
    axis="y",
    color="#D5DDE5",
    linewidth=0.8,
    alpha=0.7
)

ax_bar.set_axisbelow(True)

# ------------------------------------------------------------
# Bottom: membership matrix
# ------------------------------------------------------------

ax_matrix = fig.add_subplot(
    upset_grid[1],
    sharex=ax_bar
)

for position, membership in zip(
    bar_positions,
    memberships
):
    active_indices = [
        set_names.index(method)
        for method in membership
    ]

    # Connecting line behind active dots
    if len(active_indices) > 1:
        ax_matrix.plot(
            [position, position],
            [
                min(active_indices),
                max(active_indices)
            ],
            color="#333333",
            linewidth=2.5,
            zorder=1
        )

    for set_index, method in enumerate(
        set_names
    ):
        is_active = (
            set_index in active_indices
        )

        dot_color = (
            color_map[method]
            if is_active
            else "#DDDDDD"
        )

        dot_size = (
            15 if is_active else 12
        )

        ax_matrix.plot(
            position,
            set_index,
            "o",
            color=dot_color,
            markersize=dot_size,
            markeredgecolor=(
                "#333333"
                if is_active
                else "#DDDDDD"
            ),
            markeredgewidth=0.8,
            zorder=2
        )

ax_matrix.set_yticks(
    range(len(set_names))
)

ax_matrix.set_yticklabels(
    [
        f"{method} (total disagreements = "
        f"{set_sizes[method]:,})"
        for method in set_names
    ],
    fontsize=15
)

ax_matrix.set_xticks([])

ax_matrix.set_xlim(
    -0.7,
    number_of_intersections - 0.3
)

ax_matrix.set_ylim(
    -0.5,
    len(set_names) - 0.5
)

ax_matrix.invert_yaxis()

for spine in ax_matrix.spines.values():
    spine.set_visible(False)

ax_matrix.tick_params(
    axis="y",
    length=0,
    pad=10
)

# ------------------------------------------------------------
# Mark the groups: 1 method, 2 methods, 3 methods, ...
# ------------------------------------------------------------

group_sizes = [
    len(membership)
    for membership in memberships
]

for group_size in sorted(set(group_sizes)):

    group_positions = [
        position
        for position, size in zip(bar_positions, group_sizes)
        if size == group_size
    ]

    group_start = min(group_positions)
    group_end = max(group_positions)

    # Group label beneath the matrix
    ax_matrix.text(
        (group_start + group_end) / 2,
        -0.10,
        (
            "1 method"
            if group_size == 1
            else f"{group_size} methods"
        ),
        transform=ax_matrix.get_xaxis_transform(),
        ha="center",
        va="top",
        fontsize=15,
        fontweight="bold",
        color="#52606D"
    )

    # Bracket line spanning the group
    ax_matrix.plot(
        [group_start - 0.3, group_end + 0.3],
        [-0.05, -0.05],
        transform=ax_matrix.get_xaxis_transform(),
        color="#9AA5B1",
        linewidth=1.5,
        clip_on=False
    )

    # Divider between this group and the next one
    if group_end < number_of_intersections - 1:
        for axis in (ax_bar, ax_matrix):
            axis.axvline(
                group_end + 0.5,
                color="#C5CED8",
                linewidth=1.2,
                linestyle="--",
                zorder=0
            )

fig.text(
    0.5,
    footnote_y,
    (
        f"Rows where all four methods agree with the gold "
        f"standard: n = {all_methods_agree_count:,}. "
        "Inactive dots indicate agreement, not missing data."
    ),
    ha="center",
    va="center",
    fontsize=14,
    color="#52606D"
)

# To save a publication-quality copy, uncomment:
# fig.savefig("combined_figure.png", dpi=300, bbox_inches="tight")
# fig.savefig("combined_figure.pdf", bbox_inches="tight")

plt.show()

# ============================================================
# PART 4: Summary tables
# ============================================================

# ------------------------------------------------------------
# Display the complete performance table
# ------------------------------------------------------------

performance_summary = pd.DataFrame([
    {
        "Scoring Method": method,
        "Evaluable_n": result["n_evaluable"],
        "Gold_Standard_Positives": result["gold_positives"],
        "Method_Positives": result["method_positives"],
        "True_Positive": result["true_positive"],
        "False_Positive": result["false_positive"],
        "False_Negative": result["false_negative"],
        "True_Negative": result["true_negative"],
        "Precision": result["precision"],
        "Recall": result["recall"]
    }
    for method, result in results.items()
])

display(performance_summary)

# ------------------------------------------------------------
# Display exact intersection counts
# ------------------------------------------------------------

upset_intersection_summary = pd.DataFrame([
    {
        "Disagreeing methods": ", ".join(membership),
        "Number of methods": len(membership),
        "Exact intersection count": count
    }
    for membership, count in intersections
])

display(upset_intersection_summary)

# Display total disagreements by scoring method
upset_set_size_summary = pd.DataFrame([
    {
        "Scoring method": method,
        "Total disagreements with gold standard": (
            set_sizes[method]
        )
    }
    for method in set_names
])

display(upset_set_size_summary)

In [0]:
gold_standard_full_info.columns


In [0]:
print(
    "Unique Subject IDs:",
    gold_standard_full_info["Subject ID"].nunique()
)

In [0]:
len(set(gold_standard_full_info["Phenotype"].dropna()))

In [0]:
# Unique phenotypes in the full GPT run but not in the gold-standard dataframe
full_gpt_only_phenotypes = sorted(
    set(combined_df_full_info["Phenotype"].dropna())
    - set(gold_standard_full_info["Phenotype"].dropna())
)

print(f"Number of unique phenotypes: {len(full_gpt_only_phenotypes):,}")

#display(
#    pd.DataFrame(
#        full_gpt_only_phenotypes,
#        columns=["Phenotype"]
#    )
#)

# Save dataframes

In [0]:
# Define the base filepath
base_path = "/Workspace/Users/user/user/"

# Save each dataframe to CSV
combined_df_full_info.to_csv(f"{base_path}combined_df_full_info.csv", index=False)
combined_df_full_info_adjudications.to_csv(f"{base_path}combined_df_full_info_adjudications.csv", index=False)
gold_standard_full_info.to_csv(f"{base_path}gold_standard_full_info.csv", index=False)

print("All files saved successfully.")

In [0]:
df = gold_standard_full_info.copy()

# Treat blanks and nonnumeric entries as missing
df["Manual_Score_clean"] = pd.to_numeric(df["Manual_Score"], errors="coerce")

# Subjects with at least one manual score of 1
subjects_with_a_one = set(
    df.loc[df["Manual_Score_clean"].eq(1), "Subject ID"].dropna().unique()
)

# Every unique subject not in that set
no_manual_ones_ids = sorted(
    set(df["Subject ID"].dropna().unique()) - subjects_with_a_one
)

print(f"{len(no_manual_ones_ids):,} unique subjects have no manual scores of 1\n")

for subject_id in no_manual_ones_ids:
    print(subject_id)

In [0]:
notes_export = (
    gold_standard_full_info.loc[
        gold_standard_full_info["Subject ID"].isin(no_manual_ones_ids),
        ["Subject ID", "NOTE_TEXT"]
    ]
    .drop_duplicates()
    .sort_values("Subject ID")
    .reset_index(drop=True)
)

print(
    f"{len(notes_export):,} unique notes across "
    f"{notes_export['Subject ID'].nunique():,} subjects"
)

notes_export.to_csv(
    "subjects_no_manual_score_1_notes.csv",
    index=False
)

display(notes_export)

In [0]:
print(pd.read_csv("subjects_no_manual_score_1_notes.csv").columns.tolist())

In [0]:
import csv

notes_export = (
    gold_standard_full_info.loc[
        gold_standard_full_info["Subject ID"].isin(no_manual_ones_ids),
        ["Subject ID", "NOTE_TEXT"]
    ]
    .drop_duplicates()
    .sort_values("Subject ID")
    .reset_index(drop=True)
)

# Replace line breaks (and tabs) inside each note with a single space,
# so each note stays in one cell when the CSV is opened in Excel
notes_export["NOTE_TEXT"] = (
    notes_export["NOTE_TEXT"]
    .astype("string")
    .str.replace(r"[\r\n\t\u2028\u2029]+", " ", regex=True)
    .str.strip()
)

# Excel cannot hold more than 32,767 characters in one cell
excel_cell_limit = 32767
too_long = notes_export["NOTE_TEXT"].str.len() > excel_cell_limit

if too_long.any():
    print(
        f"WARNING: {too_long.sum():,} notes are longer than {excel_cell_limit:,} "
        "characters and will still spill into extra cells if opened in Excel. "
        "Subject IDs: " + ", ".join(map(str, notes_export.loc[too_long, "Subject ID"]))
    )

print(
    f"{len(notes_export):,} unique notes across "
    f"{notes_export['Subject ID'].nunique():,} subjects"
)

notes_export.to_csv(
    "subjects_no_manual_score_1_notes.csv",
    index=False,
    quoting=csv.QUOTE_ALL,
    encoding="utf-8-sig"
)